# DIMER Guided Notebook: Can Synthetic Defects Improve Visual Inspection?

**Profile:** `E2E` · **Mode:** `GUIDED` · **Notebook spec:** DIMER Notebook Specification 2.2 · **Status:** candidate (the combined Colab T4 run has not yet been recorded)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/flux-schnell-generation-pipeline/blob/main/tutorials/DIMER_Bosch_Synthetic_Defect_Augmentation_Capstone.ipynb)

This notebook runs a controlled experiment. You will take a small, fixed set of labelled photographs of manufactured surfaces from the Bosch Surface Defect Inspection (SDI) dataset, ask **Phi-4-multimodal-instruct** to describe what it sees, compile those descriptions into bounded text prompts, let **FLUX.1 [schnell]** generate candidate *scratch* and *spot* images from the prompts, and then test whether substituting those generated images into training improves a simple classifier on **real held-out photographs**, compared with conventional augmentation.

Success means a reproducible, interpretable comparison. A negative or null result is a valid outcome: the notebook does not require the synthetic images to help.

**Run all:** the default path downloads and verifies the dataset, audits it, freezes a grouped split, fits a majority baseline and a real-only classifier, describes training exemplars with Phi-4, generates 64 candidates with FLUX, fits three matched classifier arms with three seeds, freezes the experiment, scores the real test set once, exports the selected classifier, and rebuilds it in a fresh process to prove the export reproduces the same outputs. It needs no token, login, upload, restart, repository clone or DIMER service.

**Bring Your Own Data:** after the sample run, you can repeat the whole experiment on your own labelled images (Section 13). The same validation, split, generation, fitting, evaluation and export stages apply.

**By the end of this notebook you will be able to:**

- **explain** why accuracy misleads under class imbalance and **compare** classifiers with macro-F1;
- **identify** which parts of a model-written description are observed and which are guessed;
- **distinguish** a generated image's *intended* label from a verified annotation, and visual plausibility from measured usefulness;
- **diagnose** leakage risks: duplicate images across partitions, test-set peeking, and unequal training budgets;
- **interpret** a paired contrast between augmentation strategies with its bootstrap interval, per seed and on average;
- **apply** the same protocol to your own data and **write** a conclusion that stays within the evidence.

**This notebook does not demonstrate:** defect localisation or masks, dimensional measurement, material or severity inference, fine-tuning of Phi-4 or FLUX, physical defect simulation, certification, or a production inspection system. FLUX generates new images from text; it does not edit a supplied specimen and does not guarantee a physically valid defect.


## Before you start

**Who this is for.** You can open a Colab notebook, run cells, and read basic Python. You have met image classification before. Class imbalance, augmentation, held-out evaluation and macro-F1 are introduced here. The dataset is an international manufacturing dataset; any local relevance is by analogy, not because it contains Philippine observations.

**Runtime.** A Colab **T4 GPU** runtime (Runtime → Change runtime type → T4 GPU) with at least 15 GB of GPU memory, on Linux x86_64 (Colab, Kaggle or Linux Jupyter; the isolated environments use Linux wheels). The runtime check in Section 1 stops with an instruction if no suitable GPU is attached.

**Resource budget (estimates, not measurements).** About 53 GB of downloads (dataset 1.6 GB, Phi-4 11.2 GB, FLUX 33.7 GB, ResNet-18 0.05 GB, Python wheels about 6 GB) and about 55 GB of free disk with the default of deleting each model's weights after use. Model time is expected to be well under 90 minutes on a T4, excluding downloads. The notebook measures and records the actual download bytes, stage times, peak host memory and peak GPU memory; a later hosted run replaces these estimates.

### How to use this notebook

1. Select the T4 runtime, then choose **Runtime → Run all**. Nothing needs editing.
2. Cells titled **Infrastructure: …** set up environments, locks and helpers. Run them; you do not need to study them. Cells that begin with `%%writefile` save a stage script into `work/sdi_capstone/`; the *next* code cell runs it. The scripts are the experiment's actual logic, so read the ones a section discusses.
3. Form fields (the configuration cell in Section 1) are the only values you might change, and only for the optional activities. Every field has a safe default.
4. Each stage prints what it did. After each stage, read **What to notice**, answer the **Check your reasoning** question, then open the sample answer.
5. Outputs go to `outputs/bosch_sdi_capstone/` (or `outputs/byod_capstone/` for your own data). Nothing is uploaded anywhere.

### Why two Python environments?

Phi-4-multimodal needs `transformers 4.48.2` and `torch 2.6`; the FLUX pipeline needs `transformers 5.17`, `diffusers 0.40` and `torch 2.14`. They cannot share one interpreter. Section 1 creates two isolated environments from exact dependency locks, and each heavy stage runs as a separate process in the right environment. A process that exits returns all of its GPU and host memory, which is also how the notebook keeps Phi-4, the FLUX text encoders, the FLUX transformer and the feature extractor from ever occupying the GPU together.


## Roadmap

| § | Stage | Kind |
|---|---|---|
| 1 | Runtime check, locked environments, shared core | infrastructure |
| 2 | Dataset card, acquisition, digest verification, quality audit, frozen grouped split | data practice |
| 3 | Majority baseline and a real-only classifier (validation only) | core concept |
| 4 | Phi-4 descriptions of training exemplars: observed versus guessed | core concept |
| 5 | Bounded prompt compilation | engineering |
| 6 | FLUX generation, contact sheets, intended labels | core concept |
| 7 | Synthetic features and the synthetic-to-training audit | evaluation practice |
| 8 | Three matched arms, validation selection, frozen experiment | evaluation practice |
| 9 | One final real-test comparison | evaluation practice |
| 10 | Error analysis, uncertainty and an evidence-based conclusion | interpretation |
| 11 | Export, fresh-process reload parity, new-image inference | engineering |
| 12 | Records: environment, run summary, limitations, attribution | engineering |
| 13 | Optional: change one thing, human review, bring your own data, troubleshooting | optional |

**Fast path.** If you are short of time, run everything and read Sections 3, 6, 9 and 10.


## The question, and your prediction

> Under a fixed real-data budget, classifier, class-sampling policy and training budget, does substituting synthetic defect examples into training improve **real-test macro-F1** compared with conventional augmentation?

The experiment trains a small classifier on top of frozen image features in three **arms**, each with the same number of optimiser updates, the same batch composition and the same random draws per seed:

| Arm | Training source | Purpose |
|---|---|---|
| Majority baseline | always predicts `normal` | shows why accuracy misleads under imbalance |
| **A** real-only | 128 normal, 64 scratch, 32 spot real images, unaugmented | reference classifier |
| **B** conventional augmentation | the same real images, each drawn as one of four fixed flip/brightness/contrast views | primary practical comparator |
| **C** synthetic augmentation | as B, but each defect slot draws a generated image with probability 0.5 | tests the generated pool |

The **primary contrast is C − B**; B − A is context.

**Make a prediction now, before any model runs.** Write down (1) whether C − B will be positive, zero or negative, (2) roughly how large, and (3) which class you expect synthetic images to help most. You will compare against the evidence in Section 10.

### Input → System → Output

```
labelled Bosch training images (product A: normal / scratches / spots)
   │  audit, group duplicates, freeze a 60/20/20 grouped split, pick 128/64/32 training images
   ├─► Phi-4-multimodal (frozen, 4-bit) ─► descriptions of 3 exemplars per class
   │        └─► deterministic, bounded prompt template ─► 6 prompts
   │                 └─► FLUX.1 [schnell] (frozen, 4-bit) ─► 32 candidates per defect class (intended labels)
   ├─► ResNet-18 (frozen, ImageNet) ─► 512-d features for real + synthetic images
   └─► linear head fitted per arm × seed ─► validation selects epochs ─► FREEZE
                                             └─► real test photographs, scored once
Output: per-image predictions, macro-F1 per arm/seed, paired C − B contrast with a bootstrap interval,
        an exported classifier that a fresh process reproduces, and an evidence record.
```

**Two ideas carry the whole notebook.** First, a generated image's label is *intended* (it is the label in the prompt), not verified by anyone. Second, only held-out *real* photographs can say whether generated images were useful; how realistic they look is a different question.


<details><summary><b>Glossary (open when a term is unfamiliar)</b></summary>

- **Class imbalance** — one class (here `normal`, about 93% of images) vastly outnumbers others. A classifier that always says `normal` scores high accuracy while finding no defects.
- **Macro-F1** — compute F1 (the harmonic mean of precision and recall) separately for each class, then average the classes with equal weight. A missed rare class hurts it as much as a missed common one (Sokolova & Lapalme, 2009).
- **Balanced accuracy** — the average of per-class recall.
- **Augmentation** — label-preserving transformations of training images (here: horizontal flip, mild brightness and contrast) that add variation without new photographs.
- **Synthetic augmentation** — adding generated images to training (Trabucco et al., 2023; Wang et al., 2023).
- **Intended label** — the class named in the prompt that produced a generated image. Nobody has checked that the image shows that defect.
- **Held-out evaluation** — scoring on images that were never used for fitting or for any choice. **Validation** images are used for choices (epoch, exported arm); **test** images are used once, after everything is frozen.
- **Group / leakage** — near-identical images (duplicates, consecutive frames) must stay in the same partition; otherwise the test set contains near-copies of training images and scores are inflated.
- **Frozen features / linear head** — ResNet-18 (He et al., 2016), pretrained on ImageNet (Deng et al., 2009), turns each image into a 512-number vector and is never updated. Only a small linear layer (3 × 512 weights + 3 biases) is fitted.
- **NF4 / 4-bit** — a 4-bit weight format that lets Phi-4 and FLUX fit a 16 GB GPU (Dettmers et al., 2023). Every number from these models is the 4-bit model's.
- **Seed** — the number that fixes random draws. Three seeds show how much results move for reasons unrelated to the method.
- **Paired group bootstrap** — resample the test groups with replacement many times, re-score every arm on each resample, and read the spread of the C − B difference (Efron & Tibshirani, 1993).

</details>


## 1 · Runtime, locked environments and the shared core  <sub>(infrastructure)</sub>

The configuration cell holds every value a reader can change. The defaults run the canonical experiment; the optional fields are explained in Section 13. The Phi-4 checkpoint ships its model code as Python files inside the model repository, which `transformers` executes. `ALLOW_PHI4_REMOTE_CODE` is set to `True` so that Run all proceeds; that code is pinned to one immutable revision and each file's SHA-256 is verified before it is imported (Section 4 explains the boundary). Set it to `False` to stop before Phi-4 runs.

**BYOD contract (read before choosing `byod`).** A zip containing `manifest.csv` with columns `image_id`, `path`, `class` and optional `group`, plus the images (`.jpg`, `.jpeg`, `.png`, sides 64–4096 px). Exactly three classes: one named `normal` and two defect classes (lowercase names such as `dent`, `stain`). Paths are relative to the manifest, without `..`. A `group` (for example a specimen or session id) must never span two classes. After grouping, each class needs at least 128 / 64 / 32 independent training groups (normal / more frequent defect / less frequent defect) and at least 5 images in each of validation and test. Data stays inside this runtime; the only transmission is the images you upload to the runtime yourself. **Do not upload confidential, restricted, personal or regulated images unless you are authorised to process them in Colab.**


In [ ]:
# @title Configuration: form fields (the defaults run the canonical experiment)
DATA_SOURCE = 'bosch'  # @param ["bosch", "byod"]
BYOD_ZIP_PATH = ''  # @param {type:"string"}
NEW_IMAGE_DIR = ''  # @param {type:"string"}
ALLOW_PHI4_REMOTE_CODE = True  # @param {type:"boolean"}
SEEDS = '17,29,43'  # @param {type:"string"}
DELETE_MODEL_WEIGHTS_AFTER_USE = True  # @param {type:"boolean"}
RUN_SYNTHETIC_FRACTION_EXERCISE = True  # @param {type:"boolean"}
EXERCISE_SYNTHETIC_PROBABILITY = 0.25  # @param {type:"number"}
RUN_HUMAN_REVIEW_EXTENSION = False  # @param {type:"boolean"}
HUMAN_REVIEW_CSV = ''  # @param {type:"string"}
BYOD_CHECK_ZIP = ''  # @param {type:"string"}

import json
import time
from pathlib import Path

NOTEBOOK_STARTED = time.time()
BASE = Path.cwd()
WORK_ROOT = BASE / 'work' / 'sdi_capstone'  # scripts, locks, environments (shared by both data sources)
WORK = WORK_ROOT / DATA_SOURCE  # data, weights, features and heads for this data source
OUT = BASE / 'outputs' / ('bosch_sdi_capstone' if DATA_SOURCE == 'bosch' else 'byod_capstone')
for folder in (WORK_ROOT / 'locks', WORK / 'state', OUT / 'figures'):
    folder.mkdir(parents=True, exist_ok=True)
if DATA_SOURCE not in ('bosch', 'byod'):
    raise ValueError("DATA_SOURCE must be 'bosch' or 'byod'")
seeds = [int(s) for s in SEEDS.replace(' ', '').split(',') if s]
if not seeds or len(set(seeds)) != len(seeds):
    raise ValueError('SEEDS must be distinct integers separated by commas, for example 17,29,43')
if DATA_SOURCE == 'byod':
    print('BYOD: images stay in this runtime. Do not upload confidential, restricted, personal or regulated data '
          'unless you are authorised to process it here.')
    if not BYOD_ZIP_PATH:
        from google.colab import files  # optional interactive branch; the default path never reaches this line

        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('upload exactly one zip file (manifest.csv + images)')
        BYOD_ZIP_PATH = str(WORK / 'byod_upload.zip')
        Path(BYOD_ZIP_PATH).write_bytes(next(iter(uploaded.values())))
RUN_CONFIG = WORK / 'run_config.json'
RUN_CONFIG.write_text(json.dumps({
    'base_dir': str(BASE), 'work_dir': str(WORK), 'out_dir': str(OUT), 'data_source': DATA_SOURCE,
    'byod_zip': BYOD_ZIP_PATH, 'new_image_dir': NEW_IMAGE_DIR, 'seeds': seeds,
    'allow_phi4_remote_code': bool(ALLOW_PHI4_REMOTE_CODE),
    'delete_model_weights_after_use': bool(DELETE_MODEL_WEIGHTS_AFTER_USE), 'delete_archive_after_extract': True,
}, indent=2))
print(f'data source: {DATA_SOURCE}; seeds: {seeds}' + ('' if len(seeds) >= 3 else ' (fewer than three seeds: a SMOKE run, not the canonical comparison)'))
print(f'working files: {WORK}\noutputs: {OUT}')


The runtime check records the GPU, driver, host memory and free disk, and stops early with an instruction if the run cannot fit.


In [ ]:
# @title Infrastructure: runtime check (GPU, disk, host memory)
import platform
import shutil
import subprocess


def _smi(query):
    try:
        result = subprocess.run(['nvidia-smi', f'--query-gpu={query}', '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.TimeoutExpired):
        return ''
    return result.stdout.strip().splitlines()[0] if result.returncode == 0 and result.stdout.strip() else ''


with open('/proc/meminfo') as handle:
    MEMINFO = {line.split(':')[0]: int(line.split()[1]) for line in handle}
REQUIRED_DISK_GB = 55 if DELETE_MODEL_WEIGHTS_AFTER_USE else 70
RUNTIME = {
    'python_kernel': platform.python_version(),
    'platform': platform.platform(),
    'gpu': _smi('name') or 'none',
    'gpu_memory_mib': int(_smi('memory.total') or 0),
    'driver': _smi('driver_version') or 'none',
    'host_memory_gib': round(MEMINFO['MemTotal'] / 2**20, 1),
    'disk_free_gb_at_start': round(shutil.disk_usage(BASE).free / 1e9, 1),
    'required_disk_gb': REQUIRED_DISK_GB,
}
print(json.dumps(RUNTIME, indent=2))
if RUNTIME['gpu_memory_mib'] < 15000:
    raise RuntimeError('This notebook needs a CUDA GPU with at least 15 GB. In Colab: Runtime > Change runtime type > T4 GPU, then Run all again.')
if RUNTIME['disk_free_gb_at_start'] < REQUIRED_DISK_GB:
    raise RuntimeError(f"Only {RUNTIME['disk_free_gb_at_start']} GB of disk is free; the run needs about {REQUIRED_DISK_GB} GB. Start a fresh runtime (Runtime > Disconnect and delete runtime) and Run all again.")
if RUNTIME['host_memory_gib'] < 12:
    print('WARNING: less than 12 GiB of host memory; the 4-bit model loads may run out of memory (see Troubleshooting).')


Both environments are created from exact hash locks: every transitive package is pinned with its SHA-256, and a pinned `uv` installs wheels only with `--require-hashes --only-binary :all: --no-deps` into a uv-managed CPython 3.12.12, so nothing floats and nothing is installed into this notebook's kernel (no restart). Reruns reuse an environment whose lock digest matches. This works on Linux x86_64 only (Colab, Kaggle, Linux Jupyter).


In [ ]:
# @title Infrastructure: exact dependency locks (every transitive package pinned)
# Resolved with `uv pip compile --generate-hashes --only-binary :all:` for CPython 3.12 on x86_64 manylinux from
# tools/capstone/locks/*.in: every package is pinned and every distribution's SHA-256 is listed.
# The lab lock takes torch/torchvision from the CUDA 12.6 wheel index (these run on any CUDA 12.x or newer driver).
LOCKS = {
    'lab': r"""
# Hash lock for the capstone's 'lab' environment (2026-10-03 uv isolated environment). Same 74 pins as the
# earlier plain lock, now with the SHA-256 of every published distribution. Regenerate with:
#   uv pip compile lab.in -c <previous lab.lock> --generate-hashes --no-header --no-annotate --only-binary :all:
#     --python-platform x86_64-manylinux_2_28 --python-version 3.12 --index-url https://download.pytorch.org/whl/cu126
#     --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
# Installed with --require-hashes --only-binary :all: --no-deps (wheels only; Linux x86_64).
accelerate==1.15.0 \
    --hash=sha256:5654f8c5eaa0d4fa68b33e287a97765da6849bf6d51dcac874e73fbbddfb6134 \
    --hash=sha256:97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec
annotated-doc==0.0.5 \
    --hash=sha256:117bac03a25ede5df5440e855b32d556049ca169ead221505badf432fed4b101 \
    --hash=sha256:c7e58ce09192557605d8bbd92836d7e1d520ac9580096042c0bfd197efacf1bb
anyio==4.15.1 \
    --hash=sha256:6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101 \
    --hash=sha256:9f28306018cbd6d329e64a36d58256edff76dd996fe423bc957326e578b82a94
bitsandbytes==0.50.2 \
    --hash=sha256:4311f52a880b341bada639e4edd1a3c8d786830c9c93cdde29eaa1f062c8f8e5 \
    --hash=sha256:55348a9a4a21bfd99cf8c7b32fe67b4030ae5c2a05738e03c1747f65fa6ec283 \
    --hash=sha256:8437ab68a04ea56daf1d6ecb54230fb1d88be4b89fe2d79bc399bc0203b487cf \
    --hash=sha256:c697963c8fda3dcd0d7ebd9b5211ae4067feef7cd06e0350d4e816a434fe683d \
    --hash=sha256:d5772560dd94c4d9c57f50c9b017450a1707f7687bfd4b3dc86f7342aafe721e
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
charset-normalizer==3.5.1 \
    --hash=sha256:00668ebb0609751758682eb0b5857e7c35b9f00e84dfdef062e103244ec94d45 \
    --hash=sha256:012a22b88a77ca2e59b98ac5889b0deb604147666032f45e6d6e217634d2550d \
    --hash=sha256:01e93745f7f219b703b60ba7afead36cfc4242782be5af484673fc500df12da5 \
    --hash=sha256:04368edf83514385ffc3e1cfd4546e595f4f1272dd23ba437a93a9cc3741d47b \
    --hash=sha256:0722590aabf9dc6a6c0343d523c05458fa2b5047dbe6302fd526bb570600753f \
    --hash=sha256:07ffd07412fc5d5e84cd8952acf9ff7e4ed7a708e69d1bada19d8ba91711353f \
    --hash=sha256:09a7bba9f739468c8e78c36a75c33768e53cb1959fc638f510454c14683f00d5 \
    --hash=sha256:0b2b1b3fa5670c127b246df1d0c059defd41f689a868a3b9d79df9b1cac42d22 \
    --hash=sha256:0c6dfb5ca6723eeed15aa8e564a014d69fcb8812f94eef11fe3631e0508199f5 \
    --hash=sha256:0d929fc574b4d6fd9e7c0f5c2ede8716a41911923aa7fa5fce38e0818aa4a1ac \
    --hash=sha256:13e3afe97712e8887cd516e960c63f0b93122971e5b5e4b2622fe7701771e838 \
    --hash=sha256:15f024313246a4ed976c60f440bb8d257815513a681d212ff74fd46f7d715a90 \
    --hash=sha256:195ce897c6153c0700078142cf8efe3e6454ca4cf4357499e4078dfd83396626 \
    --hash=sha256:19a3dd5aa73cef1c99687c4fc57db016a9c17104ae1185da88ba566a5d3bebe4 \
    --hash=sha256:1d1c7a53a6c2103925cdd6d7229f8c567379f211c869793df679f2e9f738c369 \
    --hash=sha256:1f5883d77fd409a261abb5dc8ccbe335720d798b1de4abb3b1d47ccbbc76b53b \
    --hash=sha256:21b82d8082f6f5e7f456ef0bd16323d08de1266efbfeb476e64b2a91d1471a4e \
    --hash=sha256:252d099029bcbea642f2a06c4ed5046bdf8b5a8150b64afa5e027e88b106e5ee \
    --hash=sha256:256dd4d85d9e4dc595e2bc983c980e73f62ddeb3165c58b4c3dfe78c5c8548c1 \
    --hash=sha256:26422d45fd13551cf564c58932f7d72b4f58b93b0fcf18c35ba6be12b46bb102 \
    --hash=sha256:2679de311c7946dde5d3b6f44941844133ff5c7cb86099c0061ab1e8901c20a8 \
    --hash=sha256:29880d17a8eb0b5cfdfd8944b468322928059aa35f1f5fa8ff22b149ec0b42f8 \
    --hash=sha256:2bced4061f000f7187254a02ad3433ae17eaf991747ceea2f478422590a5bba9 \
    --hash=sha256:2e9cf9253119d8e5d111f05d71626786fd3d6193817316eab1ca088cdb8593cf \
    --hash=sha256:2f06b7eae9dbe77fe1d644ca244dad508de8d302870a43f3c559b521270938a0 \
    --hash=sha256:2f293479cce755c75f1697e87c409b7ae4c555c7dfecb6e988ad13abba943031 \
    --hash=sha256:329fc3ccb63ad22d867d84c2adea759a64079a37ba4a343433b02c7a2816871e \
    --hash=sha256:343fb4f2821043bd87095f7b08a1a181febc8e36ac64212143bbfd0a0e1bc235 \
    --hash=sha256:3588e376b3ea2eea84976f67273d679f229e24c66dce7b82ae45aef04ff6e072 \
    --hash=sha256:35aea775dc2bd5f54cd84a1cd2696cc3207c479cb9cf0bd346f0d343e4300ddb \
    --hash=sha256:35fe081843b35aad20ffeccec3eeffbe637b15d14f3fb22cc1b59cd8ec17e93c \
    --hash=sha256:36047af20e17097c3bb9476c2b7655f2f7aa51322c0ba58c07695bedf755a950 \
    --hash=sha256:3617ac3cfd8b9888f145ad89dd6e692285834b0201c6074a5eeaad3fd4d668c2 \
    --hash=sha256:366ec70f5547c640d3ce1985722490f23faf4eb5216a7eeba78277490e78dacb \
    --hash=sha256:394fea06235c8543390050ed5f529187074b029fb027213f6c46ac11ab5d950e \
    --hash=sha256:3d27167433c0d5f18dc850f07d0b3816221984fecdc405d6c157a6f0b8f8e9e6 \
    --hash=sha256:3e5e1224c0a6a90e05843e07adfec669edebec17801c67072f51e59561d63c0b \
    --hash=sha256:41876ee62a3dddf48ff1121ad8f0798032aa03f2fd35f21f34a4cab14f18d8d2 \
    --hash=sha256:433c5a81eade63b47e522303bad236f59dba55ea6951746f5558355eeed8c75d \
    --hash=sha256:4582c27e8c889d64811987b5967fbd3ae0c823fe1fd933b543d55ac20bb475fa \
    --hash=sha256:485a0d363cafefcd2538a73c7c838daa2035f09b2c9f9b5e3133f80c6aeb84c2 \
    --hash=sha256:494b70049a4d69aec6e8137c13af4cf8db8c9f9820a1392ac293b0dd2987a818 \
    --hash=sha256:496846868fea80e479324862fa877f02411f2fd0f83b79ccee2607aa68b2a032 \
    --hash=sha256:4abdc5f9ad448c1ecbfae2974b820535d6bc6e7eef63babbab3d81cf46968c71 \
    --hash=sha256:4b599739b93b2cbeded49645ae3c8d1405c29ddfbceac1545c87a3f9580a9e96 \
    --hash=sha256:4bea7f8ebe90bbd7f0e4a2de42ca6924ba23e3e76418c408ff82f1d46fabd687 \
    --hash=sha256:4c4fb141a727957c93edfe5c32a26ceb6b5f6461d67146e2d39f51e16170bea8 \
    --hash=sha256:4c9548dc78002099910abaebc0a72ac58b7d30931869e0351c09b507dff4ece3 \
    --hash=sha256:4d26f14f041e83dd8edfd61f4cd4fa7285d31798b5bf1f28e70c367ba6c41d61 \
    --hash=sha256:4f298bdadb8f0b9e5672877f647d1be9373ef5320c9e2f049795e26cad28b6a9 \
    --hash=sha256:52ec005752a56ae79547a05c0139ca2501a0c866390b6115008456b9f0e7cde1 \
    --hash=sha256:55261ac0d2941c42f196dd576f543d87a8ee03cd6f5e30dfb4d807b2e3b9121a \
    --hash=sha256:56490c595a28b1bb27dfc583e816152a9767721ef58b2c03b13f954d2f707420 \
    --hash=sha256:58d3e12c88e0950bca850ae1f7c256055c097639c2edb9eb123af9807d8b15e4 \
    --hash=sha256:58d4aa13a59c969dbfdf9e6a9560e242cbfd9e8a8f50c2747714df1a423adf65 \
    --hash=sha256:59171c6e45bf07d0d5cab3b0bf81d945035530f6873398b3b531c31184d46663 \
    --hash=sha256:5b6d1386bf0096d26d3a863dc0a487a5b4eb9aa93cf5ba69683d29dde6b9d60f \
    --hash=sha256:5c0ea61a470e070686aa30892fed79e297d2c8d0ab46b8bcdf027d38c51da591 \
    --hash=sha256:5c84bec0ab5ae0c64bfe73a7d2adcb5ce73b467523fc27fd6a28ab2aa6cbe35a \
    --hash=sha256:5ca0555312ae2fe82715cada7fac375530c2f3349e1eaa1bcb33d0283ac79a18 \
    --hash=sha256:5d8531a6569d025f68e2321e7638fb7978f23db58e5f69f56913837aae03816e \
    --hash=sha256:5e2d0e146dcb57034f8b97dc58d2d512cb90aba253960ce449f695fec6a82c6f \
    --hash=sha256:5fc45d653ea8c9a20479167e11d4a0f8cb2fa3470737ab6f9c827532313187b7 \
    --hash=sha256:6117b84ea48435e5356dc737f5121485c30920ba43375fa7b434fd753df0eac3 \
    --hash=sha256:6199d5606e2bbf2b096cf64d03f8b6790c91081d5ac866b8e7bb6422738cc60c \
    --hash=sha256:62b55f6722735a6c472f88361cde6640608773d9443cebdbb51abf436a1fcdd3 \
    --hash=sha256:687c9ca3035544b113bea2055e180af96fb63c0c476e22a9180f51925186e7b7 \
    --hash=sha256:6b7430cf5728e68f6c462254009a6ef4086e1bea43cf2f57aa9c55fb4f50ff96 \
    --hash=sha256:6ba32c4d2abf1d2fe7cf27d280f4cca5664233b0f885549c7761719eb977f486 \
    --hash=sha256:6c9cdde8becb25a7fde49924511aa2644d6f8081cc8df8e9452724303348d8e3 \
    --hash=sha256:6df0ec430f9a831772c23ca5a224cba36517a58a84bb32c32bb59a9fa67c47f6 \
    --hash=sha256:6e2912d4babbc65196ac13c2f53468dc57fb8b9c25ef913e8c59ddf7c6dc0e1b \
    --hash=sha256:6e5e4d73d588ca5ed09df1b7dcd1b203d1df3c542e3f50d126c947d432b10731 \
    --hash=sha256:70055ff39b97c99e7ae40ea3e393fb62aa2e44dbd9b29f8d14f42fb0025c3959 \
    --hash=sha256:706bfd38730a5ac7a365793269a00f4e988178cec121391f4248d84ad8c972e9 \
    --hash=sha256:7235dc28fc6dd9d832ac7c7bce95367dedb85929f17368a0c2bee1e080b9acbf \
    --hash=sha256:774d157f112367ff4abd29019f38f023c24e00e56edc7829c20e358a5a913ad8 \
    --hash=sha256:77efcff2b23071c349402ac1066667a3d011f62398d81408c9b88ad991747c9e \
    --hash=sha256:789b8982559ae28dad2356519f841655756cdcd96616410590ae0b17454ee64f \
    --hash=sha256:7ac76cf9afd34929d76eb7fcb63be476a4853d8a96f0dcf2d0db68a0cbdf9885 \
    --hash=sha256:7c0c10730342b0c9b35dd1d619beb8214e520bd96a1f870f452680b238aab3e0 \
    --hash=sha256:823f82903d189af463d7df250ef1f7f696f3cee08cc8d91deb565e8d425f6506 \
    --hash=sha256:838648accb3a7fd9803fd45c87bce8509648eb0c11bc34e216141300977244f2 \
    --hash=sha256:854066be00447fa8de2ccbbe893e2ffc4b123ef16d897af794c1e18bd4a714b0 \
    --hash=sha256:85d5855daafc240cc045c026d7a15fd198a09b0fc8ff6f5ecbb5297b509cb11e \
    --hash=sha256:85de3134b5379856e323ba37c19c9256d39425f7b76a63af52b09fb4664c2e8f \
    --hash=sha256:87e4f41d375c0b9be2fb5251aee4b8a689169e134535aed81bf085c3b647451e \
    --hash=sha256:88ca277405c2d3b71c4e1c2ee0e7966e807bcba86a69d11e19ba199d18ae4491 \
    --hash=sha256:88e85ab89cb822c1e635f51d6d32e488f94e002e70e2f492bdb8b945543f345a \
    --hash=sha256:8ac8c94b6539074e0f40899301273ac8402b9b3e01c7b7ba269ff30340aaaf20 \
    --hash=sha256:8fe532b3c966d1fb794e0698e4589d0444017ae77fc0b31edea13c0e35bcc449 \
    --hash=sha256:9085f87b0e38a2b92b8923059b4e8789fe40d9279712d15dcc670048d77079af \
    --hash=sha256:90b7481fb62fbe172c558bc6fd1c4c98d82004a54a7551f20e11ac9bf0b8708c \
    --hash=sha256:92caef967d287a407085d61176fce4012b1dd62daed4eb6d5ceb26d3d2538712 \
    --hash=sha256:9362dd90aa7dab48c0054a21187791ccf05473f7dba5d92b8033ae62164675e7 \
    --hash=sha256:94d78ecec2605a8d0398b0f365d5f12a63248438516f5dac536a5eff7337df4a \
    --hash=sha256:94fbf1c0c6cc0d3d5e50f9a9313a8cdca90dd696d34b381cd1704f8c9e939f20 \
    --hash=sha256:950f23cb393f85543777b0433f082cddd25b51ab398eac7971146495679efe5f \
    --hash=sha256:96eefc178f8636b9c760c5829345307fd81cfae9ab1e80997dbddeb0f54ee9a3 \
    --hash=sha256:96fef3e886d6a9874b14f27fc193fbdc69d5d8035783d86aa4e1cea594e695f9 \
    --hash=sha256:977cdbd483a9cff38179bea4fd754289a6f2195c7abd414aba85410b3e66cc5e \
    --hash=sha256:978eab16f55b4ab2c2a745be9a0a840bf8f09a7f227d9c76eb30214d078865a5 \
    --hash=sha256:994e883d17c559cdfd38c84003c8b27d25424a1077272a17e7cd27bfe0bf57b2 \
    --hash=sha256:9ac4444d8d4fd4c4bd08bf451ed3167aa9e7ec6cdb41b648794f1d1103652e36 \
    --hash=sha256:9b5db6052055d34d41230fb78d7c439c23dc536a9896f6cb039e8dd92cfc1263 \
    --hash=sha256:9d9a0dc7cbe9bec24c3f767c9122c41fe5a1bc43f47cd099d00d393e09769de4 \
    --hash=sha256:9dbdd9205662134957cf0c324f639bdc5031c0ca056e2369e238db75187c0f11 \
    --hash=sha256:9eea3ab2597a5e65fe65296e2d6a84570845a6b55532d90333d740d48bbc850a \
    --hash=sha256:a2028475ba855475b8b4d3cfeb4994269c967aea8b9892dfba907f4263a863a3 \
    --hash=sha256:a3a370082ce34d0612f421e15fe011c53bb1feff21a26d06ad4fb244dab5a375 \
    --hash=sha256:a545775cfe815855ea32d7c27731d79da358ef2055b4a25830231b1622dd18aa \
    --hash=sha256:a5cbd90ecf0fc62e64726917ad083b73001f0563657a87ec3c0b504e277dc90d \
    --hash=sha256:a6d095662e73e74f0a49988e0593373e243e3a52e27bfeea0a859e88acf4a0f5 \
    --hash=sha256:a6dac12ff6b846103483683f60c5f8fee205121adc58ffd87e90a90a3af69e99 \
    --hash=sha256:a951ad59cad9145664a730d3036b40b844e74d2d3683da40111463cd3a83845d \
    --hash=sha256:aa1099b956fb795e686d073568f6dc002a0bb89765ea6d5b055dd7d9bf1b116c \
    --hash=sha256:aa2bb0b37202dca27175591f761108b5d34096ade1191ffe4808bdf6b1571488 \
    --hash=sha256:aae2ee51122d3ae968a3837d97dc24a0aeebb0dea23694422cd172bd30017cd6 \
    --hash=sha256:ab743e9bc90c1f73552ec33e10e3331315acd2c397b36065b591b0181de533cc \
    --hash=sha256:ac00177c4831ffa650f8609e4bdddd5fe09c03b1c0c47acece7e6ea20421598b \
    --hash=sha256:ac13b004224fb341e1e25a1ed5e19d32f57cdb2a403e01f003b46f051a550f6f \
    --hash=sha256:acaf604462bf330b0d07e7a07c1d6e4adac79e5fb13e9c5140590542cafacc00 \
    --hash=sha256:ae31a1a1db2ee6cc2942fccaf695c934bc7f3db9f2133a3fef1f367cf1a4ab10 \
    --hash=sha256:ae4a097991662cd4fff0ddc74e0fe7874f82e00042fa0ea00855645ed0c79598 \
    --hash=sha256:aea996a6aba25260827c9ea511d1addfde2da9eb686ac961838509086188b7e6 \
    --hash=sha256:b39b69b347e5e47a3b5b8cfc005c68c1ba347474e3960236c4944a8ecd174962 \
    --hash=sha256:b54e7e13267d49ffbfe68e25b3cbd774dab38fa37238f71265e91b36146eb21c \
    --hash=sha256:b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08 \
    --hash=sha256:ba2f37ee79e6338845261a3c5b1784e5d1acdff2c0785b284f1b633033d136ab \
    --hash=sha256:ba501e667c17d8411f98e67a022d9604ef179aff0e459b7e292c796837c13573 \
    --hash=sha256:baf3775a2635e5a11fbd5e4e64ee69c7e86875d224a5c72aca4c141064589a90 \
    --hash=sha256:bb57753e36e4855b8ca375069482250a6246372331a3e4f3407eaebb007443f5 \
    --hash=sha256:bd6c173f04743d483881bffa1478d5a4624475b8cd1d2194956a75548e191c18 \
    --hash=sha256:be47f99644b208bff7766314013f9acf57b056b04191d570d68ad14022cf5b1d \
    --hash=sha256:c010f5581d9c612804cc59fcf7b524b707fbcb72828551237ab545bb5c7034af \
    --hash=sha256:c1dcc36dcb96abc02236e182d17e0f71430152a6c2c7447421da2d2dc144edea \
    --hash=sha256:c428c6c31eb5f4277d7f8eccaf767fbd548ddd5ce3c8b4f4cbbfab3d96b5904c \
    --hash=sha256:c658c50ac0c98cd755a2dd50b7977d3bca7df401dcc47fbdfa87db53ef7d4e8b \
    --hash=sha256:c71fb0d56c920c269cd3e2e3fe7c610e3f1fdb21a6ce60efa6430ff63676cea6 \
    --hash=sha256:c7b742bf31c88566b4bb6335a7f393bb322e580b6bb98df7bd0c25e6e3519ce8 \
    --hash=sha256:cc0329df4caaceb950d2f580b5ac716a377f7059624a0bafaeaf8a218c6ed774 \
    --hash=sha256:cc5d36d96478aa9c60654bd932525bf32964c62a7281eafdf16d85003a8d6004 \
    --hash=sha256:ce854f5f478050ade5a238731c4ca985a7d3b3cb53ff600a9b5c3b689b5f0a7a \
    --hash=sha256:ced3fdd71aaa83ce593746c2edb42b7a59cb4c19c8b5c407781c72e493aae55a \
    --hash=sha256:cee5dd7c6fb5dd52a0fe2a740f9bc6e3593f5f8b1788bde49de02086f30182b2 \
    --hash=sha256:cfa1c0cc3a8f9f53f1243a5a99ac36fd003880199383b37672e86ddda9cb07e2 \
    --hash=sha256:d1ee1e296209fdce05b81b663250eefa02213a2da7b41bf26f7829b8ba3545aa \
    --hash=sha256:d59b75732e9b6f27388e10c14b0259cc5f2e48c78627d185e6a177b58ad3cffe \
    --hash=sha256:d63600d620ad0064c3a748b950ac5ea38a80190e5498532efefa4b7b3f1da1f3 \
    --hash=sha256:dd732602a7009217f658d5863d12d79d373a4de0eebc111094bcdd3bb8e0a6cc \
    --hash=sha256:e06efa066f7dbadbc84ebc126a97c452a6451dfcf589d89d788484949e1cf795 \
    --hash=sha256:e199fb99720074809a7720f1c0b4d919eea8b87e88713e0f8f602f7bef543d9d \
    --hash=sha256:e4b018dc5a0eee4676e38fe84a47a427816c590b93b55d9025274ec4d6ffc2dc \
    --hash=sha256:e6621fb2a4988d6e53eedc455e5903e2679f3967b8acb3d639f1b63c14a2e893 \
    --hash=sha256:e71c909f353863b2b89c83de2ebed71ea6d0df8a6ef65a128193c5e650766bef \
    --hash=sha256:e90251c0c7bdd54a100a0dce3c07b7e637278c93af29dbf78ebb89a58c4bac7d \
    --hash=sha256:e9fbdce1e47394b09bc9f26ab117dfc8d6491977a11d86f592bb42c779db2fda \
    --hash=sha256:eb12fb2ba69ffa05f8695f61c69e591dc4b4a12ac3757ac8af8adb259bf56d17 \
    --hash=sha256:eda059b6bc8bc0812d626fd91a7ce01bf583df0a61296eff390fd94141a34e30 \
    --hash=sha256:f03ac127268b43ef4fe9e6ab6794a6794b49485a0cc0c1db79876d2f33f75bc7 \
    --hash=sha256:f298e218441525d3794428b4c8b8fb8662c6d3ea79925d4807ee6b9a96a3bca5 \
    --hash=sha256:f5542f9b941279d82d41eb0aa9f98eba36fe4df5c7086c651df7944935b37182 \
    --hash=sha256:f6f7deae3feb4edfa2efaf7c574fe88cbf055038a6abdb40188e4fff66d5699f \
    --hash=sha256:f9b1e28d0e8dbfa858abdba91d6b547beaf2df1a59bec6da6faae7b96a4991a9 \
    --hash=sha256:f9f8405c2c758532c74fed975dbee57be1f31a6e865c031870c79a6ed3212ada \
    --hash=sha256:fa48b1b63d639f9483e0633e092f5851e2348c352f1f9bb6c8182f87884ef876 \
    --hash=sha256:fb78f6e7fcd8ad785d28cd577168bc1aaee827b25bb8755638f694794ea98f0a \
    --hash=sha256:fbc597639158fd7c14d55e808718848319540f51b0e6746e3eefa59723a4a348 \
    --hash=sha256:fce8cbd4997efeb450bd298b54f755dcdff18d496f7a5ddbb4867c6d7c88fdc3 \
    --hash=sha256:fd0350afdc3aabd5576f60ea109228bd5538139713c7b094c5cd27c73a98bc6f \
    --hash=sha256:fd0a274c0e5f9a21565cd9d3dd749b61f96b7aa1e20a93aa1ba4029518f2e5c0 \
    --hash=sha256:fdb8a068947befafba9952162645dc2fecaeb400e64584829ed5e9b2fbe21a7f
click==8.5.0 \
    --hash=sha256:255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360 \
    --hash=sha256:ba0d2089de75ea0310e2dde03160e6ca10009947fb95a182f9b54021bb272e34
contourpy==1.4.0 \
    --hash=sha256:018227e134b73090b06f911e773a526c77c9af426be29065500ce096a5accd4c \
    --hash=sha256:072a702e2e178f4fcf96f04775d0c059e4c917925abf0d3208ebb6865df3c23d \
    --hash=sha256:0c7a4c2716a4e98342221954416a836cca77c996c14ddbf22b5f01d5d93ca09c \
    --hash=sha256:107ec46f7aa1664266d69b1181aadc953d58d39f7d8d648fa5b795d4a060b0de \
    --hash=sha256:181bea01bc742734ae672fa00c717d855dd35e1f029536406538c52e7b0cd73d \
    --hash=sha256:186ba929df36d61b6127da2e89cd1357e4cb79aca647381ab1a6feb0b152877b \
    --hash=sha256:196759a3e4db60e1546167e361090ebb6d199aa3aafc9e34ea17a5ac3b23e814 \
    --hash=sha256:20156f5a1ac4f8ce02656e39a61e82164a3d359796dc8026f75b062783d500e1 \
    --hash=sha256:20da2f86bfaed60dbba729fe738c8241b1dd22b6cf7e8cbbf30df290bd04ba28 \
    --hash=sha256:2deb580178ca19437a84bd77e4bc2cd91a8ccad212413a83c273900868b5978c \
    --hash=sha256:2e741a39dfc96babe1722561e81351aeec104526c57dbe91c167c1db606c2d55 \
    --hash=sha256:3863ef2e2b13fe93f8c0ebb08ee400cb07153b8b0e91c5acb26e4537f283634f \
    --hash=sha256:3e0a2392533e5e2abdf277c951047516acc8d2f3c42b2dd9dc34c907c9ebabf2 \
    --hash=sha256:404dbcd9233513dfd1323b66ea593fef90e899f658f6b8a9ed8e93bd0ca669db \
    --hash=sha256:417be048f7e122cefbe2a34c51a1f2b0410f8eb35796f794d45475ddb7872d9d \
    --hash=sha256:423bd5f4b3f11d54a8c597234e513382e3454bb106b8ec7ae32ba4ed63f8d99b \
    --hash=sha256:432e89f835cf01f8d2123a130a3126b17ee6e0348e4d62c0973872b30aa873bf \
    --hash=sha256:43c3ccbb32c6294b183dcc8e8c46dacf5ecef809497e3d48a5be298eef185ad0 \
    --hash=sha256:43d80072a32299bf945de0a6dd4e034ea0162e832261de2e07a274a5adbba9c9 \
    --hash=sha256:4bffcb2e8cc5c0e837631acab59dd04265fdac376550e7c3e310523877c68cb7 \
    --hash=sha256:510f7d93d94cf6ebf4e2cd0640bea16ab8affac7547230175b6745b596ce0599 \
    --hash=sha256:5450f091ac1be0be3ad3a2a3b3f23b5e443e78c670ced4fd347d626f92a28fd2 \
    --hash=sha256:5545ff38229c15d11d822fc6f7415b932da9f4975f53e305b7b96ebfb01f889c \
    --hash=sha256:5a4c89c7f38a0d7a94356d74e3391073c4325c4d47ad2fd065c3fe7dbae16c0c \
    --hash=sha256:61624f6722480aa7e168fd746164e7bfdf48f8ccb6542f1613741200dc37e2b1 \
    --hash=sha256:618137ec5778fb76d494c9ec854b104a377fddba0d128bef7136cce2c3bf0df8 \
    --hash=sha256:630d32f06cedf37b7f1dd2b6a12e4492826bd477a4c9c7b2e2fee6ce3a2cea76 \
    --hash=sha256:64039341e2d8804f1a13bda8c69083eab935167bbd7e856cf096241f17d5bd45 \
    --hash=sha256:6507a016976a75a15aee47809a34605de6244ccc056e6164bf9fb14b27eecad4 \
    --hash=sha256:6ced1670fafee703b8277ab1645072a226e74f167166eae146fb743916119b67 \
    --hash=sha256:6e697d94e69f499ff6bebb899cae97a58d5d14f0e1fe9568b43a0248d2f9af8c \
    --hash=sha256:6f963ea9f0d68d2b6a02d861e3c0aae09ae25ba21f243fcf281e9fed468b078d \
    --hash=sha256:738c44fa71735a617f36da58e32810512407d283d5d1bb5b1cecb00d7eeba7cf \
    --hash=sha256:758e7496cec3195fc28a28945eab3ceef348c47fb95c7afc2eca2f3901209c8c \
    --hash=sha256:78ed5c5f962b3e156109f0531b174a65ba80d13cc681a70227b12524c162e184 \
    --hash=sha256:79b06c60d5e569ce12c5da8f0c51217cdc484386e2349fa717c30fafb56d2871 \
    --hash=sha256:7f861af508fca2384eef392bc1d8377cfa349b2b854cb11b45d89c75780e2561 \
    --hash=sha256:80c7fc8e7ac217777ce7ba2db3cf8478652b5a4b335283111012ff9eaac9d842 \
    --hash=sha256:86d05cec773c9507a3950122e0e40ce77c23c75ceeb2fc189514e71893cbb34b \
    --hash=sha256:875f42444c9cf48d56f724f2637e60d0f73b3b12c9041e1484580a233edf9591 \
    --hash=sha256:90feb8da006803a95ba573fddc9536357d0c3faddc3a4c9e22816ca49af52878 \
    --hash=sha256:912c6afaa106e2f74ba22b30416b77ef7eb94e3bdc5a4df51ab147845dba9b6d \
    --hash=sha256:96019f059bcb3774acc0104be3360a57b36d33eaaa2e1d7f77c800ca8e07618b \
    --hash=sha256:9c0e07c691f3b3321913ed9b8161c50ea5f77fadd006f52f5e755359b0dcbfc5 \
    --hash=sha256:a03b32c2e7eda8b17757c022162c13c86f5c3d6f937dddf9ba3c3ff7b513953b \
    --hash=sha256:a1c8a74744fa746eeaa12f0f9ed7f8b4add9d8f1b14d95e3b5e133675eac888f \
    --hash=sha256:a3e67bc1a6a4618a1dac7c3053a9ffece5ddfa2046b2670b342cf430bb0b87d1 \
    --hash=sha256:a58b5f130afec61a093d743cdda435cc4047748b6791627e34a68f41ec3a9b07 \
    --hash=sha256:a5e2bb871b90cd7a52bdee63bf80ef7acb46398d27efc66d7679240016efc5aa \
    --hash=sha256:abd0f7e51ecc52bf8c95713bd80b1c17c516a71fa391c54475308a4377903c2b \
    --hash=sha256:b7794ea07cab575633daad8d6e963b662053391022294d3aacc1d9ba9ef54114 \
    --hash=sha256:b9dc493e7924e5aa32d89d0b8e50986280f3d4b284968a792787bd7bcffd3cad \
    --hash=sha256:be85d160e7c795113c2a93254dde2fc7e86b375322570eee78b420969426973c \
    --hash=sha256:bee6ff96653e0807cd0f8eaef00174a031e3a2effa5c20f49a5a8d574b05af23 \
    --hash=sha256:c76f3a5318164db7d9401132fc1b5364b784c613c93fa506e3b5e6d1bf353ec8 \
    --hash=sha256:c8be3392b84cf43373a488874dc1ebf46f60784a70977f92586e18019d71c7ed \
    --hash=sha256:c927ec747633e68c9920bcaad48b0bcefe648812b822f18c34e2d0e2270a3d2a \
    --hash=sha256:cc87f2ffa84a49a77d76cb792f28a48ff3b05a222aa0b2bddbd839ba78e1d5ca \
    --hash=sha256:dd59df9e2fb0aff8bd7fd1adb9f349b7c835245a9d0f18c2f2deeb537190f2b1 \
    --hash=sha256:ddf5a2596d716fd3793434844caf31abc7a40b1e8718431420c89858268fb909 \
    --hash=sha256:de503609fdb71f597634ca64fb4bd2f13d27c97f140e912b1e8ed93544840df0 \
    --hash=sha256:e21d1d4a9db5b3a793649c7ab0ddb4697f2818929eec871a7b9c45fce7b2a1ea \
    --hash=sha256:e439ab450c93455feb0218532ded3e946ec7a9b5f459069f122cfa44b89229f6 \
    --hash=sha256:ea09dc704029930cbd097f75cb0cc1db9852adbcf1a151fac7e9559f7bcbfd19 \
    --hash=sha256:ef47fef9a912c77e3d84b014f701e5a9340f25703e9ed3bcebe37f91fb69dc49 \
    --hash=sha256:ef9440f6f8506246269a82734f5ff9e2e4c5e775b3996fc883cc491c5257eca6 \
    --hash=sha256:f1219a8898523cba821085f2da8a1b962cc696325763f09d7f93538ba43b2d70 \
    --hash=sha256:f863c6100bf926cf47d13f3cd75f9bb8ebd98aaab230eeb4468b91f98f39a6b3 \
    --hash=sha256:fa1b787362a3856e63dd2b89449f6c384d55ee1beb8f94f4bcc871b40f02462c \
    --hash=sha256:fc9feef8f1f001c5b87decadc67c4a5d1eebb62ca39c4763d1237ff62cf2b707
cuda-bindings==12.9.9 \
    --hash=sha256:074f9d7bd14fa80c596bfcdbac7a131636abe333a3b40da769979d6f87be39f3 \
    --hash=sha256:1c91d66b9467c21190e531bd1bc09893c6384542189cf880ab15213a16266063 \
    --hash=sha256:5fd8ab78c4ddd9a909fb4f1fa9a914d5e83e223b77320df63ddfe3c46e559337 \
    --hash=sha256:607c13eb76c244cacfb955675f8bea2c5b942be9ef4affd41d5a7368d211b260 \
    --hash=sha256:71e84bcdf477a989c31994023f353d6cbbb6d5cba58deab9bc5e7d961d8b832d \
    --hash=sha256:7647dab9d264d117952528f2403810debb04ba74f756575eba3b72f66d20311f \
    --hash=sha256:88c41afe5be41acb17e57977dd534e335f86d0c96d284d71457168525f6327c0 \
    --hash=sha256:94e4f9bd6b9b21aad545e0d441707094e4ff88ab420d62b85fcdc8b6da1e039f \
    --hash=sha256:af8cb9540c9a3d04df7f52584326279a10ef017021ae8246642a2339dc94e98b \
    --hash=sha256:b4cc44252c7835d63876df352564b1a39e0091bc67cee7a9a271286b2771348f \
    --hash=sha256:c578d2ad525f844b663236b280d72446521e44ee115d2bacba320f79eba2a936 \
    --hash=sha256:d399447bf5d929d6f59ec4e2c7b26d11ef21e8be5e88995e41c705db1a7c1363 \
    --hash=sha256:d8d535cba3a3374e7069e875061bfaca48f044db988fa6497928478ab00a797f \
    --hash=sha256:e570b6ed83fe8d2ed45c2e9951cea6120ee490df258ed5a2b1ebfb2bf35541e9 \
    --hash=sha256:ed5d6cadb36313cdc8a43fc8c8818ff54e5369449b96e21f53de556c606c4d1f \
    --hash=sha256:f74777eb8e83ff076b8759a9ea65521ec5fa2b46d59d15ab93d9155d4a9f0f19 \
    --hash=sha256:fb842851be2ed9e80fce988dd68c8ea27a506a71e0e29d228b35cbd0a333dac2 \
    --hash=sha256:fc8a97efcb14e48664fc15479b621002ba4e0a6fba34e617a70b07f4bd638390
cuda-pathfinder==1.8.2 \
    --hash=sha256:4e65059febdb4d19d5cbc4798677e19db2b582f2f702f457b609e571690d357e
cuda-toolkit==12.6.3 \
    --hash=sha256:79d8605baeb6c2f695761e0efb54bc62dbc3c9e32eb0742df7669c07befaa8f7
cycler==0.12.1 \
    --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30 \
    --hash=sha256:88bb128f02ba341da8ef447245a9e138fae777f6a23943da4540077d3601eb1c
diffusers==0.40.0 \
    --hash=sha256:49f112ce52ff6d332ab68afec01f79a53e6489b42cb69104444c98b4a2f64af8 \
    --hash=sha256:5b5da7c3ddb62152fa4afc577f02e050af688c797375c34fb2d01006da3f3541
filelock==4.0.5 \
    --hash=sha256:2b155f098c4f285fb41954a22c616c4e8a0635b78c184338ba3023c1c91a4b4d \
    --hash=sha256:be9166068623636cedcad7b2ef735b04e5fb4344d978e09fd6fbd6ede06f14f0
fonttools==4.66.0 \
    --hash=sha256:02117d05dddb51e39b5c5a6eed0702ed8e0e1c341cb138101b2446eb479858f8 \
    --hash=sha256:03922992966a1830b94a750961d1ccfe1a7375d792ca61077a5afb719811788a \
    --hash=sha256:056641fcedeaad24b92e343e35e9fd1337200c102450678894b17928482d010d \
    --hash=sha256:0640e69b00e089d6b82e0a54e3a03af84b775f1f0b2cbecc1b0d7fe3980a194a \
    --hash=sha256:0cf33d1041364ebb7a9958db757e213c6d2a238eb41755fbf935db028c2db20f \
    --hash=sha256:107aae7c38f05561fef5103baa98a44400b5f4c69d87839c9a5f8125405e7d9a \
    --hash=sha256:135ce8738a6341fbefeedc061e20c2889350e84405abea7e61b1c9d87e1a1f2f \
    --hash=sha256:15315302c620e6ce8582934e07932dcf442fc3f59ab208d30564200b9a2f7ea4 \
    --hash=sha256:1d120ea0f5260b04e9b5ac0d9239efde4c23d990347d5d5889cdd37221726fec \
    --hash=sha256:221defad3b1949c1fdde2558d1fe780d42f926b683a71a46a4495a0e6a6614bd \
    --hash=sha256:2b278e2abe596872b3c9fe611f956ded85588537974e48e2adbe2d6ee159e099 \
    --hash=sha256:35684b562df7154d7a0ebfa512db9199c7fb0a93b5abfb59dac9d022dbee7aaa \
    --hash=sha256:379b94fe10651d26caf398eb646b3ecde886a9c504d3cce5e46164d58a78e32d \
    --hash=sha256:3889fdbe63e85dbf2a0d9d7c9d90746827ce60af2cdd24f0bb4738380a9540eb \
    --hash=sha256:398bfc5ce9055e0b91a6189f8cf462d4230b38f522b2ceaecb4f34cbf4fb5480 \
    --hash=sha256:3ea59aaec135c96c4cff20d79fdf1d10623993712a77990585a42cfd6a42bc5c \
    --hash=sha256:4513753259f06316bdc2039c9bf3129e74923c1cfad0722b4728a9ce19beeeb2 \
    --hash=sha256:4bdaf80590fcccb1b9d8e3b8ca72225defb14d03b79788f59781ce1bb66049b9 \
    --hash=sha256:5dda4087a7f8f0b985995489b3c0115bd117f18c0fc236013a0e78e837bb9ac2 \
    --hash=sha256:6371e7ed43cc9e23550fe4a1b72af3205275f28dcecac8c09b80a4582e326802 \
    --hash=sha256:6627f48130cdbe04c7e1189bff4192886ecda757f575626ee8468f6580eaecf3 \
    --hash=sha256:67ea5af3ca60e1e5c9b2841b1f6dba5ef16aab581bf463abb68515447154ab61 \
    --hash=sha256:6b11180c4166a4fc353bb02b147322b7c454927bba4f5040ccdef109a73f9051 \
    --hash=sha256:700c448f190cc7bbd9f059bd96131f4770e90938c9692550b142153bea73b14c \
    --hash=sha256:8009e736e1491c569392b2e9782c30dcb6ac879def102c8c2e06add8d2adc10f \
    --hash=sha256:850d50b347f31667d3a277a6e66a4219236db9d98afadb2993b8addf9bd4f593 \
    --hash=sha256:85e2960a29e6882567b981ba408b5a5ca2d8865b8f67240e1b420348997a13be \
    --hash=sha256:8739d77810aeb6d2ecd55cc2e4864d98a00c317fb6131a6d22ca58b3f50cf22a \
    --hash=sha256:886f3029ea10362c592d2e1a5326a44645b85a2a0be6fdb7c9edbdb60cedb0ab \
    --hash=sha256:8af85dfc2ad564f95b3e650024578bb2268bc4d827d418aab9650bbde4b6926a \
    --hash=sha256:8ea6265535b25f8254868f5a212a1c9dbd88c2c4050f3df18d9174949df0d6d9 \
    --hash=sha256:9b0198962bbba64a88bf15dc5d95e4bab8c5208dede22a10fb06280406ed9874 \
    --hash=sha256:a5a0624aaefb0a29e4ed7500a98732a1b0ec2d3040fe86cc8837f82e3fe4f446 \
    --hash=sha256:a9a45a23b020e667499fb2dbce0f1373b1b1b8cf88e47f3031e69ef3ea21f3ee \
    --hash=sha256:ada87643b20a8fa5763e6dd4fa0f5e2901cbdbe0833deb6d5c5a9c3186782b94 \
    --hash=sha256:aea0cc5609a5f2a2500f91bd6e1989fdd22da0f225be00dc1c4cec775838a7d0 \
    --hash=sha256:b02686b2e47115b2378ab6156abd984c776c6faa4f56898fbcc1c30ee387937d \
    --hash=sha256:b432b6b54a0f3be118c6243312c826aa086aa9bcb150160be4f0bc8a21ea3aaa \
    --hash=sha256:b46500e4d6f5708a9127ea12902ba54cb0ec594fc6bec649afc53154a1125b2b \
    --hash=sha256:ba65eb3e2c46ad0922d84ef78e287c2f48b7a8cae4420728bf3b8e583282d6d7 \
    --hash=sha256:bc7b7ddc1a1f46c363354304e9a8dd93722e4a6a24f785015650898dacf40df9 \
    --hash=sha256:c144e68572af1dfcb05fe065827d0920d4118b265504928e1d2262e3701a25a1 \
    --hash=sha256:c3a66c749b69e9abd92e519ccbdea7210522b0fffdba3f6492a8fab461d4708a \
    --hash=sha256:c519433e8632284dbe64ee4c81b609a17e205bc1afff285553723964880d1c89 \
    --hash=sha256:ce818581070527883e3678b92b1ac7ed3569e48c80f51b89b264f97d84be2608 \
    --hash=sha256:cf0033f343cb1592fb24ae3eaf8b58156f208884ac16991094293ee94c585654 \
    --hash=sha256:cffa168522a0e057d3a5628c8ebd43ddf66a34613f42e116bffb85b0b32ce5a0 \
    --hash=sha256:d2d10f89c4bf7cd42b84695da617f9130fa594647ee74673bf8c8d001fa04ef9 \
    --hash=sha256:dbee639a23c4e067aedfbcf84d2e466a71a1a1156ad04d6d09023791ed167495 \
    --hash=sha256:e4677025ee40b3b1420387ca85f55545b8095f7596c3d5d59552dbee19c91f07 \
    --hash=sha256:ee4f85d1341af630d787514eef1a0e1ad883f9d6c11a4c1465faeb9cd0bb5ced \
    --hash=sha256:ee7b9f6c835ea1a9beb5a35c8eb0c62b6a3290f8105d3d0c29a0a18c90f9b8ba \
    --hash=sha256:ef0610dfe7bb5bf574d9bdad6f597403ebc9807d124ac6f148d7604b2609be98 \
    --hash=sha256:f014bbf05f30731bf8b9f5c7d2b4c0e4f28f375926f95b187f85e0bbceb16029 \
    --hash=sha256:f2032ac005e14c56e774fd19f2bc1c22c796c0c3b39d91741c0e3f9575944d38 \
    --hash=sha256:fa995d96ced196388d8ed7a67ee001b181459d230e24bb68b193ae859f5609da \
    --hash=sha256:fd3dcb69d65c05a2fffa5d9b743b77e0cdc13832665ba435065493dad2b57f83 \
    --hash=sha256:fd9297a5f670f3e59ae697289ddad5902d3c4a86188e83e7823bde06334a560c
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
h11==0.16.0 \
    --hash=sha256:4e35b956cf45792e4caa5885e69fba00bdbc6ffafbfa020300e549b208ee5ff1 \
    --hash=sha256:63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
httpcore==1.0.9 \
    --hash=sha256:2d400746a40668fc9dec9810239072b40b4484b640a8c38fd654a024c7a1bf55 \
    --hash=sha256:6e34463af53fd2ab5d807f399a9b45ea31c3dfa2276f15a2c3f00afff6e176e8
httpx==0.28.1 \
    --hash=sha256:75e98c5f16b0f35b567856f597f06ff2270a374470a5c2392242528e3e3e42fc \
    --hash=sha256:d909fcccc110f8c7faf814ca82a9a4d816bc5a6dbfea25d6591d6985b8ba59ad
huggingface-hub==1.32.0 \
    --hash=sha256:b0c7c80561969d9cdacdd55fce67ba9584cca0b9d4ea80957a3a5c1445fac5c8 \
    --hash=sha256:ed70a45498abe86039df7c2f4e5f7575de524be908d3840e8f828d5525eafd6a
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
importlib-metadata==9.0.1 \
    --hash=sha256:ab830580bc0ef3db61ce8fae716389e5462b67e033018bab6d8f80ef17172f99 \
    --hash=sha256:bba5600596a7e21f3eef53281cf28d6a5195634d2f2b78ff9501a3272c6eaab0
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
kiwisolver==1.5.1 \
    --hash=sha256:007a5553dfc4f4e8d184f588a0200e2cd4b63a59cc8796df3c39909e679dc7a0 \
    --hash=sha256:0324cd2567259b7a095f6cf18a52b0ffc6f3de9e69528ff1bc0e7a37bd43ff1a \
    --hash=sha256:0627b9bceb9c3cdcf12b8a18655eedfed2692b038df27423383c120d0b7dc2d6 \
    --hash=sha256:06a6917674de9e0fe3f66f5430787f59a9f2ddb64af9b714eaec547e29ef5c19 \
    --hash=sha256:072bdb15a3c19a5b5dbc8f8fb1f4e1884bf4f3507eeb4cc6334401274d37a5c0 \
    --hash=sha256:0a4faea5c6db201c6a21391d2ac926ea97acf7dacdbc3c417189e1adb1a00837 \
    --hash=sha256:0ba9527afc80ae3d7814ed98b6572d02bf85eaf48065678342c5f0c6dab7a8c7 \
    --hash=sha256:0d8924877ce22e17326a99a418c3c82037da078df3c6a260b13eca677444e6e7 \
    --hash=sha256:0ebdef3eae5336568147c39a55be6a2036ffde53faa9ca2d978989ae7c2da12c \
    --hash=sha256:1209042a623ddfda5497e4066c7b77651dde8e1d3a9dd97599dc7e97f3b9b78c \
    --hash=sha256:16895f553ee6620a827d2da56b871f835fb70b9216cca5d188e885caf6e3bd23 \
    --hash=sha256:17851e5dad4484be0cbccbde3b15331deae036de9aebd45eed964487802b172f \
    --hash=sha256:1798e83840c3f627246104c4d8a9639c60fa068adf9ce92b61791781fa8a68c1 \
    --hash=sha256:18170a77ddfecf40ec60d0928268dc95880c881864e015a8f34094ed18b9b9ad \
    --hash=sha256:186884a58486651e3c217b6acea0a53eaa9498fdd472057c46f2f0fb5c25aad5 \
    --hash=sha256:18a0cfb124546a4c2e6087c5f3029c7f44b37c85b142e0ced71f73a7599ac208 \
    --hash=sha256:1983f0974a750a6f6556f368ba11105d1d8369c735b944747c9f12ae5aea7aae \
    --hash=sha256:1a7587dc335f2c0f5bd577fd0540bd16c66006bdb60f759a1059f025e6c4f071 \
    --hash=sha256:1acc7e5b7ef05e9da8bb70cd6c7c4513090213d2e1ad9720f599f0bf6c52aec5 \
    --hash=sha256:1d852545c4d0e35a72728d072cbaa59e2fa7dd84bdf01e068d670dd0ceb58eb6 \
    --hash=sha256:1ed0f5e49d0ceff8b72190824d9e59c062fbbc02c231b853112c78474b3f5ec2 \
    --hash=sha256:1fff05e239575b1481b6ed1a782f6fad616efbf1f0b1f44e6e85c4dfe426e483 \
    --hash=sha256:21e46b23a2da695c364124817bc01d970effd5483147f8d66a6a7167e3f6b851 \
    --hash=sha256:22d5e5aaad6be121f2515765e3b1c444352cb8eb4c86510801db8f2e50757316 \
    --hash=sha256:2551cf9917af48ee7c4b29cc82320489508cf96fd26a51f6fc124de661cd44c7 \
    --hash=sha256:255605693a483db7bd5c79f60437f7bf658f7f520d61aa42722e32257c941951 \
    --hash=sha256:26e8268480be5061d509e29669d59103c067a26377a56491630ece11762e3858 \
    --hash=sha256:27add358abe374ebaa3b8763ef380bc99051b5a4b18d94878366a9e4f59efef0 \
    --hash=sha256:2ae70bc59790d2af72a3f76f24b272403e135070340281108b447cb77ea70819 \
    --hash=sha256:2e10ae1bba1899188b33557c10d73affcc12033edd18adddb57d209039976a4c \
    --hash=sha256:3221f78211074f561c44ca42eac0619828171bec15a2c4cf6f7747d07df76e8e \
    --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38 \
    --hash=sha256:34ec467940442c9943016fb2d4c81d1ba84351eeca2f1a78f8bc87f1ba0d414c \
    --hash=sha256:37f801b5d7cc0e5a548921308e059fd2b057bb42972b591cfa3049f95423c4ed \
    --hash=sha256:38f6e0deb4d0a4615efe0c4efc5990b06ae450ab50a0b321c0b078b6d238c083 \
    --hash=sha256:3c24cd69455e1b00ddf770c13b6e2c33e07d6dc3f2d34add0bf9277c5c6bbd46 \
    --hash=sha256:3cc210010fd2f438a3ed430b45f1b501fd13a8618bf984dc2c5ce5b69b78752e \
    --hash=sha256:3fa5855898f6d3d01b72ccd48a2d65cbdee301251603fefe34e2025bddba219c \
    --hash=sha256:416ba7ff9f233b7036689bb5a3783537e838ad483f63558d2a800f75afe738b1 \
    --hash=sha256:431dc224a1a92a5c8f582d96e505196a3b5997a7271076678da2dfde67b77e9a \
    --hash=sha256:43844c1a7ad6d723d5b5b4c4fc7f5bd399c40e288120d16257c7c9e8765c6e85 \
    --hash=sha256:44b8faef94f1857e77fa0238f3390ff1ac51d2ea20a487e2e452a59fd2b5f5ca \
    --hash=sha256:470d420f98d368d6f010633a20659b544c5fdfa5329e6b70219f2ef08fd4a7ef \
    --hash=sha256:482676e5bd48d70ac99d9fc78863469845421e01184fa83f1f9366dc49f7e974 \
    --hash=sha256:4d4ca09bf13cff792b1884f64b98ee6c2467930d632233be25c56b442d99f10e \
    --hash=sha256:5025e36fb4fb275cef0a4e30dbb11cb4ae61d1c83deb90189cb5d7e4cafd6b55 \
    --hash=sha256:509735237ae0d849e8a843551d423d2500d2e0a9ac1611a145658b29c0fb9f85 \
    --hash=sha256:534f02c1abb31ed6dbd3515545285c330b2f12d00fdb1fdb71658b9ca5a13a6a \
    --hash=sha256:5978c3340f16a35c30f8ab2fa7bcf559973c55f1a5ef6970e1f621acf3c4db13 \
    --hash=sha256:5b973887ff782cfd6b67c9904ad8ca542e0bc5e4961503408b423b5a688b4d38 \
    --hash=sha256:5c490db2168a508088f59140dd392556a54b8bd1048fc6383c8baff13c359673 \
    --hash=sha256:5d142e352eb13facc7dd047489aebdff6ba78576c239f1ea04931979caaf0567 \
    --hash=sha256:5daa1f19e097050b9c4d9a78fcc9263cb96c9dfae08037ddc1b7c4ad1889f2a2 \
    --hash=sha256:61e9a64c7635095a6bfe483e2ff055d437c59bd45f3617a228b37277f0185d62 \
    --hash=sha256:63fb7294b768f444eb4b068965f2662f28c2fd4161e23bd60fcf3ff27b74c046 \
    --hash=sha256:685929988b208a911f1285e2f8ed54210b0d681a3dc0f03e00d599d291986e7e \
    --hash=sha256:6a797a1cefc8b9c93170db580337e1fe3d011ad18b1299943231279406342048 \
    --hash=sha256:6b92f60017dda7d877fdc546438b5e28f31c523264f49cf5a48c1d0ce1a0dfbc \
    --hash=sha256:70ed9a45c7484d2b30cdacf60d220f494a1763b9fec1ad03285c6553fa0889f2 \
    --hash=sha256:719a35fa1156db3640555f95ebb94f60a444e64d1c69626b0edef5df78eba225 \
    --hash=sha256:74ad5c3dad54a4641b4c28cd15ded70899d04459c6c7aeacafea716be97cce6d \
    --hash=sha256:74ea337e0ec3f6f342a36a4f1b5cd94dd9affddcd28ba9aae2905af932ee8c6b \
    --hash=sha256:75d9b1cf8258462dbdc1eeda718c96ea7f079324c09067f6daabfcf37712b7fe \
    --hash=sha256:77a4c8187a5948d7f8795adb765a3c7b553d07d86d88e43038fc32fc1fb9a3f3 \
    --hash=sha256:7824b5e8bdbf0bccb4ccd37bbb115849a1dc45437fb4de8351385ed07c437ee0 \
    --hash=sha256:7d38b0c279c3032e8c9cc013b405c6df8e1668dbf15465779aa7f15f61201812 \
    --hash=sha256:7e9c01d3dd7ceba4d1d436cc021d40d592466e40b9bc7f5d83dc4e98a5c9cd8c \
    --hash=sha256:7fd82debf43c6acd0a94359d232f6bb516ee13f269a7993736a9ac9f988bb5d9 \
    --hash=sha256:824c3d763a05ea9e9003610145186b0e9848c7584a5575c79bac5a8e7cd80bad \
    --hash=sha256:828f75af2b0080c8a972e75f649ab46af008e92c6104a57a759157200b835b75 \
    --hash=sha256:83f78128fa28705fa85d01c59771c72fe81c11bd0e6155edbb9f818983a7d761 \
    --hash=sha256:876bbfd276473d3daffe30e8c975df4ed9429967b41a6cb362dbb5155b6f13ad \
    --hash=sha256:886fc26012f0e8b5f69d1cfe6d711f6b11f194621539bf8e6bb1c25c5dc82724 \
    --hash=sha256:8a34616dc2521cc8dc1d7d081734da63539f021ac0450ce950908340c6e7aa2f \
    --hash=sha256:8a708a47ade1fe19e8371d5da076bac0dd4b0a5a7985ad6c637f7f7e361b6baa \
    --hash=sha256:8af9b142ad719ae3a911ebf616bc4b78b32bbab84d6a40d3ad2f129670509957 \
    --hash=sha256:8bf4df63592c2a66b4f8edc5df2544998c288aa02f96ce0acd880cd1de8c8127 \
    --hash=sha256:8de6f2a4ce7e7bd27d23dd94abf0ccafe0e0e5cc9c764b0577191f2c25f08f26 \
    --hash=sha256:8f8fddb8e323bd6eee4e54e69a39243beab22689070f4c66b472c4cc88bb89d8 \
    --hash=sha256:8fca690b00c4c48f6c2a547b0160ed511357093a4e4c9b47e0fadf3128066d89 \
    --hash=sha256:9506e892bcc3b409831d363c6f53e5985e1c8d1f6f6b0256d00358684ff85378 \
    --hash=sha256:958254518717542d02d0688d0d20cbf771da5e415e6f49543f92481c850a4540 \
    --hash=sha256:95a02752aa032eef4aed01cda6d9b687c669bd0396bf4519eef8bba22a286720 \
    --hash=sha256:96c30002424670b5e1e46495c2b8cbffef39cf77c1d79e76462029d50339785b \
    --hash=sha256:98b208a7cc42c803445ef551d6753cc42a5ea13e9cab1ee66cd8b9cb70195330 \
    --hash=sha256:9b3092d8992a1d69b7a59c3e39f35e1b9be327a17f68a7c35fc17329e337d6f2 \
    --hash=sha256:9e51c119992ea8820706871c30a4642ec76de20ae82f9b50b9a45517d8e9f810 \
    --hash=sha256:a5716a33bfabb2c6ce27b6cf03253467b3804f83e215f4d202685cf93c6c9874 \
    --hash=sha256:a5a00665d1a0e26763a7338d7e911d4598fbc1d50dd0d6b7919b7dc6c5d6569f \
    --hash=sha256:a5ca5aebae78a0bc13c1943af4af615d4966c5b650b05d5aa83b50e427196fee \
    --hash=sha256:a7b85b2cc6ea45e5f7e8c9a30bc9fabd47cda09106cbb4b967335c3e6c43b69d \
    --hash=sha256:a83ee7107df13abe42a54a6654670eef9bb39425cf2e27f65e0007465e1286ab \
    --hash=sha256:aa7d00b1700966d2917e54d278aba86897890ca9276dd8b76cf6446b6c181b92 \
    --hash=sha256:ab620eb663952455271ac37f9aaad86b73c969c02f11f53cea405b38e96a4300 \
    --hash=sha256:ad8b9671348d7c8716715652ae11f85ed0eb99e265a2df2ca490577d69860b2c \
    --hash=sha256:aefe930d113798330e9462f7874542977869c0613cba3262e2de3a8d5dee8f3a \
    --hash=sha256:b03af77d77e50edba2030fd5f7c352ff209314b09030a3cba7c14edf9a09a444 \
    --hash=sha256:b390aec180a7c054919c04898835e1c77bced23ea8383eb2c570213bf25d1a86 \
    --hash=sha256:b3d78f7bb2b9d9a30345be1474b9aaa8685430b54afb51ba3639b5c6c11e9ed6 \
    --hash=sha256:b5664603a253efd3a75716d793d1d3a6a82723b61dc6db767b2460bbbeec4c0f \
    --hash=sha256:b69602970994a2ed8bbfa78c2f0394a7435226c6040489702d9f0a0ad0c07052 \
    --hash=sha256:b6ae6a0328f0bc035741820fdeecdcd67bf4694eee03972e843663107122f450 \
    --hash=sha256:bad20d4c69c851c982a1e3606f4c293edfd5a87885786c50082412240c4b1ffd \
    --hash=sha256:bb7c99f0673c03017a3ee01e54a5c2617a05468b11eabe513b0080e063ed95b1 \
    --hash=sha256:bebb89489b279b2f5661bbbb2abcc87bcd4a46607bb4a5c966f04f1db6b8df9a \
    --hash=sha256:bfd1de989b3330420e29de39352f5c049905c9e3ee67233a50d550e3d652c148 \
    --hash=sha256:c2306e8bb53601979fcb3fa09cc65e031876d9ae01eff2fcbcd7a84ef94d5bc1 \
    --hash=sha256:c3a4e41e3096bf1f0f1b76e2ffd6d828d6547f574f702d59bdbef7acfa59db9c \
    --hash=sha256:c6834b92dd2428e2dd85ef3d85f723d3c12f20aaf43a2ddd4f944ca25d833408 \
    --hash=sha256:c90d3022d8a94778939cda8638c6c8da8fa757b8958dad7ec868ce29c87681b8 \
    --hash=sha256:ca307d6c259e5c98d3cb9ade55342b47a6839762caf2536f3d7b46ee660cc82e \
    --hash=sha256:ca7f6fe0f37ca978a1e5eb7a3a68e6413f417e78e838324947ffd420202b198b \
    --hash=sha256:cb6fae641357ed2f6e533c0d3c6504a4a5703621a50c89459e46051d56b61140 \
    --hash=sha256:cdaeeb6c350106df6bf9d873395973e5f066a9713200b72cd64f55d0a3eafab6 \
    --hash=sha256:cea20da04494e662b83c872683bf4ff2345206043d036315ed0e924b652e7294 \
    --hash=sha256:cea90547bfd93807e0013a004dc76552be44fad3bc1cc2b38610a9e889ed098f \
    --hash=sha256:d09037ca068d784ebc4aec290ef952ca27ac15dd9c0b5801a88c6e1096b83e6b \
    --hash=sha256:d27c2123977cb9269c30a49ba45f03a4323017ef693e19db4ec9dbe1299a3002 \
    --hash=sha256:d50de98e8d807dc31822fff96f50293163a62418eb65487a21b42713d72ed0b7 \
    --hash=sha256:d66a64dd5dec136040ec2ae94aa026a912ee60fdd45bc28d3db30037fd809e88 \
    --hash=sha256:d79308fa689fac89cbcfbd4dbfc80b5f95c54c5a7fd4d194be221f9d33d026e6 \
    --hash=sha256:da3275833be0edbaf4830fae08bae3dc7219f40ce0c37eaa6c25825957e06612 \
    --hash=sha256:dc1a26b8e53395a01c2c611e58602fa47461f136fba7cd5542e6db6d64be1839 \
    --hash=sha256:dc23390afe9f4ef9ac3bcc72a03a56eebbde03f4c571a32cb38f859cff9a6524 \
    --hash=sha256:e05c2f7925f1d88778e53cb44f14e0223204a3bdd09a41664750363acfb1f2ef \
    --hash=sha256:e12dfea7f5fc2a34a9080efbf79c4c44eb380ec5b9c6fea09407e08f0d1e941d \
    --hash=sha256:e4e4523d6f336708d732516e6cfca7796cf3d96c9474eb5aecf6165f2f1fefc3 \
    --hash=sha256:e4e49f7e1a4e7191bdf9dc67a974db714501b1fc52c24324103d06a86abd5c08 \
    --hash=sha256:e68e151428b5384f766cd25739bf77c7e4a3dc93b5ded7a12118d9fbfdf78ab6 \
    --hash=sha256:e8e4d953faaded9ec7ede36824e9814082d22d4c7b1eafbfa079ecba8cd0d076 \
    --hash=sha256:ee9df1f0d77b9c6e94f4ac0fec533fbddd5ea3a327807f18d7b069ae019ded80 \
    --hash=sha256:f0a887b6565bbfe80efde2b7f6e8890d7d9bbdb11bdb17028a3690c32fe0621f \
    --hash=sha256:f0f4a42db92d6ec7677ab9d12830a2a8ec145a9c6d15db2b593466bc875c78d7 \
    --hash=sha256:f1303ef2eec81262a4b708c3e858afe58d7c75ad91c1c05266eda7673369859a \
    --hash=sha256:f1d56ec54d257d05e0b50f5780d967540cd07beeaf9e5f645b26d50cce79f4d8 \
    --hash=sha256:f4167e87b397f273dc2356fcf1eaf50a6bac51e6105f45103ef7129c8efb0255 \
    --hash=sha256:f76fc85bd054c806960f917ec0f329e24e436f1712267d90588e4c39890caa63 \
    --hash=sha256:f942903fde7363d1d879057ec5de01310efda2597161784d752fa9953a01a71a \
    --hash=sha256:f9b1c4900736e489a812c529100de4b8fb617d4db075e931e213c57424b83d9b \
    --hash=sha256:fc271a6f0a2126958f4090e5507b9da5848927dae331f8f763bd4aa642b3d2cd \
    --hash=sha256:febcce10f2bcdbb80b4ea919238a6a4ac13dbc4c7cadbe8d5d75c3682f8b5404
markdown-it-py==4.2.0 \
    --hash=sha256:04a21681d6fbb623de53f6f364d352309d4094dd4194040a10fd51833e418d49 \
    --hash=sha256:9f7ebbcd14fe59494226453aed97c1070d83f8d24b6fc3a3bcf9a38092641c4a
markupsafe==3.0.3 \
    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \
    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \
    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \
    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \
    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \
    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \
    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \
    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \
    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \
    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \
    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \
    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \
    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \
    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \
    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \
    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \
    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \
    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \
    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \
    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \
    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \
    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \
    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \
    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \
    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \
    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \
    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \
    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \
    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \
    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \
    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \
    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \
    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \
    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \
    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \
    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \
    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \
    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \
    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \
    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \
    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \
    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \
    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \
    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \
    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \
    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \
    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \
    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \
    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \
    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \
    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \
    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \
    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \
    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \
    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \
    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \
    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \
    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \
    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \
    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \
    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \
    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \
    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \
    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \
    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \
    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \
    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \
    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \
    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \
    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \
    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \
    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \
    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \
    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \
    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \
    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \
    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \
    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \
    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \
    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \
    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \
    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \
    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \
    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \
    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \
    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \
    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \
    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \
    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50
matplotlib==3.11.2 \
    --hash=sha256:01dc8eaaab5a9fce9ff615eca82345728f289e4715b186ee10c6d85272fc26bb \
    --hash=sha256:02329432ae5c6af87cf208ee575b701d698bdf0b1a3bb28cc6d53c36e967e575 \
    --hash=sha256:07d9b9fa60cd4c393692f50d0bb03123242ddf61c99bb0e95e75feb354e7c1a8 \
    --hash=sha256:116cdb0eb0eb5644fc98eb2975d4b4dd4ad35e5c8e6b851c22e6976f371f7ab5 \
    --hash=sha256:1944895967f87c84c9b4bad29a707b31f5b36df4a3a2339ea1f8ea3ce5105539 \
    --hash=sha256:1a3040b209f3968b4e84161df7b174f07a9fad33b0f2d7e48ea3bbd3075e2863 \
    --hash=sha256:1b9a7ad579856284135e401ecc918c5f8a017ee30539298862a109f51b971710 \
    --hash=sha256:254d4ddb2fa8df3b4c689c0c306063aee10521df82cfb438185e499c75fe37c1 \
    --hash=sha256:2a8285cea8ef4d92aa041d1c33788bcae82248503300f93f1ca2136b9049452f \
    --hash=sha256:30ec15d7eefee71de16b7c689b42ba49715a4644650b76a6c7c70d79daf24e91 \
    --hash=sha256:399fef672f7046ef7d6a57572b2a6f9845f3f5afcff04e7b2df7a363a9f42190 \
    --hash=sha256:3aa4b8516fd26659e4363abbf317c703d9116496c5db2e9d0609a2866dd39dd2 \
    --hash=sha256:3da3bc0cbf7245e7db72cc6d29d12c5abef72cb73059c73b945f14e3545f3eb2 \
    --hash=sha256:3e8576f7c47e02fd4f21f44171302d2d1d58d4471d46da3d71fe8899d19539d9 \
    --hash=sha256:57b9ea60a835937c2012861923cbb91f47db8565775d326d1c42fc926aa10351 \
    --hash=sha256:5e1e923a3fc3326b99ec0a6ff1ab1338ac6c6cc62ad9d8a9c944197c7f8c6221 \
    --hash=sha256:643ff850d8e0f5b8319337f87ed3cb59506afb3df3cc48de777d85871233be7b \
    --hash=sha256:75b6d88402770e181b5d06a67dda4129c31da7d05004d63a21c310ec78d1b83c \
    --hash=sha256:79a258f58253dfa025af80a9e9bb228d75fced007f0e93ae7423fefbde81a74d \
    --hash=sha256:7d43ff8cebb50840648cb6429b2228621dbd709010f3117b3740abb20abf21c0 \
    --hash=sha256:7e5a90f8a707ebb6004a713b1cff091a40cc5df4c7c1cb165e5a505ebc11c292 \
    --hash=sha256:7ef7a53b66780e5d942923724f08577fbc5be1f7322da0f0f3f9dcaa45dd803d \
    --hash=sha256:854df8d7dfe9fdffcbaa6f39e44a6c24b409cb4d7561fbc09213b157d833f6a6 \
    --hash=sha256:894a9cbbecbe30ae6787d464df2e8fc7a8d475cfc68f87c3029f7c11152899b3 \
    --hash=sha256:8c8255de28f986d935a64c9ca71c0ec2d2f41d355691f5ea684725dc91413f71 \
    --hash=sha256:930efb28f59fda124e39265d177bab302625297bb147d2910de725a2fc2aef54 \
    --hash=sha256:a24d5fd36e4f0e742c3851dcd20810e56a95633a342e4bf6cb591c678e8fe61f \
    --hash=sha256:a6939df7567114b6bac7f4c5e06c84a67f197c1b2f2e4b234d4eecb3bfec9482 \
    --hash=sha256:a8756cc73d9af9a7fe0deb54ea2e75ef73b01d9e575877e72acad5458e660943 \
    --hash=sha256:af2661f6ac6bbd1d081996f54fdd9385715c625ace8cec0869055c0cfbf38981 \
    --hash=sha256:bbf1062991d826ed27e2144f3afa4461afbb8ff56e8f703043e191a8163b1ee9 \
    --hash=sha256:bc067c462a86f0e57bf52fc6e058d90171a5420007dac00c46e90153050c69a7 \
    --hash=sha256:bf3fe71fbfb8ec0e310e0bc8537c3405a01f38f25f9394ed2135e6202fed542b \
    --hash=sha256:c0b83f044ce10a98027b105b3931548719a6e8c7ef986b4362651e0b5367c8dc \
    --hash=sha256:c27e577ece613ea12a0790e00b4eb80d901c59a3e16cad474f31d8b1529690b9 \
    --hash=sha256:c5c1c68ee401fc98271263410f0e5ce88285abacf7627132914e8adf3d70ff43 \
    --hash=sha256:c9721f81275499da1feeb36a2cf8192ea086283b3bd16b7dc4c9d7aedb7396d6 \
    --hash=sha256:cc82dde2a0d3e3ad472edce04897ad7146b8d8bfd1df8a32992eebb81af18fdc \
    --hash=sha256:cec596316640f2b394b8f0daa0ea61a8eae82d017b620b9f202befb972a59ea4 \
    --hash=sha256:cf41ecd1b0c0b6f7177ed965a54c2afbe888715c7cf6054dc12d53bc1494002c \
    --hash=sha256:d3304eb5a59442a8867f6920d484591c0fa09ffc29e9260be2feec3351e25869 \
    --hash=sha256:d480038c83691532ed52ff3147db51fa902fc78cb2d8349993a1cdb684435bff \
    --hash=sha256:df4f7784aca81a94f254c0a2767d592ee25f407e488f5fa7203e51093fb6ca27 \
    --hash=sha256:e43b188f0a5b75447bcc197728258166aa64365770ed1caa36595a5e1ca4bbba \
    --hash=sha256:e60cf3047a51edecdc4535a9196bbd6a732936b8e9f8184aabf4d16165884aaf \
    --hash=sha256:eac4b07d4e3743b172451e122ea964f72f152879d4f9adcf3f3d33e518f12ead \
    --hash=sha256:eb3712dc9b464793de0a4e42a7313d50293c751f94bddaf7332a1bc71bccdda9 \
    --hash=sha256:ecea603dd2fbf8242fd31a305a8b12a4ece2de28096870c65fdd0d1e35b8d9a6 \
    --hash=sha256:ef31985c4dedb5f1424e1aec6849a47dd37689cb7fa3c20b1b82187f26806261 \
    --hash=sha256:ef752769cd962f39ea0b6ffc82d1ea43a0012c5a6157c7a075212fa509cfcff2 \
    --hash=sha256:f25446b2981717dca9786bac841cb3fd7efb568e3e3c755dd980481c5cb9228d \
    --hash=sha256:f2ac30cf5eb5dff1b584627ae0b0e1186551a4f69ae3c75073911da497a29170 \
    --hash=sha256:fea03cf56568cc1cba08b470be6a0559e71c3a5b688d54b7179bb35ba23d0821
mdurl==0.1.2 \
    --hash=sha256:84008a41e51615a49fc9966191ff91509e3c40b939176e643fd50a5c2196b8f8 \
    --hash=sha256:bb413d29f5eea38f31dd4754dd7377d4465116fb207585f97bf925588687c1ba
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
nvidia-cublas-cu12==12.6.4.1 \
    --hash=sha256:08ed2686e9875d01b58e3cb379c6896df8e76c75e0d4a7f7dace3d7b6d9ef8eb \
    --hash=sha256:235f728d6e2a409eddf1df58d5b0921cf80cfa9e72b9f2775ccb7b4a87984668 \
    --hash=sha256:9e4fa264f4d8a4eb0cdbd34beadc029f453b3bafae02401e999cf3d5a5af75f8
nvidia-cuda-cupti-cu12==12.6.80 \
    --hash=sha256:166ee35a3ff1587f2490364f90eeeb8da06cd867bd5b701bf7f9a02b78bc63fc \
    --hash=sha256:358b4a1d35370353d52e12f0a7d1769fc01ff74a191689d3870b2123156184c4 \
    --hash=sha256:6768bad6cab4f19e8292125e5f1ac8aa7d1718704012a0e3272a6f61c4bce132 \
    --hash=sha256:a3eff6cdfcc6a4c35db968a06fcadb061cbc7d6dde548609a941ff8701b98b73 \
    --hash=sha256:bbe6ae76e83ce5251b56e8c8e61a964f757175682bbad058b170b136266ab00a
nvidia-cuda-nvrtc-cu12==12.6.85 \
    --hash=sha256:3f3134f50963882373063901657554f230bedf6039d30b09f6be55c64c993a37 \
    --hash=sha256:800927308ccc5dd6246d3f61f7fcef2ed7ec4e59e199090d360d3293f78bd5a2 \
    --hash=sha256:a419e2c95e75b88b602f8bb66f82a6c5651e8475a509841c958486b1b71510bf
nvidia-cuda-runtime-cu12==12.6.77 \
    --hash=sha256:6116fad3e049e04791c0256a9778c16237837c08b27ed8c8401e2e45de8d60cd \
    --hash=sha256:86c58044c824bf3c173c49a2dbc7a6c8b53cb4e4dca50068be0bf64e9dab3f7f \
    --hash=sha256:a84d15d5e1da416dd4774cb42edf5e954a3e60cc945698dc1d5be02321c44dc8 \
    --hash=sha256:ba3b56a4f896141e25e19ab287cd71e52a6a0f4b29d0d31609f60e3b4d5219b7 \
    --hash=sha256:d461264ecb429c84c8879a7153499ddc7b19b5f8d84c204307491989a365588e
nvidia-cudnn-cu12==9.10.2.21 \
    --hash=sha256:949452be657fa16687d0930933f032835951ef0892b37d2d53824d1a84dc97a8 \
    --hash=sha256:c6288de7d63e6cf62988f0923f96dc339cea362decb1bf5b3141883392a7d65e \
    --hash=sha256:c9132cc3f8958447b4910a1720036d9eff5928cc3179b0a51fb6d167c6cc87d8
nvidia-cufft-cu12==11.3.0.4 \
    --hash=sha256:6048ebddfb90d09d2707efb1fd78d4e3a77cb3ae4dc60e19aab6be0ece2ae464 \
    --hash=sha256:768160ac89f6f7b459bee747e8d175dbf53619cfe74b2a5636264163138013ca \
    --hash=sha256:8510990de9f96c803a051822618d42bf6cb8f069ff3f48d93a8486efdacb48fb \
    --hash=sha256:ccba62eb9cef5559abd5e0d54ceed2d9934030f51163df018532142a8ec533e5 \
    --hash=sha256:d16079550df460376455cba121db6564089176d9bac9e4f360493ca4741b22a6
nvidia-cufile-cu12==1.11.1.6 \
    --hash=sha256:8f57a0051dcf2543f6dc2b98a98cb2719c37d3cee1baba8965d57f3bbc90d4db \
    --hash=sha256:cc23469d1c7e52ce6c1d55253273d32c565dd22068647f3aa59b3c6b005bf159
nvidia-curand-cu12==10.3.7.77 \
    --hash=sha256:6d6d935ffba0f3d439b7cd968192ff068fafd9018dbf1b85b37261b13cfc9905 \
    --hash=sha256:6e82df077060ea28e37f48a3ec442a8f47690c7499bff392a5938614b56c98d8 \
    --hash=sha256:7b2ed8e95595c3591d984ea3603dd66fe6ce6812b886d59049988a712ed06b6e \
    --hash=sha256:99f1a32f1ac2bd134897fc7a203f779303261268a65762a623bf30cc9fe79117 \
    --hash=sha256:a42cd1344297f70b9e39a1e4f467a4e1c10f1da54ff7a85c12197f6c652c8bdf
nvidia-cusolver-cu12==11.7.1.2 \
    --hash=sha256:0ce237ef60acde1efc457335a2ddadfd7610b892d94efee7b776c64bb1cac9e0 \
    --hash=sha256:6813f9d8073f555444a8705f3ab0296d3e1cb37a16d694c5fc8b862a0d8706d7 \
    --hash=sha256:6cf28f17f64107a0c4d7802be5ff5537b2130bfc112f25d5a30df227058ca0e6 \
    --hash=sha256:dbbe4fc38ec1289c7e5230e16248365e375c3673c9c8bac5796e2e20db07f56e \
    --hash=sha256:e9e49843a7707e42022babb9bcfa33c29857a93b88020c4e4434656a655b698c
nvidia-cusparse-cu12==12.5.4.2 \
    --hash=sha256:23749a6571191a215cb74d1cdbff4a86e7b19f1200c071b3fcf844a5bea23a2f \
    --hash=sha256:4acb8c08855a26d737398cba8fb6f8f5045d93f82612b4cfd84645a2332ccf20 \
    --hash=sha256:7556d9eca156e18184b94947ade0fba5bb47d69cec46bf8660fd2c71a4b48b73 \
    --hash=sha256:7aa32fa5470cf754f72d1116c7cbc300b4e638d3ae5304cfa4a638a5b87161b1 \
    --hash=sha256:d25b62fb18751758fe3c93a4a08eff08effedfe4edf1c6bb5afd0890fe88f887
nvidia-cusparselt-cu12==0.7.1 \
    --hash=sha256:8878dce784d0fac90131b6817b607e803c36e629ba34dc5b433471382196b6a5 \
    --hash=sha256:f1bb701d6b930d5a7cea44c19ceb973311500847f81b634d802b7b539dc55623 \
    --hash=sha256:f67fbb5831940ec829c9117b7f33807db9f9678dc2a617fbe781cac17b4e1075
nvidia-nccl-cu12==2.29.3 \
    --hash=sha256:35ad42e7d5d722a83c36a3a478e281c20a5646383deaf1b9ed1a9ab7d61bed53 \
    --hash=sha256:6351b79dc7d2cc3d654ea1523616b9eeded71fe9c8da66b71eef9a5d1b2adad4
nvidia-nvjitlink-cu12==12.9.86 \
    --hash=sha256:994a05ef08ef4b0b299829cde613a424382aff7efb08a7172c1fa616cc3af2ca \
    --hash=sha256:cc6fcec260ca843c10e34c936921a1c426b351753587fdd638e8cff7b16bb9db \
    --hash=sha256:e3f1171dbdc83c5932a45f0f4c99180a70de9bd2718c1ab77d14104f6d7147f9
nvidia-nvshmem-cu12==3.4.5 \
    --hash=sha256:042f2500f24c021db8a06c5eec2539027d57460e1c1a762055a6554f72c369bd \
    --hash=sha256:0b48363fc6964dede448029434c6abed6c5e37f823cb43c3bcde7ecfc0457e15
nvidia-nvtx-cu12==12.6.77 \
    --hash=sha256:2fb11a4af04a5e6c84073e6404d26588a34afd35379f0855a99797897efa75c0 \
    --hash=sha256:6574241a3ec5fdc9334353ab8c479fe75841dbe8f4532a8fc97ce63503330ba1 \
    --hash=sha256:adcaabb9d436c9761fca2b13959a2d237c5f9fd406c8e4b723c695409ff88059 \
    --hash=sha256:b90bed3df379fa79afbd21be8e04a0314336b8ae16768b58f2d34cb1d04cd7d2 \
    --hash=sha256:f44f8d86bb7d5629988d61c8d3ae61dddb2015dee142740536bc7481b022fe4b
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
protobuf==7.36.2 \
    --hash=sha256:497d0463ff3316681da6c0b9e8d06cb465d61abce00b613ab42226175644d1bb \
    --hash=sha256:89f23aa53c24553a2416fd4fd1ec06f74fa42b14b546d8883128813f775bbfd2 \
    --hash=sha256:912c1221170e16c08d1f086762f563dd61ff83c18b5fa6652952dfaded66f728 \
    --hash=sha256:a300819d441e078a5608c0d3c709796bb548136058fda017ae51d425b44fd353 \
    --hash=sha256:bdb3a345d48db958e6ce1f18e508beb0cc981d64f24088427549c866cd039f1e \
    --hash=sha256:cbc70b17ee27e28894c7fee8bb04be1abead49e936bc70eb60052531eee2079e \
    --hash=sha256:e11e1f0180583a2af89db6a2ecd9e8dc40aa6d2988ca175bfd0e6d12ea72d74e \
    --hash=sha256:f4fee11ec330d238b34a05c9b675f693c20415d1c5bd7d5320cc2f8a798eb9cf
psutil==7.2.2 \
    --hash=sha256:0746f5f8d406af344fd547f1c8daa5f5c33dbc293bb8d6a16d80b4bb88f59372 \
    --hash=sha256:076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9 \
    --hash=sha256:11fe5a4f613759764e79c65cf11ebdf26e33d6dd34336f8a337aa2996d71c841 \
    --hash=sha256:1a571f2330c966c62aeda00dd24620425d4b0cc86881c89861fbc04549e5dc63 \
    --hash=sha256:1a7b04c10f32cc88ab39cbf606e117fd74721c831c98a27dc04578deb0c16979 \
    --hash=sha256:1fa4ecf83bcdf6e6c8f4449aff98eefb5d0604bf88cb883d7da3d8d2d909546a \
    --hash=sha256:2edccc433cbfa046b980b0df0171cd25bcaeb3a68fe9022db0979e7aa74a826b \
    --hash=sha256:7b6d09433a10592ce39b13d7be5a54fbac1d1228ed29abc880fb23df7cb694c9 \
    --hash=sha256:8c233660f575a5a89e6d4cb65d9f938126312bca76d8fe087b947b3a1aaac9ee \
    --hash=sha256:917e891983ca3c1887b4ef36447b1e0873e70c933afc831c6b6da078ba474312 \
    --hash=sha256:ab486563df44c17f5173621c7b198955bd6b613fb87c71c161f827d3fb149a9b \
    --hash=sha256:ae0aefdd8796a7737eccea863f80f81e468a1e4cf14d926bd9b6f5f2d5f90ca9 \
    --hash=sha256:b0726cecd84f9474419d67252add4ac0cd9811b04d61123054b9fb6f57df6e9e \
    --hash=sha256:b58fabe35e80b264a4e3bb23e6b96f9e45a3df7fb7eed419ac0e5947c61e47cc \
    --hash=sha256:c7663d4e37f13e884d13994247449e9f8f574bc4655d509c3b95e9ec9e2b9dc1 \
    --hash=sha256:e452c464a02e7dc7822a05d25db4cde564444a67e58539a00f929c51eddda0cf \
    --hash=sha256:e78c8603dcd9a04c7364f1a3e670cea95d51ee865e4efb3556a3a63adef958ea \
    --hash=sha256:eb7e81434c8d223ec4a219b5fc1c47d0417b12be7ea866e24fb5ad6e84b3d988 \
    --hash=sha256:ed0cace939114f62738d808fdcecd4c869222507e266e574799e9c0faa17d486 \
    --hash=sha256:eed63d3b4d62449571547b60578c5b2c4bcccc5387148db46e0c2313dad0ee00 \
    --hash=sha256:fd04ef36b4a6d599bbdb225dd1d3f51e00105f6d48a28f006da7f9822f2606d8
pygments==2.21.0 \
    --hash=sha256:2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9 \
    --hash=sha256:610ca751c9bc2492b38eb9a38a7fbc93edbbb2d7182edaf34e66ae493dee5c8c
pyparsing==3.3.3 \
    --hash=sha256:928ae7e20211f3b6f3915a72f06a0cfd29ab9d24279dd6346b6b1a7146397d36 \
    --hash=sha256:ece8c00a69cf01b45d0b1dedabb469c90d8caf996d4fda40f147627a122849a4
python-dateutil==2.9.0.post0 \
    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \
    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
regex==2026.9.10 \
    --hash=sha256:030fa9e23624e39b3b94e46b90a5abd1a1678eb2f58fcdd3fd6c27526bf91c7e \
    --hash=sha256:032da15431c890d376f53547f0a6219f4f4cd19f3e4f11bdc321453b5bd207e4 \
    --hash=sha256:044bd4639b6bb409ec9e5d8b7accd57e02b4c4a4e2eafde916f8ae8006b3e40b \
    --hash=sha256:048a89ee797db10160bd2bd519286577a6b43a100279bd4b7d8456a3d69c80a0 \
    --hash=sha256:05fb018cfe7144585fc83882405906ff84994a2d154afc2509ecc7752c51f864 \
    --hash=sha256:07b45ba5c94b8fcb30cb6c56a11f715c57533a3017964504322ea52690a27b72 \
    --hash=sha256:0aa7589394230e0f0a422ab6b90841ff12c87e855e7aaf75d192a54a5f124548 \
    --hash=sha256:0acee94b480dd853e39434aa9a575f95385b1b4b8fa3feae56db363ca5cad782 \
    --hash=sha256:0b9ba3b2765cdfe18f0f561a69f78a69701f2896654a81c711108d35d14e5099 \
    --hash=sha256:0c32480f3371b75068decaf9e5da72c224e953830dd71e36e06cf80e30ea39d8 \
    --hash=sha256:1270cdec69248592bbe38a0b263ed58d907b891bd2b93703e225c317e421bda1 \
    --hash=sha256:13c52fc377792675f604a207a2ae5958c080f6854f7698d40d9ff034d95b1e76 \
    --hash=sha256:14caa05ce39ec70437af5aac8814c50ee6628f4a90353871c059692f448a164f \
    --hash=sha256:1562aabd9d4eb09bd88a62ad97ed06800094b529ac43419e43020b9cefec79b0 \
    --hash=sha256:175cf49ce7a994c88b8f15e3cb17cdb66a48ebb2d36de736b8205033db950f89 \
    --hash=sha256:1aa309ab7ba89a62d6cf70dbd38d4176440bce3c7001ab86256704cf4c18c6eb \
    --hash=sha256:1ad10a135fa0b4e4a462a61d07c6654d7518cfdb5cb8da08f9ff7d61384af1fe \
    --hash=sha256:1b891f77554bff991804cee24b78b40789f7d5993a24c7907bc7025fd2a70c8d \
    --hash=sha256:1e321e2c84f0e52c457f5ea5944f796d6e8e09cb99738ea98dcc1bfe402a128d \
    --hash=sha256:1e954e246466d5a1a78f563ce8364b5d7cb19e7adb0ccdec8f9c9610083187bc \
    --hash=sha256:1f0a8b4928823bc8b217a1ab7bf3d90598909dec9a70fbbfe9a52cc4eca55990 \
    --hash=sha256:1fbc8314436353e097c050e11b01a6c11433579437ed0579730157676ef59e2f \
    --hash=sha256:20e8bfb07ad79a282f8b95b56fe67f9750b1b7f775724e4ba1f23cb296115ce4 \
    --hash=sha256:217e98ba5fc8908ed8ffd4ebac04753a0c831067cbfb495b9821b94cc61eaa76 \
    --hash=sha256:239620b0e0681669367c0e218c8eb2551d9f8fe3b9fccfc8d0003377804e8348 \
    --hash=sha256:23ac9a28180f274d7dd7651fa131ad5b02d343b75df4b040737f0356223895dd \
    --hash=sha256:2479171edccced52ef02b899558f88ab2c235fe05b93180fdcae1670aacd89e1 \
    --hash=sha256:24d12a625a37c89c2b09303402a06942f55f071b95a7916a49c17034c3d47cd5 \
    --hash=sha256:2dd9286093c71afc8f55ef035c5b9d2776641fd72c6535f1febc92d0b0be9666 \
    --hash=sha256:2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0 \
    --hash=sha256:31e4df2b11d48f61d511019bc1ee9b477055f17c352b68fe72db7a98b14d603c \
    --hash=sha256:3264132d576847ab5f88bb83e7debe67854bf165b3ea613bd467312b6099536a \
    --hash=sha256:3540734dbe241ebb3b87d5713781f6749a3e4d45480f506aa5fb5cbb0c37d249 \
    --hash=sha256:35ba3bab0c45079735f55ac61526774de1d84bc4a0333cc554e1a4ab74913924 \
    --hash=sha256:3a66e40a1a20de96a2fee00ed67e11012b62d85b277688258677fd19997addb7 \
    --hash=sha256:3bdeed3318a8eb2bbadc9c56347e0ff651639e934a47e168d05a3b12929fd0e7 \
    --hash=sha256:3fb4ae8cf83ef4e9addd43b2da31a9f45be816a8036fae8af59c8998b72718e2 \
    --hash=sha256:4971776b4f2bd7fd9a83eceb2cb2592cbe2924f639fe8045e6a9de5ba4bfcf25 \
    --hash=sha256:4a761ea45f2ad74c575ef5850ea514cef97302a552d3c7c9d1a1a870d4661d6c \
    --hash=sha256:4c66d54042a14a503907d81861b8a5235e6d1f03d4fbc1d8767f652eaf957ac1 \
    --hash=sha256:4db7d00c4afbfbb55b8e17b1e371da11418ea9389b030acec63c1fa4c7ad4b86 \
    --hash=sha256:4f0407474ffac8e5e89d93ca41d60891e29f0ab8423eb66ff292d850a86a0843 \
    --hash=sha256:53e182b6b04d0011909b47d51a2d72d908de07c7b1c7f16b3adda2204d723bc1 \
    --hash=sha256:5847e22bbf959764d776937d791d034cc2d19b787e361c88d97e859e8dc68502 \
    --hash=sha256:58c01f7b81079cf0817ba831ff4d9eff5d28be4a3ac76c353e6f09bd63f4c386 \
    --hash=sha256:58da726d3e766c0b3f5a3997dfaf0275898a1107b8191cdd6b0437fe45fd817d \
    --hash=sha256:5bef622850cf760154719d4e0d74b0a855962432995168e250069899ae12fe8f \
    --hash=sha256:5ccd139b2061132e7b265cfb4b4721baeb9f8928b81415304abf1ec7e3181c26 \
    --hash=sha256:5cef9f3d14796500ea834c41dbe688f1f6b23c7024dc23e8a794d7ebaf5d71d0 \
    --hash=sha256:63bb62cf62217dc38c8a6b2b61b165b0e4eb8fa93b0aba12139251c0986a8fa3 \
    --hash=sha256:681ed38664b64c6617d3c3c332018d1948c77e139c5ea667c1886efa671e426f \
    --hash=sha256:6888065672b341e5246f391ec16dc258a29218ac784172fd67c30d941544755b \
    --hash=sha256:6aebdd9a946de328b3f6f61dbf48dd064a36eb6dddf96e34ae6651d37f6e9383 \
    --hash=sha256:6afcad14310f1311d077553ed374b42a5e538f85a8c884b4e38e52de091c8077 \
    --hash=sha256:6b34a778c695d24e77c140e3b4c95da69282e34f2f6b02b55656aa4a0379f643 \
    --hash=sha256:6fd555fc9abef50c530869690b2daca054c8811a7aff632d11f9a7b2590b2742 \
    --hash=sha256:71879292c9c7ac67b1680345b16daba1be937cb027362cfa04e68f65db2dcfdd \
    --hash=sha256:75242f44a3e283106077be4ab717bc535e4701c9d54ad69e195945c22f137a1d \
    --hash=sha256:75aa39d3f4f1650eea84e46b0d8cefe77dd5478c10e3d0aaf0b0f00493475a7a \
    --hash=sha256:75f9297b16fcb588a1f8d8a55dabef3c0c20b0c7bac43c87ceaaaf1a825c12f4 \
    --hash=sha256:79e9432995e14c749d34209413de5e621ec8e67789bf4f46dbfabea9d06a2406 \
    --hash=sha256:7abb38b8c40f3a235235a44da452c64b7b5c1d650ec6351027db0e090804f2e5 \
    --hash=sha256:7dcad477c49c4c626a6c4fcd71b39a971aa217060cc40a6569fd24edcc0fa509 \
    --hash=sha256:7e6c0b5ec6ddee4032247585dc491b0fa58627745b66a705728703a3f0331231 \
    --hash=sha256:7f8f10015866608fe4c043cec2e4fe4c39a94bb50e45091de4cdf4004b9ae4b0 \
    --hash=sha256:866de9f98df0611d7b62b3a8729d3284a64c0cc6edd90bb95a533e443a4939cb \
    --hash=sha256:87f5f75c109f08f5c602d68e1af54cead8165189c727b6ac946b30b9833a3ba4 \
    --hash=sha256:880ac684c27176464c00c3fdc456116364f5ebc70da07aad0c2d4a7ba45e98db \
    --hash=sha256:88b02aa8d0ec9b6189fe933d425775882271c23700ac11fd26d1779b0f56fde3 \
    --hash=sha256:8ba1f78bd4fef2d8f84b894ec28ac3481afe6cc07aaa253ad4717ef7b3fe6bcb \
    --hash=sha256:8c07021a4faa3f092869adbd1f35cdc7a592276c807aeebc3ceb8ff1a638f0b4 \
    --hash=sha256:8d5c4518235a2ec1611e57af85fa488d529c1106aacff12adadcedf8687012cd \
    --hash=sha256:8e127d9a80cbf1c3276bb465c6d047e8705e97b58c2b8f2f0c0a69c336b44b37 \
    --hash=sha256:94c5ce3bc41d226b4eb89ca3f842b2e28c031487fb1f34eb2153d98235831325 \
    --hash=sha256:94d096369b7cd96d15343fef5257fe39eff9d0e8758b92a0e15e358b92cdb2fc \
    --hash=sha256:968c1e33edd9a104d1bf24c8d476c72de7e3839ae7f894b37e9e4f4739fdeeca \
    --hash=sha256:990797e765d89a423880052c68b61c31afe701de94a8c060f61c40605ca6c727 \
    --hash=sha256:9ce239acb15843ab03976626af810a4424b0409689ec2bbc52088ab5479ab487 \
    --hash=sha256:9d772586951d7d6a5d162d48f414065e483b1c81ab38fd8ed97c78b05883421a \
    --hash=sha256:9fbd2e5d8002dc49a6129fb321ec51c57a025e752ed525ddce0ba9223c4350a7 \
    --hash=sha256:a41693eb3fc4b92e6127d113813c6c395237f7edd3224abf67609af48c690d11 \
    --hash=sha256:abbfc1c33bf8efddcc43844aba61e036d74a918680dc3ce8ce2538b004eda0f9 \
    --hash=sha256:b298cdc33c5cc6969ff07f0fba19cc73e0fd8576373c50935feadaca2f6b4405 \
    --hash=sha256:b43456de605c8ee77eb75f07bc1ee44ba27f9cee22207deb77d495e954b7d953 \
    --hash=sha256:b71649169a9fcf30b395ee01047fa7ad6654a4c900ca75b23c04dedcce6a1f8c \
    --hash=sha256:b91c37551bf39d75116c02b146956f65b9aa0337a4a652f4ae186983789d4001 \
    --hash=sha256:b9d36b03dc362aa40ffaaec9d9bd75e87763529563ec008c43b0e07782f5be7a \
    --hash=sha256:bafa41b0dd63669e5c0f8adf3d24819efeb73c847f492eb011212eb352e69041 \
    --hash=sha256:bb7774924f8cd69f49cba0b3c2d679a6326f777e0e67d130ad5203e4df53f0d3 \
    --hash=sha256:bf29611e5376fec8f795879bb5c6153a76c3a292573d173c26784042b01eb840 \
    --hash=sha256:c014641157e9049b0603b8daa5343bd408d9b757b709aaa0f373cd3fab2d7944 \
    --hash=sha256:c103b3b14e011774af4fb7e4617ad4d72b9171905cd3b231a70a4efd76e477d7 \
    --hash=sha256:c22df8dd6373bbe3898e77429ffc85594300e39d752fd0e68a31e59d37899376 \
    --hash=sha256:c25a754bb81a2edcfc3b65eda50f017d736f818112ed43e8aafd595cb00678ae \
    --hash=sha256:c32818b28bcd153b25b63038348a9fe9b9fbcddb60df43f204c3ab55eeb57f77 \
    --hash=sha256:c37fa93bf18bf4f90b01c0fa9f11ea567ee4b7dd8bf96e63663e5edc37aa38cf \
    --hash=sha256:c3d95d7d9538b5b726dd6fcd7b6117a71e6565202f6d64f5845fb4d8f203f533 \
    --hash=sha256:c8fbd9cb30c68c1686b94029b9ef845d5870d3d65baf66cb126b676849b9d72b \
    --hash=sha256:cb76a9c4e07a6a47849726af0ed14c41741a182f097f134a8cf29c1bc0f4dde8 \
    --hash=sha256:ce7c118cb102975f974585688357a717ffbf9dddd64ab0bb1bc93eb5b367cf95 \
    --hash=sha256:cf377960d2ac37d987394a9dbaa75e91338c41a46d41e1d25e90125e7b3ee2dc \
    --hash=sha256:d278ad30ec83b6b9202685b0f80b741a51ea3ca7f0595ebda96e7628b6398876 \
    --hash=sha256:d2d377fd1cad611b806cdd732d86b65f536c768209890cb442556548daa65a23 \
    --hash=sha256:d414c411c06fe0009eac33488fb1591c66b5c2673e342e452e7bb2fe63da8194 \
    --hash=sha256:d8c668af8f7bdb1d18739c27d30cd9f4b371495a883f75a002fb7a39d740fecd \
    --hash=sha256:dce932f8e3ba936475ea3d0d8b59f7b050a9e206e994f53f8fd80299871e87da \
    --hash=sha256:debc629e98b95abaea1cf3057ca296151f348c697c9b8a59d18013adb302c0dd \
    --hash=sha256:e0dc78251154b66dc60211563fc115345da332eaa881e4e2523fb1edae3772f4 \
    --hash=sha256:e5e4a6e0734a685d13b9685622bb503bdbb2927f8b0df025a5085f0ea067475b \
    --hash=sha256:e6b99181d184d0f5c7b36b8d12b94d1e9499cce6246594331f9edc5d2ea9fceb \
    --hash=sha256:e7327795089ddb44912dce1434e1d7244be2e9fb48fcc2d6782936af7a3062db \
    --hash=sha256:ebb2ba68e4641a994061f70bf44ed448fba0b9b1d18c94ffb9efc1cca805b39b \
    --hash=sha256:ec8855f08c17895a26fbf5f19ed829722e19b34a96629e49a43c92974924026b \
    --hash=sha256:ecb2e7acb18f8cc4a67f0ad986c0af291ea4dd385d0614ba9bc09d7f8bbb478c \
    --hash=sha256:ef4c0a9dfdc90581b90b1b95a8c3d1557f8ff8f5a2a53536d26314de699d1468 \
    --hash=sha256:ef4ce69ff97fbb44b46751cfea5e859ad0b66d1a50abf34954f0645f51e81671 \
    --hash=sha256:ef5a059ea1c6ee5d1c7e99a2484e628608d010921efe876c6f0e2029d2f35eca \
    --hash=sha256:f0e2e5d23448b660d60a6ed85c46cc03b4b48bd276b8f4041d4a5fe2a4a0626b \
    --hash=sha256:f2374c27deb189b282ec7e16106752c22ad39b056bbd8018960b1e4cc95d67a1 \
    --hash=sha256:f2f43bf4e47ff7ce9e585558706d698c6204d0f80bf2207766382ed817c8e9f4 \
    --hash=sha256:f5c629df03adec31ee505dda3c8988f106c9390e4cbd343600036eb8b3d6724f \
    --hash=sha256:f70b9f0e39c2dba1d9da6bf7ef7c377cad7277f8440e9a69be05ede529ff024c \
    --hash=sha256:f7d4656e17ab736e9415a6442a345bfc97bb8b7dcce47884bb74a37f70f08d0c \
    --hash=sha256:f8bdec659a8fa7af51a32b224b3b7c02bc415d54ffd35187b1d224176b17d607 \
    --hash=sha256:faa911fbbcf8ac90bda0e0657d60768e3390954ef0588211d63a22add1cb1cd1 \
    --hash=sha256:fbc4e2f3cb7ce8436154e6483079e7d35eeb321a952fa936e180300630d8b873 \
    --hash=sha256:fd6bd89b9fc06018d35851cab0240adb7dd84d51941b19f6574ac90cd54e3ae5 \
    --hash=sha256:ff4d7b14ea19e50c8d9d6d83f45bd9b45cbb624c07ac1fa54db0a019049abed7 \
    --hash=sha256:ff6b3267318661dfddf6b3628663e00e5946bd0a5c8fa678537a1401f0388f91 \
    --hash=sha256:ffc2da104e43db716ce30cef9f28049a1faa6aca385dd8771b033268d0730b07
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
rich==15.0.0 \
    --hash=sha256:33bd4ef74232fb73fe9279a257718407f169c09b78a87ad3d296f548e27de0bb \
    --hash=sha256:edd07a4824c6b40189fb7ac9bc4c52536e9780fbbfbddf6f1e2502c31b068c36
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
sentencepiece==0.2.2 \
    --hash=sha256:046b15ea22d8042e2e173561d464ec3b64a9c2081324df70ebce7bf7ebb3e497 \
    --hash=sha256:0e2aae42960392d6dcb9a72d8e1e65a97294c965071b43c7b3429a42f350250e \
    --hash=sha256:1120e0791540615e650b2e9bea835bf38a7362455d8ab62dee7968219c2d79a0 \
    --hash=sha256:1402d8ee36f0d851cea8eee4dbb85fea14643b7503cf4d00d102eec0fe3ca719 \
    --hash=sha256:1416b92f2f010333786fe6306ed2631121d5ea492219b0841e967b6765e64107 \
    --hash=sha256:16c84ddef8d3084a8af37208acd365b08092ca089080f1a71fbfdd911adda9b3 \
    --hash=sha256:1edb10e520e4bddf74d85b0f5ae74cc2d60c2b448885080bfb618bc2b3a49f6b \
    --hash=sha256:201a8e0f55501a76e08dbf2c54bc45f4642b379271e89c667d517bfbc2191f2a \
    --hash=sha256:252908153eeec06c3ca3a32077e64a49d572e3d89881475b4e0f02d99d9fcc7c \
    --hash=sha256:38111ed1f79268f399c505028023d5eaaf0ab4e5eafceb709468b0d3323e7838 \
    --hash=sha256:3ab3f1ae98970b5590e2209341522718900ba19bcc2c207ffaa6bd417ad960c5 \
    --hash=sha256:3d2b5e824b5622038dc7b490897efe05ebbbb9e7350fc142f3ecc8789ef9bdf6 \
    --hash=sha256:3ec27c152a1f1b24bc9168b55a5880f3c16e2334e697da6f55a1046a22405a3d \
    --hash=sha256:3f5851441ab1ef8634963a5100b733a8bbeefe623e0c5c005b1f1f3880e574cf \
    --hash=sha256:3fd9ce2ab4460c713cfdeb4aca693ca6732a11538e05fb332d5af42e3d7fde25 \
    --hash=sha256:44284adc6fbe9d5bdd480541431a3d93f674fa44736714d3ad4bcee8283ace7d \
    --hash=sha256:443ac618c7a2a1377cf5c82581fbb849591d14e656d5e5a3e4682d4e36a34e4e \
    --hash=sha256:46ba07b543add034de0ff47ac5f907e9a06682f91d85121a972764628933be6b \
    --hash=sha256:4f0603267cd15b92b68c2c0e852a441507614b70dc7773659baa6b8c214a91fd \
    --hash=sha256:524e2a85c028a0d2f9935191fa751e5ef9d9bcc39616f70ab14b28d0369c9936 \
    --hash=sha256:54a83df9260a89c1734256e620fe1f1a6bfedd7547139d4dc1384efac11a3a85 \
    --hash=sha256:59d6588712101ccfcae9b03692be3aaae1514c2078666d7b05f15ba3a702e41b \
    --hash=sha256:63250cfab8b80a1ef82a614eb2b3cadfec2c405f870cedc139d08e2f063eb708 \
    --hash=sha256:64b656f025355cf8c51abe9fbe3848540756c6d7ca5e6791b1afa664bc24c7cb \
    --hash=sha256:65d84ec36888de4a848eee5f910e67fbc79b064685ef1e10a502e14520ead9c9 \
    --hash=sha256:69e9dc8078e128286ed3b975e37c837ba96e215a50c3ef9f3f8b7ab9e5a832a0 \
    --hash=sha256:6dd76f3e5c8b2eb8a3a3efee787bbf5b9a66e52a048fe09cab85eca33fec6790 \
    --hash=sha256:70d4ca6f4d06df7f0ccab6fe4f49c8a712c8c8b6847b4f0af9a0e1dbb0e0337e \
    --hash=sha256:72b7825b331b1b7e7c45be2e674b3e3c65af608fa376bad2d851b20aaf0cdc78 \
    --hash=sha256:741b4b367140e9b5c36b5a14c72179f2c946d991ea9a7c031a2a1ee6ad097b99 \
    --hash=sha256:74f0ee601047c0c12a783088b51be4e6214a62ecd9e02278c477433cd16e0ed9 \
    --hash=sha256:76ff5814db72e7462dece042d7593cdf102b8ec82c2b1cc201a2add34ee3050d \
    --hash=sha256:77c3ce990b23441e5ecfa5bce181fd6f408b564aeb6d7e1d1e7de9c5612501c8 \
    --hash=sha256:79bac5a251f23a7341e28fda9ce0d5319edf45328239ce037c0682936f137906 \
    --hash=sha256:7c6e7bf684dc12145bfa685d3060beaea55139134ba848289bee514ed42e7383 \
    --hash=sha256:7fc14c1585139fa6b68775e616a6b90cf622ebf219f9558c0aeaf5d253ee6c9b \
    --hash=sha256:89625fb43765cccaa1443b9adb61f283e5fe4cb1536728205d06bada730caa53 \
    --hash=sha256:8b2db2056c97224e122054fd794543cde5d24b7cae28424f6e3eb79bbe08e42b \
    --hash=sha256:8d44b20234905ff022b7d535f79d1f823ad7670c9851cc4f03cdc34787cdb3ab \
    --hash=sha256:8eed98514bffe5ecac37f493f91869c351fbb05629328bfdbc08502c6c094dc0 \
    --hash=sha256:8f1f61592e7cabd45d49ce8cc0ef42ca655c091e037153754fb3fa59725b5914 \
    --hash=sha256:b23fe17779834d3c27aaf2edac9486d04cca1a7deb8f5facda35150ac6263a91 \
    --hash=sha256:bc7b0b1da20f856bfac5f84b2673fe534b167e41980b27442ca8f78c2b7eb77e \
    --hash=sha256:c62bd361cec1f5b556eb8210264ecfff37486cd990c3386cc00310f26c54090a \
    --hash=sha256:c76c9b3324efd79029eeb0fd2ced1964bdbeca7d45e030b46fa3ef3cf74f8032 \
    --hash=sha256:c798f0b327bac10dc95cdac77b9a197ab2bd7dd1e60ebd7586a12d918d4be711 \
    --hash=sha256:c8a168b040bc61681293f79a949b5d911c8e25086f4260285b8d97ab5f1195da \
    --hash=sha256:caad9566e2ef0e5640d36032c69b0edc7ac6028277b93d93815898804fac450c \
    --hash=sha256:cbce24284f51f71d10a42b7b9c964dcb9048b28f1c8e5db40bcbcb6f428cba6a \
    --hash=sha256:cd810878180a52950e5a61f25ada5248a453bbdbafe474f89514135fbc1f633d \
    --hash=sha256:d254c98ca6387655400b3959c33c83efd807f5edeb608e3aca45800ceaa77151 \
    --hash=sha256:d795c4ac689a57f9d4ba2288126ec7901d389ad5827d2f8b8533c883974fe563 \
    --hash=sha256:df88b0c34f2fa909d322f7b06b1398e1e81af4b2f42a7b8e3556f928b25d1811 \
    --hash=sha256:eb8da9d9a9b418422c21a07fd19b9d9228692b7a7468a45eec6b11642d3c808b \
    --hash=sha256:f7c06c751c19d923435a54bff4f7e66e728fad160e8da28254f133abc9725820 \
    --hash=sha256:fa9f5ef0e2a82233dd0b8b32ea3f5710e0c44afbc07ed3620219f32601e56090 \
    --hash=sha256:fd523c4992041faa5c2b3cde62253d11a96c30d73a34afe48a486e8e2254cd1c
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
shellingham==1.5.4 \
    --hash=sha256:7ecfff8f2fd72616f7481040475a65b2bf8af90a56c89140852d1120324e8686 \
    --hash=sha256:8dbca0739d487e5bd35ab3ca4b36e11c4078f3a234bfce294b0a0291363404de
six==1.17.0 \
    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \
    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
tokenizers==0.23.2 \
    --hash=sha256:12f0835dc2ee694746a76adf7b1567d4346a4a502ebe93fb1f5f80ea49799b78 \
    --hash=sha256:2e96f5699d5249c9c64aa8412e044f727aae3a4098cf830f9901ec1afc361cde \
    --hash=sha256:325fee2e0418a9dc6c9ecf736a5f5f0db7875183ace9549ae339da76f7a1fbb7 \
    --hash=sha256:41c2f84d172449b4dadb9cdc508e3e364076613c35b16e76ecfe47a60d1e3305 \
    --hash=sha256:43e4f2071e3cc8d5d86421c874aebc82659bb51a68bcdef5a0da75ee89511ccb \
    --hash=sha256:5c56bda1511921587789163e524d196ed8284174ac23abd7685d5ea8da6c4718 \
    --hash=sha256:7b7e37ba198f24150f523e1242e83c4970de4a525480586be5dcc24d9add32c5 \
    --hash=sha256:7f0f085686b9de0d0079e6f874ae053600db64c5d13049e0bbc0119926d25aac \
    --hash=sha256:85a9a357a3764aecc904ee76bdaf8cf1ad8e5a67a1b929a487c4a39b49ed0e90 \
    --hash=sha256:950d7c9426fa72406a0ffeacdbc0bb9985f5db20eb8b263f29c79aaf83105703 \
    --hash=sha256:986670e43691469dcee610ea0f846f91a8f84e91fc6f7a48d4c064414c0ec2bf \
    --hash=sha256:a37039b5dfc4af84eb3ef0a92f4307e28936c8f9adccba2629d36f652e9bf7a2 \
    --hash=sha256:bef235815a067b2648caf6dcc7a71091b0b0fff9ee8057f6451eb9335fae52ef \
    --hash=sha256:debf978920d93ba9c219bd67cc4bbfaf912c9039e41e7a28b91ec15e3728c95a \
    --hash=sha256:e49c394456dd9985787fec76132438ba3fb8911f857b1bf3d40119f9292d41aa \
    --hash=sha256:eb2f9c8a24da020ea8c11a01a19c1c2547912d92121ae4a01cfbca46125dee40 \
    --hash=sha256:f486f402f6f9abee5bb032553736813af0c710a86b2e0ca592634c55cea1f835
torch==2.14.0+cu126 \
    --hash=sha256:00d7da8a47c184fbc36cd56dd89153b61b37fdc89cf46fc1a2824c7ac1d60a53 \
    --hash=sha256:0e9c1fd980771ecbb70c6db1e9b95a37c05d0f736760c86034619ea7a28d7796 \
    --hash=sha256:11a492e40dfd18597fdb8271879a3a09496a23140deb81a6b5a005d9cff7312b \
    --hash=sha256:1d6da1e1ae557381de0269f262627dff7c602385e2aad5357153091e463655f3 \
    --hash=sha256:24224de9ebcc6f85cfa48614cb1810ceffac5428f7daded933b0808b5bd0f303 \
    --hash=sha256:25a859aca44fb130aca596237e099fcdafd29e48fd4fd93e911599d2769bb014 \
    --hash=sha256:2b84f9a1908f7b344c3f7d3e41f4fd226d0399c2103e94d75f50b0b03ec08c54 \
    --hash=sha256:51f19afa3c8cbb53f27aa451461c1de37f507fdcf0ae2f8baf4135c07f2c3277 \
    --hash=sha256:87d84eb53ba20b369d8ef499d5995caed47cf2cc591f3a151c2e8b04652752f3 \
    --hash=sha256:898b03fc60e642f1b28f59da9f647022a8ad339c1799693b1385fdcb8889f6ba \
    --hash=sha256:8b09e2ee81cfc43cc99102aa09706bf0e52f72392e7ede587e06b174254c29f0 \
    --hash=sha256:9b958b826c2b1b64afd5be9c05fe4409971df09f1e47b937ef74c1dba327cdbc \
    --hash=sha256:a46969ff0b46955ae4adc6edc07f98ef5ac5913d11263849c0bb4ada099d4c7f \
    --hash=sha256:ac9d1ce1093606f5cb1deb4f612d2fcb273ed10ff4a33703abba09cc16c5dcc8 \
    --hash=sha256:b15e1e62ab842d099ae9b19312374f62579668171efed8b676108b60fa75e5b0 \
    --hash=sha256:b1fd46592963134867e6533775eb8ef354ef450d1b1a277f50592011ed28048d \
    --hash=sha256:b5723fa3dddd9cf548273eae0503ef9c47d0cce475edbf648ccab62224021fc7 \
    --hash=sha256:c4b923a8c489d905132e2aa841cb81af93d0137371061d2b8f319779d7943328 \
    --hash=sha256:c9a5674ca12dc716dcc29df4b3e3ce3ef49c0b743dc28150cfd14669352dbeae \
    --hash=sha256:d3ab36375a5a459a85c1b183832dcba83ce569596004ff7991a3ae5b62e4ae51 \
    --hash=sha256:e3b40ebad10df3f6e91232478a73d7856c5782a011e62180fda1b24933eb3cfc \
    --hash=sha256:e9922441cb2ed269742a46246840c53634ca963f8b0a38d30182c28b4e8388da \
    --hash=sha256:eb41a67a24e2101ab0fd8e30263d66a93e9585bffef5da3b6c23e1b00bfb8647 \
    --hash=sha256:f17fd246459d01505d34da609687757a4db139d7d5e5645ae8604706f9b5b364
torchvision==0.29.0+cu126 \
    --hash=sha256:039e8fa701073c42cd373510082f600752729d785d690eaf04a27ab6dcb9b75c \
    --hash=sha256:221c331cf3f36c4420b2ab9583462b87a2bd99055023317afe2b65ba0b0c4f9d \
    --hash=sha256:31b3c9353f18d31a1140a332893c09bfd47ad1664af4fc956068a25c5107d2f6 \
    --hash=sha256:33f1a399b92cf73c0fb9c2c38df4ee3c9f76ab40ee99aa19fb967f0b32b2bfdd \
    --hash=sha256:3779ea90c4df995c1e8826bd7fa262e06429d9e82b7894e05eca40775d16e172 \
    --hash=sha256:392e51f63cdfe7e17f8e5147e633aa4466ff9cecd865eaa12a287333d1f78047 \
    --hash=sha256:596f88228e7d4c04467cfeec0a3f216c49e86e244e9a210bacb5a3d58eaca235 \
    --hash=sha256:684d61016de90114825c20e394313ed56da648a0b790e5682f6b280bf50d229c \
    --hash=sha256:6aeb6f5692db8e0abcb6095af2987935dc3ce4bbf5e4d5e84b1dc9180289786f \
    --hash=sha256:708f5515b03c21579f74d2100c9a7d1ca807b65b5f9562760d2707b4290303ac \
    --hash=sha256:7166d49e9d4a7afa770ba4ed58f796463949f834d2477e0914d833c9b03ec655 \
    --hash=sha256:86143cdc66e755d364a182e1e3f65eb2f14034c58fba30f37a96fd3df6018587 \
    --hash=sha256:8859598ffb73f785ee90ea07deaced573dfac354842776f8dc81829024fff935 \
    --hash=sha256:996519e1c9588921d4d6486bc06e03b9a54584f02e27690f318474a9505f7e03 \
    --hash=sha256:9e3ce3e74c79a6a31f27fecb31cf254a9ab953085e0c05844f38f2d0b1443ac2 \
    --hash=sha256:a4557ec86c5fc97b4d9f9477f20e2a387644701d05a1794c566621b2d413d807 \
    --hash=sha256:e48c54862e70424822972056b1eec251451686a593527a22003ad284d0e92c89 \
    --hash=sha256:eabc4310a59b5429b328abfc84bd7b247faea873780fcecbc6038c03e5473619 \
    --hash=sha256:ec7dffc3d1defeefda5e9b45c39d5346eb6a1fc753f11920ec8ae3866d0f436c \
    --hash=sha256:ee4990d285a14ea1fa3cecdd562a72d6e93121478ef99d8a8ac108751652b394 \
    --hash=sha256:eec3a66999d22f72692491e88c545caada633ad7991e82119c81e2c38bb8c3c5 \
    --hash=sha256:f048bf897d5ca930c21bd0aa23bb8b077bc2d8f9f4f5620dd9dd952a15654ea2 \
    --hash=sha256:f334e177fdf48400939b96fa50dd5f15a21bc13c4ea8752d0815828f2c12dc85 \
    --hash=sha256:f827d0d3a5a858937ce117a3bbe3461bd2803ca150c3ab0d825dc7da2b50bbdc
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
transformers==5.17.0 \
    --hash=sha256:78ec1ce21579b38dfb83950a0658cd119f87212a2fcfdff478096ce9d6c03801 \
    --hash=sha256:a153be279169b55b92d8000bf4af294aed684503d091cca7804da2dd8a9de000
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
typer==0.27.2 \
    --hash=sha256:269b7eb9d3c202ca84b4bc9618cb04ebb43d3d4d1e567e4c768607232c05f945 \
    --hash=sha256:b3a5fc4342d5fc8fda8fc3010b1cf117e9249aab7fae800c2eff62fd3842d97d
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
zipp==4.1.0 \
    --hash=sha256:25ad4e16390cd314347dd8f1de67a2ac538ae658ed4ab9db16029c07c188e97f \
    --hash=sha256:4cb57381f544315db7688e976e922a2b18cdb513d21cc194eb42232ba2a3e602
""",
    'phi4': r"""
# Hash lock for the capstone's 'phi4' environment (2026-10-03 uv isolated environment). Same 51 pins as the
# earlier plain lock, now with the SHA-256 of every published distribution. Regenerate with:
#   uv pip compile phi4.in -c <previous phi4.lock> --generate-hashes --no-header --no-annotate --only-binary :all:
#     --python-platform x86_64-manylinux_2_28 --python-version 3.12 --index-url https://pypi.org/simple
# Installed with --require-hashes --only-binary :all: --no-deps (wheels only; Linux x86_64).
accelerate==1.3.0 \
    --hash=sha256:518631c0adb80bd3d42fb29e7e2dc2256bcd7c786b0ba9119bbaa08611b36d9c \
    --hash=sha256:5788d9e6a7a9f80fed665cf09681c4dddd9dc056bea656db4140ffc285ce423e
backoff==2.2.1 \
    --hash=sha256:03f829f5bb1923180821643f8753b0502c3b682293992485b0eef2807afa5cba \
    --hash=sha256:63579f9a0628e06278f7e47b7d7d5b6ce20dc65c5e96a6f3ca99a6adca0396e8
bitsandbytes==0.45.5 \
    --hash=sha256:a5453f30cc6aab6ccaac364e6bf51a7808d3da5f71763dffeb6d9694c59136e4 \
    --hash=sha256:ed1c61b91d989d6a33fd05737d6edbf5086d8ebc89235ee632c7a19144085da2
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
cffi==2.1.1 \
    --hash=sha256:046bfc24911b37851ee1b51aab8bffe713d89c68c6a057b09484ce9fd5f69b4e \
    --hash=sha256:06c72bb76605a4b0cd0aad6930b69d4baf7dd5d806cfc409b824191099700e66 \
    --hash=sha256:0beceaabe56af686895136a2de78db54ecd8e4046b236b8fd6d6cb61389e9bf2 \
    --hash=sha256:154852545011f779917b11c78db2358d095da62a9a172b78ad0a583ee5adc0d0 \
    --hash=sha256:194cffa889098ced9976c3fc6340305e43f6303657d298da55366907c05c22d6 \
    --hash=sha256:19ee6127ee34de7d83ce3d371ebc5ed91addbdcc39f9ab15ce4eb35a4e534971 \
    --hash=sha256:1a18a57b58cfb21fc28d72e876acf10eaed67a1ed96226f92af4df681d571c4c \
    --hash=sha256:1aa5645c30469b09530c4ebca77ebf8f17618293c58f8549cb1a543a50236e7d \
    --hash=sha256:1dea0e4d7d4f11f619fe8c1d76caf49e24405b4b5743c0e3be16a500ecd930c9 \
    --hash=sha256:208f941bb9d18e768138677f0a6d2ce01f590df56043dda1df1535ac57c88517 \
    --hash=sha256:210019b6c7cf07f081b4c54635c8cf744377001350e29cc0f81c4377b4797735 \
    --hash=sha256:246fa40ce8645a614ff682e0b70f37134e460eaf93a775e0cbe3cca585a67a80 \
    --hash=sha256:25792eac27877609e7bb06d42ff88278a6624fff2ba9bbb523c09616b117e80f \
    --hash=sha256:27350daa11d4f10c540e6e89dada4c54feb7256ad03e9a4dc075ebad7ba360d1 \
    --hash=sha256:28907ab9bfb6aa13184cfc17c6b8e1023c5ab6fd7076d8c20a35e59fe04f8f29 \
    --hash=sha256:2ae64be792b8966f2c69538199728b290e34726562896df1e5dc8ffd8d8188e8 \
    --hash=sha256:31348097ff5bbe827ccc41795d4dd099d9f0625e7def00ee653c137a490c2a6c \
    --hash=sha256:3143d81e29e1e20a9ce10901ec369012947876596f75a222235965f2b7ae832e \
    --hash=sha256:3222ba5d678f80a030e6afbcc33dc1ae5cb45facabb61cee2c7016b8432fde48 \
    --hash=sha256:3311ed60d36f83378794e1009ac6258bafbf81f7888b4caa7b35a521e3f95813 \
    --hash=sha256:334644fbac4eff73d985a17a91226df55d0f394160c4cfb880e084c8f7161cac \
    --hash=sha256:34e261f78cb6ceaaa36f42f2613f4380d94d9c759a9c73c769ee6e0247364632 \
    --hash=sha256:363e05fa78e15116c3c32c210ee36884fd6b9afa6d440e47112c3bd511d64cb6 \
    --hash=sha256:398aff33cee2767e3e781d2554c54bd0dff386bb437581e0d8011fde1a942ec1 \
    --hash=sha256:3d22a20b1fb1632cc72c22f95f7b0d2961c3e1c235f245ba4c606c4771035659 \
    --hash=sha256:42a494cee34437f05546455144f2b5d9ac09b1face62bcfce597d2e521066688 \
    --hash=sha256:42e2f76b9455f5a9a844f770bf3e200ed3da0e15f5df3db9c31fe80b04b3d004 \
    --hash=sha256:42f6930c31dc7f50732c9ae793c2786c7b6b044195967bbdde40bb9be81c4cc0 \
    --hash=sha256:456a61fa52d579ebf9df2e9552ead5129855dbaff6c1e5a9b1bc408809bdc062 \
    --hash=sha256:471cee653ae88de62096552e6d24ccb4a5adb8c8c9f10b5054d0122c15bf2779 \
    --hash=sha256:49cbc70e6542d4ccccb936558d1064a8012541e78f821f955cff24e357776c94 \
    --hash=sha256:4a7c934f7360e8cd64fe9efadcbd10c7c6364f531e432b9a4bf5ccbc9e0e8b50 \
    --hash=sha256:4be96343e422f2dfcd12ab5c9f5aebe03f82f737c6bffeca6830b3875cb44aab \
    --hash=sha256:4f42141fc14250de6dde5ee7ea4432be017252d91f19c5ad043c084cea629cac \
    --hash=sha256:507a24c282e0f42f8ed737cf048572cbf580468da5555764a8331735e9c736b6 \
    --hash=sha256:51b31d1c98274844cfd7838ce00bfc27c7423a4dc00fc0772fc3331c2cc90676 \
    --hash=sha256:58acb8ab8e295e6c5ea12f888cbb13cf21511ef2a3303a23f4325c29d17fe5c1 \
    --hash=sha256:5a59cc1c4442bc3d5c703bf720b51138d0bfc173618807c9ee2490a7541dd3d9 \
    --hash=sha256:5bb4e7ea95dcd6a014a6fef62e62467d67d8e582326443f3d68e71d6320a9fcf \
    --hash=sha256:5c58fe613dc5e5336357eff555824a314d8e43282600435c8d1cb6a7a2fedd13 \
    --hash=sha256:5e7cecbaadb83884793e05828cee59b210b24583b9c7425d0ba6a754fe22eb4e \
    --hash=sha256:616f097f2fe415bc92a247f02e11f634e1f9e9a83d327e3c915c15089c87869e \
    --hash=sha256:63bbfd5ded17c4840ac07cd8f1c21ba9d9708141f840b324f422f41b207e3973 \
    --hash=sha256:64faea20f4e2613363a1a9b9c7dd73058f3ecd00133a511e72ad7c511658f527 \
    --hash=sha256:661c298b4821edebead0c91edd2b00374d67ad7c5a1f7a91d4442633b79d6a72 \
    --hash=sha256:68e62fe11f30d5ca8289242866f0a5291402d8529ca2178ab8afc5c9694ae890 \
    --hash=sha256:6a8dddef476fab96d066d578fc88526767b836ab5ab21754e1d5bf3879c31c7c \
    --hash=sha256:6e192623c49c94421616a5778fba35cf0d5a8d000650c1967ef4448ee5cdd990 \
    --hash=sha256:7225e4514edb64eb6740324353e0da0711954fd8d7da4576755b1c6e09b697cd \
    --hash=sha256:75f80557d1389eddbd0de2681f6a390a0c5338c31ddaa821381c203fc3fd50d9 \
    --hash=sha256:770de9db11e84213beec501cfcaa013b019820ca881e03344dea5844f7876d94 \
    --hash=sha256:7750c6449dff7864bb9bb27ddfb0267756189201a3afc911d82b3caacd70dfc3 \
    --hash=sha256:7bde5e4cc5c10140859842b9d383af292b22639a4dffb725314baf45968cef80 \
    --hash=sha256:7ce713ace7c0e4520535b42b77eaa742c16dab813978064913e5a3cf82973b41 \
    --hash=sha256:7da0c5eff80f0197f3b3d1232ec5a682a9325f4ae9016a78f5f5ca35f9ced1f5 \
    --hash=sha256:7dbb61fe3a7699468030f71bbe5f8a0e326a151daa91beb11a6fc1f980c55e1c \
    --hash=sha256:811bd1e21d32de12efca32393a0ab3f5133b54fce9bd44b8bd77ab07da14bf6a \
    --hash=sha256:8ef53b2de9bcb9197d31854256575d59dbac0cba72ac627bb291ef5eceb74be4 \
    --hash=sha256:937c0052c05a31ca1daf18de3158eed4dbfcb9cc107adbea227728d647be701e \
    --hash=sha256:9d2055050ea716bd38b7f7f1579c275386646b4894c155a3e2f3cd62ed41b7c6 \
    --hash=sha256:9f8d177621de5cb38ee3e731eda45d421db093ec0739f46a5594babda7987a98 \
    --hash=sha256:a2d7755bef5a12ed488f4ef1f1b69ee9191d7396083b755a5d2295f6edb4768b \
    --hash=sha256:a48d62ab9d6f4f98c983223a547af44be6ca3691074c31cecced6facd3ba2dc1 \
    --hash=sha256:a4f00aa42f75d6e4595e8866e748cc1705adc0cddfeb2ca86d0d03993d63ba03 \
    --hash=sha256:a6e721d4b0e45d5b65e87534470e67b18dcd092c83f68fba09f152b9cbc061af \
    --hash=sha256:a730a083190634c65cca36ba5f489531576ebd79bcd5c8e172130f6453127231 \
    --hash=sha256:a931079504ecc49efed7744c476a5c343a92fabf66dec2db95edb1b2fdc770e2 \
    --hash=sha256:aa9511c62d14da7aacc9b4bf51f3f697a621e83b2d6919008243c3aad168eea3 \
    --hash=sha256:ab36d55f9ed2d067327667c2fea18dda018eb628dd6347aa01dda6cf1f5d3836 \
    --hash=sha256:ad2c86c495b899d862ea0f4b42891b8713a3bd45dd4105c7fd51c2a72f39f3a5 \
    --hash=sha256:aeae0e330c9f6acd681f647d46cefd30c29f93e3392882e792e82080c9691399 \
    --hash=sha256:b0431303acaea1089ad4b3e9ce4e6518193def1118d4073ca848635ee4ea2e96 \
    --hash=sha256:b5bdfd1c873d4e093aabc0ca84c4ca6dbc4f752afb5c86f146d9742580c9da2e \
    --hash=sha256:baed1e86cc735622097354b9d1281406caf42ff42a886d29faa8e8d1630333be \
    --hash=sha256:c1453022f490d2459a11819d83ad1d586e9ff65a12ac3e705ffebd46d3685dcf \
    --hash=sha256:c26608d2222fb1e94487e4a387d85f13eb55d5ed725cb25a0c589ac4ee60e7bc \
    --hash=sha256:c7659f22557c5a0bc4855cd635f55edec690cc008a40768527762cb9fb263455 \
    --hash=sha256:c8c69575568085ba0b1b10c0249d779a214aea6f6522e949a0fc9fb0fcb449d0 \
    --hash=sha256:c8d2c9fd1f2d16f780d15127abb050d13d1a76c03a4bd87d7e4980e45e511e12 \
    --hash=sha256:ca82be1a1d406ecfe1d25dc16cb33488e5a16bf4438c9fb590484ea29d92478b \
    --hash=sha256:cc572dace3f60ef98d7b12ff411d20f5362feb31a0439eab0085bbfd349982d7 \
    --hash=sha256:d18e5ac0f2f03f4f518d3e23db0f0cad7faa1da8620e9c09461d443bbf6e6692 \
    --hash=sha256:d28630f5854ab07ab1fd4aba756de52326c82e6be15d414b12793f1975048b54 \
    --hash=sha256:d9c275eaacd24aa73f94ffd6de08fc3f932424d8b6c376f4bed7cde376fe7bc3 \
    --hash=sha256:da0e573f9f97159390c89d9f1a9e41908b66d408cc5b58d08cf3847d844c531b \
    --hash=sha256:dd31f52ea1086513bb9df30f8fcee9b8918323ae067a3d5b78bc826a000712be \
    --hash=sha256:dddad92b554513a31f272570678ba307fb9f618f05e3d4a5eacafff9eae03e1d \
    --hash=sha256:df423d40ee8654634421812bc3b196da3f9bd7d32929da813f8394c4348a5358 \
    --hash=sha256:df913725b79db7bcf03448f36b7bf8815363417d5b58deecf9305e3e30f0f21a \
    --hash=sha256:e0bcb7e0f677f543555d2adff3bf19c05f66cdb4796e5ff602442ab2fe3c4ef7 \
    --hash=sha256:e2d65b31f36619cda3999b78b2aa9632e76b78448e7a56fc4240824200e7c4fc \
    --hash=sha256:e6e8cff14d6fb0be70a09c0bdc58096f501952d04624ebf867e0e56da2df8960 \
    --hash=sha256:f16c709686a78c727bbbf059f92b0bf41c6fc60deec706d2dc19f529175a6125 \
    --hash=sha256:f24fb43132a4c6b4cb4eb029492919b2db645be6808d738f244fd146c03c32cb \
    --hash=sha256:f53e442b08449d42821fa4a4fba000095af9f62742a500f978a9f557ec44339a \
    --hash=sha256:f5cfbc5fe74540d335175b656c725d74d90e3730c626d92575eea35029d9afaa \
    --hash=sha256:f81b3b8f3d4e343550fa4baa0e479bba9f2d29ce9c2e9b51d1ce1718d7442fcf \
    --hash=sha256:f8ec5e643a9a937f64e1999eb9f75d072263751912dc5cd06d3c85f8f44be7c3 \
    --hash=sha256:fb92203a88b3d3053034db775110081c49d28be6551923805e039924093761e4 \
    --hash=sha256:fcd22650c908d7b7da162bbfaab594a1227a15d1643a98c68b122ac642fa2264
charset-normalizer==3.5.1 \
    --hash=sha256:00668ebb0609751758682eb0b5857e7c35b9f00e84dfdef062e103244ec94d45 \
    --hash=sha256:012a22b88a77ca2e59b98ac5889b0deb604147666032f45e6d6e217634d2550d \
    --hash=sha256:01e93745f7f219b703b60ba7afead36cfc4242782be5af484673fc500df12da5 \
    --hash=sha256:04368edf83514385ffc3e1cfd4546e595f4f1272dd23ba437a93a9cc3741d47b \
    --hash=sha256:0722590aabf9dc6a6c0343d523c05458fa2b5047dbe6302fd526bb570600753f \
    --hash=sha256:07ffd07412fc5d5e84cd8952acf9ff7e4ed7a708e69d1bada19d8ba91711353f \
    --hash=sha256:09a7bba9f739468c8e78c36a75c33768e53cb1959fc638f510454c14683f00d5 \
    --hash=sha256:0b2b1b3fa5670c127b246df1d0c059defd41f689a868a3b9d79df9b1cac42d22 \
    --hash=sha256:0c6dfb5ca6723eeed15aa8e564a014d69fcb8812f94eef11fe3631e0508199f5 \
    --hash=sha256:0d929fc574b4d6fd9e7c0f5c2ede8716a41911923aa7fa5fce38e0818aa4a1ac \
    --hash=sha256:13e3afe97712e8887cd516e960c63f0b93122971e5b5e4b2622fe7701771e838 \
    --hash=sha256:15f024313246a4ed976c60f440bb8d257815513a681d212ff74fd46f7d715a90 \
    --hash=sha256:195ce897c6153c0700078142cf8efe3e6454ca4cf4357499e4078dfd83396626 \
    --hash=sha256:19a3dd5aa73cef1c99687c4fc57db016a9c17104ae1185da88ba566a5d3bebe4 \
    --hash=sha256:1d1c7a53a6c2103925cdd6d7229f8c567379f211c869793df679f2e9f738c369 \
    --hash=sha256:1f5883d77fd409a261abb5dc8ccbe335720d798b1de4abb3b1d47ccbbc76b53b \
    --hash=sha256:21b82d8082f6f5e7f456ef0bd16323d08de1266efbfeb476e64b2a91d1471a4e \
    --hash=sha256:252d099029bcbea642f2a06c4ed5046bdf8b5a8150b64afa5e027e88b106e5ee \
    --hash=sha256:256dd4d85d9e4dc595e2bc983c980e73f62ddeb3165c58b4c3dfe78c5c8548c1 \
    --hash=sha256:26422d45fd13551cf564c58932f7d72b4f58b93b0fcf18c35ba6be12b46bb102 \
    --hash=sha256:2679de311c7946dde5d3b6f44941844133ff5c7cb86099c0061ab1e8901c20a8 \
    --hash=sha256:29880d17a8eb0b5cfdfd8944b468322928059aa35f1f5fa8ff22b149ec0b42f8 \
    --hash=sha256:2bced4061f000f7187254a02ad3433ae17eaf991747ceea2f478422590a5bba9 \
    --hash=sha256:2e9cf9253119d8e5d111f05d71626786fd3d6193817316eab1ca088cdb8593cf \
    --hash=sha256:2f06b7eae9dbe77fe1d644ca244dad508de8d302870a43f3c559b521270938a0 \
    --hash=sha256:2f293479cce755c75f1697e87c409b7ae4c555c7dfecb6e988ad13abba943031 \
    --hash=sha256:329fc3ccb63ad22d867d84c2adea759a64079a37ba4a343433b02c7a2816871e \
    --hash=sha256:343fb4f2821043bd87095f7b08a1a181febc8e36ac64212143bbfd0a0e1bc235 \
    --hash=sha256:3588e376b3ea2eea84976f67273d679f229e24c66dce7b82ae45aef04ff6e072 \
    --hash=sha256:35aea775dc2bd5f54cd84a1cd2696cc3207c479cb9cf0bd346f0d343e4300ddb \
    --hash=sha256:35fe081843b35aad20ffeccec3eeffbe637b15d14f3fb22cc1b59cd8ec17e93c \
    --hash=sha256:36047af20e17097c3bb9476c2b7655f2f7aa51322c0ba58c07695bedf755a950 \
    --hash=sha256:3617ac3cfd8b9888f145ad89dd6e692285834b0201c6074a5eeaad3fd4d668c2 \
    --hash=sha256:366ec70f5547c640d3ce1985722490f23faf4eb5216a7eeba78277490e78dacb \
    --hash=sha256:394fea06235c8543390050ed5f529187074b029fb027213f6c46ac11ab5d950e \
    --hash=sha256:3d27167433c0d5f18dc850f07d0b3816221984fecdc405d6c157a6f0b8f8e9e6 \
    --hash=sha256:3e5e1224c0a6a90e05843e07adfec669edebec17801c67072f51e59561d63c0b \
    --hash=sha256:41876ee62a3dddf48ff1121ad8f0798032aa03f2fd35f21f34a4cab14f18d8d2 \
    --hash=sha256:433c5a81eade63b47e522303bad236f59dba55ea6951746f5558355eeed8c75d \
    --hash=sha256:4582c27e8c889d64811987b5967fbd3ae0c823fe1fd933b543d55ac20bb475fa \
    --hash=sha256:485a0d363cafefcd2538a73c7c838daa2035f09b2c9f9b5e3133f80c6aeb84c2 \
    --hash=sha256:494b70049a4d69aec6e8137c13af4cf8db8c9f9820a1392ac293b0dd2987a818 \
    --hash=sha256:496846868fea80e479324862fa877f02411f2fd0f83b79ccee2607aa68b2a032 \
    --hash=sha256:4abdc5f9ad448c1ecbfae2974b820535d6bc6e7eef63babbab3d81cf46968c71 \
    --hash=sha256:4b599739b93b2cbeded49645ae3c8d1405c29ddfbceac1545c87a3f9580a9e96 \
    --hash=sha256:4bea7f8ebe90bbd7f0e4a2de42ca6924ba23e3e76418c408ff82f1d46fabd687 \
    --hash=sha256:4c4fb141a727957c93edfe5c32a26ceb6b5f6461d67146e2d39f51e16170bea8 \
    --hash=sha256:4c9548dc78002099910abaebc0a72ac58b7d30931869e0351c09b507dff4ece3 \
    --hash=sha256:4d26f14f041e83dd8edfd61f4cd4fa7285d31798b5bf1f28e70c367ba6c41d61 \
    --hash=sha256:4f298bdadb8f0b9e5672877f647d1be9373ef5320c9e2f049795e26cad28b6a9 \
    --hash=sha256:52ec005752a56ae79547a05c0139ca2501a0c866390b6115008456b9f0e7cde1 \
    --hash=sha256:55261ac0d2941c42f196dd576f543d87a8ee03cd6f5e30dfb4d807b2e3b9121a \
    --hash=sha256:56490c595a28b1bb27dfc583e816152a9767721ef58b2c03b13f954d2f707420 \
    --hash=sha256:58d3e12c88e0950bca850ae1f7c256055c097639c2edb9eb123af9807d8b15e4 \
    --hash=sha256:58d4aa13a59c969dbfdf9e6a9560e242cbfd9e8a8f50c2747714df1a423adf65 \
    --hash=sha256:59171c6e45bf07d0d5cab3b0bf81d945035530f6873398b3b531c31184d46663 \
    --hash=sha256:5b6d1386bf0096d26d3a863dc0a487a5b4eb9aa93cf5ba69683d29dde6b9d60f \
    --hash=sha256:5c0ea61a470e070686aa30892fed79e297d2c8d0ab46b8bcdf027d38c51da591 \
    --hash=sha256:5c84bec0ab5ae0c64bfe73a7d2adcb5ce73b467523fc27fd6a28ab2aa6cbe35a \
    --hash=sha256:5ca0555312ae2fe82715cada7fac375530c2f3349e1eaa1bcb33d0283ac79a18 \
    --hash=sha256:5d8531a6569d025f68e2321e7638fb7978f23db58e5f69f56913837aae03816e \
    --hash=sha256:5e2d0e146dcb57034f8b97dc58d2d512cb90aba253960ce449f695fec6a82c6f \
    --hash=sha256:5fc45d653ea8c9a20479167e11d4a0f8cb2fa3470737ab6f9c827532313187b7 \
    --hash=sha256:6117b84ea48435e5356dc737f5121485c30920ba43375fa7b434fd753df0eac3 \
    --hash=sha256:6199d5606e2bbf2b096cf64d03f8b6790c91081d5ac866b8e7bb6422738cc60c \
    --hash=sha256:62b55f6722735a6c472f88361cde6640608773d9443cebdbb51abf436a1fcdd3 \
    --hash=sha256:687c9ca3035544b113bea2055e180af96fb63c0c476e22a9180f51925186e7b7 \
    --hash=sha256:6b7430cf5728e68f6c462254009a6ef4086e1bea43cf2f57aa9c55fb4f50ff96 \
    --hash=sha256:6ba32c4d2abf1d2fe7cf27d280f4cca5664233b0f885549c7761719eb977f486 \
    --hash=sha256:6c9cdde8becb25a7fde49924511aa2644d6f8081cc8df8e9452724303348d8e3 \
    --hash=sha256:6df0ec430f9a831772c23ca5a224cba36517a58a84bb32c32bb59a9fa67c47f6 \
    --hash=sha256:6e2912d4babbc65196ac13c2f53468dc57fb8b9c25ef913e8c59ddf7c6dc0e1b \
    --hash=sha256:6e5e4d73d588ca5ed09df1b7dcd1b203d1df3c542e3f50d126c947d432b10731 \
    --hash=sha256:70055ff39b97c99e7ae40ea3e393fb62aa2e44dbd9b29f8d14f42fb0025c3959 \
    --hash=sha256:706bfd38730a5ac7a365793269a00f4e988178cec121391f4248d84ad8c972e9 \
    --hash=sha256:7235dc28fc6dd9d832ac7c7bce95367dedb85929f17368a0c2bee1e080b9acbf \
    --hash=sha256:774d157f112367ff4abd29019f38f023c24e00e56edc7829c20e358a5a913ad8 \
    --hash=sha256:77efcff2b23071c349402ac1066667a3d011f62398d81408c9b88ad991747c9e \
    --hash=sha256:789b8982559ae28dad2356519f841655756cdcd96616410590ae0b17454ee64f \
    --hash=sha256:7ac76cf9afd34929d76eb7fcb63be476a4853d8a96f0dcf2d0db68a0cbdf9885 \
    --hash=sha256:7c0c10730342b0c9b35dd1d619beb8214e520bd96a1f870f452680b238aab3e0 \
    --hash=sha256:823f82903d189af463d7df250ef1f7f696f3cee08cc8d91deb565e8d425f6506 \
    --hash=sha256:838648accb3a7fd9803fd45c87bce8509648eb0c11bc34e216141300977244f2 \
    --hash=sha256:854066be00447fa8de2ccbbe893e2ffc4b123ef16d897af794c1e18bd4a714b0 \
    --hash=sha256:85d5855daafc240cc045c026d7a15fd198a09b0fc8ff6f5ecbb5297b509cb11e \
    --hash=sha256:85de3134b5379856e323ba37c19c9256d39425f7b76a63af52b09fb4664c2e8f \
    --hash=sha256:87e4f41d375c0b9be2fb5251aee4b8a689169e134535aed81bf085c3b647451e \
    --hash=sha256:88ca277405c2d3b71c4e1c2ee0e7966e807bcba86a69d11e19ba199d18ae4491 \
    --hash=sha256:88e85ab89cb822c1e635f51d6d32e488f94e002e70e2f492bdb8b945543f345a \
    --hash=sha256:8ac8c94b6539074e0f40899301273ac8402b9b3e01c7b7ba269ff30340aaaf20 \
    --hash=sha256:8fe532b3c966d1fb794e0698e4589d0444017ae77fc0b31edea13c0e35bcc449 \
    --hash=sha256:9085f87b0e38a2b92b8923059b4e8789fe40d9279712d15dcc670048d77079af \
    --hash=sha256:90b7481fb62fbe172c558bc6fd1c4c98d82004a54a7551f20e11ac9bf0b8708c \
    --hash=sha256:92caef967d287a407085d61176fce4012b1dd62daed4eb6d5ceb26d3d2538712 \
    --hash=sha256:9362dd90aa7dab48c0054a21187791ccf05473f7dba5d92b8033ae62164675e7 \
    --hash=sha256:94d78ecec2605a8d0398b0f365d5f12a63248438516f5dac536a5eff7337df4a \
    --hash=sha256:94fbf1c0c6cc0d3d5e50f9a9313a8cdca90dd696d34b381cd1704f8c9e939f20 \
    --hash=sha256:950f23cb393f85543777b0433f082cddd25b51ab398eac7971146495679efe5f \
    --hash=sha256:96eefc178f8636b9c760c5829345307fd81cfae9ab1e80997dbddeb0f54ee9a3 \
    --hash=sha256:96fef3e886d6a9874b14f27fc193fbdc69d5d8035783d86aa4e1cea594e695f9 \
    --hash=sha256:977cdbd483a9cff38179bea4fd754289a6f2195c7abd414aba85410b3e66cc5e \
    --hash=sha256:978eab16f55b4ab2c2a745be9a0a840bf8f09a7f227d9c76eb30214d078865a5 \
    --hash=sha256:994e883d17c559cdfd38c84003c8b27d25424a1077272a17e7cd27bfe0bf57b2 \
    --hash=sha256:9ac4444d8d4fd4c4bd08bf451ed3167aa9e7ec6cdb41b648794f1d1103652e36 \
    --hash=sha256:9b5db6052055d34d41230fb78d7c439c23dc536a9896f6cb039e8dd92cfc1263 \
    --hash=sha256:9d9a0dc7cbe9bec24c3f767c9122c41fe5a1bc43f47cd099d00d393e09769de4 \
    --hash=sha256:9dbdd9205662134957cf0c324f639bdc5031c0ca056e2369e238db75187c0f11 \
    --hash=sha256:9eea3ab2597a5e65fe65296e2d6a84570845a6b55532d90333d740d48bbc850a \
    --hash=sha256:a2028475ba855475b8b4d3cfeb4994269c967aea8b9892dfba907f4263a863a3 \
    --hash=sha256:a3a370082ce34d0612f421e15fe011c53bb1feff21a26d06ad4fb244dab5a375 \
    --hash=sha256:a545775cfe815855ea32d7c27731d79da358ef2055b4a25830231b1622dd18aa \
    --hash=sha256:a5cbd90ecf0fc62e64726917ad083b73001f0563657a87ec3c0b504e277dc90d \
    --hash=sha256:a6d095662e73e74f0a49988e0593373e243e3a52e27bfeea0a859e88acf4a0f5 \
    --hash=sha256:a6dac12ff6b846103483683f60c5f8fee205121adc58ffd87e90a90a3af69e99 \
    --hash=sha256:a951ad59cad9145664a730d3036b40b844e74d2d3683da40111463cd3a83845d \
    --hash=sha256:aa1099b956fb795e686d073568f6dc002a0bb89765ea6d5b055dd7d9bf1b116c \
    --hash=sha256:aa2bb0b37202dca27175591f761108b5d34096ade1191ffe4808bdf6b1571488 \
    --hash=sha256:aae2ee51122d3ae968a3837d97dc24a0aeebb0dea23694422cd172bd30017cd6 \
    --hash=sha256:ab743e9bc90c1f73552ec33e10e3331315acd2c397b36065b591b0181de533cc \
    --hash=sha256:ac00177c4831ffa650f8609e4bdddd5fe09c03b1c0c47acece7e6ea20421598b \
    --hash=sha256:ac13b004224fb341e1e25a1ed5e19d32f57cdb2a403e01f003b46f051a550f6f \
    --hash=sha256:acaf604462bf330b0d07e7a07c1d6e4adac79e5fb13e9c5140590542cafacc00 \
    --hash=sha256:ae31a1a1db2ee6cc2942fccaf695c934bc7f3db9f2133a3fef1f367cf1a4ab10 \
    --hash=sha256:ae4a097991662cd4fff0ddc74e0fe7874f82e00042fa0ea00855645ed0c79598 \
    --hash=sha256:aea996a6aba25260827c9ea511d1addfde2da9eb686ac961838509086188b7e6 \
    --hash=sha256:b39b69b347e5e47a3b5b8cfc005c68c1ba347474e3960236c4944a8ecd174962 \
    --hash=sha256:b54e7e13267d49ffbfe68e25b3cbd774dab38fa37238f71265e91b36146eb21c \
    --hash=sha256:b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08 \
    --hash=sha256:ba2f37ee79e6338845261a3c5b1784e5d1acdff2c0785b284f1b633033d136ab \
    --hash=sha256:ba501e667c17d8411f98e67a022d9604ef179aff0e459b7e292c796837c13573 \
    --hash=sha256:baf3775a2635e5a11fbd5e4e64ee69c7e86875d224a5c72aca4c141064589a90 \
    --hash=sha256:bb57753e36e4855b8ca375069482250a6246372331a3e4f3407eaebb007443f5 \
    --hash=sha256:bd6c173f04743d483881bffa1478d5a4624475b8cd1d2194956a75548e191c18 \
    --hash=sha256:be47f99644b208bff7766314013f9acf57b056b04191d570d68ad14022cf5b1d \
    --hash=sha256:c010f5581d9c612804cc59fcf7b524b707fbcb72828551237ab545bb5c7034af \
    --hash=sha256:c1dcc36dcb96abc02236e182d17e0f71430152a6c2c7447421da2d2dc144edea \
    --hash=sha256:c428c6c31eb5f4277d7f8eccaf767fbd548ddd5ce3c8b4f4cbbfab3d96b5904c \
    --hash=sha256:c658c50ac0c98cd755a2dd50b7977d3bca7df401dcc47fbdfa87db53ef7d4e8b \
    --hash=sha256:c71fb0d56c920c269cd3e2e3fe7c610e3f1fdb21a6ce60efa6430ff63676cea6 \
    --hash=sha256:c7b742bf31c88566b4bb6335a7f393bb322e580b6bb98df7bd0c25e6e3519ce8 \
    --hash=sha256:cc0329df4caaceb950d2f580b5ac716a377f7059624a0bafaeaf8a218c6ed774 \
    --hash=sha256:cc5d36d96478aa9c60654bd932525bf32964c62a7281eafdf16d85003a8d6004 \
    --hash=sha256:ce854f5f478050ade5a238731c4ca985a7d3b3cb53ff600a9b5c3b689b5f0a7a \
    --hash=sha256:ced3fdd71aaa83ce593746c2edb42b7a59cb4c19c8b5c407781c72e493aae55a \
    --hash=sha256:cee5dd7c6fb5dd52a0fe2a740f9bc6e3593f5f8b1788bde49de02086f30182b2 \
    --hash=sha256:cfa1c0cc3a8f9f53f1243a5a99ac36fd003880199383b37672e86ddda9cb07e2 \
    --hash=sha256:d1ee1e296209fdce05b81b663250eefa02213a2da7b41bf26f7829b8ba3545aa \
    --hash=sha256:d59b75732e9b6f27388e10c14b0259cc5f2e48c78627d185e6a177b58ad3cffe \
    --hash=sha256:d63600d620ad0064c3a748b950ac5ea38a80190e5498532efefa4b7b3f1da1f3 \
    --hash=sha256:dd732602a7009217f658d5863d12d79d373a4de0eebc111094bcdd3bb8e0a6cc \
    --hash=sha256:e06efa066f7dbadbc84ebc126a97c452a6451dfcf589d89d788484949e1cf795 \
    --hash=sha256:e199fb99720074809a7720f1c0b4d919eea8b87e88713e0f8f602f7bef543d9d \
    --hash=sha256:e4b018dc5a0eee4676e38fe84a47a427816c590b93b55d9025274ec4d6ffc2dc \
    --hash=sha256:e6621fb2a4988d6e53eedc455e5903e2679f3967b8acb3d639f1b63c14a2e893 \
    --hash=sha256:e71c909f353863b2b89c83de2ebed71ea6d0df8a6ef65a128193c5e650766bef \
    --hash=sha256:e90251c0c7bdd54a100a0dce3c07b7e637278c93af29dbf78ebb89a58c4bac7d \
    --hash=sha256:e9fbdce1e47394b09bc9f26ab117dfc8d6491977a11d86f592bb42c779db2fda \
    --hash=sha256:eb12fb2ba69ffa05f8695f61c69e591dc4b4a12ac3757ac8af8adb259bf56d17 \
    --hash=sha256:eda059b6bc8bc0812d626fd91a7ce01bf583df0a61296eff390fd94141a34e30 \
    --hash=sha256:f03ac127268b43ef4fe9e6ab6794a6794b49485a0cc0c1db79876d2f33f75bc7 \
    --hash=sha256:f298e218441525d3794428b4c8b8fb8662c6d3ea79925d4807ee6b9a96a3bca5 \
    --hash=sha256:f5542f9b941279d82d41eb0aa9f98eba36fe4df5c7086c651df7944935b37182 \
    --hash=sha256:f6f7deae3feb4edfa2efaf7c574fe88cbf055038a6abdb40188e4fff66d5699f \
    --hash=sha256:f9b1e28d0e8dbfa858abdba91d6b547beaf2df1a59bec6da6faae7b96a4991a9 \
    --hash=sha256:f9f8405c2c758532c74fed975dbee57be1f31a6e865c031870c79a6ed3212ada \
    --hash=sha256:fa48b1b63d639f9483e0633e092f5851e2348c352f1f9bb6c8182f87884ef876 \
    --hash=sha256:fb78f6e7fcd8ad785d28cd577168bc1aaee827b25bb8755638f694794ea98f0a \
    --hash=sha256:fbc597639158fd7c14d55e808718848319540f51b0e6746e3eefa59723a4a348 \
    --hash=sha256:fce8cbd4997efeb450bd298b54f755dcdff18d496f7a5ddbb4867c6d7c88fdc3 \
    --hash=sha256:fd0350afdc3aabd5576f60ea109228bd5538139713c7b094c5cd27c73a98bc6f \
    --hash=sha256:fd0a274c0e5f9a21565cd9d3dd749b61f96b7aa1e20a93aa1ba4029518f2e5c0 \
    --hash=sha256:fdb8a068947befafba9952162645dc2fecaeb400e64584829ed5e9b2fbe21a7f
filelock==4.0.5 \
    --hash=sha256:2b155f098c4f285fb41954a22c616c4e8a0635b78c184338ba3023c1c91a4b4d \
    --hash=sha256:be9166068623636cedcad7b2ef735b04e5fb4344d978e09fd6fbd6ede06f14f0
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
markupsafe==3.0.3 \
    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \
    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \
    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \
    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \
    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \
    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \
    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \
    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \
    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \
    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \
    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \
    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \
    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \
    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \
    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \
    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \
    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \
    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \
    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \
    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \
    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \
    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \
    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \
    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \
    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \
    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \
    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \
    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \
    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \
    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \
    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \
    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \
    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \
    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \
    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \
    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \
    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \
    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \
    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \
    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \
    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \
    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \
    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \
    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \
    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \
    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \
    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \
    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \
    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \
    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \
    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \
    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \
    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \
    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \
    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \
    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \
    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \
    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \
    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \
    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \
    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \
    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \
    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \
    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \
    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \
    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \
    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \
    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \
    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \
    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \
    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \
    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \
    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \
    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \
    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \
    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \
    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \
    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \
    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \
    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \
    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \
    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \
    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \
    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \
    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \
    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \
    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \
    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \
    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
numpy==1.26.4 \
    --hash=sha256:03a8c78d01d9781b28a6989f6fa1bb2c4f2d51201cf99d3dd875df6fbd96b23b \
    --hash=sha256:08beddf13648eb95f8d867350f6a018a4be2e5ad54c8d8caed89ebca558b2818 \
    --hash=sha256:1af303d6b2210eb850fcf03064d364652b7120803a0b872f5211f5234b399f20 \
    --hash=sha256:1dda2e7b4ec9dd512f84935c5f126c8bd8b9f2fc001e9f54af255e8c5f16b0e0 \
    --hash=sha256:2a02aba9ed12e4ac4eb3ea9421c420301a0c6460d9830d74a9df87efa4912010 \
    --hash=sha256:2e4ee3380d6de9c9ec04745830fd9e2eccb3e6cf790d39d7b98ffd19b0dd754a \
    --hash=sha256:3373d5d70a5fe74a2c1bb6d2cfd9609ecf686d47a2d7b1d37a8f3b6bf6003aea \
    --hash=sha256:47711010ad8555514b434df65f7d7b076bb8261df1ca9bb78f53d3b2db02e95c \
    --hash=sha256:4c66707fabe114439db9068ee468c26bbdf909cac0fb58686a42a24de1760c71 \
    --hash=sha256:50193e430acfc1346175fcbdaa28ffec49947a06918b7b92130744e81e640110 \
    --hash=sha256:52b8b60467cd7dd1e9ed082188b4e6bb35aa5cdd01777621a1658910745b90be \
    --hash=sha256:60dedbb91afcbfdc9bc0b1f3f402804070deed7392c23eb7a7f07fa857868e8a \
    --hash=sha256:62b8e4b1e28009ef2846b4c7852046736bab361f7aeadeb6a5b89ebec3c7055a \
    --hash=sha256:666dbfb6ec68962c033a450943ded891bed2d54e6755e35e5835d63f4f6931d5 \
    --hash=sha256:675d61ffbfa78604709862923189bad94014bef562cc35cf61d3a07bba02a7ed \
    --hash=sha256:679b0076f67ecc0138fd2ede3a8fd196dddc2ad3254069bcb9faf9a79b1cebcd \
    --hash=sha256:7349ab0fa0c429c82442a27a9673fc802ffdb7c7775fad780226cb234965e53c \
    --hash=sha256:7ab55401287bfec946ced39700c053796e7cc0e3acbef09993a9ad2adba6ca6e \
    --hash=sha256:7e50d0a0cc3189f9cb0aeb3a6a6af18c16f59f004b866cd2be1c14b36134a4a0 \
    --hash=sha256:95a7476c59002f2f6c590b9b7b998306fba6a5aa646b1e22ddfeaf8f78c3a29c \
    --hash=sha256:96ff0b2ad353d8f990b63294c8986f1ec3cb19d749234014f4e7eb0112ceba5a \
    --hash=sha256:9fad7dcb1aac3c7f0584a5a8133e3a43eeb2fe127f47e3632d43d677c66c102b \
    --hash=sha256:9ff0f4f29c51e2803569d7a51c2304de5554655a60c5d776e35b4a41413830d0 \
    --hash=sha256:a354325ee03388678242a4d7ebcd08b5c727033fcff3b2f536aea978e15ee9e6 \
    --hash=sha256:a4abb4f9001ad2858e7ac189089c42178fcce737e4169dc61321660f1a96c7d2 \
    --hash=sha256:ab47dbe5cc8210f55aa58e4805fe224dac469cde56b9f731a4c098b91917159a \
    --hash=sha256:afedb719a9dcfc7eaf2287b839d8198e06dcd4cb5d276a3df279231138e83d30 \
    --hash=sha256:b3ce300f3644fb06443ee2222c2201dd3a89ea6040541412b8fa189341847218 \
    --hash=sha256:b97fe8060236edf3662adfc2c633f56a08ae30560c56310562cb4f95500022d5 \
    --hash=sha256:bfe25acf8b437eb2a8b2d49d443800a5f18508cd811fea3181723922a8a82b07 \
    --hash=sha256:cd25bcecc4974d09257ffcd1f098ee778f7834c3ad767fe5db785be9a4aa9cb2 \
    --hash=sha256:d209d8969599b27ad20994c8e41936ee0964e6da07478d6c35016bc386b66ad4 \
    --hash=sha256:d5241e0a80d808d70546c697135da2c613f30e28251ff8307eb72ba696945764 \
    --hash=sha256:edd8b5fe47dab091176d21bb6de568acdd906d1887a4584a15a9a96a1dca06ef \
    --hash=sha256:f870204a840a60da0b12273ef34f7051e98c3b5961b61b0c2c1be6dfd64fbcd3 \
    --hash=sha256:ffa75af20b44f8dba823498024771d5ac50620e6915abac414251bd971b4529f
nvidia-cublas-cu12==12.4.5.8 \
    --hash=sha256:0f8aa1706812e00b9f19dfe0cdb3999b092ccb8ca168c0db5b8ea712456fd9b3 \
    --hash=sha256:2fc8da60df463fdefa81e323eef2e36489e1c94335b5358bcb38360adf75ac9b \
    --hash=sha256:5a796786da89203a0657eda402bcdcec6180254a8ac22d72213abc42069522dc
nvidia-cuda-cupti-cu12==12.4.127 \
    --hash=sha256:5688d203301ab051449a2b1cb6690fbe90d2b372f411521c86018b950f3d7922 \
    --hash=sha256:79279b35cf6f91da114182a5ce1864997fd52294a87a16179ce275773799458a \
    --hash=sha256:9dec60f5ac126f7bb551c055072b69d85392b13311fcc1bcda2202d172df30fb
nvidia-cuda-nvrtc-cu12==12.4.127 \
    --hash=sha256:0eedf14185e04b76aa05b1fea04133e59f465b6f960c0cbf4e37c3cb6b0ea198 \
    --hash=sha256:a178759ebb095827bd30ef56598ec182b85547f1508941a3d560eb7ea1fbf338 \
    --hash=sha256:a961b2f1d5f17b14867c619ceb99ef6fcec12e46612711bcec78eb05068a60ec
nvidia-cuda-runtime-cu12==12.4.127 \
    --hash=sha256:09c2e35f48359752dfa822c09918211844a3d93c100a715d79b59591130c5e1e \
    --hash=sha256:64403288fa2136ee8e467cdc9c9427e0434110899d07c779f25b5c068934faa5 \
    --hash=sha256:961fe0e2e716a2a1d967aab7caee97512f71767f852f67432d572e36cb3a11f3
nvidia-cudnn-cu12==9.1.0.70 \
    --hash=sha256:165764f44ef8c61fcdfdfdbe769d687e06374059fbb388b6c89ecb0e28793a6f \
    --hash=sha256:6278562929433d68365a07a4a1546c237ba2849852c0d4b2262a486e805b977a
nvidia-cufft-cu12==11.2.1.3 \
    --hash=sha256:5dad8008fc7f92f5ddfa2101430917ce2ffacd86824914c82e28990ad7f00399 \
    --hash=sha256:d802f4954291101186078ccbe22fc285a902136f974d369540fd4a5333d1440b \
    --hash=sha256:f083fc24912aa410be21fa16d157fed2055dab1cc4b6934a0e03cba69eb242b9
nvidia-curand-cu12==10.3.5.147 \
    --hash=sha256:1f173f09e3e3c76ab084aba0de819c49e56614feae5c12f69883f4ae9bb5fad9 \
    --hash=sha256:a88f583d4e0bb643c49743469964103aa59f7f708d862c3ddb0fc07f851e3b8b \
    --hash=sha256:f307cc191f96efe9e8f05a87096abc20d08845a841889ef78cb06924437f6771
nvidia-cusolver-cu12==11.6.1.9 \
    --hash=sha256:19e33fa442bcfd085b3086c4ebf7e8debc07cfe01e11513cc6d332fd918ac260 \
    --hash=sha256:d338f155f174f90724bbde3758b7ac375a70ce8e706d70b018dd3375545fc84e \
    --hash=sha256:e77314c9d7b694fcebc84f58989f3aa4fb4cb442f12ca1a9bde50f5e8f6d1b9c
nvidia-cusparse-cu12==12.3.1.170 \
    --hash=sha256:9bc90fb087bc7b4c15641521f31c0371e9a612fc2ba12c338d3ae032e6b6797f \
    --hash=sha256:9d32f62896231ebe0480efd8a7f702e143c98cfaa0e8a76df3386c1ba2b54df3 \
    --hash=sha256:ea4f11a2904e2a8dc4b1833cc1b5181cde564edd0d5cd33e3c168eff2d1863f1
nvidia-cusparselt-cu12==0.6.2 \
    --hash=sha256:0057c91d230703924c0422feabe4ce768841f9b4b44d28586b6f6d2eb86fbe70 \
    --hash=sha256:067a7f6d03ea0d4841c85f0c6f1991c5dda98211f6302cb83a4ab234ee95bef8 \
    --hash=sha256:df2c24502fd76ebafe7457dbc4716b2fec071aabaed4fb7691a201cde03704d9
nvidia-nccl-cu12==2.21.5 \
    --hash=sha256:8579076d30a8c24988834445f8d633c697d42397e92ffc3f63fa26766d25e0a0
nvidia-nvjitlink-cu12==12.4.127 \
    --hash=sha256:06b3b9b25bf3f8af351d664978ca26a16d2c5127dbd53c0497e28d1fb9611d57 \
    --hash=sha256:4abe7fef64914ccfa909bc2ba39739670ecc9e820c83ccc7a6ed414122599b83 \
    --hash=sha256:fd9020c501d27d135f983c6d3e244b197a7ccad769e34df53a42e276b0e25fa1
nvidia-nvtx-cu12==12.4.127 \
    --hash=sha256:641dccaaa1139f3ffb0d3164b4b84f9d253397e38246a4f2f36728b48566d485 \
    --hash=sha256:781e950d9b9f60d8241ccea575b32f5105a5baf4c2351cab5256a24869f12a1a \
    --hash=sha256:7959ad635db13edf4fc65c06a6e9f9e55fc2f92596db928d169c0bb031e88ef3
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
peft==0.13.2 \
    --hash=sha256:0e0cbd40ebdf5fe4ea79f255880d02f96712d18899509369a2cc5768ad46d672 \
    --hash=sha256:d4e0951ec78eac11c45a051801c569913436888c578d48e5ce86996b715bc6ef
pillow==11.1.0 \
    --hash=sha256:015c6e863faa4779251436db398ae75051469f7c903b043a48f078e437656f83 \
    --hash=sha256:0a2f91f8a8b367e7a57c6e91cd25af510168091fb89ec5146003e424e1558a96 \
    --hash=sha256:11633d58b6ee5733bde153a8dafd25e505ea3d32e261accd388827ee987baf65 \
    --hash=sha256:2062ffb1d36544d42fcaa277b069c88b01bb7298f4efa06731a7fd6cc290b81a \
    --hash=sha256:31eba6bbdd27dde97b0174ddf0297d7a9c3a507a8a1480e1e60ef914fe23d352 \
    --hash=sha256:3362c6ca227e65c54bf71a5f88b3d4565ff1bcbc63ae72c34b07bbb1cc59a43f \
    --hash=sha256:368da70808b36d73b4b390a8ffac11069f8a5c85f29eff1f1b01bcf3ef5b2a20 \
    --hash=sha256:36ba10b9cb413e7c7dfa3e189aba252deee0602c86c309799da5a74009ac7a1c \
    --hash=sha256:3764d53e09cdedd91bee65c2527815d315c6b90d7b8b79759cc48d7bf5d4f114 \
    --hash=sha256:3a5fe20a7b66e8135d7fd617b13272626a28278d0e578c98720d9ba4b2439d49 \
    --hash=sha256:3cdcdb0b896e981678eee140d882b70092dac83ac1cdf6b3a60e2216a73f2b91 \
    --hash=sha256:4637b88343166249fe8aa94e7c4a62a180c4b3898283bb5d3d2fd5fe10d8e4e0 \
    --hash=sha256:4db853948ce4e718f2fc775b75c37ba2efb6aaea41a1a5fc57f0af59eee774b2 \
    --hash=sha256:4dd43a78897793f60766563969442020e90eb7847463eca901e41ba186a7d4a5 \
    --hash=sha256:54251ef02a2309b5eec99d151ebf5c9904b77976c8abdcbce7891ed22df53884 \
    --hash=sha256:54ce1c9a16a9561b6d6d8cb30089ab1e5eb66918cb47d457bd996ef34182922e \
    --hash=sha256:593c5fd6be85da83656b93ffcccc2312d2d149d251e98588b14fbc288fd8909c \
    --hash=sha256:5bb94705aea800051a743aa4874bb1397d4695fb0583ba5e425ee0328757f196 \
    --hash=sha256:67cd427c68926108778a9005f2a04adbd5e67c442ed21d95389fe1d595458756 \
    --hash=sha256:70ca5ef3b3b1c4a0812b5c63c57c23b63e53bc38e758b37a951e5bc466449861 \
    --hash=sha256:73ddde795ee9b06257dac5ad42fcb07f3b9b813f8c1f7f870f402f4dc54b5269 \
    --hash=sha256:758e9d4ef15d3560214cddbc97b8ef3ef86ce04d62ddac17ad39ba87e89bd3b1 \
    --hash=sha256:7d33d2fae0e8b170b6a6c57400e077412240f6f5bb2a342cf1ee512a787942bb \
    --hash=sha256:7fdadc077553621911f27ce206ffcbec7d3f8d7b50e0da39f10997e8e2bb7f6a \
    --hash=sha256:8000376f139d4d38d6851eb149b321a52bb8893a88dae8ee7d95840431977081 \
    --hash=sha256:837060a8599b8f5d402e97197d4924f05a2e0d68756998345c829c33186217b1 \
    --hash=sha256:89dbdb3e6e9594d512780a5a1c42801879628b38e3efc7038094430844e271d8 \
    --hash=sha256:8c730dc3a83e5ac137fbc92dfcfe1511ce3b2b5d7578315b63dbbb76f7f51d90 \
    --hash=sha256:8e275ee4cb11c262bd108ab2081f750db2a1c0b8c12c1897f27b160c8bd57bbc \
    --hash=sha256:9044b5e4f7083f209c4e35aa5dd54b1dd5b112b108648f5c902ad586d4f945c5 \
    --hash=sha256:93a18841d09bcdd774dcdc308e4537e1f867b3dec059c131fde0327899734aa1 \
    --hash=sha256:9409c080586d1f683df3f184f20e36fb647f2e0bc3988094d4fd8c9f4eb1b3b3 \
    --hash=sha256:96f82000e12f23e4f29346e42702b6ed9a2f2fea34a740dd5ffffcc8c539eb35 \
    --hash=sha256:9aa9aeddeed452b2f616ff5507459e7bab436916ccb10961c4a382cd3e03f47f \
    --hash=sha256:9ee85f0696a17dd28fbcfceb59f9510aa71934b483d1f5601d1030c3c8304f3c \
    --hash=sha256:a07dba04c5e22824816b2615ad7a7484432d7f540e6fa86af60d2de57b0fcee2 \
    --hash=sha256:a3cd561ded2cf2bbae44d4605837221b987c216cff94f49dfeed63488bb228d2 \
    --hash=sha256:a697cd8ba0383bba3d2d3ada02b34ed268cb548b369943cd349007730c92bddf \
    --hash=sha256:a76da0a31da6fcae4210aa94fd779c65c75786bc9af06289cd1c184451ef7a65 \
    --hash=sha256:a85b653980faad27e88b141348707ceeef8a1186f75ecc600c395dcac19f385b \
    --hash=sha256:a8d65b38173085f24bc07f8b6c505cbb7418009fa1a1fcb111b1f4961814a442 \
    --hash=sha256:aa8dd43daa836b9a8128dbe7d923423e5ad86f50a7a14dc688194b7be5c0dea2 \
    --hash=sha256:ab8a209b8485d3db694fa97a896d96dd6533d63c22829043fd9de627060beade \
    --hash=sha256:abc56501c3fd148d60659aae0af6ddc149660469082859fa7b066a298bde9482 \
    --hash=sha256:ad5db5781c774ab9a9b2c4302bbf0c1014960a0a7be63278d13ae6fdf88126fe \
    --hash=sha256:ae98e14432d458fc3de11a77ccb3ae65ddce70f730e7c76140653048c71bfcbc \
    --hash=sha256:b20be51b37a75cc54c2c55def3fa2c65bb94ba859dde241cd0a4fd302de5ae0a \
    --hash=sha256:b523466b1a31d0dcef7c5be1f20b942919b62fd6e9a9be199d035509cbefc0ec \
    --hash=sha256:b5d658fbd9f0d6eea113aea286b21d3cd4d3fd978157cbf2447a6035916506d3 \
    --hash=sha256:b6123aa4a59d75f06e9dd3dac5bf8bc9aa383121bb3dd9a7a612e05eabc9961a \
    --hash=sha256:bd165131fd51697e22421d0e467997ad31621b74bfc0b75956608cb2906dda07 \
    --hash=sha256:bf902d7413c82a1bfa08b06a070876132a5ae6b2388e2712aab3a7cbc02205c6 \
    --hash=sha256:c12fc111ef090845de2bb15009372175d76ac99969bdf31e2ce9b42e4b8cd88f \
    --hash=sha256:c1eec9d950b6fe688edee07138993e54ee4ae634c51443cfb7c1e7613322718e \
    --hash=sha256:c640e5a06869c75994624551f45e5506e4256562ead981cce820d5ab39ae2192 \
    --hash=sha256:cc1331b6d5a6e144aeb5e626f4375f5b7ae9934ba620c0ac6b3e43d5e683a0f0 \
    --hash=sha256:cfd5cd998c2e36a862d0e27b2df63237e67273f2fc78f47445b14e73a810e7e6 \
    --hash=sha256:d3d8da4a631471dfaf94c10c85f5277b1f8e42ac42bade1ac67da4b4a7359b73 \
    --hash=sha256:d44ff19eea13ae4acdaaab0179fa68c0c6f2f45d66a4d8ec1eda7d6cecbcc15f \
    --hash=sha256:dd0052e9db3474df30433f83a71b9b23bd9e4ef1de13d92df21a52c0303b8ab6 \
    --hash=sha256:dd0e081319328928531df7a0e63621caf67652c8464303fd102141b785ef9547 \
    --hash=sha256:dda60aa465b861324e65a78c9f5cf0f4bc713e4309f83bc387be158b077963d9 \
    --hash=sha256:e06695e0326d05b06833b40b7ef477e475d0b1ba3a6d27da1bb48c23209bf457 \
    --hash=sha256:e1abe69aca89514737465752b4bcaf8016de61b3be1397a8fc260ba33321b3a8 \
    --hash=sha256:e267b0ed063341f3e60acd25c05200df4193e15a4a5807075cd71225a2386e26 \
    --hash=sha256:e5449ca63da169a2e6068dd0e2fcc8d91f9558aba89ff6d02121ca8ab11e79e5 \
    --hash=sha256:e63e4e5081de46517099dc30abe418122f54531a6ae2ebc8680bcd7096860eab \
    --hash=sha256:f189805c8be5ca5add39e6f899e6ce2ed824e65fb45f3c28cb2841911da19070 \
    --hash=sha256:f7955ecf5609dee9442cbface754f2c6e541d9e6eda87fad7f7a989b0bdb9d71 \
    --hash=sha256:f86d3a7a9af5d826744fabf4afd15b9dfef44fe69a98541f666f66fbb8d3fef9 \
    --hash=sha256:fbd43429d0d7ed6533b25fc993861b8fd512c42d04514a0dd6337fb3ccf22761
psutil==7.2.2 \
    --hash=sha256:0746f5f8d406af344fd547f1c8daa5f5c33dbc293bb8d6a16d80b4bb88f59372 \
    --hash=sha256:076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9 \
    --hash=sha256:11fe5a4f613759764e79c65cf11ebdf26e33d6dd34336f8a337aa2996d71c841 \
    --hash=sha256:1a571f2330c966c62aeda00dd24620425d4b0cc86881c89861fbc04549e5dc63 \
    --hash=sha256:1a7b04c10f32cc88ab39cbf606e117fd74721c831c98a27dc04578deb0c16979 \
    --hash=sha256:1fa4ecf83bcdf6e6c8f4449aff98eefb5d0604bf88cb883d7da3d8d2d909546a \
    --hash=sha256:2edccc433cbfa046b980b0df0171cd25bcaeb3a68fe9022db0979e7aa74a826b \
    --hash=sha256:7b6d09433a10592ce39b13d7be5a54fbac1d1228ed29abc880fb23df7cb694c9 \
    --hash=sha256:8c233660f575a5a89e6d4cb65d9f938126312bca76d8fe087b947b3a1aaac9ee \
    --hash=sha256:917e891983ca3c1887b4ef36447b1e0873e70c933afc831c6b6da078ba474312 \
    --hash=sha256:ab486563df44c17f5173621c7b198955bd6b613fb87c71c161f827d3fb149a9b \
    --hash=sha256:ae0aefdd8796a7737eccea863f80f81e468a1e4cf14d926bd9b6f5f2d5f90ca9 \
    --hash=sha256:b0726cecd84f9474419d67252add4ac0cd9811b04d61123054b9fb6f57df6e9e \
    --hash=sha256:b58fabe35e80b264a4e3bb23e6b96f9e45a3df7fb7eed419ac0e5947c61e47cc \
    --hash=sha256:c7663d4e37f13e884d13994247449e9f8f574bc4655d509c3b95e9ec9e2b9dc1 \
    --hash=sha256:e452c464a02e7dc7822a05d25db4cde564444a67e58539a00f929c51eddda0cf \
    --hash=sha256:e78c8603dcd9a04c7364f1a3e670cea95d51ee865e4efb3556a3a63adef958ea \
    --hash=sha256:eb7e81434c8d223ec4a219b5fc1c47d0417b12be7ea866e24fb5ad6e84b3d988 \
    --hash=sha256:ed0cace939114f62738d808fdcecd4c869222507e266e574799e9c0faa17d486 \
    --hash=sha256:eed63d3b4d62449571547b60578c5b2c4bcccc5387148db46e0c2313dad0ee00 \
    --hash=sha256:fd04ef36b4a6d599bbdb225dd1d3f51e00105f6d48a28f006da7f9822f2606d8
pycparser==3.0 \
    --hash=sha256:600f49d217304a5902ac3c37e1281c9fe94e4d0489de643a9504c5cdfdfc6b29 \
    --hash=sha256:b727414169a36b7d524c1c3e31839a521725078d7b2ff038656844266160a992
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
regex==2026.9.10 \
    --hash=sha256:030fa9e23624e39b3b94e46b90a5abd1a1678eb2f58fcdd3fd6c27526bf91c7e \
    --hash=sha256:032da15431c890d376f53547f0a6219f4f4cd19f3e4f11bdc321453b5bd207e4 \
    --hash=sha256:044bd4639b6bb409ec9e5d8b7accd57e02b4c4a4e2eafde916f8ae8006b3e40b \
    --hash=sha256:048a89ee797db10160bd2bd519286577a6b43a100279bd4b7d8456a3d69c80a0 \
    --hash=sha256:05fb018cfe7144585fc83882405906ff84994a2d154afc2509ecc7752c51f864 \
    --hash=sha256:07b45ba5c94b8fcb30cb6c56a11f715c57533a3017964504322ea52690a27b72 \
    --hash=sha256:0aa7589394230e0f0a422ab6b90841ff12c87e855e7aaf75d192a54a5f124548 \
    --hash=sha256:0acee94b480dd853e39434aa9a575f95385b1b4b8fa3feae56db363ca5cad782 \
    --hash=sha256:0b9ba3b2765cdfe18f0f561a69f78a69701f2896654a81c711108d35d14e5099 \
    --hash=sha256:0c32480f3371b75068decaf9e5da72c224e953830dd71e36e06cf80e30ea39d8 \
    --hash=sha256:1270cdec69248592bbe38a0b263ed58d907b891bd2b93703e225c317e421bda1 \
    --hash=sha256:13c52fc377792675f604a207a2ae5958c080f6854f7698d40d9ff034d95b1e76 \
    --hash=sha256:14caa05ce39ec70437af5aac8814c50ee6628f4a90353871c059692f448a164f \
    --hash=sha256:1562aabd9d4eb09bd88a62ad97ed06800094b529ac43419e43020b9cefec79b0 \
    --hash=sha256:175cf49ce7a994c88b8f15e3cb17cdb66a48ebb2d36de736b8205033db950f89 \
    --hash=sha256:1aa309ab7ba89a62d6cf70dbd38d4176440bce3c7001ab86256704cf4c18c6eb \
    --hash=sha256:1ad10a135fa0b4e4a462a61d07c6654d7518cfdb5cb8da08f9ff7d61384af1fe \
    --hash=sha256:1b891f77554bff991804cee24b78b40789f7d5993a24c7907bc7025fd2a70c8d \
    --hash=sha256:1e321e2c84f0e52c457f5ea5944f796d6e8e09cb99738ea98dcc1bfe402a128d \
    --hash=sha256:1e954e246466d5a1a78f563ce8364b5d7cb19e7adb0ccdec8f9c9610083187bc \
    --hash=sha256:1f0a8b4928823bc8b217a1ab7bf3d90598909dec9a70fbbfe9a52cc4eca55990 \
    --hash=sha256:1fbc8314436353e097c050e11b01a6c11433579437ed0579730157676ef59e2f \
    --hash=sha256:20e8bfb07ad79a282f8b95b56fe67f9750b1b7f775724e4ba1f23cb296115ce4 \
    --hash=sha256:217e98ba5fc8908ed8ffd4ebac04753a0c831067cbfb495b9821b94cc61eaa76 \
    --hash=sha256:239620b0e0681669367c0e218c8eb2551d9f8fe3b9fccfc8d0003377804e8348 \
    --hash=sha256:23ac9a28180f274d7dd7651fa131ad5b02d343b75df4b040737f0356223895dd \
    --hash=sha256:2479171edccced52ef02b899558f88ab2c235fe05b93180fdcae1670aacd89e1 \
    --hash=sha256:24d12a625a37c89c2b09303402a06942f55f071b95a7916a49c17034c3d47cd5 \
    --hash=sha256:2dd9286093c71afc8f55ef035c5b9d2776641fd72c6535f1febc92d0b0be9666 \
    --hash=sha256:2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0 \
    --hash=sha256:31e4df2b11d48f61d511019bc1ee9b477055f17c352b68fe72db7a98b14d603c \
    --hash=sha256:3264132d576847ab5f88bb83e7debe67854bf165b3ea613bd467312b6099536a \
    --hash=sha256:3540734dbe241ebb3b87d5713781f6749a3e4d45480f506aa5fb5cbb0c37d249 \
    --hash=sha256:35ba3bab0c45079735f55ac61526774de1d84bc4a0333cc554e1a4ab74913924 \
    --hash=sha256:3a66e40a1a20de96a2fee00ed67e11012b62d85b277688258677fd19997addb7 \
    --hash=sha256:3bdeed3318a8eb2bbadc9c56347e0ff651639e934a47e168d05a3b12929fd0e7 \
    --hash=sha256:3fb4ae8cf83ef4e9addd43b2da31a9f45be816a8036fae8af59c8998b72718e2 \
    --hash=sha256:4971776b4f2bd7fd9a83eceb2cb2592cbe2924f639fe8045e6a9de5ba4bfcf25 \
    --hash=sha256:4a761ea45f2ad74c575ef5850ea514cef97302a552d3c7c9d1a1a870d4661d6c \
    --hash=sha256:4c66d54042a14a503907d81861b8a5235e6d1f03d4fbc1d8767f652eaf957ac1 \
    --hash=sha256:4db7d00c4afbfbb55b8e17b1e371da11418ea9389b030acec63c1fa4c7ad4b86 \
    --hash=sha256:4f0407474ffac8e5e89d93ca41d60891e29f0ab8423eb66ff292d850a86a0843 \
    --hash=sha256:53e182b6b04d0011909b47d51a2d72d908de07c7b1c7f16b3adda2204d723bc1 \
    --hash=sha256:5847e22bbf959764d776937d791d034cc2d19b787e361c88d97e859e8dc68502 \
    --hash=sha256:58c01f7b81079cf0817ba831ff4d9eff5d28be4a3ac76c353e6f09bd63f4c386 \
    --hash=sha256:58da726d3e766c0b3f5a3997dfaf0275898a1107b8191cdd6b0437fe45fd817d \
    --hash=sha256:5bef622850cf760154719d4e0d74b0a855962432995168e250069899ae12fe8f \
    --hash=sha256:5ccd139b2061132e7b265cfb4b4721baeb9f8928b81415304abf1ec7e3181c26 \
    --hash=sha256:5cef9f3d14796500ea834c41dbe688f1f6b23c7024dc23e8a794d7ebaf5d71d0 \
    --hash=sha256:63bb62cf62217dc38c8a6b2b61b165b0e4eb8fa93b0aba12139251c0986a8fa3 \
    --hash=sha256:681ed38664b64c6617d3c3c332018d1948c77e139c5ea667c1886efa671e426f \
    --hash=sha256:6888065672b341e5246f391ec16dc258a29218ac784172fd67c30d941544755b \
    --hash=sha256:6aebdd9a946de328b3f6f61dbf48dd064a36eb6dddf96e34ae6651d37f6e9383 \
    --hash=sha256:6afcad14310f1311d077553ed374b42a5e538f85a8c884b4e38e52de091c8077 \
    --hash=sha256:6b34a778c695d24e77c140e3b4c95da69282e34f2f6b02b55656aa4a0379f643 \
    --hash=sha256:6fd555fc9abef50c530869690b2daca054c8811a7aff632d11f9a7b2590b2742 \
    --hash=sha256:71879292c9c7ac67b1680345b16daba1be937cb027362cfa04e68f65db2dcfdd \
    --hash=sha256:75242f44a3e283106077be4ab717bc535e4701c9d54ad69e195945c22f137a1d \
    --hash=sha256:75aa39d3f4f1650eea84e46b0d8cefe77dd5478c10e3d0aaf0b0f00493475a7a \
    --hash=sha256:75f9297b16fcb588a1f8d8a55dabef3c0c20b0c7bac43c87ceaaaf1a825c12f4 \
    --hash=sha256:79e9432995e14c749d34209413de5e621ec8e67789bf4f46dbfabea9d06a2406 \
    --hash=sha256:7abb38b8c40f3a235235a44da452c64b7b5c1d650ec6351027db0e090804f2e5 \
    --hash=sha256:7dcad477c49c4c626a6c4fcd71b39a971aa217060cc40a6569fd24edcc0fa509 \
    --hash=sha256:7e6c0b5ec6ddee4032247585dc491b0fa58627745b66a705728703a3f0331231 \
    --hash=sha256:7f8f10015866608fe4c043cec2e4fe4c39a94bb50e45091de4cdf4004b9ae4b0 \
    --hash=sha256:866de9f98df0611d7b62b3a8729d3284a64c0cc6edd90bb95a533e443a4939cb \
    --hash=sha256:87f5f75c109f08f5c602d68e1af54cead8165189c727b6ac946b30b9833a3ba4 \
    --hash=sha256:880ac684c27176464c00c3fdc456116364f5ebc70da07aad0c2d4a7ba45e98db \
    --hash=sha256:88b02aa8d0ec9b6189fe933d425775882271c23700ac11fd26d1779b0f56fde3 \
    --hash=sha256:8ba1f78bd4fef2d8f84b894ec28ac3481afe6cc07aaa253ad4717ef7b3fe6bcb \
    --hash=sha256:8c07021a4faa3f092869adbd1f35cdc7a592276c807aeebc3ceb8ff1a638f0b4 \
    --hash=sha256:8d5c4518235a2ec1611e57af85fa488d529c1106aacff12adadcedf8687012cd \
    --hash=sha256:8e127d9a80cbf1c3276bb465c6d047e8705e97b58c2b8f2f0c0a69c336b44b37 \
    --hash=sha256:94c5ce3bc41d226b4eb89ca3f842b2e28c031487fb1f34eb2153d98235831325 \
    --hash=sha256:94d096369b7cd96d15343fef5257fe39eff9d0e8758b92a0e15e358b92cdb2fc \
    --hash=sha256:968c1e33edd9a104d1bf24c8d476c72de7e3839ae7f894b37e9e4f4739fdeeca \
    --hash=sha256:990797e765d89a423880052c68b61c31afe701de94a8c060f61c40605ca6c727 \
    --hash=sha256:9ce239acb15843ab03976626af810a4424b0409689ec2bbc52088ab5479ab487 \
    --hash=sha256:9d772586951d7d6a5d162d48f414065e483b1c81ab38fd8ed97c78b05883421a \
    --hash=sha256:9fbd2e5d8002dc49a6129fb321ec51c57a025e752ed525ddce0ba9223c4350a7 \
    --hash=sha256:a41693eb3fc4b92e6127d113813c6c395237f7edd3224abf67609af48c690d11 \
    --hash=sha256:abbfc1c33bf8efddcc43844aba61e036d74a918680dc3ce8ce2538b004eda0f9 \
    --hash=sha256:b298cdc33c5cc6969ff07f0fba19cc73e0fd8576373c50935feadaca2f6b4405 \
    --hash=sha256:b43456de605c8ee77eb75f07bc1ee44ba27f9cee22207deb77d495e954b7d953 \
    --hash=sha256:b71649169a9fcf30b395ee01047fa7ad6654a4c900ca75b23c04dedcce6a1f8c \
    --hash=sha256:b91c37551bf39d75116c02b146956f65b9aa0337a4a652f4ae186983789d4001 \
    --hash=sha256:b9d36b03dc362aa40ffaaec9d9bd75e87763529563ec008c43b0e07782f5be7a \
    --hash=sha256:bafa41b0dd63669e5c0f8adf3d24819efeb73c847f492eb011212eb352e69041 \
    --hash=sha256:bb7774924f8cd69f49cba0b3c2d679a6326f777e0e67d130ad5203e4df53f0d3 \
    --hash=sha256:bf29611e5376fec8f795879bb5c6153a76c3a292573d173c26784042b01eb840 \
    --hash=sha256:c014641157e9049b0603b8daa5343bd408d9b757b709aaa0f373cd3fab2d7944 \
    --hash=sha256:c103b3b14e011774af4fb7e4617ad4d72b9171905cd3b231a70a4efd76e477d7 \
    --hash=sha256:c22df8dd6373bbe3898e77429ffc85594300e39d752fd0e68a31e59d37899376 \
    --hash=sha256:c25a754bb81a2edcfc3b65eda50f017d736f818112ed43e8aafd595cb00678ae \
    --hash=sha256:c32818b28bcd153b25b63038348a9fe9b9fbcddb60df43f204c3ab55eeb57f77 \
    --hash=sha256:c37fa93bf18bf4f90b01c0fa9f11ea567ee4b7dd8bf96e63663e5edc37aa38cf \
    --hash=sha256:c3d95d7d9538b5b726dd6fcd7b6117a71e6565202f6d64f5845fb4d8f203f533 \
    --hash=sha256:c8fbd9cb30c68c1686b94029b9ef845d5870d3d65baf66cb126b676849b9d72b \
    --hash=sha256:cb76a9c4e07a6a47849726af0ed14c41741a182f097f134a8cf29c1bc0f4dde8 \
    --hash=sha256:ce7c118cb102975f974585688357a717ffbf9dddd64ab0bb1bc93eb5b367cf95 \
    --hash=sha256:cf377960d2ac37d987394a9dbaa75e91338c41a46d41e1d25e90125e7b3ee2dc \
    --hash=sha256:d278ad30ec83b6b9202685b0f80b741a51ea3ca7f0595ebda96e7628b6398876 \
    --hash=sha256:d2d377fd1cad611b806cdd732d86b65f536c768209890cb442556548daa65a23 \
    --hash=sha256:d414c411c06fe0009eac33488fb1591c66b5c2673e342e452e7bb2fe63da8194 \
    --hash=sha256:d8c668af8f7bdb1d18739c27d30cd9f4b371495a883f75a002fb7a39d740fecd \
    --hash=sha256:dce932f8e3ba936475ea3d0d8b59f7b050a9e206e994f53f8fd80299871e87da \
    --hash=sha256:debc629e98b95abaea1cf3057ca296151f348c697c9b8a59d18013adb302c0dd \
    --hash=sha256:e0dc78251154b66dc60211563fc115345da332eaa881e4e2523fb1edae3772f4 \
    --hash=sha256:e5e4a6e0734a685d13b9685622bb503bdbb2927f8b0df025a5085f0ea067475b \
    --hash=sha256:e6b99181d184d0f5c7b36b8d12b94d1e9499cce6246594331f9edc5d2ea9fceb \
    --hash=sha256:e7327795089ddb44912dce1434e1d7244be2e9fb48fcc2d6782936af7a3062db \
    --hash=sha256:ebb2ba68e4641a994061f70bf44ed448fba0b9b1d18c94ffb9efc1cca805b39b \
    --hash=sha256:ec8855f08c17895a26fbf5f19ed829722e19b34a96629e49a43c92974924026b \
    --hash=sha256:ecb2e7acb18f8cc4a67f0ad986c0af291ea4dd385d0614ba9bc09d7f8bbb478c \
    --hash=sha256:ef4c0a9dfdc90581b90b1b95a8c3d1557f8ff8f5a2a53536d26314de699d1468 \
    --hash=sha256:ef4ce69ff97fbb44b46751cfea5e859ad0b66d1a50abf34954f0645f51e81671 \
    --hash=sha256:ef5a059ea1c6ee5d1c7e99a2484e628608d010921efe876c6f0e2029d2f35eca \
    --hash=sha256:f0e2e5d23448b660d60a6ed85c46cc03b4b48bd276b8f4041d4a5fe2a4a0626b \
    --hash=sha256:f2374c27deb189b282ec7e16106752c22ad39b056bbd8018960b1e4cc95d67a1 \
    --hash=sha256:f2f43bf4e47ff7ce9e585558706d698c6204d0f80bf2207766382ed817c8e9f4 \
    --hash=sha256:f5c629df03adec31ee505dda3c8988f106c9390e4cbd343600036eb8b3d6724f \
    --hash=sha256:f70b9f0e39c2dba1d9da6bf7ef7c377cad7277f8440e9a69be05ede529ff024c \
    --hash=sha256:f7d4656e17ab736e9415a6442a345bfc97bb8b7dcce47884bb74a37f70f08d0c \
    --hash=sha256:f8bdec659a8fa7af51a32b224b3b7c02bc415d54ffd35187b1d224176b17d607 \
    --hash=sha256:faa911fbbcf8ac90bda0e0657d60768e3390954ef0588211d63a22add1cb1cd1 \
    --hash=sha256:fbc4e2f3cb7ce8436154e6483079e7d35eeb321a952fa936e180300630d8b873 \
    --hash=sha256:fd6bd89b9fc06018d35851cab0240adb7dd84d51941b19f6574ac90cd54e3ae5 \
    --hash=sha256:ff4d7b14ea19e50c8d9d6d83f45bd9b45cbb624c07ac1fa54db0a019049abed7 \
    --hash=sha256:ff6b3267318661dfddf6b3628663e00e5946bd0a5c8fa678537a1401f0388f91 \
    --hash=sha256:ffc2da104e43db716ce30cef9f28049a1faa6aca385dd8771b033268d0730b07
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
safetensors==0.5.3 \
    --hash=sha256:1077f3e94182d72618357b04b5ced540ceb71c8a813d3319f1aba448e68a770d \
    --hash=sha256:11bce6164887cd491ca75c2326a113ba934be596e22b28b1742ce27b1d076467 \
    --hash=sha256:21d01c14ff6c415c485616b8b0bf961c46b3b343ca59110d38d744e577f9cce7 \
    --hash=sha256:32c3ef2d7af8b9f52ff685ed0bc43913cdcde135089ae322ee576de93eae5135 \
    --hash=sha256:37f1521be045e56fc2b54c606d4455573e717b2d887c579ee1dbba5f868ece04 \
    --hash=sha256:391ac8cab7c829452175f871fcaf414aa1e292b5448bd02620f675a7f3e7abb9 \
    --hash=sha256:4a243be3590bc3301c821da7a18d87224ef35cbd3e5f5727e4e0728b8172411e \
    --hash=sha256:799021e78287bac619c7b3f3606730a22da4cda27759ddf55d37c8db7511c74b \
    --hash=sha256:836cbbc320b47e80acd40e44c8682db0e8ad7123209f69b093def21ec7cafd11 \
    --hash=sha256:8bd84b12b1670a6f8e50f01e28156422a2bc07fb16fc4e98bded13039d688a0d \
    --hash=sha256:b6b0d6ecacec39a4fdd99cc19f4576f5219ce858e6fd8dbe7609df0b8dc56965 \
    --hash=sha256:bd20eb133db8ed15b40110b7c00c6df51655a2998132193de2f75f72d99c7073 \
    --hash=sha256:cead1fa41fc54b1e61089fa57452e8834f798cb1dc7a09ba3524f1eb08e0317a \
    --hash=sha256:cfc0ec0846dcf6763b0ed3d1846ff36008c6e7290683b61616c4b040f6a54ace \
    --hash=sha256:df26da01aaac504334644e1b7642fa000bfec820e7cef83aeac4e355e03195ff
scipy==1.15.2 \
    --hash=sha256:01edfac9f0798ad6b46d9c4c9ca0e0ad23dbf0b1eb70e96adb9fa7f525eff0bf \
    --hash=sha256:03205d57a28e18dfd39f0377d5002725bf1f19a46f444108c29bdb246b6c8a11 \
    --hash=sha256:08b57a9336b8e79b305a143c3655cc5bdbe6d5ece3378578888d2afbb51c4e37 \
    --hash=sha256:11e7ad32cf184b74380f43d3c0a706f49358b904fa7d5345f16ddf993609184d \
    --hash=sha256:28a0d2c2075946346e4408b211240764759e0fabaeb08d871639b5f3b1aca8a0 \
    --hash=sha256:2b871df1fe1a3ba85d90e22742b93584f8d2b8e6124f8372ab15c71b73e428b8 \
    --hash=sha256:302093e7dfb120e55515936cb55618ee0b895f8bcaf18ff81eca086c17bd80af \
    --hash=sha256:42dabaaa798e987c425ed76062794e93a243be8f0f20fff6e7a89f4d61cb3d40 \
    --hash=sha256:447ce30cee6a9d5d1379087c9e474628dab3db4a67484be1b7dc3196bfb2fac9 \
    --hash=sha256:4c6676490ad76d1c2894d77f976144b41bd1a4052107902238047fb6a473e971 \
    --hash=sha256:54c462098484e7466362a9f1672d20888f724911a74c22ae35b61f9c5919183d \
    --hash=sha256:597a0c7008b21c035831c39927406c6181bcf8f60a73f36219b69d010aa04737 \
    --hash=sha256:5a6fd6eac1ce74a9f77a7fc724080d507c5812d61e72bd5e4c489b042455865e \
    --hash=sha256:5ea7ed46d437fc52350b028b1d44e002646e28f3e8ddc714011aaf87330f2f32 \
    --hash=sha256:601881dfb761311045b03114c5fe718a12634e5608c3b403737ae463c9885d53 \
    --hash=sha256:62ca1ff3eb513e09ed17a5736929429189adf16d2d740f44e53270cc800ecff1 \
    --hash=sha256:69ea6e56d00977f355c0f84eba69877b6df084516c602d93a33812aa04d90a3d \
    --hash=sha256:6a8e34cf4c188b6dd004654f88586d78f95639e48a25dfae9c5e34a6dc34547e \
    --hash=sha256:6d0194c37037707b2afa7a2f2a924cf7bac3dc292d51b6a925e5fcb89bc5c776 \
    --hash=sha256:6f223753c6ea76983af380787611ae1291e3ceb23917393079dcc746ba60cfb5 \
    --hash=sha256:6f5e296ec63c5da6ba6fa0343ea73fd51b8b3e1a300b0a8cae3ed4b1122c7462 \
    --hash=sha256:7cd5b77413e1855351cdde594eca99c1f4a588c2d63711388b6a1f1c01f62274 \
    --hash=sha256:869269b767d5ee7ea6991ed7e22b3ca1f22de73ab9a49c44bad338b725603301 \
    --hash=sha256:87994da02e73549dfecaed9e09a4f9d58a045a053865679aeb8d6d43747d4df3 \
    --hash=sha256:888307125ea0c4466287191e5606a2c910963405ce9671448ff9c81c53f85f58 \
    --hash=sha256:92233b2df6938147be6fa8824b8136f29a18f016ecde986666be5f4d686a91a4 \
    --hash=sha256:9412f5e408b397ff5641080ed1e798623dbe1ec0d78e72c9eca8992976fa65aa \
    --hash=sha256:9b18aa747da280664642997e65aab1dd19d0c3d17068a04b3fe34e2559196cb9 \
    --hash=sha256:9de9d1416b3d9e7df9923ab23cd2fe714244af10b763975bea9e4f2e81cebd27 \
    --hash=sha256:a2ec871edaa863e8213ea5df811cd600734f6400b4af272e1c011e69401218e9 \
    --hash=sha256:a5080a79dfb9b78b768cebf3c9dcbc7b665c5875793569f48bf0e2b1d7f68f6f \
    --hash=sha256:a8bf5cb4a25046ac61d38f8d3c3426ec11ebc350246a4642f2f315fe95bda655 \
    --hash=sha256:b09ae80010f52efddb15551025f9016c910296cf70adbf03ce2a8704f3a5ad20 \
    --hash=sha256:b5e025e903b4f166ea03b109bb241355b9c42c279ea694d8864d033727205e65 \
    --hash=sha256:bad78d580270a4d32470563ea86c6590b465cb98f83d760ff5b0990cb5518a93 \
    --hash=sha256:bae43364d600fdc3ac327db99659dcb79e6e7ecd279a75fe1266669d9a652828 \
    --hash=sha256:c4697a10da8f8765bb7c83e24a470da5797e37041edfd77fd95ba3811a47c4fd \
    --hash=sha256:c90ebe8aaa4397eaefa8455a8182b164a6cc1d59ad53f79943f266d99f68687f \
    --hash=sha256:cd58a314d92838f7e6f755c8a2167ead4f27e1fd5c1251fd54289569ef3495ec \
    --hash=sha256:cf72ff559a53a6a6d77bd8eefd12a17995ffa44ad86c77a5df96f533d4e6c6bb \
    --hash=sha256:def751dd08243934c884a3221156d63e15234a3155cf25978b0a668409d45eb6 \
    --hash=sha256:e7c68b6a43259ba0aab737237876e5c2c549a031ddb7abc28c7b47f22e202ded \
    --hash=sha256:ecf797d2d798cf7c838c6d98321061eb3e72a74710e6c40540f0e8087e3b499e \
    --hash=sha256:f031846580d9acccd0044efd1a90e6f4df3a6e12b4b6bd694a7bc03a89892b28 \
    --hash=sha256:fb530e4794fc8ea76a4a21ccb67dea33e5e0e60f07fc38a49e821e1eae3b71a0 \
    --hash=sha256:fe8a9eb875d430d81755472c5ba75e84acc980e4a8f6204d402849234d3017db
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
soundfile==0.13.1 \
    --hash=sha256:03267c4e493315294834a0870f31dbb3b28a95561b80b134f0bd3cf2d5f0e618 \
    --hash=sha256:1e70a05a0626524a69e9f0f4dd2ec174b4e9567f4d8b6c11d38b5c289be36ee9 \
    --hash=sha256:743f12c12c4054921e15736c6be09ac26b3b3d603aef6fd69f9dde68748f2593 \
    --hash=sha256:82dc664d19831933fe59adad199bf3945ad06d84bc111a5b4c0d3089a5b9ec33 \
    --hash=sha256:9c9e855f5a4d06ce4213f31918653ab7de0c5a8d8107cd2427e44b42df547deb \
    --hash=sha256:a23c717560da2cf4c7b5ae1142514e0fd82d6bbd9dfc93a50423447142f2c445 \
    --hash=sha256:b2c68dab1e30297317080a5b43df57e302584c49e2942defdde0acccc53f0e5b \
    --hash=sha256:c734564fab7c5ddf8e9be5bf70bab68042cd17e9c214c06e365e20d64f9a69d5
sympy==1.13.1 \
    --hash=sha256:9cebf7e04ff162015ce31c9c6c9144daa34a93bd082f54fd8f12deca4f47515f \
    --hash=sha256:db36cdc64bf61b9b24578b6f7bab1ecdd2452cf008f34faa33776680c26d66f8
tokenizers==0.21.4 \
    --hash=sha256:1340ff877ceedfa937544b7d79f5b7becf33a4cfb58f89b3b49927004ef66f78 \
    --hash=sha256:2107ad649e2cda4488d41dfd031469e9da3fcbfd6183e74e4958fa729ffbf9c6 \
    --hash=sha256:2ccc10a7c3bcefe0f242867dc914fc1226ee44321eb618cfe3019b5df3400133 \
    --hash=sha256:39b376f5a1aee67b4d29032ee85511bbd1b99007ec735f7f35c8a2eb104eade5 \
    --hash=sha256:3c1f4317576e465ac9ef0d165b247825a2a4078bcd01cba6b54b867bdf9fdd8b \
    --hash=sha256:3c73012da95afafdf235ba80047699df4384fdc481527448a078ffd00e45a7d9 \
    --hash=sha256:475d807a5c3eb72c59ad9b5fcdb254f6e17f53dfcbb9903233b0dfa9c943b597 \
    --hash=sha256:51b7eabb104f46c1c50b486520555715457ae833d5aee9ff6ae853d1130506ff \
    --hash=sha256:5e2f601a8e0cd5be5cc7506b20a79112370b9b3e9cb5f13f68ab11acd6ca7d60 \
    --hash=sha256:6c42a930bc5f4c47f4ea775c91de47d27910881902b0f20e4990ebe045a415d0 \
    --hash=sha256:714b05b2e1af1288bd1bc56ce496c4cebb64a20d158ee802887757791191e6e2 \
    --hash=sha256:c212aa4e45ec0bb5274b16b6f31dd3f1c41944025c2358faaa5782c754e84c24 \
    --hash=sha256:cc88bb34e23a54cc42713d6d98af5f1bf79c07653d24fe984d2d695ba2c922a2 \
    --hash=sha256:f23186c40395fc390d27f519679a58023f368a0aad234af145e0f39ad1212732 \
    --hash=sha256:fa23f85fbc9a02ec5c6978da172cdcbac23498c3ca9f3645c5c68740ac007880
torch==2.6.0 \
    --hash=sha256:09e06f9949e1a0518c5b09fe95295bc9661f219d9ecb6f9893e5123e10696628 \
    --hash=sha256:265f70de5fd45b864d924b64be1797f86e76c8e48a02c2a3a6fc7ec247d2226c \
    --hash=sha256:2bb8987f3bb1ef2675897034402373ddfc8f5ef0e156e2d8cfc47cacafdda4a9 \
    --hash=sha256:46763dcb051180ce1ed23d1891d9b1598e07d051ce4c9d14307029809c4d64f7 \
    --hash=sha256:4874a73507a300a5d089ceaff616a569e7bb7c613c56f37f63ec3ffac65259cf \
    --hash=sha256:510c73251bee9ba02ae1cb6c9d4ee0907b3ce6020e62784e2d7598e0cfa4d6cc \
    --hash=sha256:56eeaf2ecac90da5d9e35f7f35eb286da82673ec3c582e310a8d1631a1c02341 \
    --hash=sha256:683410f97984103148e31b38a8631acf31c3034c020c0f4d26171e7626d8317a \
    --hash=sha256:6860df13d9911ac158f4c44031609700e1eba07916fff62e21e6ffa0a9e01961 \
    --hash=sha256:7979834102cd5b7a43cc64e87f2f3b14bd0e1458f06e9f88ffa386d07c7446e1 \
    --hash=sha256:7e1448426d0ba3620408218b50aa6ada88aeae34f7a239ba5431f6c8774b1239 \
    --hash=sha256:94fc63b3b4bedd327af588696559f68c264440e2503cc9e6954019473d74ae21 \
    --hash=sha256:9a610afe216a85a8b9bc9f8365ed561535c93e804c2a317ef7fabcc5deda0989 \
    --hash=sha256:9ea955317cfcd3852b1402b62af258ce735c2edeee42ca9419b6bc889e5ae053 \
    --hash=sha256:a0d5e1b9874c1a6c25556840ab8920569a7a4137afa8a63a32cee0bc7d89bd4b \
    --hash=sha256:b789069020c5588c70d5c2158ac0aa23fd24a028f34a8b4fcb8fcb4d7efcf5fb \
    --hash=sha256:bb2c6c3e65049f081940f5ab15c9136c7de40d3f01192541c920a07c7c585b7e \
    --hash=sha256:c4f103a49830ce4c7561ef4434cc7926e5a5fe4e5eb100c19ab36ea1e2b634ab \
    --hash=sha256:ccbd0320411fe1a3b3fec7b4d3185aa7d0c52adac94480ab024b5c8f74a0bf1d \
    --hash=sha256:ff96f4038f8af9f7ec4231710ed4549da1bdebad95923953a25045dcf6fd87e2
torchaudio==2.6.0 \
    --hash=sha256:04803a969710bdb77a4ddfdb85a32fa9b9e0310dc91f7eb7e54d6083dd69bfab \
    --hash=sha256:0eda1cd876f44fc014dc04aa680db2fa355a83df5d834398db6dd5f5cd911f4c \
    --hash=sha256:0f0db5c997d031c34066d8be1c0ce7d2a1f2b6c016a92885b20b00bfeb17b753 \
    --hash=sha256:22798d5d8e37869bd5875d37f42270efbeb8ae94bda97fed40c1c5e0e1c62fa3 \
    --hash=sha256:377b177a3d683a9163e4cab5a06f0346dac9ff96fa527477338fd90fc6a2a4b6 \
    --hash=sha256:393fa74ec40d167f0170728ea21c9b5e0f830648fd02df7db2bf7e62f64245ec \
    --hash=sha256:52182f6de4e7b342d139e54b703185d428de9cce3c4cf914a9b2ab2359d192a3 \
    --hash=sha256:52f15185349c370fc1faa84e8b8b2782c007472db9d586a16bba314130b322f2 \
    --hash=sha256:6291d9507dc1d6b4ffe8843fbfb201e6c8270dd8c42ad70bb76226c0ebdcad56 \
    --hash=sha256:66f2e0bd5ab56fd81419d2f5afb74a9a70141688594646441756c8c24f424a73 \
    --hash=sha256:715aa21f6bdbd085454c313ae3a2c7cc07bf2e8cf05752f819afb5b4c57f4e6f \
    --hash=sha256:72e77055d8e742475c6dfacf59fab09b1fc94d4423e14897e188b67cad3851c6 \
    --hash=sha256:7d0e4b08c42325bf4b887de9a25c44ed882997001740e1bd7d901f65581cf1ab \
    --hash=sha256:86d6239792bf94741a41acd6fe3d549faaf0d50e7275d17d076a190bd007e2f9 \
    --hash=sha256:8c1a4d08e35a9ceaadadbff6e60bcb3442482f800369be350103dfd08b4ddf52 \
    --hash=sha256:9d8e07789452efdb8132d62afe21f2293a72805f26c2891c6c53e4e4df38ddf6 \
    --hash=sha256:b521ea9618fb4c29a6f8071628170c222291f46a48a3bf424cfeb488f54af714 \
    --hash=sha256:c12fc41241b8dfce3ccc1917f1c81a0f92f532d9917706600046f1eb21d2d765 \
    --hash=sha256:c6386bfa478afae2137715bb60f35520e3b05f5fc6d3bcc6969cf9cdfb11c09c \
    --hash=sha256:d855da878a28c2e5e6fb3d76fcddd544f4d957a320b29602cea5af2fe0ad1f3a
torchvision==0.21.0 \
    --hash=sha256:044ea420b8c6c3162a234cada8e2025b9076fa82504758cd11ec5d0f8cd9fa37 \
    --hash=sha256:084ac3f5a1f50c70d630a488d19bf62f323018eae1b1c1232f2b7047d3a7b76d \
    --hash=sha256:110d115333524d60e9e474d53c7d20f096dbd8a080232f88dddb90566f90064c \
    --hash=sha256:3891cd086c5071bda6b4ee9d266bb2ac39c998c045c2ebcd1e818b8316fb5d41 \
    --hash=sha256:49bcfad8cfe2c27dee116c45d4f866d7974bcf14a5a9fbef893635deae322f2f \
    --hash=sha256:5045a3a5f21ec3eea6962fa5f2fa2d4283f854caec25ada493fcf4aab2925467 \
    --hash=sha256:5083a5b1fec2351bf5ea9900a741d54086db75baec4b1d21e39451e00977f1b1 \
    --hash=sha256:54454923a50104c66a9ab6bd8b73a11c2fc218c964b1006d5d1fe5b442c3dcb6 \
    --hash=sha256:54815e0a56dde95cc6ec952577f67e0dc151eadd928e8d9f6a7f821d69a4a734 \
    --hash=sha256:5568c5a1ff1b2ec33127b629403adb530fab81378d9018ca4ed6508293f76e2b \
    --hash=sha256:5c22caeaae8b3c36d93459f1a5294e6f43306cff856ed243189a229331a404b4 \
    --hash=sha256:659b76c86757cb2ee4ca2db245e0740cfc3081fef46f0f1064d11adb4a8cee31 \
    --hash=sha256:669575b290ec27304569e188a960d12b907d5173f9cd65e86621d34c4e5b6c30 \
    --hash=sha256:6bdce3890fa949219de129e85e4f6d544598af3c073afe5c44e14aed15bdcbb2 \
    --hash=sha256:6eb75d41e3bbfc2f7642d0abba9383cc9ae6c5a4ca8d6b00628c225e1eaa63b3 \
    --hash=sha256:7e9e9afa150e40cd2a8f0701c43cb82a8d724f512896455c0918b987f94b84a4 \
    --hash=sha256:8c44b6924b530d0702e88ff383b65c4b34a0eaf666e8b399a73245574d546947 \
    --hash=sha256:9147f5e096a9270684e3befdee350f3cacafd48e0c54ab195f45790a9c146d67 \
    --hash=sha256:97a5814a93c793aaf0179cfc7f916024f4b63218929aee977b645633d074a49f \
    --hash=sha256:abbf1d7b9d52c00d2af4afa8dac1fb3e2356f662a4566bd98dfaaa3634f4eb34 \
    --hash=sha256:b0c0b264b89ab572888244f2e0bad5b7eaf5b696068fc0b93e96f7c3c198953f \
    --hash=sha256:b578bcad8a4083b40d34f689b19ca9f7c63e511758d806510ea03c29ac568f7b \
    --hash=sha256:e6572227228ec521618cea9ac3a368c45b7f96f1f8622dc9f1afe891c044051f \
    --hash=sha256:ff96666b94a55e802ea6796cabe788541719e6f4905fc59c380fed3517b6a64d \
    --hash=sha256:ffa2a16499508fe6798323e455f312c7c55f2a88901c9a7c0fb1efa86cf7e327
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
transformers==4.48.2 \
    --hash=sha256:493bc5b0268b116eff305edf6656367fc89cf570e7a9d5891369e04751db698a \
    --hash=sha256:dcfb73473e61f22fb3366fe2471ed2e42779ecdd49527a1bdf1937574855d516
triton==3.2.0 \
    --hash=sha256:30ceed0eff2c4a73b14eb63e052992f44bbdf175f3fad21e1ac8097a772de7ee \
    --hash=sha256:8009a1fb093ee8546495e96731336a33fb8856a38e45bb4ab6affd6dbc3ba220 \
    --hash=sha256:8d9b215efc1c26fa7eefb9a157915c92d52e000d2bf83e5f69704047e63f125c \
    --hash=sha256:b3e54983cd51875855da7c68ec05c05cf8bb08df361b1d5b69e05e40b0c9bd62 \
    --hash=sha256:e5dfa23ba84541d7c0a531dfce76d8bcd19159d50a4a8b14ad01e91734a5c1b0
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
""",
}
for _name, _text in LOCKS.items():
    (WORK_ROOT / 'locks' / f'{_name}.lock').write_text(_text.strip() + '\n')
print({name: sum('==' in line for line in text.splitlines()) for name, text in LOCKS.items()}, 'pinned packages')


In [ ]:
# @title Infrastructure: create the two locked environments (about 5-8 minutes on first run)
# Nothing is installed into this notebook's kernel, so Run all needs no restart. A pinned uv binary (checked by size
# and SHA-256) builds a uv-managed CPython 3.12.12 per environment and installs wheels only, each checked against the
# lock's SHA-256 (--require-hashes --only-binary :all:). Linux x86_64 only (Colab, Kaggle, Linux Jupyter).
import hashlib
import io
import os
import urllib.request
import zipfile

UV_VERSION = '0.12.19'
UV_URL = ('https://files.pythonhosted.org/packages/76/71/b47cec536d8ee7b09017c1d9db211dfc2e7ce5c87d0482918d8b3411ec48/'
          'uv-0.12.19-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl')
UV_SHA256 = 'a63d18a0aa38ee9f21a5406afbbaeb41303bcd954be9d6b7c1b95ac275e53958'
UV_WHEEL_BYTES = 20478749
PYTHON_REQUEST = '3.12.12'
# The kernel's own Python settings must not leak into the isolated interpreters.
UV_ENV = {k: v for k, v in os.environ.items() if k not in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN')}
UV_ENV.update(MPLBACKEND='Agg', UV_PYTHON_INSTALL_DIR=str(WORK_ROOT / 'python'))
INDEX_ARGS = {
    'lab': ['--index-url', 'https://download.pytorch.org/whl/cu126', '--extra-index-url', 'https://pypi.org/simple', '--index-strategy', 'unsafe-best-match'],
    'phi4': ['--index-url', 'https://pypi.org/simple'],
}
PROBE = """
import importlib.metadata as md, json, platform, sys
out = {'python': platform.python_version()}
for d in ('torch', 'torchvision', 'transformers', 'diffusers', 'bitsandbytes', 'accelerate', 'peft', 'huggingface-hub', 'safetensors', 'numpy', 'pillow', 'matplotlib'):
    try:
        out[d] = md.version(d)
    except md.PackageNotFoundError:
        pass
import torch
out['cuda_available'] = torch.cuda.is_available()
out['torch_cuda'] = torch.version.cuda
out['device'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print(json.dumps(out))
"""
UV = WORK_ROOT / 'bin' / 'uv'
if not (UV.exists() and (UV.parent / '.uv_sha256').exists() and (UV.parent / '.uv_sha256').read_text() == UV_SHA256):
    if platform.system() != 'Linux' or platform.machine() != 'x86_64':
        raise RuntimeError(f'this notebook runs on Linux x86_64 only (Colab, Kaggle, Linux Jupyter); found {platform.system()} {platform.machine()}')
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=120) as response:
                wheel = response.read(UV_WHEEL_BYTES + 1)
            break
        except OSError:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    if len(wheel) != UV_WHEEL_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('CONTRACT FAILURE: the uv wheel size or SHA-256 differs from the pin; re-run the cell, never skip the check')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
        UV.parent.mkdir(parents=True, exist_ok=True)
        UV.write_bytes(archive.read(member))
    UV.chmod(0o700)
    (UV.parent / '.uv_sha256').write_text(UV_SHA256)
ENV_PYTHON, ENVIRONMENTS = {}, {}
for name in ('lab', 'phi4'):
    lock = WORK_ROOT / 'locks' / f'{name}.lock'
    root = WORK_ROOT / 'envs' / name
    python = root / 'bin' / 'python'
    digest = hashlib.sha256(lock.read_bytes()).hexdigest()
    marker = root / '.lock_sha256'
    started = time.time()
    if not (python.exists() and marker.exists() and marker.read_text() == f'{digest} {PYTHON_REQUEST}'):
        shutil.rmtree(root, ignore_errors=True)
        subprocess.run([str(UV), 'venv', '--quiet', '--managed-python', '--python', PYTHON_REQUEST, str(root)], env=UV_ENV, check=True)
        subprocess.run([str(UV), 'pip', 'install', '--quiet', '--python', str(python), '--require-hashes', '--only-binary', ':all:', '--no-deps', '--no-cache',
                        '--link-mode', 'copy', '-r', str(lock), *INDEX_ARGS[name]], env=UV_ENV, check=True)
        marker.write_text(f'{digest} {PYTHON_REQUEST}')
    probe = json.loads(subprocess.run([str(python), '-c', PROBE], capture_output=True, text=True, check=True, env=UV_ENV).stdout)
    if not probe['cuda_available']:
        raise RuntimeError(f"the {name} environment cannot see the GPU (torch {probe['torch']}, CUDA {probe['torch_cuda']}, driver {RUNTIME['driver']}); see Troubleshooting")
    ENVIRONMENTS[name] = {**probe, 'lock_sha256': digest, 'uv': UV_VERSION, 'uv_wheel_sha256': UV_SHA256, 'install_seconds': round(time.time() - started, 1)}
    ENV_PYTHON[name] = str(python)
    print(f"{name}: Python {probe['python']}, torch {probe['torch']} (CUDA {probe['torch_cuda']}), transformers {probe.get('transformers')}, "
          f"diffusers {probe.get('diffusers', '-')}, bitsandbytes {probe.get('bitsandbytes')} on {probe['device']} ({ENVIRONMENTS[name]['install_seconds']} s)")
(WORK / 'state' / 'environments.json').write_text(json.dumps(ENVIRONMENTS, indent=2))


### The shared core

The next cells save `sdi_core.py`, the small library every stage imports: acquisition and archive safety, image validation, duplicate grouping, the split, prompt compilation, candidate eligibility, preprocessing and augmentation views, the matched slot schedule, metrics, the bootstrap, the frozen record and the classifier artifact. It uses only the Python standard library, NumPy and Pillow, so the same code runs in both environments. Each section is one cell; later sections of this notebook point back to the parts they use. You may run these cells without reading them now.


In [ ]:
%%writefile work/sdi_capstone/sdi_core.py
"""Shared helpers for the Bosch synthetic-defect capstone notebook (standard library + NumPy + Pillow only).

The notebook writes this file into its working directory and every stage script imports it, so acquisition,
validation, grouping, splitting, prompt compilation, slot schedules, metrics, the bootstrap and the classifier
artifact are implemented once and used identically by every stage. Nothing here imports a model library.
"""
# ruff: noqa: E501
from __future__ import annotations

import csv
import hashlib
import io
import json
import os
import re
import shutil
import stat
import time
import urllib.error
import urllib.request
import zipfile
from collections.abc import Callable, Iterable, Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageDraw, ImageEnhance, ImageOps


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Source, constants and errors

CORE_VERSION = "sdi-core/1"
SOURCE_REPOSITORY = "https://github.com/boschresearch/The-Surface-Defect-Inspection-Dataset"
SOURCE_COMMIT = "c6e0afe66e9dbd9d99326c986ea884ddf2aad9f8"
ARCHIVE_NAME = "SDI_DATASET_v1.zip"
ARCHIVE_URL = f"https://media.githubusercontent.com/media/boschresearch/The-Surface-Defect-Inspection-Dataset/{SOURCE_COMMIT}/{ARCHIVE_NAME}"
ARCHIVE_BYTES = 1_647_097_112
ARCHIVE_SHA256 = "d33ea340151cdf909f3807a37e00d335c66eb8960b0c2eb568fd9fc75d96fa7c"
DATASET_LICENSE = "CC BY-SA 4.0"
DATASET_NOTES_URL = f"{SOURCE_REPOSITORY}/blob/{SOURCE_COMMIT}/DT_GAN_BMVC_SDI_dataset.pdf"
# Counts published in the dataset notes: expectations, not measurements. The notebook measures the archive.
DOCUMENTED_COUNTS = {
    "A": {"normal": 6250, "scratches": 340, "spots": 108},
    "B": {"normal": 6250, "scratches": 167, "spots": 670},
    "C": {"normal": 6250, "scratches": 121, "spots": 258},
}
# Extraction ceilings set from the inspected archive (18,841 entries, 1,644,493,674 expanded bytes,
# largest member 232,910 bytes) with a small margin; anything larger is refused rather than extracted.
MAX_ARCHIVE_ENTRIES = 19_000
MAX_EXPANDED_BYTES = 1_700_000_000
MAX_MEMBER_BYTES = 4_000_000
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png")
MIN_IMAGE_SIDE = 64
MAX_IMAGE_SIDE = 4096
SIGNATURE_SIDE = 32
# Perceptual grouping threshold: Pearson correlation of 32x32 grayscale thumbnails. In product A the exact-pixel
# duplicates score >= 0.99 and the 18 pairs between 0.98 and 0.99 are consecutive capture frames, but the distribution
# has no gap here: hundreds of pairs sit between 0.97 and 0.98 and ~32,000 exceed 0.95 because the surfaces share
# one texture. 0.98 is a declared cut, not a natural boundary; the data stage reports how many pairs sit just below
# it and how many similar pairs cross partitions.
NEAR_DUPLICATE_THRESHOLD = 0.98
AUDIT_SIMILARITY = 0.95
SPLIT_VERSION = "sdi-a-grouped-602020-v1"
SPLIT_FRACTIONS = (("train", 0.6), ("val", 0.2), ("test", 0.2))
BUDGET_BY_ROLE = (128, 64, 32)  # normal, more frequent defect, less frequent defect
MIN_HELDOUT_PER_CLASS = 5
EXEMPLARS_PER_CLASS = 3
NORMAL = "normal"
CLASS_NAME_RE = re.compile(r"^[a-z][a-z0-9_]{0,31}$")
IMAGE_ID_RE = re.compile(r"^[A-Za-z0-9._-]{1,128}$")


class ContractError(ValueError):
    """A validation failure that names the broken contract and the corrective action."""


def run_main(entry: Callable[..., None], *args: Any) -> None:
    """Run a stage entry point; a ContractError is printed as one actionable line and exits with status 2."""
    try:
        entry(*args)
    except ContractError as exc:
        print(f"\nCONTRACT FAILURE: {exc}", flush=True)
        raise SystemExit(2) from None


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: str | Path, chunk: int = 8 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()


def canonical_json(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=True)


def write_json(path: str | Path, obj: Any) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(tmp, path)
    return path


def read_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_jsonl(path: str | Path, rows: Iterable[Mapping[str, Any]]) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, sort_keys=True) + "\n")
    os.replace(tmp, path)
    return path


def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Acquisition and archive safety


def fetch_archive(
    url: str,
    dest: str | Path,
    *,
    expected_bytes: int,
    expected_sha256: str,
    timeout: float = 60.0,
    retries: int = 4,
    chunk: int = 8 << 20,
    log: Callable[[str], None] = print,
    opener: Callable[..., Any] | None = None,
) -> dict[str, Any]:
    """Stream `url` to `dest` with timeouts, bounded resumable retries and atomic completion, then verify size and
    SHA-256 before the file is given its final name. A mismatch is refused; nothing substitutes another dataset."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and dest.stat().st_size == expected_bytes and sha256_file(dest) == expected_sha256:
        return {"reused": True, "downloaded_bytes": 0, "attempts": 0, "sha256": expected_sha256, "bytes": expected_bytes}
    part = dest.with_name(dest.name + ".part")
    urlopen = opener or urllib.request.urlopen
    downloaded, attempts, complete = 0, 0, False
    started = time.time()
    for attempt in range(1, retries + 2):
        attempts = attempt
        have = part.stat().st_size if part.exists() else 0
        if have > expected_bytes:
            part.unlink()
            have = 0
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with urlopen(urllib.request.Request(url, headers=headers), timeout=timeout) as response:
                resumed = bool(have) and getattr(response, "status", 200) == 206
                mode = "ab" if resumed else "wb"
                have = have if resumed else 0
                next_report = have + (256 << 20)
                with open(part, mode) as handle:
                    while block := response.read(chunk):
                        handle.write(block)
                        downloaded += len(block)
                        have += len(block)
                        if have >= next_report:
                            log(f"  {have / 1e9:.2f} / {expected_bytes / 1e9:.2f} GB")
                            next_report += 256 << 20
        except (urllib.error.URLError, TimeoutError, ConnectionError, OSError) as exc:
            wait = min(2**attempt, 30)
            log(f"  attempt {attempt} interrupted ({type(exc).__name__}: {exc}); retrying in {wait} s")
            time.sleep(wait)
            continue
        size = part.stat().st_size
        if size == expected_bytes:
            complete = True
            break
        if size < 1024 and expected_bytes > 1024:
            text = part.read_bytes()[:200].decode("utf-8", "replace")
            if "git-lfs" in text:
                part.unlink()
                raise ContractError(f"{url} returned a Git LFS pointer, not the archive; the LFS media host must serve the object")
        log(f"  attempt {attempt} ended at {size} of {expected_bytes} bytes; resuming")
    if not complete:
        raise ContractError(
            f"download of {url} did not complete after {attempts} attempts. A timeout is an access problem, not "
            "evidence of a corrupt dataset: re-run this cell (completed bytes are kept and resumed)."
        )
    digest = sha256_file(part)
    if digest != expected_sha256:
        part.unlink()
        raise ContractError(f"archive SHA-256 {digest} differs from the pinned {expected_sha256}; refusing to extract it")
    os.replace(part, dest)
    return {"reused": False, "downloaded_bytes": downloaded, "attempts": attempts, "sha256": digest, "bytes": expected_bytes, "seconds": round(time.time() - started, 1)}


def check_zip_members(
    archive: zipfile.ZipFile,
    *,
    allowed_suffixes: Sequence[str],
    allowed_names: Sequence[str] = (),
    max_entries: int = MAX_ARCHIVE_ENTRIES,
    max_total: int = MAX_EXPANDED_BYTES,
    max_member: int = MAX_MEMBER_BYTES,
) -> list[zipfile.ZipInfo]:
    """Refuse absolute paths, traversal, symlinks, encrypted or duplicate entries, unsupported file types and
    archives whose declared expanded size exceeds the ceiling. Returns the file entries to extract."""
    infos = archive.infolist()
    if len(infos) > max_entries:
        raise ContractError(f"archive has {len(infos)} entries; the ceiling is {max_entries}")
    seen: set[str] = set()
    files: list[zipfile.ZipInfo] = []
    total = 0
    for info in infos:
        name = info.filename
        if stat.S_ISLNK((info.external_attr >> 16) & 0xFFFF):
            raise ContractError(f"symlink entry refused: {name!r}")
        if name.startswith(("/", "\\")) or re.match(r"^[A-Za-z]:", name) or "\\" in name or "\x00" in name:
            raise ContractError(f"absolute or non-POSIX path refused: {name!r}")
        if any(part == ".." for part in name.split("/")):
            raise ContractError(f"path traversal refused: {name!r}")
        if name in seen:
            raise ContractError(f"duplicate archive entry refused: {name!r}")
        seen.add(name)
        if info.is_dir():
            continue
        if info.flag_bits & 0x1:
            raise ContractError(f"encrypted entry refused: {name!r}")
        base = name.rsplit("/", 1)[-1]
        if base not in allowed_names and Path(base).suffix.lower() not in allowed_suffixes:
            raise ContractError(f"unsupported file refused: {name!r}; allowed: {sorted(allowed_suffixes) + sorted(allowed_names)} (remove OS metadata such as __MACOSX/ or .DS_Store and re-zip)")
        if info.file_size > max_member:
            raise ContractError(f"entry {name!r} expands to {info.file_size} bytes; the per-file ceiling is {max_member}")
        total += info.file_size
        if total > max_total:
            raise ContractError(f"archive expands beyond the {max_total}-byte ceiling")
        files.append(info)
    return files


def safe_extract(zip_path: str | Path, root: str | Path, **limits: Any) -> dict[str, Any]:
    """Extract into `<root>.partial` with every member checked, count the bytes actually written against the
    ceilings, and rename to `root` only when everything succeeded."""
    root = Path(root)
    staging = root.with_name(root.name + ".partial")
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    base = staging.resolve()
    max_total = limits.get("max_total", MAX_EXPANDED_BYTES)
    written = 0
    with zipfile.ZipFile(zip_path) as archive:
        files = check_zip_members(archive, **limits)
        for info in files:
            target = (staging / info.filename).resolve()
            if not target.is_relative_to(base):
                raise ContractError(f"entry {info.filename!r} resolves outside the extraction root")
            target.parent.mkdir(parents=True, exist_ok=True)
            count = 0
            with archive.open(info) as source, open(target, "wb") as sink:
                while block := source.read(1 << 20):
                    count += len(block)
                    if count > info.file_size or written + count > max_total:
                        raise ContractError(f"entry {info.filename!r} expanded beyond its declared size")
                    sink.write(block)
            written += count
    shutil.rmtree(root, ignore_errors=True)
    os.replace(staging, root)
    return {"files": len(files), "bytes": written}


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Inventory and image validation

BOSCH_OK = re.compile(r"^SDI_DATASET_v1/([ABC])_ok/([ABC])_ok_(\d{5})\.jpg$")
BOSCH_NOK = re.compile(r"^SDI_DATASET_v1/([ABC])_nok/(train|val|test)/(scratches|spots)/([ABC])_(sc|sp)_(train|val|test)_(\d{5})\.jpg$")


def parse_bosch_path(relpath: str) -> dict[str, str]:
    """Map an archive path to product, label and the source's own split assignment (defects only)."""
    match = BOSCH_OK.match(relpath)
    if match and match.group(1) == match.group(2):
        return {"product": match.group(1), "label": NORMAL, "official_split": ""}
    match = BOSCH_NOK.match(relpath)
    if match:
        product, split, folder, product2, abbrev, split2, _ = match.groups()
        if product == product2 and split == split2 and abbrev == folder[:2]:
            return {"product": product, "label": folder, "official_split": split}
    raise ContractError(f"unexpected archive path {relpath!r}; the documented layout is <P>_ok/*.jpg and <P>_nok/<split>/<class>/*.jpg")


def load_gray(path: str | Path) -> tuple[Image.Image, str]:
    """Decode an image and return it as 8-bit grayscale plus its original mode (conversions are reported)."""
    with Image.open(path) as image:
        image.load()
        mode = image.mode
        if mode == "L":
            return image.copy(), mode
        if mode in ("RGB", "RGBA", "P", "LA", "I;16", "I", "F"):
            if mode in ("I;16", "I", "F"):
                array = np.asarray(image, dtype=np.float64)
                if not np.isfinite(array).all():
                    raise ContractError(f"{path}: non-finite pixel values")
                return Image.fromarray(np.clip(array, 0, 255).astype(np.uint8)), mode
            return image.convert("L"), mode
    raise ContractError(f"{path}: unsupported image mode {mode!r}")


def load_verified_gray(path: str | Path, row: Mapping[str, Any]) -> Image.Image:
    """Load a manifest-listed image only if its file bytes and decoded pixels still match the digests recorded in the
    frozen split manifest. Every stage that consumes a listed image goes through here, so a changed or substituted
    file is refused before it can be scored under its old label."""
    data = Path(path).read_bytes()
    if sha256_bytes(data) != row["file_sha256"]:
        raise ContractError(f"{row['image_id']}: the file at {path} no longer matches the SHA-256 recorded in split_manifest.csv; restore the original file or start a new experiment from Section 2")
    image, _ = load_gray(io.BytesIO(data))
    if pixel_digest(np.asarray(image)) != row["pixel_sha256"]:
        raise ContractError(f"{row['image_id']}: decoded pixels differ from split_manifest.csv; restore the original file or start a new experiment from Section 2")
    return image


def signature(image: Image.Image) -> np.ndarray:
    """32x32 grayscale thumbnail, zero mean, unit norm: the dot product of two signatures is their Pearson correlation."""
    thumb = np.asarray(image.convert("L").resize((SIGNATURE_SIDE, SIGNATURE_SIDE), Image.BILINEAR), dtype=np.float32).ravel()
    thumb = thumb - thumb.mean()
    norm = float(np.linalg.norm(thumb))
    return thumb / norm if norm > 1e-6 else np.zeros_like(thumb)


def inspect_image(path: str | Path) -> dict[str, Any]:
    """File digest, decodability, size, mode, decoded-pixel digest and signature. Never raises for a bad image:
    failures are recorded so the inventory can report them."""
    path = Path(path)
    record: dict[str, Any] = {"bytes": path.stat().st_size, "file_sha256": sha256_file(path)}
    try:
        image, mode = load_gray(path)
        array = np.asarray(image)
        if not (min(image.size) >= MIN_IMAGE_SIDE and max(image.size) <= MAX_IMAGE_SIDE):
            raise ContractError(f"side {image.size} outside {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE}")
        record.update(
            decode_ok=True,
            width=image.size[0],
            height=image.size[1],
            mode=mode,
            pixel_sha256=sha256_bytes(f"{array.shape}|".encode() + array.tobytes()),
            signature=signature(image),
            error="",
        )
    except Exception as exc:  # corrupt files are data to report, not a crash
        record.update(decode_ok=False, width=0, height=0, mode="", pixel_sha256="", signature=None, error=f"{type(exc).__name__}: {exc}"[:200])
    return record


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Duplicate grouping, split and budget


class _UnionFind:
    def __init__(self, n: int) -> None:
        self.parent = list(range(n))

    def find(self, i: int) -> int:
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]
            i = self.parent[i]
        return i

    def union(self, a: int, b: int) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)


def near_duplicate_pairs(signatures: np.ndarray, threshold: float = NEAR_DUPLICATE_THRESHOLD, block: int = 2048) -> list[tuple[int, int, float]]:
    """All pairs i < j whose signature correlation is >= threshold (blockwise, so memory stays bounded)."""
    pairs: list[tuple[int, int, float]] = []
    n = len(signatures)
    for start in range(0, n, block):
        sims = signatures[start : start + block] @ signatures.T
        rows, cols = np.nonzero(sims >= threshold)
        for r, c in zip(rows.tolist(), cols.tolist(), strict=True):
            i = start + r
            if i < c:
                pairs.append((i, c, round(float(sims[r, c]), 5)))
    return pairs


def max_similarity(queries: np.ndarray, references: np.ndarray, block: int = 2048) -> tuple[np.ndarray, np.ndarray]:
    """For each query signature, the highest correlation with any reference signature and its index."""
    if len(references) == 0 or len(queries) == 0:
        return np.full(len(queries), -1.0, dtype=np.float32), np.full(len(queries), -1)
    best = np.empty(len(queries), dtype=np.float32)
    where = np.empty(len(queries), dtype=np.int64)
    for start in range(0, len(queries), block):
        sims = queries[start : start + block] @ references.T
        where[start : start + block] = sims.argmax(axis=1)
        best[start : start + block] = sims.max(axis=1)
    return best, where


def assign_groups(records: Sequence[Mapping[str, Any]], signatures: np.ndarray, *, threshold: float = NEAR_DUPLICATE_THRESHOLD) -> dict[str, Any]:
    """Union exact decoded-pixel duplicates, perceptual near duplicates and any user-declared groups. Returns the
    group id of every record plus the evidence. Groups approximate shared origin; they do not prove independent
    physical specimens."""
    n = len(records)
    uf = _UnionFind(n)
    by_pixels: dict[str, int] = {}
    exact = 0
    for i, rec in enumerate(records):
        key = rec["pixel_sha256"]
        if key in by_pixels:
            uf.union(by_pixels[key], i)
            exact += 1
        else:
            by_pixels[key] = i
    declared: dict[str, int] = {}
    for i, rec in enumerate(records):
        user = rec.get("declared_group") or ""
        if user:
            if user in declared:
                uf.union(declared[user], i)
            else:
                declared[user] = i
    pairs = near_duplicate_pairs(signatures, threshold)
    for i, j, _ in pairs:
        uf.union(i, j)
    members: dict[int, list[int]] = {}
    for i in range(n):
        members.setdefault(uf.find(i), []).append(i)
    group_of = [""] * n
    for idx in members.values():
        gid = "g_" + min(records[i]["pixel_sha256"] for i in idx)[:16]
        for i in idx:
            group_of[i] = gid
    return {"group_of": group_of, "exact_duplicate_links": exact, "near_duplicate_pairs": pairs, "groups": len(members)}


def _rank(*parts: Any) -> str:
    return sha256_bytes("|".join(str(p) for p in parts).encode())


def assign_split(groups: Mapping[str, Mapping[str, Any]], *, version: str = SPLIT_VERSION, fractions: Sequence[tuple[str, float]] = SPLIT_FRACTIONS) -> dict[str, str]:
    """Deterministic, approximately class-stratified group assignment: within each class, groups are visited in a
    hash order and each goes to the partition with the largest relative shortfall against its target."""
    out: dict[str, str] = {}
    labels = sorted({g["label"] for g in groups.values()})
    for label in labels:
        gids = sorted((gid for gid, g in groups.items() if g["label"] == label), key=lambda gid: _rank(version, "split", gid))
        total = sum(groups[gid]["size"] for gid in gids)
        counts = {name: 0 for name, _ in fractions}
        for gid in gids:
            name = max(fractions, key=lambda item: ((item[1] * total - counts[item[0]]) / (item[1] * total), -[f[0] for f in fractions].index(item[0])))[0]
            out[gid] = name
            counts[name] += groups[gid]["size"]
    return out


def check_group_leakage(rows: Sequence[Mapping[str, Any]]) -> None:
    """Every group must sit in exactly one partition; raises naming the groups that cross."""
    seen: dict[str, set[str]] = {}
    for row in rows:
        if row["split"] in ("train", "val", "test"):
            seen.setdefault(row["group_id"], set()).add(row["split"])
    crossing = sorted(gid for gid, splits in seen.items() if len(splits) > 1)
    if crossing:
        raise ContractError(f"{len(crossing)} group(s) cross partitions, e.g. {crossing[:5]}; the split is invalid")


def class_order_from_counts(counts: Mapping[str, int]) -> list[str]:
    """normal first, then defect classes by descending count (ties by name)."""
    if NORMAL not in counts:
        raise ContractError(f"a class named {NORMAL!r} is required; found {sorted(counts)}")
    defects = sorted((c for c in counts if c != NORMAL), key=lambda c: (-counts[c], c))
    if len(defects) != 2:
        raise ContractError(f"exactly two defect classes are supported besides {NORMAL!r}; found {defects}")
    return [NORMAL, *defects]


def build_split_rows(records: Sequence[dict[str, Any]], *, version: str = SPLIT_VERSION, budgets: Sequence[int] = BUDGET_BY_ROLE) -> dict[str, Any]:
    """Exclude undecodable images and label-conflict groups, assign groups to partitions, check leakage, then
    select the fixed real-data training budget (one image per group) and the captioning exemplars."""
    groups: dict[str, dict[str, Any]] = {}
    for rec in records:
        if not rec["decode_ok"]:
            rec["split"], rec["exclusion_reason"] = "excluded", "decode-failure"
            continue
        g = groups.setdefault(rec["group_id"], {"labels": set(), "size": 0, "members": []})
        g["labels"].add(rec["label"])
        g["size"] += 1
        g["members"].append(rec)
    conflicts = sorted(gid for gid, g in groups.items() if len(g["labels"]) > 1)
    usable = {}
    for gid, g in groups.items():
        if gid in conflicts:
            for rec in g["members"]:
                rec["split"], rec["exclusion_reason"] = "excluded", "label-conflict-duplicate"
        else:
            usable[gid] = {"label": next(iter(g["labels"])), "size": g["size"], "members": g["members"]}
    assignment = assign_split(usable, version=version)
    for gid, g in usable.items():
        for rec in g["members"]:
            rec["split"], rec["exclusion_reason"] = assignment[gid], ""
    check_group_leakage(records)
    counts = {}
    for rec in records:
        if rec["split"] != "excluded":
            counts[rec["label"]] = counts.get(rec["label"], 0) + 1
    order = class_order_from_counts(counts)
    budget = dict(zip(order, budgets, strict=True))
    for rec in records:
        rec["selected"] = False
        rec["exemplar"] = False
        rec["budget_rank"] = -1
    shortfalls = []
    for label in order:
        train_groups = sorted((gid for gid, g in usable.items() if g["label"] == label and assignment[gid] == "train"), key=lambda gid: _rank(version, "budget", gid))
        if len(train_groups) < budget[label]:
            shortfalls.append(f"{label}: {len(train_groups)} training groups for a budget of {budget[label]}")
            continue
        for rank, gid in enumerate(train_groups[: budget[label]]):
            rep = min(usable[gid]["members"], key=lambda r: _rank(version, "representative", r["file_sha256"]))
            rep["selected"], rep["budget_rank"], rep["exemplar"] = True, rank, rank < EXEMPLARS_PER_CLASS
        for split in ("val", "test"):
            n = sum(1 for r in records if r["label"] == label and r["split"] == split)
            if n < MIN_HELDOUT_PER_CLASS:
                shortfalls.append(f"{label}: {n} {split} images (minimum {MIN_HELDOUT_PER_CLASS})")
    if shortfalls:
        raise ContractError("insufficient independent examples under the grouped split: " + "; ".join(shortfalls) + ". Revise the versioned split design explicitly; held-out images are never borrowed and grouping is never relaxed silently.")
    return {"class_order": order, "budget": budget, "label_conflict_groups": conflicts, "counts": counts}


SPLIT_COLUMNS = ("image_id", "relpath", "product", "label", "official_split", "group_id", "split", "selected", "budget_rank", "exemplar", "exclusion_reason", "file_sha256", "pixel_sha256", "width", "height", "mode")


def write_split_manifest(path: str | Path, records: Sequence[Mapping[str, Any]]) -> str:
    """Deterministic CSV (sorted by image id, fixed columns, LF endings); returns its SHA-256."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(SPLIT_COLUMNS)
        for rec in sorted(records, key=lambda r: r["image_id"]):
            writer.writerow([int(rec[c]) if isinstance(rec[c], bool) else rec[c] for c in SPLIT_COLUMNS])
    return sha256_file(path)


def read_split_manifest(path: str | Path) -> list[dict[str, Any]]:
    with open(path, encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        row["selected"] = row["selected"] == "1"
        row["exemplar"] = row["exemplar"] == "1"
        row["budget_rank"] = int(row["budget_rank"])
    return rows


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Bring-your-own-data manifest

BYOD_MANIFEST = "manifest.csv"
BYOD_REQUIRED = ("image_id", "path", "class")


def load_byod_manifest(root: str | Path) -> list[dict[str, Any]]:
    """Validate a BYOD folder: manifest.csv with image_id, path, class and an optional group column; safe relative
    paths to existing images; exactly one `normal` class and two defect classes; groups that never span classes."""
    root = Path(root)
    manifest = root / BYOD_MANIFEST
    if not manifest.is_file():
        candidates = sorted(root.glob(f"*/{BYOD_MANIFEST}"))
        if len(candidates) == 1:
            root, manifest = candidates[0].parent, candidates[0]
        else:
            raise ContractError(f"{BYOD_MANIFEST} not found at the top of the BYOD folder (or one level down)")
    with open(manifest, encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        header = [h.strip() for h in (reader.fieldnames or [])]
        rows = list(reader)
    if len(set(header)) != len(header):
        raise ContractError(f"{BYOD_MANIFEST} has duplicate column names: {header}")
    missing = [c for c in BYOD_REQUIRED if c not in header]
    if missing:
        raise ContractError(f"{BYOD_MANIFEST} is missing required column(s) {missing}; expected {list(BYOD_REQUIRED)} plus optional 'group'")
    out, ids, group_labels = [], set(), {}
    base = root.resolve()
    for n, raw in enumerate(rows, start=2):
        row = {k.strip(): (v or "").strip() for k, v in raw.items() if k}
        image_id, rel, label, group = row["image_id"], row["path"], row["class"], row.get("group", "")
        if not IMAGE_ID_RE.match(image_id):
            raise ContractError(f"line {n}: image_id {image_id!r} must match {IMAGE_ID_RE.pattern}")
        if image_id in ids:
            raise ContractError(f"line {n}: duplicate image_id {image_id!r}")
        ids.add(image_id)
        if not CLASS_NAME_RE.match(label):
            raise ContractError(f"line {n}: class {label!r} must match {CLASS_NAME_RE.pattern}")
        if rel.startswith(("/", "\\")) or ".." in rel.split("/") or "\\" in rel:
            raise ContractError(f"line {n}: path {rel!r} must be relative to the manifest without '..'")
        target = (root / rel).resolve()
        if not target.is_relative_to(base) or not target.is_file():
            raise ContractError(f"line {n}: image {rel!r} not found inside the BYOD folder")
        if target.suffix.lower() not in IMAGE_SUFFIXES:
            raise ContractError(f"line {n}: {rel!r} is not one of {IMAGE_SUFFIXES}")
        if group:
            if not IMAGE_ID_RE.match(group):
                raise ContractError(f"line {n}: group {group!r} must match {IMAGE_ID_RE.pattern}")
            if group_labels.setdefault(group, label) != label:
                raise ContractError(f"line {n}: group {group!r} spans classes {group_labels[group]!r} and {label!r}; a specimen/session group must carry one label")
        out.append({"image_id": image_id, "path": target, "relpath": rel, "label": label, "declared_group": group, "product": "byod", "official_split": ""})
    if not out:
        raise ContractError(f"{BYOD_MANIFEST} has no rows")
    counts: dict[str, int] = {}
    for row in out:
        counts[row["label"]] = counts.get(row["label"], 0) + 1
    class_order_from_counts(counts)
    return out


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Descriptions to bounded prompts

VOCABULARY = {
    "texture": ("horizontal lines", "horizontal stripes", "vertical lines", "parallel lines", "striped", "grooves", "grainy", "granular", "fine texture", "rough", "smooth", "brushed", "uniform", "noisy"),
    "illumination": ("even lighting", "uniform lighting", "low contrast", "high contrast", "bright", "dark", "shadow", "reflective", "glossy", "matte"),
    "shape": ("thin", "thick", "long", "short", "straight", "curved", "diagonal", "linear", "elongated", "small", "large", "round", "circular", "oval", "irregular", "blob", "dot", "speck", "cluster", "streak", "line", "patch"),
}
TERM_LIMITS = {"texture": 2, "illumination": 1, "shape": 3}
MAX_DESCRIPTION_CHARS = 1500
MAX_PROMPT_CHARS = 300
PROMPT_BASE = "close-up grayscale industrial inspection photograph of a flat manufactured surface"
SINGULAR = {"scratches": "scratch", "spots": "spot"}
PROMPT_CHARSET = re.compile(r"^[a-z0-9 ,.\-]+$")


def extract_terms(text: str) -> dict[str, list[str]]:
    """Whitelisted descriptor terms found in a description, in vocabulary order, capped per category. Only these
    terms can reach a prompt: model output is treated as evidence to match, never as text to paste or execute."""
    lowered = text.lower()
    found: dict[str, list[str]] = {}
    for category, terms in VOCABULARY.items():
        hits = [t for t in terms if re.search(rf"(?<![a-z]){re.escape(t)}(?![a-z])", lowered)]
        found[category] = hits[: TERM_LIMITS[category]]
    return found


def description_problems(text: Any, *, need: Sequence[str]) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return ["empty description"]
    problems = []
    if len(text) > MAX_DESCRIPTION_CHARS:
        problems.append(f"longer than {MAX_DESCRIPTION_CHARS} characters")
    terms = extract_terms(text)
    if not any(terms[c] for c in need):
        problems.append(f"no whitelisted {'/'.join(need)} term")
    return problems


def defect_word(label: str) -> str:
    return SINGULAR.get(label, label.replace("_", " "))


def compile_prompts(caption_records: Sequence[Mapping[str, Any]], class_order: Sequence[str]) -> list[dict[str, Any]]:
    """One bounded prompt per (defect class, exemplar k): background terms from normal exemplar k, mark terms from
    defect exemplar k. A description that fails validation falls back to the neutral class template, recorded as
    such and never presented as Phi-4-guided."""
    by_class: dict[str, list[Mapping[str, Any]]] = {}
    for rec in sorted(caption_records, key=lambda r: (r["label"], r["exemplar_rank"])):
        by_class.setdefault(rec["label"], []).append(rec)
    normals = by_class.get(class_order[0], [])
    prompts = []
    for label in class_order[1:]:
        defects = by_class.get(label, [])
        if not defects:
            raise ContractError(f"no caption records for class {label!r}")
        for k, defect in enumerate(defects):
            normal = normals[k] if k < len(normals) else None
            defect_text, normal_text = defect.get("response", ""), (normal or {}).get("response", "")
            problems = description_problems(defect_text, need=("shape", "texture"))
            background = extract_terms(normal_text) if normal and not description_problems(normal_text, need=("texture", "illumination")) else {"texture": [], "illumination": []}
            mark = extract_terms(defect_text) if not problems else {"texture": [], "illumination": [], "shape": []}
            texture = background["texture"] or mark["texture"]
            illumination = background["illumination"] or mark["illumination"]
            if problems:
                text = f"{PROMPT_BASE}, showing one {defect_word(label)} defect, sharp focus, no text"
            else:
                parts = [PROMPT_BASE, *texture, *illumination]
                shape = " ".join(mark["shape"])
                parts.append(f"showing one {defect_word(label)} defect" + (f" as a {shape} mark" if shape else ""))
                parts += ["sharp focus", "no text"]
                text = ", ".join(parts)
            if len(text) > MAX_PROMPT_CHARS or not PROMPT_CHARSET.match(text):
                raise ContractError(f"compiled prompt violates the bound: {text!r}")
            prompts.append({
                "prompt_id": f"{label}-{k}",
                "label": label,
                "exemplar_rank": k,
                "text": text,
                "phi4_guided": not problems,
                "fallback_reason": "; ".join(problems),
                "terms": {"texture": texture, "illumination": illumination, "shape": mark["shape"]},
                "source_image_ids": [defect["image_id"]] + ([normal["image_id"]] if normal else []),
            })
    return prompts


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Synthetic candidate eligibility

GENERATION_CANDIDATES = 32
MIN_USABLE_CANDIDATES = 24


def pixel_digest(array: np.ndarray) -> str:
    return sha256_bytes(f"{array.shape}|".encode() + np.ascontiguousarray(array).tobytes())


def candidate_eligibility(pixels: np.ndarray | None, *, side: int, seen: set[str], real: set[str]) -> tuple[str, str, str]:
    """Default eligibility checks only decoding, dimensions, finite values and exact duplicates: never classifier
    confidence, CLIP score, test performance or model approval. Returns (status, reason, digest of the uint8 image)."""
    if pixels is None:
        return "rejected", "decode-failure", ""
    if pixels.shape[:2] != (side, side) or pixels.ndim not in (2, 3):
        return "rejected", f"dimensions {tuple(pixels.shape)}", ""
    if not np.isfinite(pixels).all():
        return "rejected", "non-finite pixel values", ""
    gray = np.asarray(Image.fromarray(to_uint8(pixels)).convert("L"))
    digest = pixel_digest(gray)
    if digest in real:
        return "rejected", "exact duplicate of a real image", digest
    if digest in seen:
        return "rejected", "exact duplicate of an earlier candidate", digest
    return "eligible", "", digest


def to_uint8(pixels: np.ndarray) -> np.ndarray:
    """[-1, 1] float (VAE output) or already-uint8 pixels to uint8."""
    if pixels.dtype == np.uint8:
        return pixels
    return np.clip((pixels.astype(np.float32) + 1.0) * 127.5, 0, 255).round().astype(np.uint8)


def generation_status(rows: Sequence[Mapping[str, Any]], defect_classes: Sequence[str], *, expected: int = GENERATION_CANDIDATES, minimum: int = MIN_USABLE_CANDIDATES) -> dict[str, Any]:
    usable = {c: sum(1 for r in rows if r["intended_label"] == c and r["status"] == "eligible") for c in defect_classes}
    attempted = {c: sum(1 for r in rows if r["intended_label"] == c) for c in defect_classes}
    shortfall = [f"{c}: {usable[c]} usable of {attempted[c]} attempts (minimum {minimum})" for c in defect_classes if usable[c] < minimum or attempted[c] != expected]
    return {"complete": not shortfall, "usable": usable, "attempted": attempted, "shortfall": shortfall}


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Preprocessing and fixed augmentation views

INPUT_SIDE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
N_VIEWS = 4
AUGMENT = {"hflip_probability": 0.5, "brightness": (0.9, 1.1), "contrast": (0.9, 1.1)}
PREPROCESSING = {
    "input": "8-bit grayscale (RGB inputs converted with PIL 'L')",
    "letterbox": f"resize the longer side to {INPUT_SIDE} px (bilinear, antialiased), centre on a {INPUT_SIDE}x{INPUT_SIDE} black canvas",
    "channels": "grayscale replicated to 3 channels",
    "normalisation": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD, "scale": "pixel / 255"},
}


def letterbox(image: Image.Image, side: int = INPUT_SIDE) -> Image.Image:
    image = image.convert("L")
    scale = side / max(image.size)
    size = (max(1, round(image.size[0] * scale)), max(1, round(image.size[1] * scale)))
    resized = image.resize(size, Image.BILINEAR, reducing_gap=None)
    canvas = Image.new("L", (side, side), 0)
    canvas.paste(resized, ((side - size[0]) // 2, (side - size[1]) // 2))
    return canvas


def to_model_array(image: Image.Image) -> np.ndarray:
    """(3, 224, 224) float32 normalised array from a letterboxed grayscale image."""
    gray = np.asarray(letterbox(image), dtype=np.float32) / 255.0
    return np.stack([(gray - m) / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD, strict=True)]).astype(np.float32)


def view_params(key: str, aug_seed: int, n_views: int = N_VIEWS) -> list[dict[str, Any]]:
    """Fixed, recorded label-preserving transforms for one image: horizontal flip and mild brightness/contrast.
    No crop, so a defect cannot be cut out of the frame."""
    rng = np.random.default_rng([aug_seed, int(key[:15], 16)])
    params = []
    for view in range(n_views):
        params.append({
            "view": view,
            "hflip": bool(rng.random() < AUGMENT["hflip_probability"]),
            "brightness": round(float(rng.uniform(*AUGMENT["brightness"])), 4),
            "contrast": round(float(rng.uniform(*AUGMENT["contrast"])), 4),
        })
    return params


def apply_view(image: Image.Image, params: Mapping[str, Any]) -> Image.Image:
    out = image.convert("L")
    if params["hflip"]:
        out = ImageOps.mirror(out)
    out = ImageEnhance.Brightness(out).enhance(params["brightness"])
    return ImageEnhance.Contrast(out).enhance(params["contrast"])


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Matched training schedule

HEAD = {"optimizer": "AdamW", "learning_rate": 1e-3, "weight_decay": 1e-4, "batch_size": 32, "epochs": 30, "updates_per_epoch": 32}
SEEDS = (17, 29, 43)
CANONICAL_SEED = 17
SYNTHETIC_PROBABILITY = 0.5
ARMS = ("A", "B", "C")
ARM_NAMES = {"majority": "majority baseline", "A": "A: real-only", "B": "B: conventional augmentation", "C": "C: synthetic augmentation"}


def slot_schedule(seed: int, class_sizes: Sequence[int], synth_sizes: Sequence[int], *, n_views: int = N_VIEWS, epochs: int = HEAD["epochs"], updates: int = HEAD["updates_per_epoch"], batch: int = HEAD["batch_size"]) -> dict[str, np.ndarray]:
    """Every random draw the three arms need, made once per seed so the arms differ only by where a slot's image
    comes from. Each batch is class-balanced (11/11/10, rotating which class gets 10)."""
    k = len(class_sizes)
    rng = np.random.default_rng([seed, 5107])
    total_updates = epochs * updates
    base, extra = divmod(batch, k)
    cls = np.empty((total_updates, batch), dtype=np.int64)
    for u in range(total_updates):
        per_class = [base + (1 if ((c - u) % k) < extra else 0) for c in range(k)]
        cls[u] = np.repeat(np.arange(k), per_class)
    cls = cls.ravel()
    uniform = rng.random(cls.shape)
    real_idx = np.floor(uniform * np.asarray(class_sizes)[cls]).astype(np.int64)
    view = rng.integers(0, n_views, size=cls.shape)
    coin = rng.random(cls.shape)
    synth_uniform = rng.random(cls.shape)
    synth_idx = np.floor(synth_uniform * np.maximum(np.asarray(synth_sizes), 1)[cls]).astype(np.int64)
    return {"cls": cls, "real_idx": real_idx, "view": view, "coin": coin, "synth_idx": synth_idx, "updates": np.int64(total_updates), "batch": np.int64(batch)}


def arm_sources(schedule: Mapping[str, np.ndarray], arm: str, *, synth_probability: float = SYNTHETIC_PROBABILITY, synth_sizes: Sequence[int] | None = None) -> dict[str, np.ndarray]:
    """Resolve each slot to (is_synthetic, index, view); view -1 means the deterministic (unaugmented) features.
    Normal slots (class 0) always draw real images; a defect slot in arm C draws its synthetic pool with probability
    `synth_probability`."""
    cls = schedule["cls"]
    if arm == "A":
        return {"synthetic": np.zeros_like(cls, dtype=bool), "index": schedule["real_idx"], "view": np.full_like(cls, -1)}
    if arm == "B":
        return {"synthetic": np.zeros_like(cls, dtype=bool), "index": schedule["real_idx"], "view": schedule["view"]}
    if arm == "C":
        available = np.asarray(synth_sizes if synth_sizes is not None else [0] * (int(cls.max()) + 1)) > 0
        synthetic = (cls > 0) & available[cls] & (schedule["coin"] < synth_probability)
        index = np.where(synthetic, schedule["synth_idx"], schedule["real_idx"])
        return {"synthetic": synthetic, "index": index, "view": schedule["view"]}
    raise ValueError(f"unknown arm {arm!r}")


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Metrics and the paired group bootstrap


def confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, k: int) -> np.ndarray:
    return np.bincount(np.asarray(y_true) * k + np.asarray(y_pred), minlength=k * k).reshape(k, k)


def metrics_from_confusion(cm: np.ndarray, class_names: Sequence[str]) -> dict[str, Any]:
    """Per-class precision/recall/F1/support with undefined values reported as None, plus accuracy, balanced
    accuracy and macro-F1 over the classes present in the evaluation set (missing classes are listed)."""
    cm = np.asarray(cm, dtype=np.int64)
    tp = np.diag(cm)
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    per_class, f1s, recalls = {}, [], []
    for i, name in enumerate(class_names):
        precision = float(tp[i] / predicted[i]) if predicted[i] else None
        recall = float(tp[i] / support[i]) if support[i] else None
        denominator = 2 * tp[i] + (predicted[i] - tp[i]) + (support[i] - tp[i])
        f1 = float(2 * tp[i] / denominator) if support[i] else None
        per_class[name] = {"precision": precision, "recall": recall, "f1": f1, "support": int(support[i]), "predicted": int(predicted[i])}
        if support[i]:
            f1s.append(f1 if denominator else 0.0)
            recalls.append(recall)
    total = int(cm.sum())
    return {
        "macro_f1": float(np.mean(f1s)) if f1s else None,
        "balanced_accuracy": float(np.mean(recalls)) if recalls else None,
        "accuracy": float(tp.sum() / total) if total else None,
        "per_class": per_class,
        "missing_classes": [n for i, n in enumerate(class_names) if not support[i]],
        "undefined": [f"{n}.precision" for n, v in per_class.items() if v["precision"] is None],
        "confusion": cm.tolist(),
        "n": total,
    }


def macro_f1_batch(y_true: np.ndarray, y_pred: np.ndarray, k: int) -> float:
    return metrics_from_confusion(confusion_matrix(y_true, y_pred, k), [str(i) for i in range(k)])["macro_f1"]


def group_bootstrap_indices(labels: np.ndarray, groups: Sequence[str], *, n_resamples: int, seed: int) -> list[np.ndarray]:
    """Class-stratified group bootstrap: within each class stratum, resample its groups with replacement (same
    group count) and take all their members. Every arm is scored on the same resamples."""
    labels = np.asarray(labels)
    strata: dict[int, dict[str, list[int]]] = {}
    for i, (label, gid) in enumerate(zip(labels.tolist(), groups, strict=True)):
        strata.setdefault(label, {}).setdefault(gid, []).append(i)
    rng = np.random.default_rng([seed, 9091])
    ordered = [(label, [np.asarray(v) for _, v in sorted(g.items())]) for label, g in sorted(strata.items())]
    out = []
    for _ in range(n_resamples):
        parts = []
        for _, members in ordered:
            picks = rng.integers(0, len(members), size=len(members))
            parts.extend(members[p] for p in picks)
        out.append(np.concatenate(parts))
    return out


def paired_contrast(predictions: Mapping[tuple[str, int], np.ndarray], y_true: np.ndarray, groups: Sequence[str], *, arm: str, reference: str, seeds: Sequence[int], k: int, n_resamples: int = 1000, seed: int = 2026) -> dict[str, Any]:
    """Mean over seeds of macro-F1(arm) - macro-F1(reference), with an approximate percentile interval from the
    paired, class-stratified group bootstrap. Conditional on this split and this generated pool."""
    per_seed = {s: macro_f1_batch(y_true, predictions[(arm, s)], k) - macro_f1_batch(y_true, predictions[(reference, s)], k) for s in seeds}
    observed = float(np.mean(list(per_seed.values())))
    draws = []
    for idx in group_bootstrap_indices(y_true, groups, n_resamples=n_resamples, seed=seed):
        yt = y_true[idx]
        draws.append(np.mean([macro_f1_batch(yt, predictions[(arm, s)][idx], k) - macro_f1_batch(yt, predictions[(reference, s)][idx], k) for s in seeds]))
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return {"contrast": f"{arm} - {reference}", "mean_difference": observed, "per_seed": {str(s): float(v) for s, v in per_seed.items()}, "interval_95": [float(lo), float(hi)], "resamples": n_resamples, "bootstrap_seed": seed, "fraction_above_zero": float(np.mean(np.asarray(draws) > 0))}


def softmax(logits: np.ndarray) -> np.ndarray:
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Frozen experiment record and classifier artifact

ARTIFACT_FORMAT = "org.valcorza.bosch-sdi-capstone.linear-head.v1"
ARTIFACT_WEIGHTS = "head.safetensors"
ARTIFACT_MANIFEST = "manifest.json"
FEATURE_DIM = 512
PARITY_TOLERANCE = 1e-5


STAGE_SOURCES = ("sdi_core.py", "stage_data.py", "stage_features.py", "stage_fit.py", "stage_phi4.py", "stage_prompts.py", "stage_flux.py", "stage_evaluate.py", "stage_export.py", "stage_reload.py")
DECISION_RULE = "argmax over logits; softmax scores are uncalibrated"


def source_files(script_dir: str | Path) -> dict[str, Path]:
    """The stage code and shared core that define preprocessing, fitting, evaluation and export. Frozen with the
    experiment so that changing any of them after the freeze is detected rather than silently mixed in."""
    root = Path(script_dir)
    missing = [n for n in STAGE_SOURCES if not (root / n).is_file()]
    if missing:
        raise ContractError(f"stage sources missing from {root}: {missing}")
    return {f"source/{n}": root / n for n in STAGE_SOURCES}


def freeze_record(path: str | Path, record: Mapping[str, Any], referenced: Mapping[str, str | Path], base: str | Path) -> dict[str, Any]:
    """Write the experiment record with the SHA-256 of every file it depends on and of itself. Nothing downstream
    may change after this point without `verify_frozen` failing."""
    base = Path(base)
    files = {}
    for name, p in sorted(referenced.items()):
        p = Path(p)
        files[name] = {"path": os.path.relpath(p, base), "sha256": sha256_file(p), "bytes": p.stat().st_size}
    body = dict(record)
    body["frozen_files"] = files
    body["record_sha256"] = sha256_bytes(canonical_json(body).encode())
    write_json(path, body)
    return body


def verify_frozen(path: str | Path, base: str | Path) -> dict[str, Any]:
    body = read_json(path)
    claimed = body.pop("record_sha256", None)
    if claimed != sha256_bytes(canonical_json(body).encode()):
        raise ContractError(f"{path}: the frozen experiment record was modified after freezing")
    for name, entry in body["frozen_files"].items():
        p = Path(base) / entry["path"]
        if not p.is_file():
            raise ContractError(f"frozen input {name} ({entry['path']}) is missing")
        if sha256_file(p) != entry["sha256"]:
            raise ContractError(f"frozen input {name} ({entry['path']}) changed after the experiment was frozen")
    body["record_sha256"] = claimed
    return body


def write_artifact(out_dir: str | Path, weight: np.ndarray, bias: np.ndarray, manifest: Mapping[str, Any]) -> dict[str, Any]:
    from safetensors.numpy import save_file

    out = Path(out_dir)
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    save_file({"weight": np.ascontiguousarray(weight, dtype=np.float32), "bias": np.ascontiguousarray(bias, dtype=np.float32)}, str(out / ARTIFACT_WEIGHTS))
    body = dict(manifest)
    body["format"] = ARTIFACT_FORMAT
    body["format_version"] = "1.0"
    body["files"] = {ARTIFACT_WEIGHTS: {"sha256": sha256_file(out / ARTIFACT_WEIGHTS), "bytes": (out / ARTIFACT_WEIGHTS).stat().st_size}}
    write_json(out / ARTIFACT_MANIFEST, body)
    return body


def load_artifact(artifact_dir: str | Path) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    """Reconstruct the head from files only: manifest format, exact file set, byte counts, digests, tensor names,
    shapes, dtype and finiteness are all checked before the tensors are used. safetensors holds plain arrays, so
    nothing is unpickled."""
    from safetensors.numpy import load_file

    root = Path(artifact_dir)
    manifest_path = root / ARTIFACT_MANIFEST
    if not manifest_path.is_file():
        raise ContractError(f"{manifest_path} is missing")
    manifest = read_json(manifest_path)
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ContractError(f"artifact format {manifest.get('format')!r} is not {ARTIFACT_FORMAT!r}")
    present = sorted(p.name for p in root.iterdir())
    expected = sorted([ARTIFACT_MANIFEST, *manifest.get("files", {})])
    if present != expected:
        raise ContractError(f"artifact files {present} differ from the manifest's {expected}")
    for name, entry in manifest["files"].items():
        p = root / name
        if p.stat().st_size != entry["bytes"] or sha256_file(p) != entry["sha256"]:
            raise ContractError(f"{name} does not match its manifest digest; the artifact was modified or truncated")
    classes = manifest.get("classes")
    if not isinstance(classes, list) or not classes or len(set(classes)) != len(classes):
        raise ContractError(f"artifact classes {classes!r} must be a non-empty list of unique names")
    if manifest.get("class_index") != {c: i for i, c in enumerate(classes)}:
        raise ContractError(f"artifact class_index {manifest.get('class_index')} is inconsistent with its class list")
    tensors = load_file(str(root / ARTIFACT_WEIGHTS))
    if sorted(tensors) != ["bias", "weight"]:
        raise ContractError(f"{ARTIFACT_WEIGHTS} holds {sorted(tensors)}; expected ['bias', 'weight']")
    k = len(manifest["classes"])
    weight, bias = tensors["weight"], tensors["bias"]
    if weight.shape != (k, FEATURE_DIM) or bias.shape != (k,) or weight.dtype != np.float32 or bias.dtype != np.float32:
        raise ContractError(f"head tensors have shapes {weight.shape}/{bias.shape} and dtypes {weight.dtype}/{bias.dtype}; expected ({k}, {FEATURE_DIM})/({k},) float32")
    if not (np.isfinite(weight).all() and np.isfinite(bias).all()):
        raise ContractError("head tensors contain non-finite values")
    return weight, bias, manifest


def _normalised(obj: Any) -> str:
    return canonical_json(json.loads(json.dumps(obj)))


def check_artifact_semantics(manifest: Mapping[str, Any], *, record: Mapping[str, Any], preprocessing: Mapping[str, Any], backbone: Mapping[str, Any]) -> None:
    """The artifact means what the frozen experiment evaluated: same class names in the same order with consistent
    indices, the same decision rule, preprocessing and backbone, and the same selected head. Numeric parity alone
    cannot show this, because a relabelled class list leaves every logit unchanged."""
    classes = manifest.get("classes")
    if not isinstance(classes, list) or not all(isinstance(c, str) for c in classes) or len(set(classes)) != len(classes):
        raise ContractError(f"artifact classes {classes!r} must be a list of unique names")
    if classes != list(record["class_order"]):
        raise ContractError(f"artifact classes {classes} differ from the frozen experiment's class order {record['class_order']}")
    if manifest.get("class_index") != {c: i for i, c in enumerate(classes)}:
        raise ContractError(f"artifact class_index {manifest.get('class_index')} is inconsistent with its class list")
    if manifest.get("decision_rule") != DECISION_RULE:
        raise ContractError(f"artifact decision rule {manifest.get('decision_rule')!r} is not {DECISION_RULE!r}")
    if _normalised(manifest.get("preprocessing")) != _normalised(preprocessing):
        raise ContractError("artifact preprocessing differs from the preprocessing this notebook applies; reconstruction would not reproduce the evaluated classifier")
    if _normalised(manifest.get("backbone")) != _normalised(backbone):
        raise ContractError("artifact backbone identity differs from the pinned backbone")
    if manifest.get("experiment_record_sha256") != record["record_sha256"]:
        raise ContractError("artifact was exported from a different frozen experiment record")
    selected = record["selection"]
    got = manifest.get("selection") or {}
    if (got.get("arm"), got.get("seed"), got.get("epoch")) != (selected["arm"], selected["seed"], selected["epoch"]):
        raise ContractError(f"artifact selection {got} differs from the frozen selection {selected}")
    frozen_head = record["frozen_files"].get(f"heads/{selected['arm']}_s{selected['seed']}", {})
    if manifest["files"][ARTIFACT_WEIGHTS]["sha256"] != frozen_head.get("sha256"):
        raise ContractError("artifact head tensors differ from the frozen head of the selected arm and seed")


def replay_check(logits: np.ndarray, expected_logits: np.ndarray, expected_labels: Sequence[str], classes: Sequence[str]) -> dict[str, Any]:
    """Parity passes only if every logit is within tolerance AND every replayed decision names the same class."""
    diff = float(np.abs(np.asarray(logits) - np.asarray(expected_logits)).max()) if len(expected_labels) else 0.0
    labels = [classes[int(i)] for i in np.asarray(logits).argmax(1)] if len(expected_labels) else []
    same = labels == list(expected_labels)
    return {"examples": len(expected_labels), "max_abs_logit_difference": diff, "tolerance": PARITY_TOLERANCE, "same_predictions": same, "replayed_labels": labels, "expected_labels": list(expected_labels), "passed": diff <= PARITY_TOLERANCE and same and len(expected_labels) > 0}


NEW_IMAGE_FIELDS = ("image_id", "source", "known_label", "input_mode", "input_size", "predicted_label")


def write_new_predictions(path: str | Path, rows: Sequence[Mapping[str, Any]], classes: Sequence[str]) -> Path:
    """Fixed header (identifiers, then one uncalibrated score per class in artifact order), so an empty cohort still
    yields a valid, header-only CSV."""
    path = Path(path)
    with open(path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=[*NEW_IMAGE_FIELDS, *[f"score_{c}" for c in classes]], lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    return path


def head_logits(features: np.ndarray, weight: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return (np.asarray(features, dtype=np.float32) @ weight.T + bias).astype(np.float32)


In [ ]:
%%writefile -a work/sdi_capstone/sdi_core.py

# %% [section] Contact sheets


def contact_sheet(items: Sequence[tuple[Image.Image, str]], *, columns: int = 8, thumb: int = 160, title: str = "") -> Image.Image:
    """Grid of labelled thumbnails (each caption drawn under its image) for quick visual inspection."""
    rows = max(1, -(-len(items) // columns))
    caption_h, title_h = 28, (24 if title else 0)
    sheet = Image.new("RGB", (columns * thumb, title_h + rows * (thumb + caption_h)), "white")
    draw = ImageDraw.Draw(sheet)
    if title:
        draw.text((6, 5), title, fill="black")
    for n, (image, caption) in enumerate(items):
        x, y = (n % columns) * thumb, title_h + (n // columns) * (thumb + caption_h)
        sheet.paste(image.convert("RGB").resize((thumb - 4, thumb - 4), Image.BILINEAR), (x + 2, y + 2))
        for line_no, line in enumerate(caption.split("\n")[:2]):
            draw.text((x + 3, y + thumb + 1 + 12 * line_no), line[:26], fill="black")
    return sheet


The stage runner starts each script in its environment, streams its output, and measures time and memory.


In [ ]:
# @title Infrastructure: stage runner and display helpers (run it; no need to study it)
import os
import threading

from IPython.display import Image as IPImage
from IPython.display import Markdown, display

STAGE_LOG = {}
CHILD_ENV = {k: v for k, v in os.environ.items() if k not in ('HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP')}  # no credential; no kernel Python settings
CHILD_ENV.update(MPLBACKEND='Agg', PYTHONHASHSEED='0', HF_HOME=str(WORK_ROOT / 'hf_home'), HF_HUB_DISABLE_TELEMETRY='1', HF_HUB_DISABLE_PROGRESS_BARS='1',
                 TOKENIZERS_PARALLELISM='false', CUBLAS_WORKSPACE_CONFIG=':4096:8', PYTHONUNBUFFERED='1',
                 HF_XET_CHUNK_CACHE_SIZE_BYTES='0')  # no hidden multi-GB download cache: weights live only where the stages put them


def _gpu_used_mib():
    value = _smi('memory.used')
    return int(value) if value else None


def _mem_available_mib():
    with open('/proc/meminfo') as handle:
        for line in handle:
            if line.startswith('MemAvailable:'):
                return int(line.split()[1]) // 1024
    return None


def run_stage(name, env, script, *args):
    """Run one stage script in its locked environment, stream its output, and record wall time, the child's peak
    resident memory, peak GPU memory in use and the lowest available host memory while it ran."""
    cmd = [ENV_PYTHON[env], '-u', str(WORK_ROOT / script), str(RUN_CONFIG), *map(str, args)]
    peaks = {'peak_gpu_used_mib': 0, 'min_host_available_mib': None}
    stop = threading.Event()

    def sample():
        while not stop.is_set():
            gpu, mem = _gpu_used_mib(), _mem_available_mib()
            if gpu is not None:
                peaks['peak_gpu_used_mib'] = max(peaks['peak_gpu_used_mib'], gpu)
            if mem is not None:
                low = peaks['min_host_available_mib']
                peaks['min_host_available_mib'] = mem if low is None else min(low, mem)
            stop.wait(1.0)

    disk_before = shutil.disk_usage(BASE).free
    started = time.time()
    monitor = threading.Thread(target=sample, daemon=True)
    monitor.start()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=CHILD_ENV, cwd=BASE)
    for line in proc.stdout:
        print(line, end='')
    _, status, usage = os.wait4(proc.pid, 0)
    proc.returncode = os.waitstatus_to_exitcode(status)
    stop.set()
    monitor.join()
    record = {'env': env, 'script': script, 'args': list(map(str, args)), 'exit_code': proc.returncode, 'seconds': round(time.time() - started, 1),
              'peak_child_rss_mib': usage.ru_maxrss // 1024, **peaks,
              'disk_free_gb_before': round(disk_before / 1e9, 1), 'disk_free_gb_after': round(shutil.disk_usage(BASE).free / 1e9, 1)}
    STAGE_LOG[name] = record
    (WORK / 'state' / 'stage_log.json').write_text(json.dumps(STAGE_LOG, indent=2))
    print(f"\n[{name}] {record['seconds']} s | peak child RSS {record['peak_child_rss_mib']} MiB | peak GPU in use {record['peak_gpu_used_mib']} MiB | "
          f"lowest host memory available {record['min_host_available_mib']} MiB | disk free {record['disk_free_gb_after']} GB")
    if proc.returncode != 0:
        raise RuntimeError(f"stage '{name}' failed with exit code {proc.returncode}. Read the last lines above, then see Troubleshooting in Section 13.")
    return record


def load(name):
    path = OUT / name
    return json.loads(path.read_text()) if name.endswith('.json') else [json.loads(x) for x in path.read_text().splitlines() if x.strip()]


def table(headers, rows):
    lines = ['| ' + ' | '.join(map(str, headers)) + ' |', '|' + '---|' * len(headers)]
    lines += ['| ' + ' | '.join('—' if v is None else str(v) for v in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


def fmt(value, digits=3):
    return '—' if value is None else f'{value:.{digits}f}'


def show(name, width=None):
    path = OUT / 'figures' / name
    if path.exists():
        display(IPImage(filename=str(path), width=width))
    else:
        print(f'(no figure {name})')


## 2 · The dataset: acquisition, audit and a frozen grouped split  <sub>(data practice)</sub>

**Dataset card.** The Bosch Surface Defect Inspection (SDI) dataset (source repository pinned at commit `c6e0afe6…`) contains grayscale photographs of three products (A, B, C), each labelled `normal`, `scratches` or `spots`. It was released with the DT-GAN study (Wang et al., 2023); this notebook is a new experiment, not a reproduction of that paper's results. The repository declares **CC BY-SA 4.0**: attribution is required and adapted dataset material you redistribute must carry the same licence. The dataset's licence does not automatically apply to the notebook code, the models, or generated images; Section 12 records each separately.

**Acquisition.** The archive is a 1,647,097,112-byte Git LFS object. The data stage streams it with timeouts, resumes on interruption, verifies its byte count and SHA-256 before extraction, and refuses path traversal, symlinks, non-image files and anything beyond an expanded-size ceiling set from the inspected archive. It never substitutes another dataset.

**Audit and split.** Before any model sees an image, the stage inventories every file, compares the counts with the published notes (documented counts are expectations, measured counts are facts), finds exact decoded-pixel duplicates and near duplicates, and forms groups. Groups, not images, are assigned to train / validation / test (60/20/20, approximately class-stratified, deterministic). The source supplies no specimen or session identifiers, so groups approximate shared capture and this is an **exploratory image-group benchmark**, not a claim about independent physical specimens. Only product A is used. From the training partition, a fixed budget of 128 normal, 64 scratch and 32 spot images is selected (one per group); the rest of the training partition stays unused.

**Question for this stage:** do the measured counts match the documentation, and could the same picture end up in both training and test?


The next cell saves the data stage: acquisition, extraction, inventory, grouping, the split and the budget (with a `--check-byod` dry-run mode used in Section 13). The cell after it runs the stage.


In [ ]:
%%writefile work/sdi_capstone/stage_data.py
"""Stage 1 (lab environment): acquire, verify, extract, inventory, audit, group, split and select the real-data budget."""
# ruff: noqa: E501
from __future__ import annotations

import argparse
import shutil
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402

# Digest of split_manifest.csv produced from the pinned archive when this notebook was written (Pillow 11.3.0,
# NumPy 2.5.3). A different digest means grouping or assignment changed; the run records it rather than hiding it.
REFERENCE_SPLIT_SHA256 = "8b5ef942a82bdacdabf98f2a0b3ab5d3b5adb60ce02dc57831f73147323e65aa"


def acquire_bosch(cfg: dict, work: Path) -> tuple[Path, dict]:
    root = work / "data" / "sdi"
    marker = root / ".extracted.json"
    if marker.is_file() and core.read_json(marker).get("archive_sha256") == core.ARCHIVE_SHA256:
        info = core.read_json(marker)
        info["reused_extraction"] = True
        print(f"reusing verified extraction at {root} ({info['files']} files)")
        return root, info
    zip_path = work / "downloads" / core.ARCHIVE_NAME
    print(f"downloading {core.ARCHIVE_URL}\n  expected {core.ARCHIVE_BYTES:,} bytes, SHA-256 {core.ARCHIVE_SHA256}")
    fetched = core.fetch_archive(core.ARCHIVE_URL, zip_path, expected_bytes=core.ARCHIVE_BYTES, expected_sha256=core.ARCHIVE_SHA256)
    print(f"verified: {fetched}")
    extracted = core.safe_extract(zip_path, root, allowed_suffixes=(".jpg",))
    info = {"archive_sha256": core.ARCHIVE_SHA256, "archive_bytes": core.ARCHIVE_BYTES, "download": fetched, **extracted}
    core.write_json(marker, info)
    if cfg["delete_archive_after_extract"]:
        zip_path.unlink()
        info["archive_deleted_after_extract"] = True
    print(f"extracted {extracted['files']:,} files, {extracted['bytes']:,} bytes (every entry checked before writing)")
    return root, info


def inventory_bosch(root: Path) -> list[dict]:
    records = []
    files = sorted(p for p in root.rglob("*") if p.is_file() and not p.name.startswith("."))
    for n, path in enumerate(files, 1):
        rel = path.relative_to(root).as_posix()
        rec = {"image_id": path.stem, "relpath": rel, "path": path, **core.parse_bosch_path(rel), **core.inspect_image(path)}
        records.append(rec)
        if n % 5000 == 0:
            print(f"  inspected {n:,} / {len(files):,}")
    ids = [r["image_id"] for r in records]
    if len(set(ids)) != len(ids):
        raise core.ContractError("image ids are not unique across the archive")
    return records


def acquire_byod(cfg: dict, work: Path) -> tuple[list[dict], dict]:
    source = Path(cfg["byod_zip"])
    if not source.is_file():
        raise core.ContractError(f"BYOD_ZIP_PATH {source} does not exist; upload a zip with manifest.csv and images")
    root = work / "data" / "byod"
    extracted = core.safe_extract(source, root, allowed_suffixes=core.IMAGE_SUFFIXES, allowed_names=(core.BYOD_MANIFEST,), max_entries=50_000, max_total=8_000_000_000, max_member=50_000_000)
    rows = core.load_byod_manifest(root)
    records = []
    for row in rows:
        rec = {**row, **core.inspect_image(row["path"])}
        rec["relpath"] = Path(row["path"]).relative_to(root.resolve()).as_posix()
        records.append(rec)
    return records, {"byod_zip_sha256": core.sha256_file(source), **extracted}


def check_byod(cfg: dict, work: Path, zip_path: str) -> None:
    """Dry run for a BYOD zip: the same extraction, manifest, image, grouping, split and budget rules, no models."""
    cfg = {**cfg, "byod_zip": zip_path}
    scratch = work / "byod_check"
    records, _ = acquire_byod(cfg, scratch)
    bad = [r["relpath"] for r in records if not r["decode_ok"]]
    ok = [r for r in records if r["decode_ok"]]
    grouping = core.assign_groups(ok, np.stack([r["signature"] for r in ok]).astype(np.float32))
    for r, gid in zip(ok, grouping["group_of"], strict=True):
        r["group_id"] = gid
    for r in records:
        if not r["decode_ok"]:
            r["group_id"] = "x_" + r["file_sha256"][:16]
    split = core.build_split_rows(records)
    counts: dict = {}
    for r in records:
        counts.setdefault(r["split"], {}).setdefault(r["label"], 0)
        counts[r["split"]][r["label"]] += 1
    shutil.rmtree(scratch, ignore_errors=True)
    print(core.canonical_json({"byod_check": "PASS", "images": len(records), "undecodable": bad, "groups": grouping["groups"], "class_order": split["class_order"], "split_counts": counts}))


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("config")
    parser.add_argument("--check-byod", default=None, help="validate a BYOD zip without running the experiment")
    args = parser.parse_args()
    cfg = core.read_json(args.config)
    if args.check_byod:
        check_byod(cfg, Path(cfg["work_dir"]), args.check_byod)
        return
    work, out = Path(cfg["work_dir"]), Path(cfg["out_dir"])
    started = time.time()
    if cfg["data_source"] == "bosch":
        root, acquisition = acquire_bosch(cfg, work)
        records = inventory_bosch(root)
        measured: dict = {}
        for r in records:
            measured.setdefault(r["product"], {}).setdefault(r["label"], 0)
            measured[r["product"]][r["label"]] += 1
        official: dict = {}
        for r in records:
            if r["official_split"]:
                key = f"{r['product']}/{r['label']}/{r['official_split']}"
                official[key] = official.get(key, 0) + 1
        inventory = {
            "entries": len(records),
            "measured_counts": measured,
            "documented_counts": core.DOCUMENTED_COUNTS,
            "differences": {p: {c: measured.get(p, {}).get(c, 0) - n for c, n in cls.items()} for p, cls in core.DOCUMENTED_COUNTS.items()},
            "official_split_counts": official,
            "official_split_note": "the source assigns only defect images to train/val/test; normal images carry no official split",
            "modes": sorted({f"{r['mode']} {r['width']}x{r['height']}" for r in records if r["decode_ok"]}),
            "undecodable": [r["relpath"] for r in records if not r["decode_ok"]],
            "specimen_or_session_metadata": "none supplied by the source (unknown, not inferred)",
        }
        experiment = [r for r in records if r["product"] == "A"]
    else:
        experiment, acquisition = acquire_byod(cfg, work)
        inventory = {"entries": len(experiment), "undecodable": [r["relpath"] for r in experiment if not r["decode_ok"]], "declared_groups": sum(1 for r in experiment if r.get("declared_group"))}
    print(f"inventory: {inventory['entries']:,} images, {len(inventory['undecodable'])} undecodable")

    ok = [r for r in experiment if r["decode_ok"]]
    signatures = np.stack([r["signature"] for r in ok]).astype(np.float32)
    grouping = core.assign_groups(ok, signatures)
    for r, gid in zip(ok, grouping["group_of"], strict=True):
        r["group_id"] = gid
    for r in experiment:
        if not r["decode_ok"]:
            r["group_id"] = "x_" + r["file_sha256"][:16]
            r.update(width=0, height=0, mode="", pixel_sha256="")
    split = core.build_split_rows(experiment)
    split_sha = core.write_split_manifest(out / "split_manifest.csv", experiment)
    by_split: dict = {}
    for r in experiment:
        by_split.setdefault(r["split"], {}).setdefault(r["label"], 0)
        by_split[r["split"]][r["label"]] += 1
    selected = {c: sum(1 for r in experiment if r["selected"] and r["label"] == c) for c in split["class_order"]}
    pairs = grouping["near_duplicate_pairs"]
    ids = [r["image_id"] for r in ok]
    conflict_members = sorted(r["image_id"] for r in experiment if r["exclusion_reason"] == "label-conflict-duplicate")
    split_of = [r["split"] for r in ok]
    label_of = [r["label"] for r in ok]
    similar = core.near_duplicate_pairs(signatures, core.AUDIT_SIMILARITY)
    crossing: dict = {}
    for i, j, _ in similar:
        if split_of[i] != split_of[j] and "excluded" not in (split_of[i], split_of[j]):
            key = "/".join(sorted((label_of[i], label_of[j])))
            crossing[key] = crossing.get(key, 0) + 1
    below = sum(1 for _, _, s in similar if core.NEAR_DUPLICATE_THRESHOLD - 0.005 <= s < core.NEAR_DUPLICATE_THRESHOLD)
    official_vs_custom: dict = {}
    for r in experiment:
        if r["official_split"]:
            key = f"official {r['official_split']} -> {r['split']}"
            official_vs_custom[key] = official_vs_custom.get(key, 0) + 1
    manifest = {
        "data_source": cfg["data_source"],
        "source": {"repository": core.SOURCE_REPOSITORY, "commit": core.SOURCE_COMMIT, "archive": core.ARCHIVE_NAME, "url": core.ARCHIVE_URL, "bytes": core.ARCHIVE_BYTES, "sha256": core.ARCHIVE_SHA256, "license": core.DATASET_LICENSE, "notes": core.DATASET_NOTES_URL} if cfg["data_source"] == "bosch" else {"byod": True},
        "acquisition": acquisition,
        "inventory": inventory,
        "experiment_scope": "product A, classes normal / scratches / spots" if cfg["data_source"] == "bosch" else "user-supplied manifest",
        "grouping": {
            "rule": "union of exact decoded-pixel duplicates, 32x32 signature correlation >= threshold, and declared groups",
            "threshold": core.NEAR_DUPLICATE_THRESHOLD,
            "groups": grouping["groups"],
            "exact_duplicate_links": grouping["exact_duplicate_links"],
            "near_duplicate_pairs": len(pairs),
            "example_pairs": [[ids[i], ids[j], s] for i, j, s in pairs[:25]],
            "label_conflict_groups": len(split["label_conflict_groups"]),
            "label_conflict_images": conflict_members,
            "pairs_just_below_threshold": {"range": [core.NEAR_DUPLICATE_THRESHOLD - 0.005, core.NEAR_DUPLICATE_THRESHOLD], "count": below},
            "similar_pairs_crossing_partitions": {"correlation_at_least": core.AUDIT_SIMILARITY, "by_label_pair": crossing},
            "interpretation": "exploratory image-group benchmark: duplicate groups approximate shared capture; they do not prove independent physical specimens, and similar images below the threshold can sit in different partitions",
        },
        "split": {
            "version": core.SPLIT_VERSION,
            "design": "deterministic, approximately class-stratified 60/20/20 group assignment (custom; not the source's split)",
            "fractions": dict(core.SPLIT_FRACTIONS),
            "counts": by_split,
            "official_vs_custom": official_vs_custom,
            "manifest_sha256": split_sha,
            "reference_sha256": REFERENCE_SPLIT_SHA256 if cfg["data_source"] == "bosch" else None,
            "matches_reference": (split_sha == REFERENCE_SPLIT_SHA256) if cfg["data_source"] == "bosch" else None,
        },
        "class_order": split["class_order"],
        "budget": split["budget"],
        "selected": selected,
        "unused_training_images": sum(1 for r in experiment if r["split"] == "train" and not r["selected"]),
        "new_image_candidates": {c: sum(1 for r in experiment if r["split"] == "train" and not r["selected"] and r["label"] == c) for c in split["class_order"]},
        "exemplars": sorted((r["label"], r["budget_rank"], r["image_id"]) for r in experiment if r["exemplar"]),
        "core_version": core.CORE_VERSION,
        "seconds": round(time.time() - started, 1),
    }
    core.write_json(out / "data_manifest.json", manifest)
    if not all(manifest["new_image_candidates"].values()):
        print(f"NOTE: unused training images per class {manifest['new_image_candidates']}: Section 11's new-image preview needs spare images or NEW_IMAGE_DIR; without them it is recorded as not run")
    audit = work / "audit"
    audit.mkdir(parents=True, exist_ok=True)
    np.savez(audit / "signatures.npz", image_id=np.array(ids), signature=signatures.astype(np.float16))
    # Training-only views: selected budget images, and near-duplicate pairs whose members are both training images.
    from PIL import Image

    items = []
    for label in split["class_order"]:
        chosen = sorted((r for r in experiment if r["selected"] and r["label"] == label), key=lambda r: r["budget_rank"])[:8]
        items += [(Image.open(r["path"]), f"{label}\n{r['image_id']}") for r in chosen]
    figures = out / "figures"
    figures.mkdir(parents=True, exist_ok=True)
    core.contact_sheet(items, title="Selected TRAINING images only (held-out images are never previewed)").save(figures / "training_examples.png")
    by_id = {r["image_id"]: r for r in experiment}
    train_pairs = [(ids[i], ids[j], s) for i, j, s in pairs if by_id[ids[i]]["split"] == "train" and by_id[ids[j]]["split"] == "train"][:4]
    pair_items = []
    for a, b, s in train_pairs:
        pair_items += [(Image.open(by_id[a]["path"]), f"{a}\ncorr {s}"), (Image.open(by_id[b]["path"]), f"{b}\nsame group")]
    if pair_items:
        core.contact_sheet(pair_items, columns=4, title="Near-duplicate training pairs grouped together").save(figures / "near_duplicate_pairs.png")
    print(core.canonical_json({"split_counts": by_split, "selected": selected, "split_sha256": split_sha, "matches_reference": manifest["split"]["matches_reference"], "label_conflict_images": len(conflict_members), "near_duplicate_pairs": len(pairs), "similar_pairs_crossing": crossing, "just_below_threshold": below, "seconds": manifest["seconds"]}))


if __name__ == "__main__":
    core.run_main(main)


In [ ]:
run_stage('data', 'lab', 'stage_data.py')
DATA = load('data_manifest.json')
if DATA_SOURCE == 'bosch':
    inv = DATA['inventory']
    table(['product', 'class', 'documented', 'measured', 'difference'],
          [[p, c, n, inv['measured_counts'].get(p, {}).get(c, 0), inv['differences'][p][c]] for p, cls in inv['documented_counts'].items() for c, n in cls.items()])
    print('image formats:', inv['modes'], '| undecodable:', len(inv['undecodable']), '|', inv['specimen_or_session_metadata'])
    print('official split of product A defects -> this notebook\'s split:', DATA['split']['official_vs_custom'])
g = DATA['grouping']
print(f"groups: {g['groups']:,}; exact-duplicate links: {g['exact_duplicate_links']}; near-duplicate pairs (corr >= {g['threshold']}): {g['near_duplicate_pairs']}; "
      f"pairs just below the threshold: {g['pairs_just_below_threshold']['count']}")
print(f"label-conflict duplicates excluded: {g['label_conflict_groups']} groups ({len(g['label_conflict_images'])} images): {g['label_conflict_images']}")
print('similar pairs (corr >= 0.95) that cross partitions, by label pair:', g['similar_pairs_crossing_partitions']['by_label_pair'])
order = DATA['class_order']
table(['partition', *order], [[s, *[DATA['split']['counts'].get(s, {}).get(c, 0) for c in order]] for s in ('train', 'val', 'test', 'excluded')])
print(f"selected training budget: {DATA['selected']}; unused training images: {DATA['unused_training_images']:,}")
print(f"split manifest SHA-256 {DATA['split']['manifest_sha256'][:16]}...; matches the reference audit: {DATA['split']['matches_reference']}")
show('training_examples.png')
show('near_duplicate_pairs.png')


**What to notice.**

- The defect counts match the published notes, but the normal counts do not (product A has 66 fewer normal images than documented; product C about 1,490 fewer). The notebook reports this; it does not guess why.
- Five pairs of images are pixel-for-pixel identical yet labelled `scratches` in one copy and `spots` in the other, and several of those pairs sit in *different* official partitions. Those ten images are excluded as label conflicts: neither label can be trusted, and keeping them would put the same picture in two partitions.
- The near-duplicate threshold (correlation 0.98 of 32 × 32 thumbnails) is a declared cut, not a natural boundary; the printout shows how many pairs sit just below it and how many similar pairs cross partitions. Most crossing pairs are normal/normal, because these surfaces share one texture.
- The test set has only about 21 spot images. Every spot metric later rests on those few pictures.
- `matches the reference audit: True` means your split is byte-identical to the one this notebook was written against. `False` means grouping or assignment changed (for example a different image decoder); the run continues with its own frozen manifest and records the difference.


> **Check your reasoning.** Why is a random image-level split risky for this dataset, and what does grouping by duplicates not protect against?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Consecutive capture frames and duplicated files are near-copies. With an image-level random split, a near-copy of a test image can sit in training, so the classifier is partly tested on what it memorised and scores look better than they would on new parts. Grouping keeps each detected cluster in one partition. It cannot detect similarity below the threshold, and it cannot tell whether two dissimilar images come from the same physical part, line or session, because the source supplies no such metadata. That is why the notebook calls this an image-group benchmark and not a specimen-level one.

</details>


## 3 · Baselines first: majority class and a real-only classifier  <sub>(core concept)</sub>

Before any generated image exists, it is worth knowing what "good" looks like. The feature stage turns each selected training image into a 512-number vector with a frozen ImageNet ResNet-18 (He et al., 2016), plus four fixed augmented views of each image for the conventional arm; validation images get one deterministic vector each. Preprocessing is the same everywhere: grayscale, letterbox to 224 × 224 without cropping, replicate to three channels, ImageNet normalisation.

Then a **preview** fits the linear head for arm A only (real images, no augmentation) and scores it on **validation**. The majority baseline always predicts `normal`.

**Predict first:** the majority baseline's accuracy on validation will be around what number? Its macro-F1?


The next cell saves the feature stage. `--part real` handles the training budget and validation; `--part synthetic` (Section 7) handles generated candidates.


In [ ]:
%%writefile work/sdi_capstone/stage_features.py
"""Stage 3 / 7 (lab environment): frozen ResNet-18 features.

`--part real` (before any generation): the selected training budget (deterministic + four fixed views) and validation.
`--part synthetic` (after generation): the eligible synthetic pool and the synthetic-to-training near-duplicate audit.
"""
# ruff: noqa: E501
from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402

BACKBONE = {
    "id": "timm/resnet18.tv_in1k",
    "revision": "bbd144b3e5565108aad885f145491d11bc6ce807",
    "file": "model.safetensors",
    "bytes": 46_807_446,
    "sha256": "694f673df6520a3158624e8a89af086f59923ee4cd7436fe5bc3bc71d295ad81",
    "license": "bsd-3-clause",
    "architecture": "torchvision.models.resnet18 (the torchvision ImageNet-1k weights, repackaged as safetensors)",
    "feature": "512-d global-average-pooled output of layer4; the ImageNet classifier (fc) is replaced by identity",
}
AUG_SEED = 20260929
BATCH = 64


def stage_backbone(work: Path) -> Path:
    from huggingface_hub import hf_hub_download

    root = work / "weights" / "resnet18-tv-in1k"
    path = root / BACKBONE["file"]
    if not path.is_file() or path.stat().st_size != BACKBONE["bytes"]:
        hf_hub_download(BACKBONE["id"], BACKBONE["file"], revision=BACKBONE["revision"], local_dir=str(root))
    if core.sha256_file(path) != BACKBONE["sha256"]:
        raise core.ContractError(f"{path} fails its pinned digest; refusing to load it")
    return path


def build_backbone(weights: Path, device: str):
    import torch
    import torchvision
    from safetensors.torch import load_file

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    model = torchvision.models.resnet18(weights=None)
    model.load_state_dict(load_file(str(weights)), strict=True)
    model.fc = torch.nn.Identity()
    for p in model.parameters():
        p.requires_grad_(False)
    return model.eval().to(device)


def embed(model, images, device: str) -> np.ndarray:
    import torch

    out = []
    for start in range(0, len(images), BATCH):
        batch = np.stack([core.to_model_array(im) for im in images[start : start + BATCH]])
        with torch.inference_mode():
            out.append(model(torch.from_numpy(batch).to(device)).float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, core.FEATURE_DIM), dtype=np.float32)


def features_with_views(model, items, device: str, log: list) -> tuple[np.ndarray, np.ndarray]:
    """items: (key, grayscale image). Deterministic features plus the fixed four-view bank."""
    det = embed(model, [im for _, im in items], device)
    views = np.zeros((len(items), core.N_VIEWS, core.FEATURE_DIM), dtype=np.float32)
    params = [core.view_params(key, AUG_SEED) for key, _ in items]
    for v in range(core.N_VIEWS):
        views[:, v] = embed(model, [core.apply_view(im, params[n][v]) for n, (_, im) in enumerate(items)], device)
    log.extend({"key": key, "views": p} for (key, _), p in zip(items, params, strict=True))
    return det, views


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("config")
    parser.add_argument("--part", choices=("real", "synthetic"), required=True)
    args = parser.parse_args()
    cfg = core.read_json(args.config)
    work, out = Path(cfg["work_dir"]), Path(cfg["out_dir"])
    started = time.time()
    import torch
    from PIL import Image

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = build_backbone(stage_backbone(work), device)
    order = core.read_json(out / "data_manifest.json")["class_order"]
    rows = core.read_split_manifest(out / "split_manifest.csv")
    data_root = work / "data" / ("sdi" if cfg["data_source"] == "bosch" else "byod")
    feat_dir = work / "features"
    feat_dir.mkdir(parents=True, exist_ok=True)
    figures = out / "figures"
    figures.mkdir(parents=True, exist_ok=True)
    aug_log: list = []
    train = sorted((r for r in rows if r["selected"]), key=lambda r: (order.index(r["label"]), r["budget_rank"]))

    if args.part == "real":
        train_items = [(r["file_sha256"], core.load_verified_gray(data_root / r["relpath"], r)) for r in train]
        det, views = features_with_views(model, train_items, device, aug_log)
        np.savez(feat_dir / "train.npz", image_id=np.array([r["image_id"] for r in train]), label=np.array([order.index(r["label"]) for r in train]), det=det, views=views)
        print(f"training budget: {len(train)} images x (1 deterministic + {core.N_VIEWS} views)")
        val = sorted((r for r in rows if r["split"] == "val"), key=lambda r: r["image_id"])
        vdet = embed(model, [core.load_verified_gray(data_root / r["relpath"], r) for r in val], device)
        np.savez(feat_dir / "val.npz", image_id=np.array([r["image_id"] for r in val]), label=np.array([order.index(r["label"]) for r in val]), group=np.array([r["group_id"] for r in val]), det=vdet)
        print(f"validation: {len(val)} images (deterministic preprocessing only)")
        core.write_jsonl(out / "augmentation_views.jsonl", aug_log)
        sheet = []
        for label in order:
            r = next(x for x in train if x["label"] == label)
            image = core.load_verified_gray(data_root / r["relpath"], r)
            params = core.view_params(r["file_sha256"], AUG_SEED)
            sheet.append((image, f"{label} original"))
            sheet += [(core.apply_view(image, p), f"view {p['view']} flip={int(p['hflip'])}\nb{p['brightness']} c{p['contrast']}") for p in params]
        core.contact_sheet(sheet, columns=5, title="Conventional augmentation views of TRAINING images (label must survive every view)").save(figures / "augmentation_views.png")
        state = {"backbone": BACKBONE, "device": device, "aug_seed": AUG_SEED, "augment": core.AUGMENT, "n_views": core.N_VIEWS, "preprocessing": core.PREPROCESSING, "seconds": round(time.time() - started, 1), "torch": torch.__version__}
        core.write_json(work / "state" / "features_stage.json", state)
        print(core.canonical_json({"device": device, "train": len(train), "val": len(val), "seconds": state["seconds"]}))
        return

    generated = [r for r in core.read_jsonl(out / "generation_manifest.jsonl") if r["status"] == "eligible"]
    synth_items = [(r["pixel_sha256"], Image.open(out / r["file"]).convert("L")) for r in generated]
    sdet, sviews = features_with_views(model, synth_items, device, aug_log)
    np.savez(feat_dir / "synthetic.npz", candidate_id=np.array([r["candidate_id"] for r in generated]), label=np.array([order.index(r["intended_label"]) for r in generated], dtype=np.int64), det=sdet, views=sviews)
    core.write_jsonl(out / "synthetic_augmentation_views.jsonl", aug_log)
    print(f"synthetic pool: {len(generated)} eligible candidates (converted to grayscale, letterboxed from 512 px)")
    # Synthetic-to-training near-duplicate audit (before fitting): training partition only.
    sig = np.load(work / "audit" / "signatures.npz")
    split_of = {r["image_id"]: r["split"] for r in rows}
    keep = np.array([split_of.get(i) == "train" for i in sig["image_id"]])
    train_sigs, train_ids = sig["signature"][keep].astype(np.float32), sig["image_id"][keep]
    synth_sigs = np.stack([core.signature(im) for _, im in synth_items]) if synth_items else np.zeros((0, core.SIGNATURE_SIDE**2), np.float32)
    best, where = core.max_similarity(synth_sigs, train_sigs)
    audit = {
        "threshold": core.NEAR_DUPLICATE_THRESHOLD,
        "reference": "all training-partition real images",
        "max_correlation": float(best.max()) if len(best) else None,
        "flagged": [{"candidate_id": generated[n]["candidate_id"], "closest_training_image": str(train_ids[where[n]]), "correlation": round(float(best[n]), 4)} for n in np.nonzero(best >= core.NEAR_DUPLICATE_THRESHOLD)[0]],
        "per_candidate": {generated[n]["candidate_id"]: round(float(best[n]), 4) for n in range(len(generated))},
        "policy": "reported, not used to filter candidates",
    }
    core.write_json(out / "synthetic_training_overlap.json", audit)
    print(f"synthetic-to-training audit: max correlation {audit['max_correlation']}, {len(audit['flagged'])} flagged at >= {core.NEAR_DUPLICATE_THRESHOLD}")
    print(core.canonical_json({"device": device, "synthetic": len(generated), "seconds": round(time.time() - started, 1)}))


if __name__ == "__main__":
    core.run_main(main)


The next cell saves the fitting stage. It is used three ways: `--preview` (below), the canonical fit and freeze (Section 8), and validation-only extensions (Section 13). It never reads test images.


In [ ]:
%%writefile work/sdi_capstone/stage_fit.py
"""Stage 6 (lab environment): fit the linear heads for arms A/B/C on matched slot schedules, select epochs on
validation, and freeze the experiment. The test partition is not read here.

Modes: canonical (default); `--preview` (majority baseline and arm A on validation, before any generation);
`--synthetic-probability P --tag NAME` (change-one-thing exercise, validation only);
`--review CSV --tag NAME` (optional human-reviewed pool, validation only). Only the canonical mode freezes.
"""
# ruff: noqa: E501
from __future__ import annotations

import argparse
import csv
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402


def load_features(work: Path, *, with_synthetic: bool = True) -> dict:
    f = work / "features"
    feats = {name: dict(np.load(f / f"{name}.npz")) for name in ("train", "val")}
    empty = {"candidate_id": np.array([], dtype=str), "label": np.array([], dtype=np.int64), "det": np.zeros((0, core.FEATURE_DIM), np.float32), "views": np.zeros((0, core.N_VIEWS, core.FEATURE_DIM), np.float32)}
    feats["synthetic"] = dict(np.load(f / "synthetic.npz")) if with_synthetic else empty
    return feats


def pools(labels: np.ndarray, k: int) -> tuple[list[int], list[int]]:
    sizes = [int((labels == c).sum()) for c in range(k)]
    offsets = [int(sum(sizes[:c])) for c in range(k)]
    if any(np.any(labels[offsets[c] : offsets[c] + sizes[c]] != c) for c in range(k)):
        raise core.ContractError("feature rows are not grouped by class")
    return sizes, offsets


def slot_features(feats: dict, sched: dict, src: dict, real_off: list[int], synth_off: list[int]) -> np.ndarray:
    cls, index, view, synthetic = sched["cls"], src["index"], src["view"], src["synthetic"]
    x = np.empty((len(cls), core.FEATURE_DIM), dtype=np.float32)
    real = ~synthetic
    rows = np.asarray(real_off)[cls] + index
    det = view < 0
    x[real & det] = feats["train"]["det"][rows[real & det]]
    x[real & ~det] = feats["train"]["views"][rows[real & ~det], view[real & ~det]]
    if synthetic.any():
        srows = np.asarray(synth_off)[cls[synthetic]] + index[synthetic]
        x[synthetic] = feats["synthetic"]["views"][srows, view[synthetic]]
    return x


def train_head(x: np.ndarray, y: np.ndarray, init: dict, val_x: np.ndarray, val_y: np.ndarray, k: int) -> dict:
    import torch

    torch.use_deterministic_algorithms(True)
    head = torch.nn.Linear(core.FEATURE_DIM, k)
    head.load_state_dict({n: t.clone() for n, t in init.items()})
    opt = torch.optim.AdamW(head.parameters(), lr=core.HEAD["learning_rate"], weight_decay=core.HEAD["weight_decay"])
    xt, yt, vx = torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(val_x)
    batch, updates = core.HEAD["batch_size"], core.HEAD["updates_per_epoch"]
    curve, best = [], None
    for epoch in range(1, core.HEAD["epochs"] + 1):
        losses = []
        for u in range(updates):
            start = ((epoch - 1) * updates + u) * batch
            loss = torch.nn.functional.cross_entropy(head(xt[start : start + batch]), yt[start : start + batch])
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        with torch.no_grad():
            pred = head(vx).argmax(1).numpy()
        m = core.metrics_from_confusion(core.confusion_matrix(val_y, pred, k), [str(i) for i in range(k)])
        curve.append({"epoch": epoch, "train_loss": round(float(np.mean(losses)), 5), "val_macro_f1": m["macro_f1"], "val_balanced_accuracy": m["balanced_accuracy"]})
        if best is None or m["macro_f1"] > best["val_macro_f1"]:  # strict: the earlier epoch wins a tie
            best = {"epoch": epoch, "val_macro_f1": m["macro_f1"], "weight": head.weight.detach().numpy().copy(), "bias": head.bias.detach().numpy().copy()}
    return {**best, "curve": curve}


def read_review(path: Path, candidates: list[str]) -> set[str]:
    with open(path, encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    known, accepted = set(candidates), set()
    for n, row in enumerate(rows, start=2):
        cid, decision = row.get("candidate_id", "").strip(), row.get("decision", "").strip().lower()
        if cid not in known:
            continue  # the template lists every attempt; rejected attempts have no features
        if decision not in ("accept", "reject", "uncertain"):
            raise core.ContractError(f"{path} line {n}: decision {decision!r} must be accept, reject or uncertain")
        if decision == "accept":
            accepted.add(cid)
    return accepted


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("config")
    parser.add_argument("--synthetic-probability", type=float, default=None)
    parser.add_argument("--review", default=None)
    parser.add_argument("--tag", default=None)
    parser.add_argument("--preview", action="store_true")
    args = parser.parse_args()
    cfg = core.read_json(args.config)
    base, work, out = Path(cfg["base_dir"]), Path(cfg["work_dir"]), Path(cfg["out_dir"])
    canonical = args.synthetic_probability is None and args.review is None and not args.preview
    if args.preview:
        args.tag = "real_only_preview"
    started = time.time()
    import torch

    data = core.read_json(out / "data_manifest.json")
    order = data["class_order"]
    k = len(order)
    seeds = [int(s) for s in cfg["seeds"]]
    feats = load_features(work, with_synthetic=not args.preview)
    real_sizes, real_off = pools(feats["train"]["label"], k)
    gen_status = core.read_json(out / "generation_status.json") if not args.preview else {"complete": False, "shortfall": ["not generated yet (preview)"]}
    synth = feats["synthetic"]
    if args.review:
        accepted = read_review(Path(args.review), list(synth["candidate_id"]))
        keep = np.array([c in accepted for c in synth["candidate_id"]], dtype=bool)
        synth = {key: value[keep] for key, value in synth.items()}
        feats["synthetic"] = synth
        print(f"human-review extension: {int(keep.sum())} accepted candidates of {len(keep)}")
    synth_sizes = [int((synth["label"] == c).sum()) for c in range(k)]
    synth_off = [int(sum(synth_sizes[:c])) for c in range(k)]
    if len(synth["label"]) and np.any(np.diff(synth["label"]) < 0):
        raise core.ContractError("synthetic feature rows are not grouped by class")
    probability = core.SYNTHETIC_PROBABILITY if args.synthetic_probability is None else args.synthetic_probability
    run_c = gen_status["complete"] or not canonical
    arms = list(core.ARMS if canonical else ("A",) if args.preview else ("C",))
    if canonical and not run_c:
        arms.remove("C")
        print("generation shortfall: " + "; ".join(gen_status["shortfall"]) + " -> arm C is not run and no synthetic comparison is reported")
    val_x, val_y = feats["val"]["det"], feats["val"]["label"]
    results: dict = {"majority": {}}
    majority = int(np.argmax(real_sizes))
    majority_metrics = core.metrics_from_confusion(core.confusion_matrix(val_y, np.full_like(val_y, majority), k), order)
    heads_dir = work / "heads" / (args.tag or "canonical")
    heads_dir.mkdir(parents=True, exist_ok=True)
    from safetensors.numpy import save_file

    for seed in seeds:
        sched = core.slot_schedule(seed, real_sizes, synth_sizes)
        torch.manual_seed(seed)
        init = {n: t.detach().clone() for n, t in torch.nn.Linear(core.FEATURE_DIM, k).state_dict().items()}
        results["majority"][str(seed)] = {"class": order[majority], "val": majority_metrics}
        for arm in arms:
            src = core.arm_sources(sched, arm, synth_probability=probability, synth_sizes=synth_sizes)
            x = slot_features(feats, sched, src, real_off, synth_off)
            fit = train_head(x, sched["cls"], init, val_x, val_y, k)
            pred = (val_x @ fit["weight"].T + fit["bias"]).argmax(1)
            val_metrics = core.metrics_from_confusion(core.confusion_matrix(val_y, pred, k), order)
            path = heads_dir / f"{arm}_s{seed}.safetensors"
            save_file({"weight": fit["weight"].astype(np.float32), "bias": fit["bias"].astype(np.float32)}, str(path))
            results.setdefault(arm, {})[str(seed)] = {
                "selected_epoch": fit["epoch"], "val": val_metrics, "curve": fit["curve"], "head": str(path.relative_to(base)),
                "synthetic_slot_fraction": float(src["synthetic"].mean()), "synthetic_share_of_defect_slots": float(src["synthetic"][sched["cls"] > 0].mean()),
            }
            print(f"seed {seed} arm {arm}: epoch {fit['epoch']:>2} val macro-F1 {val_metrics['macro_f1']:.4f} (synthetic share of defect slots {results[arm][str(seed)]['synthetic_share_of_defect_slots']:.3f})")
    if not canonical:
        body = {"tag": args.tag, "arms": arms, "synthetic_probability": None if args.preview else probability, "review_csv": args.review, "synthetic_sizes": synth_sizes, "evidence": "validation only; exploratory; does not alter the canonical frozen results", "results": results}
        core.write_json(out / "extensions" / f"{args.tag}.json", body)
        print(core.canonical_json({"tag": args.tag, "val_macro_f1": {a: {s: v["val"]["macro_f1"] for s, v in results[a].items()} for a in arms}, "majority_val_macro_f1": majority_metrics["macro_f1"]}))
        return
    # Export choice: canonical seed, arm with the best validation macro-F1, ties to A, then B, then C.
    canon = str(core.CANONICAL_SEED if core.CANONICAL_SEED in seeds else seeds[0])
    ranked = sorted(arms, key=lambda a: (-results[a][canon]["val"]["macro_f1"], core.ARMS.index(a)))
    selection = {"seed": int(canon), "arm": ranked[0], "val_macro_f1": results[ranked[0]][canon]["val"]["macro_f1"], "epoch": results[ranked[0]][canon]["selected_epoch"], "rule": "canonical seed; highest validation macro-F1; ties favour A, then B, then C"}
    record = {
        "question": "Under a fixed real-data budget, classifier, class-sampling policy and training budget, does substituting synthetic defect examples into training improve real-test macro-F1 compared with conventional augmentation?",
        "primary_contrast": "C - B", "contextual_contrast": "B - A", "primary_metric": "macro-F1 on the test partition",
        "class_order": order, "budget": data["budget"], "split_version": data["split"]["version"], "split_sha256": data["split"]["manifest_sha256"],
        "seeds": seeds, "smoke_run": len(seeds) < 3, "head": core.HEAD, "synthetic_probability": probability,
        "slot_policy": "class-balanced batches (11/11/10 rotating); normal slots always real; in arm C each defect slot draws its synthetic pool with the stated probability; A uses deterministic features, B and C draw one of four fixed views",
        "arms_run": arms, "generation_status": gen_status, "real_pool_sizes": real_sizes, "synthetic_pool_sizes": synth_sizes,
        "preprocessing": core.PREPROCESSING, "augment": core.AUGMENT, "features": core.read_json(work / "state" / "features_stage.json"),
        "validation": results, "selection": selection, "majority_class": order[majority], "core_version": core.CORE_VERSION,
    }
    referenced = {name: out / name for name in ("data_manifest.json", "split_manifest.csv", "caption_records.jsonl", "prompts.json", "generation_manifest.jsonl", "generation_status.json", "synthetic_training_overlap.json", "augmentation_views.jsonl", "synthetic_augmentation_views.jsonl")}
    referenced.update({f"features/{n}": work / "features" / f"{n}.npz" for n in ("train", "synthetic", "val")})
    referenced.update({f"heads/{p.stem}": p for p in sorted(heads_dir.glob("*.safetensors"))})
    # The thumbnail signatures used by the post-freeze overlap audit, and the code that defines preprocessing,
    # fitting, evaluation and export, are bound too: changing any of them after this point fails verify_frozen.
    referenced["audit/signatures"] = work / "audit" / "signatures.npz"
    referenced.update(core.source_files(Path(__file__).resolve().parent))
    frozen = core.freeze_record(out / "experiment_config.json", record, referenced, base)
    print(f"experiment frozen: record SHA-256 {frozen['record_sha256'][:16]}..., {len(frozen['frozen_files'])} inputs pinned; export choice {selection} ({time.time() - started:.1f} s)")


if __name__ == "__main__":
    core.run_main(main)


In [ ]:
run_stage('features_real', 'lab', 'stage_features.py', '--part', 'real')
show('augmentation_views.png')
run_stage('preview', 'lab', 'stage_fit.py', '--preview')
PREVIEW = load('extensions/real_only_preview.json')
rows = []
for arm in ('majority', 'A'):
    for seed, r in PREVIEW['results'][arm].items():
        v = r['val']
        rows.append([arm, seed, fmt(v['accuracy']), fmt(v['balanced_accuracy']), fmt(v['macro_f1']), *[fmt(v['per_class'][c]['recall']) for c in order]])
table(['arm', 'seed', 'val accuracy', 'val balanced acc.', 'val macro-F1', *[f'recall {c}' for c in order]], rows)


**What to notice.** The majority baseline's accuracy is about 0.93 while its macro-F1 is about 0.32 (F1 of roughly 0.96 for `normal` and 0 for both defects, averaged). High accuracy with zero defect recall is exactly the failure an inspection system cannot afford. The real-only head trades some `normal` accuracy for defect recall. Look at the augmentation sheet: a flip or a ±10% brightness/contrast change leaves every scratch and spot visible, which is the evidence that these transforms preserve labels. The three seeds of arm A differ only in the head's initial weights and the order of training draws, so their spread is a first look at noise that has nothing to do with the method.

These are validation numbers from the preview. The canonical arms are fitted again in Section 8 with identical settings, and only Section 9 touches the test set.


> **Check your reasoning.** A colleague reports 94% accuracy for a defect classifier on this dataset. What would you ask before being impressed?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Ask for per-class recall and macro-F1, the class distribution of the evaluation set, and whether the evaluation images were held out by group. At about 93% `normal`, a classifier that finds no defects already scores 93% accuracy, so 94% could mean almost nothing.

</details>


## 4 · Phi-4 describes training exemplars: observed versus guessed  <sub>(core concept)</sub>

Phi-4-multimodal-instruct (Abouelenin et al., 2025) is a 5.6 B-parameter model that reads images and text. Here it is **frozen** and loaded 4-bit (NF4, float16 compute; the vision encoder, projector, embeddings and the checkpoint's own LoRA stay float16). It sees three exemplars per class, all drawn from the selected training images, and it is **not told the label**. Decoding is greedy, at most 96 new tokens per image.

The fixed instruction asks for visible texture, illumination and the shape of any mark, offers `material: unknown`, and forbids guessing alloy, process or severity:

> Describe only what is visible in this grayscale close-up photograph of a surface, in at most three short sentences. Mention the texture, the illumination, and the shape and size of any visible mark. If you cannot tell what the material is, write 'material: unknown'. Do not guess the alloy, the manufacturing process or how severe any mark is.

**Trust boundary.** The checkpoint `microsoft/Phi-4-multimodal-instruct` at the immutable revision `93f923e1…` implements its model, configuration and processor in five Python files inside the model repository, and `transformers 4.48.2` has no built-in implementation, so `trust_remote_code=True` is unavoidable. The stage refuses to run unless `ALLOW_PHI4_REMOTE_CODE` is `True`, pins every file (weights, configuration, tokenizer and the five code files) by SHA-256, and verifies them before `transformers` imports anything. Pinning bounds *which* code runs; it does not make the code safe by itself. The model is MIT-licensed. Its weights (11.2 GB) are deleted after this stage by default.


The next cell saves the Phi-4 description stage; the cell after it runs it in the `phi4` environment.


In [ ]:
%%writefile work/sdi_capstone/stage_phi4.py
"""Stage 2 (phi4 environment): describe the selected training exemplars with Phi-4-multimodal-instruct (frozen, NF4)."""
# ruff: noqa: E501
from __future__ import annotations

import gc
import shutil
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"
MODEL_REVISION = "93f923e1a7727d1c4f446756212d9d3e8fcc5d81"  # immutable commit; the same pin as phi4-multimodal-pipeline
MODEL_LICENSE = "MIT"
# (path, bytes, sha256) for every file the image path needs, from the pipeline's committed snapshot manifest.
FILES = (
    ("LICENSE", 1141, "c2cfccb812fe482101a8f04597dfc5a9991a6b2748266c47ac91b6a5aae15383"),
    ("added_tokens.json", 249, "d4f2aceb0f20b71dd1f4bcc7e052e4412946bf281840b8f83d39f259571af486"),
    ("config.json", 4631, "49e1c05f93d43d7f17715b779a2576235b019f587285d7d914e5b05156253f62"),
    ("configuration_phi4mm.py", 11014, "bd9609bd47ba0c87788011e5158a8bd3e1e93165a82a9b764eb7cb048006c949"),
    ("generation_config.json", 190, "757daa0d0e89171fe48fc3286341833e95b90bdb7dd3b02a2f8920fb09f85a38"),
    ("merges.txt", 2418348, "856ce61180bb689282eed6b3a6838bb1f438399be23aefe9d20eb379791fb4ad"),
    ("model-00001-of-00003.safetensors", 4997504848, "c46bb03332d82f6a3eaf85bd20af388dd4d4d68b198c2203c965c7381a466094"),
    ("model-00002-of-00003.safetensors", 4952333128, "b3e812c0c8acef4e7f5e34d6c9f77a7640ee4a2b93ea351921365ac62f19918d"),
    ("model-00003-of-00003.safetensors", 1199389232, "7be96b7339303752634b202d3f377bcf312a03046586eca6cea23347ace1e65a"),
    ("model.safetensors.index.json", 239890, "b67dbc7062e1ccf472faba4222d631dc42929c827fbdaed1ec8e34fe0601819a"),
    ("modeling_phi4mm.py", 116057, "e2b44eb7a66d6cc54524cee1ff9ba92d0658d435ea8900329ea0dbdb85c6439d"),
    ("preprocessor_config.json", 482, "9db19b9663fb86f04c0f11d3a9b7f65a19f13d4543fb4a15bd33f82b0c92d64f"),
    ("processing_phi4mm.py", 32775, "84914d3e12256b4e2186e040c9830c11408468b6774f42afe85e6f8de2626d50"),
    ("processor_config.json", 121, "798fc4cd09c067053af27f07f0d2b329b471c5b2eb923ceb06efa41dee660c05"),
    ("special_tokens_map.json", 473, "57491904f8680d4b52ed440f1f7ba48cad1c31ecf3eb453b03484e6ff4723ae8"),
    ("speech_conformer_encoder.py", 110521, "3742827e945732cc5deea4a95e14004da037044431a94e3f3fac26239e614e3a"),
    ("tokenizer.json", 15524479, "4c1b9f641d4f8b7247b8d5007dd3b6a9f6a87cb5123134fe0d326f14d10c0585"),
    ("tokenizer_config.json", 3248, "a5da2e45718db78924ad5135a58a80b0303596acf54a1dc5c912c98436ddcaf3"),
    ("vision_siglip_navit.py", 78218, "7d5c053341ee9c099126fe675d5dcdc0ed5c0246f92fffdec78a1ab2f804e28d"),
    ("vocab.json", 3910310, "6cb65a857824fa6615bb1782d95d882617a8bbce1da0317118586b36f39e98bd"),
)
# Upstream Python that transformers executes under trust_remote_code=True; each is digest-verified above first.
REMOTE_CODE_FILES = ("configuration_phi4mm.py", "modeling_phi4mm.py", "processing_phi4mm.py", "speech_conformer_encoder.py", "vision_siglip_navit.py")
# Language-model linears load 4-bit; the vision encoder, projector, embeddings and the checkpoint's LoRA stay float16.
NF4_SKIP_MODULES = ("lm_head", "embed_tokens", "embed_tokens_extend", "image_embed", "audio_embed", "img_processor", "encoder", "lora_A", "lora_B")
INSTRUCTION = (
    "Describe only what is visible in this grayscale close-up photograph of a surface, in at most three short sentences. "
    "Mention the texture, the illumination, and the shape and size of any visible mark. "
    "If you cannot tell what the material is, write 'material: unknown'. "
    "Do not guess the alloy, the manufacturing process or how severe any mark is."
)
MAX_NEW_TOKENS = 96
DECODING = {"do_sample": False, "strategy": "greedy", "max_new_tokens": MAX_NEW_TOKENS, "attention": "eager", "quantization": "nf4 (bitsandbytes, double quantisation, float16 compute)"}


def stage_and_verify(root: Path) -> dict:
    from huggingface_hub import hf_hub_download

    fetched = 0
    for rel, size, _ in FILES:
        path = root / rel
        if not path.is_file() or path.stat().st_size != size:
            print(f"  fetching {rel} ({size / 1e9:.2f} GB)")
            hf_hub_download(MODEL_ID, rel, revision=MODEL_REVISION, local_dir=str(root))
            fetched += size
    for rel, size, digest in FILES:
        path = root / rel
        if path.stat().st_size != size or core.sha256_file(path) != digest:
            raise core.ContractError(f"{MODEL_ID}@{MODEL_REVISION}: {rel} fails its pinned digest; refusing to load (no fallback model is used)")
    print(f"verified {len(FILES)} files against their pinned SHA-256 digests")
    return {"downloaded_bytes": fetched, "files": len(FILES), "total_bytes": sum(s for _, s, _ in FILES)}


def install_adapter_switch(model) -> None:
    """The checkpoint routes images through its built-in 'vision' LoRA and calls set_lora_adapter on every forward.
    Upstream's version also flips requires_grad on 4-bit tensors; this replacement only switches the active adapter."""
    from peft.tuners.tuners_utils import BaseTunerLayer

    layers = [m for m in model.modules() if isinstance(m, BaseTunerLayer)]

    def switch(name: str) -> None:
        for m in layers:
            m._active_adapter = [name]
            m._disable_adapters = False

    def unset() -> None:
        for m in layers:
            m._disable_adapters = True

    model.set_lora_adapter = switch
    model.unset_lora_adapter = unset


def main(cfg_path: str) -> None:
    cfg = core.read_json(cfg_path)
    work, out = Path(cfg["work_dir"]), Path(cfg["out_dir"])
    if not cfg["allow_phi4_remote_code"]:
        raise SystemExit("Phi-4-multimodal ships its model code as Python files in the model repository. Set ALLOW_PHI4_REMOTE_CODE = True after reviewing the trust boundary described above the cell.")
    rows = [r for r in core.read_split_manifest(out / "split_manifest.csv") if r["exemplar"]]
    if any(r["split"] != "train" or not r["selected"] for r in rows):
        raise core.ContractError("a captioning exemplar is not a selected training image; refusing to describe it")
    order = core.read_json(out / "data_manifest.json")["class_order"]
    rows.sort(key=lambda r: (order.index(r["label"]), r["budget_rank"]))
    data_root = work / "data" / ("sdi" if cfg["data_source"] == "bosch" else "byod")
    root = work / "weights" / "phi4-multimodal-instruct"
    t0 = time.time()
    staging = stage_and_verify(root)
    stage_seconds = round(time.time() - t0, 1)

    import torch
    from PIL import Image
    from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig, GenerationConfig

    if not torch.cuda.is_available():
        raise SystemExit("Phi-4 is loaded 4-bit with bitsandbytes and needs a CUDA GPU: Runtime > Change runtime type > T4 GPU")
    torch.manual_seed(0)
    t1 = time.time()
    processor = AutoProcessor.from_pretrained(str(root), trust_remote_code=True)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True, llm_int8_skip_modules=list(NF4_SKIP_MODULES))
    model = AutoModelForCausalLM.from_pretrained(str(root), trust_remote_code=True, device_map="cuda", _attn_implementation="eager", torch_dtype=torch.float16, quantization_config=quant).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    install_adapter_switch(model)
    generation_config = GenerationConfig.from_pretrained(str(root))
    load_seconds = round(time.time() - t1, 1)
    print(f"loaded 4-bit in {load_seconds} s; GPU memory allocated {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
    prompt = f"<|user|><|image_1|>{INSTRUCTION}<|end|><|assistant|>"
    records = []
    for r in rows:
        path = data_root / r["relpath"]
        image = Image.open(path).convert("RGB")
        inputs = processor(text=prompt, images=[image], return_tensors="pt").to("cuda")
        t = time.time()
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, generation_config=generation_config)
        new_tokens = generated[:, inputs["input_ids"].shape[1]:]
        text = processor.batch_decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        n = int(new_tokens.shape[1])
        eos = generation_config.eos_token_id
        eos_ids = set(eos if isinstance(eos, list | tuple) else [eos]) - {None}
        last = int(new_tokens[0, -1]) if n else None
        stop_reason = "end_of_sequence" if last in eos_ids else ("max_new_tokens" if n >= MAX_NEW_TOKENS else "other")
        records.append({
            "image_id": r["image_id"], "label": r["label"], "exemplar_rank": r["budget_rank"], "split": r["split"],
            "image_sha256": r["file_sha256"], "instruction": INSTRUCTION, "response": text, "new_tokens": n,
            "stop_reason": stop_reason, "eos_token_ids": sorted(eos_ids), "decoding": DECODING, "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
            "label_shown_to_model": False, "seconds": round(time.time() - t, 2),
        })
        print(f"[{r['label']} #{r['budget_rank']}] {r['image_id']}: {text}")
    core.write_jsonl(out / "caption_records.jsonl", records)
    summary = {
        "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "license": MODEL_LICENSE, "remote_code_files": {rel: d for rel, _, d in FILES if rel in REMOTE_CODE_FILES}},
        "staging": staging, "stage_seconds": stage_seconds, "load_seconds": load_seconds,
        "peak_gpu_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
        "records": len(records), "decoding": DECODING,
    }
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    if cfg["delete_model_weights_after_use"]:
        shutil.rmtree(root, ignore_errors=True)
        summary["weights_deleted_after_use"] = True
    core.write_json(work / "state" / "phi4_stage.json", summary)
    print(core.canonical_json({k: summary[k] for k in ("stage_seconds", "load_seconds", "peak_gpu_allocated_gib", "records")}))


if __name__ == "__main__":
    core.run_main(main, sys.argv[1])


In [ ]:
run_stage('phi4', 'phi4', 'stage_phi4.py')
CAPTIONS = load('caption_records.jsonl')
table(['class', '#', 'image', 'tokens', 'stop reason', 'description'], [[r['label'], r['exemplar_rank'], r['image_id'], r['new_tokens'], r['stop_reason'], r['response'].replace('|', '/')] for r in CAPTIONS])


**What to notice.** Read each description and sort its claims into three bins: *observed* (a line, a bright blob, horizontal banding, even lighting), *inferred but plausible* (a "scratch" where a thin bright line is visible), and *guessed* (a material, a process, a cause, a severity). Descriptions of `normal` images sometimes report a mark, and defect descriptions sometimes miss one; the model was not told the label. The `stop reason` column says how each description ended: `end_of_sequence` means the model finished; `max_new_tokens` means it was cut off at the 96-token limit (a description can use all 96 tokens and still end on its own, so the token count alone does not show truncation). Greedy decoding makes the text repeatable for the same model and inputs, not correct.


> **Check your reasoning.** Why is it important that Phi-4 only saw selected training images, and never validation or test images?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Anything that shapes the generated images (descriptions, prompts, which candidates are kept) is part of training. If held-out images informed those choices, information about the test set would flow into training through the prompts, and the final comparison would no longer be an independent test.

</details>


## 5 · From descriptions to bounded prompts  <sub>(engineering)</sub>

Model output never becomes a prompt verbatim. `compile_prompts` (core section *Descriptions to bounded prompts*) searches each description for a fixed whitelist of texture, illumination and shape terms and fills a fixed template:

`close-up grayscale industrial inspection photograph of a flat manufactured surface, <texture>, <illumination>, showing one <defect> defect as a <shape> mark, sharp focus, no text`

Background terms come from normal exemplar *k*, mark terms from defect exemplar *k*, giving three prompts per defect class. A description that is empty, too long or contains no whitelisted term falls back to the neutral class template, recorded as a fallback and never described as Phi-4-guided. The template never names a material. The prompts are written to `prompts.json` and are frozen from here on.


The next cell saves the prompt compilation stage.


In [ ]:
%%writefile work/sdi_capstone/stage_prompts.py
"""Stage 3 (lab environment): compile Phi-4 descriptions into bounded FLUX prompts and freeze them."""
# ruff: noqa: E501
from __future__ import annotations

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402


def main(cfg_path: str) -> None:
    cfg = core.read_json(cfg_path)
    out = Path(cfg["out_dir"])
    order = core.read_json(out / "data_manifest.json")["class_order"]
    records = core.read_jsonl(out / "caption_records.jsonl")
    selected = {r["image_id"] for r in core.read_split_manifest(out / "split_manifest.csv") if r["selected"] and r["split"] == "train"}
    outside = sorted({r["image_id"] for r in records} - selected)
    if outside:
        raise core.ContractError(f"caption records reference images outside the selected training budget: {outside}")
    for r in records:
        terms = core.extract_terms(r["response"])
        print(f"[{r['label']} #{r['exemplar_rank']}] matched terms: {terms}")
    prompts = core.compile_prompts(records, order)
    body = {"template_base": core.PROMPT_BASE, "vocabulary": core.VOCABULARY, "term_limits": core.TERM_LIMITS, "max_prompt_chars": core.MAX_PROMPT_CHARS, "prompts": prompts}
    core.write_json(out / "prompts.json", body)
    for p in prompts:
        tag = "Phi-4-guided" if p["phi4_guided"] else f"FALLBACK ({p['fallback_reason']})"
        print(f"{p['prompt_id']:<12} {tag}\n    {p['text']}")
    print(core.canonical_json({"prompts": len(prompts), "phi4_guided": sum(p["phi4_guided"] for p in prompts), "prompts_sha256": core.sha256_file(out / "prompts.json")}))


if __name__ == "__main__":
    core.run_main(main, sys.argv[1])


In [ ]:
run_stage('prompts', 'lab', 'stage_prompts.py')
PROMPTS = load('prompts.json')['prompts']
table(['prompt', 'guided by Phi-4', 'fallback reason', 'text'], [[p['prompt_id'], p['phi4_guided'], p['fallback_reason'] or '', p['text']] for p in PROMPTS])


**What to notice.** The whitelist deliberately throws information away: a rich description and a terse one can compile to the same prompt. That is the price of a bounded, auditable interface between two models. If most prompts are fallbacks, the Phi-4 step contributed little, and the notebook says so rather than calling the result Phi-4-guided.


> **Check your reasoning.** A description says 'a deep gouge in brushed aluminium caused by tool chatter'. What can reach the prompt, and why is that a deliberate loss?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Only whitelisted terms can: here `brushed` (texture), and possibly a shape word if one is present. 'Aluminium', 'tool chatter' and 'deep' are guesses about material, cause and severity that the image cannot establish; pasting them into a prompt would let unverified claims shape the synthetic data. The bounded template trades expressiveness for an interface that can be audited and cannot carry instructions or free text from one model into another.

</details>


## 6 · FLUX generates candidate defects  <sub>(core concept)</sub>

FLUX.1 [schnell] (Black Forest Labs, 2024) is a 12 B-parameter text-to-image model distilled to produce an image in about four steps without guidance. The weights are the ungated mirror `unsloth/FLUX.1-schnell` at revision `9df3faa7…` of the identity of record `black-forest-labs/FLUX.1-schnell` at `741f7c3c…` (Apache-2.0); every one of the 23 files is verified by SHA-256 before loading. The stage never keeps the text encoders and the transformer on the GPU together: it loads CLIP-L and T5-XXL (float16), encodes the six prompts, keeps the embeddings, releases the encoders, then loads the transformer 4-bit (NF4) with a float32 VAE.

Exactly **32 candidates per defect class** are generated at 512 × 512, 4 steps, guidance 0, from the fixed seeds 100000 + *i* (scratches) and 200000 + *i* (spots); candidate *i* uses prompt *i* mod 3. Every attempt is kept with its disposition. Default eligibility checks only that the image decodes, has the right size, has finite pixel values and is not an exact duplicate of an earlier candidate or of a selected training image. It never uses classifier confidence, CLIP scores, test performance or Phi-4 approval, because filtering on any of those would quietly select for whatever makes the result look good. If either class ends with fewer than 24 usable candidates, the notebook reports a generation failure and does not present a synthetic comparison. It does not keep generating until the pictures look right.

**Predict first:** will the generated images look like the Bosch photographs? What will differ?


The next cell saves the FLUX generation stage.


In [ ]:
%%writefile work/sdi_capstone/stage_flux.py
"""Stage 4 (lab environment): generate exactly 32 FLUX.1 [schnell] candidates per defect class from fixed seeds."""
# ruff: noqa: E501
from __future__ import annotations

import contextlib
import gc
import shutil
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402

MODEL_ID = "black-forest-labs/FLUX.1-schnell"  # identity of record (gated upstream)
MODEL_REVISION = "741f7c3ce8b383c54771c7003378a50191e9efe9"
MODEL_LICENSE = "apache-2.0"
STAGING_ID = "unsloth/FLUX.1-schnell"  # ungated, unmodified mirror that serves the bytes
STAGING_REVISION = "9df3faa7ae3b6ddf0b2b69bb78616372897cc65c"
# (path, bytes, sha256) of the 23-file diffusers snapshot, from flux-schnell-generation-pipeline's committed manifest.
FILES = (
    ("model_index.json", 536, "24946df21ff25e210486b5f6b14208983a90c9c73f8d48cfa724c0e4e03f7201"),
    ("scheduler/scheduler_config.json", 274, "b129cebacf8f851867ec5c7c4d3f4bf787e232525a53becf4df5a72278a788d5"),
    ("text_encoder/config.json", 613, "d79d5c8c6ce85112a923d621a5412886ddbbb0636210fc0f72f450582e675542"),
    ("text_encoder/model.safetensors", 246144352, "893d67a23f4693ed42cdab4cbad7fe3e727cf59609c40da28a46b5470f9ed082"),
    ("text_encoder_2/config.json", 782, "9001e5a8ae0571a362f806b87b6105dd1a15c33dca237b606d2561164109beeb"),
    ("text_encoder_2/model-00001-of-00002.safetensors", 4994582224, "ec87bffd1923e8b2774a6d240c922a41f6143081d52cf83b8fe39e9d838c893e"),
    ("text_encoder_2/model-00002-of-00002.safetensors", 4530066360, "a5640855b301fcdbceddfa90ae8066cd9414aff020552a201a255ecf2059da00"),
    ("text_encoder_2/model.safetensors.index.json", 19885, "3bacec0f0cf392399d4a385908f67dd73df99c9e9cfee669f148858ba9fbdb0a"),
    ("tokenizer/merges.txt", 524619, "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"),
    ("tokenizer/special_tokens_map.json", 588, "2cdb3b8331a60c92fc1e55a13e9fd61fd2293c5a51275fdcccd62b780052530e"),
    ("tokenizer/tokenizer_config.json", 705, "6bdcee9ccce2a16ca2b4c0c5ed00b42c50ea225f4472a8c4c1e963a2902c2881"),
    ("tokenizer/vocab.json", 1059962, "e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"),
    ("tokenizer_2/special_tokens_map.json", 2543, "7a1985a994c41886db38c719d2a3d2f40606663cc19d7c5d6a85d349320e06d2"),
    ("tokenizer_2/spiece.model", 791656, "d60acb128cf7b7f2536e8f38a5b18a05535c9e14c7a355904270e15b0945ea86"),
    ("tokenizer_2/tokenizer.json", 2424235, "f5dfec163765e18e270537fe896c49f5fad74db1525641d9b255a3008b999596"),
    ("tokenizer_2/tokenizer_config.json", 20817, "1a3d2db64215ed77854dd4208aac5f8361c1b5471cabd19c0ef1472d1a895eb0"),
    ("transformer/config.json", 321, "397cfb92299488013ec3af6142a2a877366f8d2e44efbb4f3e33479e7960d3d0"),
    ("transformer/diffusion_pytorch_model-00001-of-00003.safetensors", 9962580296, "9b633dbe87316385c5b1c262bd4b5a01e3d955170661d63dcec8a01e89c0d820"),
    ("transformer/diffusion_pytorch_model-00002-of-00003.safetensors", 9949328904, "58b4434078f0c2567ddc54e3b5cbf39626ab55fbd9d5c22956e183668f535dec"),
    ("transformer/diffusion_pytorch_model-00003-of-00003.safetensors", 3870584832, "e2cbc25471ed5186e69a9b51098300cb2f612556453e38a372c851a220ed238d"),
    ("transformer/diffusion_pytorch_model.safetensors.index.json", 120822, "783f857a5872f069e75daf4a5abe5efd6ff9ec2f37d71159767910cebfe048a6"),
    ("vae/config.json", 774, "bc1e208f414a315365fbecf426838f43b87c9d5c051219e0968a56e2644b2998"),
    ("vae/diffusion_pytorch_model.safetensors", 167666902, "f5b59a26851551b67ae1fe58d32e76486e1e812def4696a4bea97f16604d40a3"),
)
SIDE = 512
STEPS = 4
GUIDANCE_SCALE = 0.0  # schnell is guidance-distilled
MAX_T5_TOKENS = 256
PRECISION = "transformer nf4 (bitsandbytes, double quantisation, float16 compute); CLIP-L and T5-XXL float16; VAE float32"
SEED_BASE = 100_000  # candidate i of defect role r uses seed SEED_BASE * r + i


def stage_and_verify(root: Path) -> dict:
    from huggingface_hub import hf_hub_download

    fetched = 0
    for rel, size, _ in FILES:
        path = root / rel
        if not path.is_file() or path.stat().st_size != size:
            print(f"  fetching {rel} ({size / 1e9:.2f} GB) from {STAGING_ID}@{STAGING_REVISION[:8]}")
            hf_hub_download(STAGING_ID, rel, revision=STAGING_REVISION, local_dir=str(root))
            fetched += size
    for rel, size, digest in FILES:
        path = root / rel
        if path.stat().st_size != size or core.sha256_file(path) != digest:
            raise core.ContractError(f"{rel} fails its pinned digest; refusing to load (no fallback model, size or precision is used)")
    print(f"verified {len(FILES)} files against their pinned SHA-256 digests")
    return {"downloaded_bytes": fetched, "files": len(FILES), "total_bytes": sum(s for _, s, _ in FILES)}


def peak_rss_gib() -> float:
    import resource

    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20


def preload_nvidia_libraries() -> None:
    """Load the pip-installed CUDA libraries globally so bitsandbytes resolves them on images whose system CUDA is older."""
    import ctypes
    import os

    for site in sys.path:
        nvidia = Path(site) / "nvidia"
        if nvidia.is_dir():
            os.environ["LD_LIBRARY_PATH"] = ":".join([str(p) for p in nvidia.glob("*/lib")] + [os.environ.get("LD_LIBRARY_PATH", "")])
            for so in sorted(nvidia.rglob("lib*.so*")):
                with contextlib.suppress(OSError):
                    ctypes.CDLL(str(so), mode=getattr(ctypes, "RTLD_GLOBAL", 0))


def main(cfg_path: str) -> None:
    cfg = core.read_json(cfg_path)
    work, out = Path(cfg["work_dir"]), Path(cfg["out_dir"])
    order = core.read_json(out / "data_manifest.json")["class_order"]
    defects = order[1:]
    prompts_path = out / "prompts.json"
    prompts = core.read_json(prompts_path)["prompts"]
    rows = core.read_split_manifest(out / "split_manifest.csv")
    real_hashes = {r["pixel_sha256"] for r in rows if r["selected"]}  # generation review may consult selected training images only
    root = work / "weights" / "flux1-schnell"
    t0 = time.time()
    staging = stage_and_verify(root)
    stage_seconds = round(time.time() - t0, 1)

    preload_nvidia_libraries()
    import torch
    from diffusers import AutoencoderKL, BitsAndBytesConfig, FlowMatchEulerDiscreteScheduler, FluxPipeline, FluxTransformer2DModel
    from PIL import Image
    from transformers import AutoTokenizer, CLIPTextModel, T5EncoderModel

    if not torch.cuda.is_available():
        raise SystemExit("FLUX.1 [schnell] is loaded 4-bit with bitsandbytes and needs a CUDA GPU")
    fp16 = torch.float16
    tokenizer = AutoTokenizer.from_pretrained(str(root / "tokenizer"))
    tokenizer_2 = AutoTokenizer.from_pretrained(str(root / "tokenizer_2"))

    # 1) Text encoders only: encode every distinct prompt, keep the embeddings on the CPU, release the encoders.
    # The checkpoints are bfloat16. Without device_map, from_pretrained converts the whole model to float16 in host
    # RAM first (9.5 GB for T5-XXL), which exhausts a 12.7 GiB Colab VM. device_map="cuda" converts and moves one
    # tensor at a time, so host memory holds at most one tensor; every model load in this stage streams this way.
    t1 = time.time()
    clip = CLIPTextModel.from_pretrained(str(root), subfolder="text_encoder", torch_dtype=fp16, device_map="cuda").eval()
    t5 = T5EncoderModel.from_pretrained(str(root), subfolder="text_encoder_2", torch_dtype=fp16, device_map="cuda").eval()
    cache, token_counts = {}, {}
    with torch.inference_mode():
        for p in prompts:
            text = p["text"]
            clip_ids = tokenizer(text, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt").input_ids
            pooled = clip(clip_ids.to("cuda"), output_hidden_states=False).pooler_output
            t5_batch = tokenizer_2(text, padding="max_length", max_length=MAX_T5_TOKENS, truncation=True, return_tensors="pt")
            embeds = t5(t5_batch.input_ids.to("cuda"), output_hidden_states=False)[0]
            cache[p["prompt_id"]] = (embeds[0].to("cpu"), pooled[0].to("cpu"))
            n_clip = len(tokenizer(text).input_ids)
            n_t5 = len(tokenizer_2(text).input_ids)
            token_counts[p["prompt_id"]] = {"clip_tokens": n_clip, "clip_limit": tokenizer.model_max_length, "clip_truncated": n_clip > tokenizer.model_max_length, "t5_tokens": n_t5, "t5_limit": MAX_T5_TOKENS, "t5_truncated": n_t5 > MAX_T5_TOKENS}
    encoder_peak = torch.cuda.max_memory_allocated() / 2**30
    encode_seconds = round(time.time() - t1, 1)
    del clip, t5
    gc.collect()
    torch.cuda.empty_cache()
    print(f"encoded {len(cache)} prompts in {encode_seconds} s (peak {encoder_peak:.2f} GiB); encoders released, {torch.cuda.memory_allocated() / 2**30:.2f} GiB still allocated; peak host RSS so far {peak_rss_gib():.2f} GiB")

    # 2) 4-bit transformer + float32 VAE; generation from the cached embeddings.
    torch.cuda.reset_peak_memory_stats()
    t2 = time.time()
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=fp16)
    transformer = FluxTransformer2DModel.from_pretrained(str(root), subfolder="transformer", quantization_config=quant, torch_dtype=fp16, device_map="cuda").eval()
    vae = AutoencoderKL.from_pretrained(str(root), subfolder="vae", torch_dtype=torch.float32, device_map="cuda").eval()
    scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(str(root), subfolder="scheduler")
    pipe = FluxPipeline(scheduler=scheduler, vae=vae, text_encoder=None, tokenizer=tokenizer, text_encoder_2=None, tokenizer_2=tokenizer_2, transformer=transformer)
    pipe.set_progress_bar_config(disable=True)
    load_seconds = round(time.time() - t2, 1)
    print(f"4-bit transformer loaded in {load_seconds} s; {torch.cuda.memory_allocated() / 2**30:.2f} GiB allocated; peak host RSS so far {peak_rss_gib():.2f} GiB")

    gen_dir = out / "generated"
    shutil.rmtree(gen_dir, ignore_errors=True)
    manifest, seen = [], set()
    t3 = time.time()
    for role, label in enumerate(defects, start=1):
        class_prompts = [p for p in prompts if p["label"] == label]
        (gen_dir / label).mkdir(parents=True, exist_ok=True)
        for i in range(core.GENERATION_CANDIDATES):
            p = class_prompts[i % len(class_prompts)]
            seed = SEED_BASE * role + i
            cand_id = f"syn_{label}_{i:02d}"
            row = {"candidate_id": cand_id, "intended_label": label, "label_status": "intended by prompt; not an independently validated annotation", "prompt_id": p["prompt_id"], "prompt": p["text"], "phi4_guided": p["phi4_guided"], "source_image_ids": p["source_image_ids"], "seed": seed, "model_id": MODEL_ID, "model_revision": MODEL_REVISION, "staging": f"{STAGING_ID}@{STAGING_REVISION}", "precision": PRECISION, "width": SIDE, "height": SIDE, "steps": STEPS, "guidance_scale": GUIDANCE_SCALE, "scheduler": "FlowMatchEulerDiscreteScheduler (upstream config)", "review_status": "not reviewed (canonical run)"}
            t = time.time()
            pixels = None
            try:
                embeds, pooled = cache[p["prompt_id"]]
                generator = torch.Generator(device="cpu").manual_seed(seed)
                with torch.inference_mode():
                    latents = pipe(prompt_embeds=embeds[None].to("cuda", fp16), pooled_prompt_embeds=pooled[None].to("cuda", fp16), num_inference_steps=STEPS, guidance_scale=GUIDANCE_SCALE, height=SIDE, width=SIDE, max_sequence_length=MAX_T5_TOKENS, generator=generator, output_type="latent").images
                    latents = FluxPipeline._unpack_latents(latents, SIDE, SIDE, pipe.vae_scale_factor).float()
                    latents = latents / vae.config.scaling_factor + vae.config.shift_factor
                    decoded = vae.decode(latents, return_dict=False)[0][0]
                pixels = decoded.permute(1, 2, 0).float().cpu().numpy()
            except Exception as exc:  # a failed attempt is recorded, never silently retried
                row["error"] = f"{type(exc).__name__}: {exc}"[:200]
            status, reason, digest = core.candidate_eligibility(pixels, side=SIDE, seen=seen, real=real_hashes)
            row.update(status=status, reason=reason, pixel_sha256=digest, seconds=round(time.time() - t, 2))
            if pixels is not None and np.isfinite(pixels).all():
                path = gen_dir / label / f"{cand_id}.png"
                Image.fromarray(core.to_uint8(pixels)).save(path)
                row.update(file=str(path.relative_to(out)), png_sha256=core.sha256_file(path))
            if status == "eligible":
                seen.add(digest)
            manifest.append(row)
            print(f"{cand_id} seed {seed} {status}{(' (' + reason + ')') if reason else ''} {row['seconds']} s")
    generate_seconds = round(time.time() - t3, 1)
    transformer_peak = torch.cuda.max_memory_allocated() / 2**30
    core.write_jsonl(out / "generation_manifest.jsonl", manifest)
    status = core.generation_status(manifest, defects)
    core.write_json(out / "generation_status.json", status)
    figures = out / "figures"
    figures.mkdir(parents=True, exist_ok=True)
    for label in defects:
        items = [(Image.open(out / r["file"]), f"{r['candidate_id'][-2:]} {r['status']}\nintended: {label}") for r in manifest if r["intended_label"] == label and r.get("file")]
        core.contact_sheet(items, title=f"FLUX candidates, intended label '{label}' (prompt label, not a verified annotation)").save(figures / f"generated_{label}.png")
    with open(out / "human_review_template.csv", "w", encoding="utf-8") as handle:
        handle.write("candidate_id,intended_label,decision,reason\n")
        for r in manifest:
            handle.write(f"{r['candidate_id']},{r['intended_label']},,\n")
    summary = {"model": {"id": MODEL_ID, "revision": MODEL_REVISION, "license": MODEL_LICENSE, "staging": {"repo": STAGING_ID, "revision": STAGING_REVISION}}, "staging": staging, "stage_seconds": stage_seconds, "encode_seconds": encode_seconds, "encoder_peak_gpu_gib": round(encoder_peak, 2), "load_seconds": load_seconds, "generate_seconds": generate_seconds, "transformer_peak_gpu_gib": round(transformer_peak, 2), "peak_host_rss_gib": round(peak_rss_gib(), 2), "token_counts": token_counts, "prompts_sha256": core.sha256_file(prompts_path), "generation_status": status, "precision": PRECISION, "steps": STEPS, "side": SIDE}
    del pipe, transformer, vae
    gc.collect()
    torch.cuda.empty_cache()
    if cfg["delete_model_weights_after_use"]:
        shutil.rmtree(root, ignore_errors=True)
        summary["weights_deleted_after_use"] = True
    core.write_json(work / "state" / "flux_stage.json", summary)
    print(core.canonical_json({"usable": status["usable"], "complete": status["complete"], "generate_seconds": generate_seconds, "encoder_peak_gpu_gib": summary["encoder_peak_gpu_gib"], "transformer_peak_gpu_gib": summary["transformer_peak_gpu_gib"]}))


if __name__ == "__main__":
    core.run_main(main, sys.argv[1])


In [ ]:
run_stage('flux', 'lab', 'stage_flux.py')
GEN = load('generation_manifest.jsonl')
STATUS = load('generation_status.json')
print('usable candidates per class:', STATUS['usable'], '| complete:', STATUS['complete'], '|', '; '.join(STATUS['shortfall']) or 'no shortfall')
table(['status', 'reason', 'count'], [[s, r or '', sum(1 for g in GEN if g['status'] == s and g['reason'] == r)] for s, r in sorted({(g['status'], g['reason']) for g in GEN})])
for label in order[1:]:
    show(f'generated_{label}.png')


**What to notice, and questions to answer while looking (execution does not pause):**

1. Do the candidates share the Bosch images' horizontal banding, grain and lighting, or do they look like a generic "metal surface"? This gap between generated and real images is **domain mismatch**.
2. In how many `scratches` candidates can you actually see a scratch? A spot? Is any candidate plainly the wrong class, or defect-free?
3. The label under each thumbnail is the *intended* label. If you disagree with some, that is label noise the classifier will learn from in arm C.
4. Would you trust these images more if they looked more realistic? Why is realism not the same as usefulness?

An optional human-review extension (Section 13) lets you record accept / reject / uncertain for each candidate and run a separately identified experiment. It never changes the canonical results.


> **Check your reasoning.** Why does the notebook refuse to filter candidates by how confidently a classifier recognises them?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Filtering by classifier confidence keeps only the generated images that already look like what a classifier expects. That makes the synthetic pool easier and more similar to the real training data, and it uses a model's judgement where the experiment claims to be testing unfiltered generation. Tuned against validation or test scores, it would also leak held-out information into training. A fixed, minimal eligibility rule keeps the pool honest; human review is allowed only as a separately labelled extension.

</details>


## 7 · Synthetic features and the synthetic-to-training audit  <sub>(evaluation practice)</sub>

Each eligible candidate is converted to grayscale, letterboxed from 512 px to 224 px and passed through the same frozen ResNet-18, with four fixed augmented views, exactly as the real images were. Before any fitting, every candidate's thumbnail is compared with every **training-partition** image: a correlation of 0.98 or more would mean FLUX reproduced a real training photograph. Matches are reported, not used to filter.


In [ ]:
run_stage('features_synthetic', 'lab', 'stage_features.py', '--part', 'synthetic')
AUDIT = load('synthetic_training_overlap.json')
print(f"highest candidate-to-training correlation: {fmt(AUDIT['max_correlation'])}; flagged at >= {AUDIT['threshold']}: {len(AUDIT['flagged'])}")


**What to notice.** A text-to-image model that never saw these photographs is not expected to reproduce one, so the highest correlation should sit well below 0.98. It is still worth checking, because a generator trained on web images may have seen public datasets; the notebook cannot rule out pretraining overlap for Phi-4, FLUX or ResNet-18.


## 8 · Three matched arms, validation selection and a frozen experiment  <sub>(evaluation practice)</sub>

This is the controlled comparison. For each seed (17, 29, 43), `slot_schedule` (core section *Matched training schedule*) draws every random number the three arms need **once**: 30 epochs × 32 updates × 32 slots = 30,720 training slots, each batch holding 11 / 11 / 10 slots of the three classes, rotating which class gets 10. Every slot has a class, a real-image index, a view index, a coin and a synthetic-image index. The arms then differ only in where a slot's features come from:

- **A** uses the real image's deterministic features;
- **B** uses one of the real image's four fixed augmented views;
- **C** does the same as B, except that a **defect** slot whose coin is below 0.5 uses a synthetic candidate's view instead. Normal slots are always real.

So every arm sees the same number of updates, the same class balance and the same head initialisation per seed. C does not get more training; it gets a different distribution of defect examples. The head is fitted with AdamW (learning rate 0.001, weight decay 0.0001), batch 32, on the CPU with deterministic algorithms. After each epoch it is scored on **validation**; the epoch with the highest validation macro-F1 is kept (the earlier epoch wins a tie). Among the arms fitted with the canonical seed 17, the one with the highest validation macro-F1 is chosen for export (ties go to A, then B, then C).

Then the experiment is **frozen**: `experiment_config.json` records every setting and the SHA-256 of every input (split, prompts, generation manifest, features, heads, the thumbnail signatures used by the overlap audit) and of the code that defines preprocessing, fitting, evaluation and export (the shared core and every stage script). The test, export and reload stages refuse to run if any of them changes, and every held-out image is re-checked against its recorded digests before it is scored.

**Predict first:** on validation, will C beat B for every seed, some seeds, or none?


The next three cells save the stages that run **after** the freeze: the test stage (Section 9), the export stage and the fresh-process reload stage (Section 11). They are saved here, before the fit, so that the freeze binds their code together with the core, the data, feature and fitting stages: changing any of them afterwards makes the test, export and reload stages refuse to run. The test stage starts by verifying the frozen record and re-checking every test image against its recorded digests.


In [ ]:
%%writefile work/sdi_capstone/stage_evaluate.py
"""Stage 7 (lab environment): verify the frozen experiment, then score every arm and seed once on the test partition."""
# ruff: noqa: E501
from __future__ import annotations

import csv
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402
import stage_features as sf  # noqa: E402


def main(cfg_path: str) -> None:
    cfg = core.read_json(cfg_path)
    base, work, out = Path(cfg["base_dir"]), Path(cfg["work_dir"]), Path(cfg["out_dir"])
    started = time.time()
    record = core.verify_frozen(out / "experiment_config.json", base)
    print(f"frozen record verified ({len(record['frozen_files'])} inputs unchanged); scoring the test partition once")
    import torch
    from PIL import Image
    from safetensors.numpy import load_file

    order, seeds, arms = record["class_order"], record["seeds"], record["arms_run"]
    k = len(order)
    rows = core.read_split_manifest(out / "split_manifest.csv")
    data_root = work / "data" / ("sdi" if cfg["data_source"] == "bosch" else "byod")
    test = sorted((r for r in rows if r["split"] == "test"), key=lambda r: r["image_id"])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = sf.build_backbone(sf.stage_backbone(work), device)
    # Each test image is re-verified against the frozen manifest's file and pixel digests before it is scored.
    x = sf.embed(model, [core.load_verified_gray(data_root / r["relpath"], r) for r in test], device)
    y = np.array([order.index(r["label"]) for r in test])
    groups = [r["group_id"] for r in test]

    predictions, scores, per_run = {}, {}, {}
    majority = order.index(record["majority_class"])
    for seed in seeds:
        predictions[("majority", seed)] = np.full_like(y, majority)
        scores[("majority", seed)] = np.eye(k, dtype=np.float32)[np.full_like(y, majority)]
        for arm in arms:
            head = load_file(str(base / record["validation"][arm][str(seed)]["head"]))
            logits = core.head_logits(x, head["weight"], head["bias"])
            predictions[(arm, seed)] = logits.argmax(1)
            scores[(arm, seed)] = core.softmax(logits)
    for (arm, seed), pred in predictions.items():
        per_run.setdefault(arm, {})[str(seed)] = core.metrics_from_confusion(core.confusion_matrix(y, pred, k), order)
    summary = {}
    for arm, runs in per_run.items():
        summary[arm] = {m: {"mean": float(np.mean([r[m] for r in runs.values()])), "min": float(np.min([r[m] for r in runs.values()])), "max": float(np.max([r[m] for r in runs.values()])), "per_seed": {s: r[m] for s, r in runs.items()}} for m in ("macro_f1", "balanced_accuracy", "accuracy")}
    contrasts = {}
    if "C" in arms:
        contrasts["C - B"] = core.paired_contrast(predictions, y, groups, arm="C", reference="B", seeds=seeds, k=k)
    contrasts["B - A"] = core.paired_contrast(predictions, y, groups, arm="B", reference="A", seeds=seeds, k=k)

    # Held-out overlap audit, run only now that everything is frozen. A consequential match is reported, not removed.
    sig = np.load(work / "audit" / "signatures.npz")
    split_of = {r["image_id"]: r["split"] for r in rows}
    held = np.array([split_of.get(i) in ("val", "test") for i in sig["image_id"]])
    generated = [r for r in core.read_jsonl(out / "generation_manifest.jsonl") if r["status"] == "eligible"]
    synth_sigs = np.stack([core.signature(Image.open(out / r["file"]).convert("L")) for r in generated]) if generated else np.zeros((0, core.SIGNATURE_SIDE**2), np.float32)
    best, where = core.max_similarity(synth_sigs, sig["signature"][held].astype(np.float32))
    held_ids = sig["image_id"][held]
    flagged = [{"candidate_id": generated[n]["candidate_id"], "closest_heldout_image": str(held_ids[where[n]]), "correlation": round(float(best[n]), 4)} for n in np.nonzero(best >= core.NEAR_DUPLICATE_THRESHOLD)[0]]
    pixel_split: dict = {}
    for r in rows:
        if r["split"] in ("train", "val", "test"):
            pixel_split.setdefault(r["pixel_sha256"], set()).add(r["split"])
    cross_pixels = sum(1 for s in pixel_split.values() if len(s) > 1)
    overlap = {"synthetic_vs_heldout": {"max_correlation": float(best.max()) if len(best) else None, "flagged": flagged}, "exact_pixel_duplicates_across_partitions": cross_pixels}
    valid = not flagged and cross_pixels == 0
    metrics = {
        "evidence": f"tutorial evidence from one grouped split ({core.read_json(out / 'data_manifest.json')['experiment_scope']}) and one generated pool; not factory-level or production performance",
        "test_counts": {c: int((y == i).sum()) for i, c in enumerate(order)},
        "arms": {a: core.ARM_NAMES[a] for a in per_run},
        "per_run": per_run, "summary": summary, "contrasts": contrasts,
        "interval_note": "approximate 95% percentile interval from 1,000 class-stratified group bootstrap resamples of the test partition, paired across arms; conditional on this split and generated pool; it does not include training or generation variability",
        "decision_rule": "argmax of the linear head's logits (no threshold tuning); scores are softmax of uncalibrated logits, not calibrated probabilities",
        "overlap_audit": overlap, "comparison_valid": valid,
        "validity_note": "" if valid else "a held-out overlap was found after freezing; the comparison is reported as invalid rather than repaired",
        "generation_complete": record["generation_status"]["complete"], "smoke_run": record["smoke_run"],
        "frozen_record_sha256": record["record_sha256"], "seconds": round(time.time() - started, 1),
    }
    core.write_json(out / "metrics.json", metrics)
    with open(out / "predictions.csv", "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, lineterminator="\n")
        writer.writerow(["arm", "seed", "image_id", "group_id", "true_label", "predicted_label", *[f"score_{c}" for c in order]])
        for (arm, seed), pred in sorted(predictions.items(), key=lambda kv: (["majority", *core.ARMS].index(kv[0][0]), kv[0][1])):
            for n, r in enumerate(test):
                writer.writerow([arm, seed, r["image_id"], r["group_id"], r["label"], order[pred[n]], *[f"{v:.6f}" for v in scores[(arm, seed)][n]]])

    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    figures = out / "figures"
    shown = ["majority", *arms]
    fig, axes = plt.subplots(len(shown), len(seeds), figsize=(3.1 * len(seeds), 2.9 * len(shown)), squeeze=False)
    for i, arm in enumerate(shown):
        for j, seed in enumerate(seeds):
            cm = np.array(per_run[arm][str(seed)]["confusion"])
            ax = axes[i][j]
            ax.imshow(np.log1p(cm), cmap="Blues")
            for (a, b), v in np.ndenumerate(cm):
                ax.text(b, a, str(v), ha="center", va="center", fontsize=9)
            ax.set_xticks(range(k), order, fontsize=7)
            ax.set_yticks(range(k), order, fontsize=7)
            ax.set_title(f"{core.ARM_NAMES[arm]}\nseed {seed}, macro-F1 {per_run[arm][str(seed)]['macro_f1']:.3f}", fontsize=8)
            ax.set_xlabel("predicted", fontsize=7)
            ax.set_ylabel("true", fontsize=7)
    fig.tight_layout()
    fig.savefig(figures / "confusion_matrices.png", dpi=110)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    for i, arm in enumerate(shown):
        values = [per_run[arm][str(s)]["macro_f1"] for s in seeds]
        ax.scatter([i] * len(values), values, color="0.2", zorder=3)
        ax.hlines(np.mean(values), i - 0.25, i + 0.25, color="tab:blue")
    ax.set_xticks(range(len(shown)), [core.ARM_NAMES[a] for a in shown], fontsize=7)
    ax.set_ylabel("test macro-F1")
    ax.set_title("Each dot is one seed; the bar is the mean", fontsize=9)
    fig.tight_layout()
    fig.savefig(figures / "macro_f1_by_arm.png", dpi=110)
    plt.close(fig)
    # Error gallery for the canonical seed (test images may be viewed now that the experiment is frozen).
    canon = record["selection"]["seed"]
    items = []
    for arm in [a for a in ("B", "C") if a in arms]:
        wrong = [n for n in range(len(test)) if predictions[(arm, canon)][n] != y[n] and y[n] != majority][:6] + [n for n in range(len(test)) if predictions[(arm, canon)][n] != y[n] and y[n] == majority][:2]
        items += [(core.load_verified_gray(data_root / test[n]["relpath"], test[n]), f"{arm}: true {order[y[n]]}\npred {order[predictions[(arm, canon)][n]]}") for n in wrong]
    if items:
        core.contact_sheet(items, title=f"Misclassified TEST images, seed {canon} (arm: true -> predicted)").save(figures / "error_gallery.png")
    print(core.canonical_json({"macro_f1_mean": {a: round(s["macro_f1"]["mean"], 4) for a, s in summary.items()}, "contrasts": {c: [round(v["mean_difference"], 4), [round(z, 4) for z in v["interval_95"]]] for c, v in contrasts.items()}, "comparison_valid": valid}))


if __name__ == "__main__":
    core.run_main(main, sys.argv[1])


In [ ]:
%%writefile work/sdi_capstone/stage_export.py
"""Stage 8 (lab environment): export the validation-selected canonical-seed classifier and its reference logits."""
# ruff: noqa: E501
from __future__ import annotations

import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402
import stage_features as sf  # noqa: E402

REFERENCE_PER_CLASS = 3


def reference_rows(rows: list[dict], order: list[str]) -> list[dict]:
    """Fixed real examples for the reload check: the first three test images of each class by image id."""
    test = sorted((r for r in rows if r["split"] == "test"), key=lambda r: r["image_id"])
    return [r for label in order for r in [x for x in test if x["label"] == label][:REFERENCE_PER_CLASS]]


def main(cfg_path: str) -> None:
    cfg = core.read_json(cfg_path)
    base, work, out = Path(cfg["base_dir"]), Path(cfg["work_dir"]), Path(cfg["out_dir"])
    record = core.verify_frozen(out / "experiment_config.json", base)
    from safetensors.numpy import load_file

    selection, order = record["selection"], record["class_order"]
    head = load_file(str(base / record["validation"][selection["arm"]][str(selection["seed"])]["head"]))
    manifest = {
        "classes": order,
        "class_index": {c: i for i, c in enumerate(order)},
        "decision_rule": core.DECISION_RULE,
        "backbone": sf.BACKBONE,
        "preprocessing": core.PREPROCESSING,
        "selection": {**selection, "arm_name": core.ARM_NAMES[selection["arm"]]},
        "experiment_record_sha256": record["record_sha256"],
        "training_data": "no images or features are embedded; the head was fitted on features of the selected training budget (and, for arm C, generated images)",
        "intended_use": "tutorial reconstruction and inference on product-A-like surface images; not a production inspection system",
    }
    rows = core.read_split_manifest(out / "split_manifest.csv")
    ref = reference_rows(rows, order)
    data_root = work / "data" / ("sdi" if cfg["data_source"] == "bosch" else "byod")
    model = sf.build_backbone(sf.stage_backbone(work), "cpu")  # CPU on both sides of the reload check
    feats = sf.embed(model, [core.load_verified_gray(data_root / r["relpath"], r) for r in ref], "cpu")
    logits = core.head_logits(feats, head["weight"], head["bias"])
    # Expected decisions are named with the frozen experiment's class order, not the artifact's own list.
    core.write_json(out / "reload_reference.json", {"device": "cpu", "tolerance": core.PARITY_TOLERANCE, "examples": [{"image_id": r["image_id"], "relpath": r["relpath"], "label": r["label"], "expected_prediction": order[int(logits[n].argmax())], "logits": [float(v) for v in logits[n]]} for n, r in enumerate(ref)]})
    manifest["reload_reference_sha256"] = core.sha256_file(out / "reload_reference.json")
    written = core.write_artifact(out / "classifier", head["weight"], head["bias"], manifest)
    core.check_artifact_semantics(written, record=record, preprocessing=core.PREPROCESSING, backbone=sf.BACKBONE)
    print(core.canonical_json({"exported": written["selection"], "head_sha256": written["files"][core.ARTIFACT_WEIGHTS]["sha256"], "reference_examples": len(ref)}))


if __name__ == "__main__":
    core.run_main(main, sys.argv[1])


In [ ]:
%%writefile work/sdi_capstone/stage_reload.py
"""Stage 9 (fresh lab process): rebuild inference from the exported files alone, check logit parity, then score new images.

Nothing from the training process is imported or unpickled: the head comes from safetensors, the backbone from its
digest-verified safetensors file, and preprocessing from the definition in sdi_core.
"""
# ruff: noqa: E501
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import sdi_core as core  # noqa: E402
import stage_features as sf  # noqa: E402

NEW_PER_CLASS = 2


def main(cfg_path: str) -> None:
    cfg = core.read_json(cfg_path)
    base, work, out = Path(cfg["base_dir"]), Path(cfg["work_dir"]), Path(cfg["out_dir"])
    weight, bias, manifest = core.load_artifact(out / "classifier")
    # Meaning before numbers: the artifact must name the frozen experiment, its class order and indices, decision
    # rule, preprocessing, backbone and selected head. A relabelled class list would leave every logit unchanged.
    record = core.verify_frozen(out / "experiment_config.json", base)
    core.check_artifact_semantics(manifest, record=record, preprocessing=core.PREPROCESSING, backbone=sf.BACKBONE)
    reference_path = out / "reload_reference.json"
    if core.sha256_file(reference_path) != manifest.get("reload_reference_sha256"):
        raise core.ContractError("reload_reference.json differs from the reference recorded in the artifact manifest")
    model = sf.build_backbone(sf.stage_backbone(work), "cpu")
    data_root = work / "data" / ("sdi" if cfg["data_source"] == "bosch" else "byod")
    rows = core.read_split_manifest(out / "split_manifest.csv")
    by_id = {r["image_id"]: r for r in rows}
    reference = core.read_json(reference_path)
    feats = sf.embed(model, [core.load_verified_gray(data_root / e["relpath"], by_id[e["image_id"]]) for e in reference["examples"]], "cpu")
    order = manifest["classes"]
    expected = np.array([e["logits"] for e in reference["examples"]], dtype=np.float32)
    parity = core.replay_check(core.head_logits(feats, weight, bias), expected, [e["expected_prediction"] for e in reference["examples"]], order)
    parity["semantic_checks"] = "classes, class_index, decision rule, preprocessing, backbone, experiment record and selected head match the frozen experiment"
    print(f"semantic checks: PASS ({parity['semantic_checks']})")
    print(f"reload parity: max |logit difference| = {parity['max_abs_logit_difference']:.2e} (tolerance {core.PARITY_TOLERANCE:.0e}); replayed decisions identical: {parity['same_predictions']} -> {'PASS' if parity['passed'] else 'FAIL'}")
    if not parity["passed"]:
        core.write_json(out / "reload_parity.json", parity)
        raise core.ContractError("the reloaded classifier does not reproduce the exported logits and decisions; the artifact is not trustworthy")

    # New-image inference: training-partition images never used for fitting, captioning or generation, plus any
    # images in NEW_IMAGE_DIR. Unknown labels stay empty. An empty cohort is recorded as not run, never as run.
    unused = [r for r in rows if r["split"] == "train" and not r["selected"]]
    chosen = []
    for label in order:
        pool = sorted((r for r in unused if r["label"] == label), key=lambda r: core.sha256_bytes(f"new|{r['file_sha256']}".encode()))
        chosen += [(r["image_id"], data_root / r["relpath"], label, "unused training-partition image", r) for r in pool[:NEW_PER_CLASS]]
    new_dir = cfg.get("new_image_dir") or ""
    rejected = []
    if new_dir:
        for p in sorted(Path(new_dir).iterdir()):
            if p.suffix.lower() in core.IMAGE_SUFFIXES and p.is_file():
                chosen.append((p.stem, p, "", "NEW_IMAGE_DIR", None))
    results = []
    for image_id, path, label, source, row in chosen:
        info = core.inspect_image(path)
        if not info["decode_ok"]:
            rejected.append({"image": str(path), "reason": info["error"]})
            continue
        image, mode = (core.load_verified_gray(path, row), info["mode"]) if row is not None else core.load_gray(path)
        score = core.softmax(core.head_logits(sf.embed(model, [image], "cpu"), weight, bias))[0]
        results.append({"image_id": image_id, "source": source, "known_label": label, "input_mode": mode, "input_size": f"{info['width']}x{info['height']}", "predicted_label": order[int(score.argmax())], **{f"score_{c}": round(float(score[i]), 6) for i, c in enumerate(order)}})
    core.write_new_predictions(out / "new_image_predictions.csv", results, order)
    status = "ran" if results else ("not_run_all_inputs_rejected" if chosen else "not_run_no_inputs")
    parity["new_image_inference"] = {"status": status, "scored": len(results), "candidates": len(chosen), "rejected": rejected}
    core.write_json(out / "reload_parity.json", parity)
    for r in results:
        print(f"{r['image_id']:<22} known={r['known_label'] or '?':<10} predicted={r['predicted_label']:<10} " + " ".join(f"{c}={r['score_' + c]:.3f}" for c in order))
    if rejected:
        print(f"rejected {len(rejected)} new image(s): {rejected}")
    if not results:
        print(f"new-image inference NOT RUN ({status}): no unused training-partition images and no usable NEW_IMAGE_DIR images. "
              "The export and reload checks above still stand; set NEW_IMAGE_DIR to score your own images.")


if __name__ == "__main__":
    core.run_main(main, sys.argv[1])


In [ ]:
run_stage('fit', 'lab', 'stage_fit.py')
RECORD = load('experiment_config.json')
table(['arm', 'seed', 'kept epoch', 'val macro-F1', 'val balanced acc.', 'synthetic share of defect slots'],
      [[a, s, r['selected_epoch'], fmt(r['val']['macro_f1']), fmt(r['val']['balanced_accuracy']), fmt(r['synthetic_share_of_defect_slots'], 2)]
       for a in RECORD['arms_run'] for s, r in RECORD['validation'][a].items()])
print('export choice:', RECORD['selection'])
print(f"frozen: {len(RECORD['frozen_files'])} inputs pinned, record SHA-256 {RECORD['record_sha256'][:16]}...; smoke run: {RECORD['smoke_run']}")


**What to notice.** About half of the defect slots in arm C are synthetic, as designed; A and B have none. Kept epochs vary by arm and seed. Validation scores were used to choose epochs and the export, so they are slightly optimistic and are *not* the evidence for the question; the test set is. If generation fell short, arm C is absent and the notebook says so instead of comparing.


> **Check your reasoning.** Why not simply add the 64 synthetic images to the training set of arm B and train for the same number of epochs?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Adding images changes several things at once: the number of defect examples, the class balance, and (with a fixed number of epochs) the number of updates. Any difference in the result could come from any of these. Substituting synthetic images into a fixed slot schedule keeps updates, batch composition, class prior and random draws identical, so a difference between C and B can be attributed to the change in the defect-image distribution.

</details>


## 9 · One final comparison on real test photographs  <sub>(evaluation practice)</sub>

The test stage first verifies the frozen record, then extracts features for the test partition (never seen until now), scores every arm and seed once, and computes:

- **macro-F1** (primary), balanced accuracy, accuracy, and per-class precision, recall, F1 and support for each arm and seed;
- the **C − B** contrast (primary) and **B − A** (context): the mean over seeds of the macro-F1 difference, with an approximate 95% interval from 1,000 **paired, class-stratified group bootstrap** resamples of the test set (every arm is scored on the same resamples; groups, not images, are resampled);
- after freezing, an **overlap audit** of every synthetic candidate against validation and test images and a check that no exact duplicate crosses partitions. A consequential match marks the comparison invalid; it is never quietly removed.

The decision rule is argmax over the head's logits. Softmax scores in `predictions.csv` are uncalibrated and are not probabilities of a defect.

Scoring the test set again after changing something, and picking the better result, would turn it into a second validation set. The notebook therefore scores it once.


In [ ]:
run_stage('evaluate', 'lab', 'stage_evaluate.py')
METRICS = load('metrics.json')
print('test counts:', METRICS['test_counts'], '| comparison valid:', METRICS['comparison_valid'], METRICS['validity_note'])
rows = []
for arm, summ in METRICS['summary'].items():
    rows.append([METRICS['arms'][arm], *[fmt(summ['macro_f1']['per_seed'][str(s)]) for s in RECORD['seeds']], fmt(summ['macro_f1']['mean']),
                 f"{fmt(summ['macro_f1']['min'])}–{fmt(summ['macro_f1']['max'])}", fmt(summ['balanced_accuracy']['mean']), fmt(summ['accuracy']['mean'])])
table(['arm', *[f'macro-F1 seed {s}' for s in RECORD['seeds']], 'mean', 'range', 'balanced acc. (mean)', 'accuracy (mean)'], rows)
for name, c in METRICS['contrasts'].items():
    lo, hi = c['interval_95']
    verdict = 'the interval includes zero: no clear difference in this run' if lo <= 0 <= hi else ('the interval is above zero' if lo > 0 else 'the interval is below zero')
    print(f"{name}: mean difference {c['mean_difference']:+.4f} (per seed {', '.join(f'{v:+.4f}' for v in c['per_seed'].values())}); "
          f"approximate 95% interval [{lo:+.4f}, {hi:+.4f}] -> {verdict}")
show('macro_f1_by_arm.png')
show('confusion_matrices.png')


**What to notice.** Read the per-seed columns before the mean: if the sign of C − B changes between seeds, the mean hides disagreement. Compare the interval's width with the difference itself. The interval is **conditional on this split and this generated pool**: it reflects which test groups happened to be drawn, not what would happen with a different split, a new set of 64 generated images or another factory. In the confusion matrices, look at where each arm's errors go: false alarms on `normal`, missed defects, or scratch/spot confusion.


> **Check your reasoning.** Why does the bootstrap resample groups rather than individual images, and why are the arms scored on the same resamples?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Images in one group are near-copies, so they are not independent evidence; resampling them one by one would pretend there is more independent data than there is and make the interval too narrow. Scoring every arm on the same resample pairs the comparison: the difference C − B is computed on identical test images each time, so variation that affects both arms equally (an unusually hard set of spot images) cancels out instead of widening the interval.

</details>


## 10 · Error analysis, uncertainty and a bounded conclusion  <sub>(interpretation)</sub>

Now that the experiment is frozen, test images may be inspected. The gallery shows misclassified test images for the canonical seed in arms B and C.


In [ ]:
show('error_gallery.png')
rows = []
for arm in METRICS['per_run']:
    r = METRICS['per_run'][arm][str(RECORD['selection']['seed'])]
    for c in order:
        pc = r['per_class'][c]
        rows.append([arm, c, fmt(pc['precision']), fmt(pc['recall']), fmt(pc['f1']), pc['support'], pc['predicted']])
table(['arm (canonical seed)', 'class', 'precision', 'recall', 'F1', 'support', 'predicted'], rows)
undefined = {a: r[str(RECORD['selection']['seed'])]['undefined'] for a, r in METRICS['per_run'].items() if r[str(RECORD['selection']['seed'])]['undefined']}
print('undefined metrics (no predictions of that class):', undefined or 'none')


**What to notice.** Precision for the majority baseline's defect classes is undefined (it never predicts them) and is reported as such, not as zero. A change of one or two spot predictions moves spot recall by about 0.05, because the test set has about 21 spots.

> **Check your reasoning.** Suppose C − B is +0.02 with an interval of [−0.01, +0.05]. Your manager asks: 'So synthetic defects work?' What do you answer?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Not established. On this split and this generated pool, the average difference favours synthetic substitution slightly, but the interval includes zero and the result may reverse with other test groups; it says nothing about other generated pools, other products or production. The honest statement is that this run found no clear evidence either way, and the next step is repeating the experiment with independently generated pools and another product (the P1 items in Section 13).

</details>

### Write your conclusion

Complete these sentences in your own words, using only numbers printed above:

1. **Task:** *On product A of the Bosch SDI dataset (or: on my own dataset, described by its source and classes), with 128 / 64 / 32 real training images and a frozen ResNet-18 with a linear head, we compared …*
2. **Principal result:** *Mean test macro-F1 was … for A, … for B and … for C; C − B was … (approximate 95% interval …), with per-seed differences of …*
3. **Baseline / reference:** *The majority baseline scored accuracy … but macro-F1 …*
4. **Important failure mode or uncertainty:** *Most remaining errors were …; the spot class rests on … test images; …*
5. **Limitations:** *One grouped split of one product without specimen metadata; one generated pool with intended, unverified labels; frozen models in 4-bit; no calibration; …*
6. **What would change my mind:** *…*

Your prediction from the start: was it right? What in the evidence would have to differ for the opposite conclusion?


## 11 · Export, fresh-process reload and new-image inference  <sub>(engineering)</sub>

The exported classifier is the validation-selected arm for the predeclared canonical seed 17, chosen from validation scores only. The artifact in `classifier/` contains `head.safetensors` (weight 3 × 512 and bias, float32) and `manifest.json` (format, class order and indices, decision rule, backbone identity with revision and SHA-256, preprocessing definition, selection record, file digests). No images or features are embedded.

The export stage also computes reference logits and the decisions they imply for nine fixed real test images on the CPU. The reload stage is a **new process** that imports nothing from training. It checks the manifest, the file set and every digest; checks the artifact's **meaning** against the frozen experiment (class names, order and indices, decision rule, preprocessing, backbone, the selected head and the experiment record); loads the head with `safetensors` (plain arrays; nothing is unpickled); rebuilds ResNet-18 from its verified weights; re-verifies each reference image against its recorded digests; recomputes the nine logits; and requires them to match within an absolute tolerance of 1e-5 **and** to name the same class for every example. It then scores new images: up to six unused training-partition images (two per class) that were never used for fitting, captioning or generation, plus any images in `NEW_IMAGE_DIR`. If your data has no spare training images and `NEW_IMAGE_DIR` is empty, this step is recorded as not run and the CSV has only its header.


In [ ]:
run_stage('export', 'lab', 'stage_export.py')
run_stage('reload', 'lab', 'stage_reload.py')
PARITY = load('reload_parity.json')
print(f"reload parity: max |logit difference| {PARITY['max_abs_logit_difference']:.2e} <= {PARITY['tolerance']:.0e}; replayed decisions identical: {PARITY['same_predictions']} -> {PARITY['passed']}")
NEW = PARITY['new_image_inference']
print(f"new-image inference: {NEW['status']} ({NEW['scored']} scored of {NEW['candidates']} candidates, {len(NEW['rejected'])} rejected)")


**What to notice.** Parity compares numbers, not just file loading. The new-image table has one row per image with a known label where available, the predicted class and one uncalibrated score per class in the artifact's class order. Unused training-partition images are distinct from the fitted images but come from the same partition; they are an inference demonstration, not additional evidence.


> **Check your reasoning.** The reload stage loaded the files without error. Why is that not enough, and what does the parity check add?

<details><summary><b>Sample answer (open after you have answered)</b></summary>

Loading proves that files parse, not that they reproduce the classifier that was evaluated. Two different checks are needed. Numbers: recomputing logits for fixed real images in a new process and requiring agreement within 1e-5 shows the tensors, the pinned backbone file and the preprocessing code reproduce the exported outputs. Meaning: a reversed class list would leave every logit identical while swapping what the columns are called, so numeric parity cannot catch it. The reload stage therefore also checks the manifest against the frozen experiment (class order and indices, decision rule, preprocessing, backbone, selected head, experiment record) and requires every replayed decision to name the same class. What neither check covers: whether the preprocessing is right for new kinds of images, or whether the scores are calibrated.

</details>


## 12 · Records: environment, run summary, limitations and attribution  <sub>(engineering)</sub>

The last default cell writes the remaining records and prints a terminal summary. `run_summary.json` holds stage times, peak child memory, peak GPU memory in use, the lowest available host memory and disk use for every stage, measured in this runtime; they are measurements of this run, not guarantees for another.


In [ ]:
# @title Write environment, run summary, limitations and attribution records
FLUX_STATE = json.loads((WORK / 'state' / 'flux_stage.json').read_text())
PHI4_STATE = json.loads((WORK / 'state' / 'phi4_stage.json').read_text())
FEATURE_STATE = json.loads((WORK / 'state' / 'features_stage.json').read_text())
environment = {'runtime': RUNTIME, 'environments': ENVIRONMENTS, 'uv': UV_VERSION, 'uv_wheel_sha256': UV_SHA256, 'python_isolated': PYTHON_REQUEST,
               'notebook': {'name': 'DIMER_Bosch_Synthetic_Defect_Augmentation_Capstone.ipynb', 'profile': 'E2E', 'mode': 'GUIDED', 'notebook_spec': '2.2'},
               'models': {'phi4': PHI4_STATE['model'], 'flux': FLUX_STATE['model'], 'backbone': FEATURE_STATE['backbone']},
               'precision': {'phi4': 'nf4 weights, float16 compute', 'flux': FLUX_STATE['precision'], 'backbone': 'float32', 'head': 'float32 on CPU, deterministic algorithms'}}
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2))
download_bytes = {'dataset': DATA['acquisition'].get('download', {}).get('downloaded_bytes'), 'phi4': PHI4_STATE['staging']['downloaded_bytes'], 'flux': FLUX_STATE['staging']['downloaded_bytes']}
C_B = METRICS['contrasts'].get('C - B')
summary = {
    'question': RECORD['question'], 'data_source': DATA_SOURCE, 'seeds': RECORD['seeds'], 'smoke_run': RECORD['smoke_run'],
    'split_sha256': DATA['split']['manifest_sha256'], 'split_matches_reference': DATA['split']['matches_reference'],
    'generation': STATUS, 'phi4_guided_prompts': sum(p['phi4_guided'] for p in PROMPTS), 'prompts': len(PROMPTS),
    'test_macro_f1_mean': {a: s['macro_f1']['mean'] for a, s in METRICS['summary'].items()},
    'contrasts': {k: {'mean_difference': v['mean_difference'], 'interval_95': v['interval_95'], 'per_seed': v['per_seed']} for k, v in METRICS['contrasts'].items()},
    'comparison_valid': METRICS['comparison_valid'], 'exported': RECORD['selection'], 'reload_parity': PARITY,
    'frozen_record_sha256': RECORD['record_sha256'], 'stages': STAGE_LOG, 'download_bytes': download_bytes,
    'peak_gpu_allocated_gib': {'phi4': PHI4_STATE['peak_gpu_allocated_gib'], 'flux_encoders': FLUX_STATE['encoder_peak_gpu_gib'], 'flux_transformer': FLUX_STATE['transformer_peak_gpu_gib']},
    'model_time_seconds_excluding_installs': round(sum(r['seconds'] for r in STAGE_LOG.values()), 1),
    'wall_seconds_since_configuration': round(time.time() - NOTEBOOK_STARTED, 1),
    'evidence': 'tutorial evidence from one grouped split and one generated pool; not a benchmark or production result',
}
(OUT / 'run_summary.json').write_text(json.dumps(summary, indent=2))
G = DATA['grouping']
if DATA_SOURCE == 'bosch':
    scope_line = '- One custom, deterministic 60/20/20 grouped split of product A. Groups come from exact and perceptual duplicates only; the source supplies no specimen or session identifiers, so this is an exploratory image-group benchmark, not evidence about independent physical specimens.'
else:
    declared = DATA['inventory'].get('declared_groups', 0)
    scope_line = (f"- One deterministic 60/20/20 grouped split of the user-supplied dataset ({DATA['inventory']['entries']} images). "
                  + (f'{declared} images carried a declared group; groups were also merged by exact and perceptual duplicates.' if declared else 'No groups were declared, so groups come from exact and perceptual duplicates only and the evaluation is an exploratory image-group benchmark.'))
limitations = [
    '# Limitations of this run', '',
    scope_line,
    f"- The test partition holds {METRICS['test_counts']} images; per-class metrics for the rarest class rest on few images.",
    '- One generated pool of 32 candidates per defect class from fixed seeds. Generated labels are the prompt\'s intended labels and were not validated by anyone. Results may differ for another pool.',
    '- Phi-4 and FLUX run frozen in 4-bit (NF4); every number is the 4-bit models\'. ' + ('Pretraining overlap of Phi-4, FLUX or ResNet-18 with this public dataset cannot be excluded.' if DATA_SOURCE == 'bosch' else 'If your images are, or resemble, publicly available images, pretraining overlap of Phi-4, FLUX or ResNet-18 with them cannot be excluded.'),
    '- The bootstrap interval is approximate and conditional on this split and pool; it does not include training or generation variability.',
    '- Scores are softmax of uncalibrated logits under an argmax rule; no deployment threshold was chosen or validated.',
    *(['- Documented dataset counts differ from the measured archive for normal images.'] if DATA_SOURCE == 'bosch' else []),
    f"- {len(G['label_conflict_images'])} images in {G['label_conflict_groups']} label-conflict duplicate groups were excluded.",
    '- This is not a production inspection system and makes no claim about factory-level performance.',
]
if RECORD['smoke_run']:
    limitations.append('- Fewer than three seeds were run: this is a smoke run, not the canonical comparison.')
if not STATUS['complete']:
    limitations.append('- Generation fell short (' + '; '.join(STATUS['shortfall']) + '); no synthetic comparison is reported.')
if not METRICS['comparison_valid']:
    limitations.append('- ' + METRICS['validity_note'])
(OUT / 'limitations.md').write_text('\n'.join(limitations) + '\n')
if DATA_SOURCE == 'bosch':
    dataset_terms = ('`split_manifest.csv` and the figures that show dataset images are adapted dataset material: if you redistribute them, credit the source above and apply CC BY-SA 4.0.\n\n'
                     'Cite the dataset\'s accompanying paper: Wang, R., Hoppe, S., Monari, E., & Huber, M. F. (2023). Defect Transfer GAN: Diverse defect synthesis for data augmentation. https://doi.org/10.48550/arXiv.2302.08366')
    dataset_row = '| Dataset | Bosch Surface Defect Inspection dataset, https://github.com/boschresearch/The-Surface-Defect-Inspection-Dataset at commit c6e0afe66e9dbd9d99326c986ea884ddf2aad9f8, archive SHA-256 d33ea340151cdf909f3807a37e00d335c66eb8960b0c2eb568fd9fc75d96fa7c | CC BY-SA 4.0 (attribution; share-alike for adapted material) |'
else:
    dataset_terms = ('The dataset was supplied by the user. Its terms are set by its owner and are not determined by this notebook; `split_manifest.csv` and the figures '
                     'that show its images carry those terms. No Bosch dataset material was used in this run.')
    dataset_row = f"| Dataset | user-supplied BYOD zip, SHA-256 {DATA['acquisition'].get('byod_zip_sha256')} | set by the data owner; not determined by this notebook |"
attribution = f"""# Attribution and licence record

Each component keeps its own licence; they are not assumed to share one.

| Component | Identity | Licence |
|---|---|---|
{dataset_row}
| Description model | {PHI4_STATE['model']['id']} @ {PHI4_STATE['model']['revision']} | {PHI4_STATE['model']['license']} |
| Generator | {FLUX_STATE['model']['id']} @ {FLUX_STATE['model']['revision']}, bytes from {FLUX_STATE['model']['staging']['repo']} @ {FLUX_STATE['model']['staging']['revision']} | {FLUX_STATE['model']['license']} |
| Feature extractor | {FEATURE_STATE['backbone']['id']} @ {FEATURE_STATE['backbone']['revision']} | {FEATURE_STATE['backbone']['license']} |
| Notebook code | kurtvalcorza/flux-schnell-generation-pipeline | see the repository's LICENSE |

Changes made to dataset material in this run: {DATA['experiment_scope']}; {len(G['label_conflict_images'])} label-conflict duplicate images excluded; images converted to three-channel 224 x 224 letterboxed tensors for feature extraction; augmented views (horizontal flip, brightness and contrast within +/-10%).

{dataset_terms}

Generated images in `generated/` were produced by FLUX.1 [schnell] from prompts compiled from descriptions of the dataset images above. Their licensing depends on the model licence, the source images' terms and how you use them; this record does not decide that question for you.
"""
(OUT / 'ATTRIBUTION.md').write_text(attribution)
(OUT / 'completion_record_template.md').write_text('# Completion record (a learning aid, not a submission)\n\n- My prediction before running:\n- C - B (mean, interval, per seed):\n- What the majority baseline taught me:\n- One description claim that was guessed, not observed:\n- One generated candidate I would reject, and why:\n- My bounded conclusion:\n- What I would run next:\n')
print('=' * 100)
print(f"DATA        {DATA_SOURCE}: split {DATA['split']['manifest_sha256'][:12]}... (reference match: {DATA['split']['matches_reference']}), budget {DATA['selected']}")
print(f"PROMPTS     {summary['phi4_guided_prompts']} of {len(PROMPTS)} Phi-4-guided | GENERATION usable {STATUS['usable']} complete={STATUS['complete']}")
print('TEST MACRO-F1 (mean over seeds) ' + ', '.join(f'{a}={v:.4f}' for a, v in summary['test_macro_f1_mean'].items()))
for name, c in METRICS['contrasts'].items():
    print(f"CONTRAST    {name}: {c['mean_difference']:+.4f}, approx. 95% interval [{c['interval_95'][0]:+.4f}, {c['interval_95'][1]:+.4f}]")
print(f"VALIDITY    comparison valid: {METRICS['comparison_valid']} | smoke run: {RECORD['smoke_run']}")
print(f"EXPORT      arm {RECORD['selection']['arm']} seed {RECORD['selection']['seed']} | reload parity {PARITY['passed']} (max diff {PARITY['max_abs_logit_difference']:.1e}, same decisions {PARITY['same_predictions']}) | new-image inference {PARITY['new_image_inference']['status']}")
print(f"RESOURCES   model stages {summary['model_time_seconds_excluding_installs']} s; wall {summary['wall_seconds_since_configuration']} s; "
      f"peak GPU in use {max(r['peak_gpu_used_mib'] for r in STAGE_LOG.values())} MiB; peak child RSS {max(r['peak_child_rss_mib'] for r in STAGE_LOG.values())} MiB; "
      f"lowest host memory available {min(r['min_host_available_mib'] for r in STAGE_LOG.values() if r['min_host_available_mib'] is not None)} MiB")
print(f'OUTPUTS     {OUT}')
for path in sorted(p for p in OUT.rglob('*') if p.is_file() and 'generated' not in p.parts):
    print('   ', path.relative_to(OUT))
print('=' * 100)


## 13 · Optional activities, troubleshooting and further exploration  <sub>(optional)</sub>

None of these is needed for the default path, and a failure here does not affect the results above. The synthetic-fraction activity (13.1) is switched on by default because it reuses the features already computed, touches only validation data and takes seconds; set `RUN_SYNTHETIC_FRACTION_EXERCISE = False` to skip it. Its results are written to `extensions/` and are not part of `run_summary.json`, which was completed in Section 12. Both 13.1 and 13.2 are skipped with an explanation when the canonical arm C was not run.

### 13.1 Change one thing: the synthetic fraction (Predict → Change → Run → Observe → Explain)

**Predict:** if each defect slot in arm C used a synthetic image with probability 0.25 instead of 0.5, would validation macro-F1 go up or down? **Change:** set `EXERCISE_SYNTHETIC_PROBABILITY` in the configuration cell (default 0.25; try 0.75 or 1.0). **Run** the next cell. **Observe** the per-seed validation macro-F1 against the canonical C. **Explain** the pattern in terms of domain mismatch and label noise.

This activity uses **validation only** and only after the canonical result exists. Re-scoring the test set for each variant would turn repeated inspection of one test set into model selection; any such result would be exploratory, not new confirmatory evidence.


In [ ]:
if RUN_SYNTHETIC_FRACTION_EXERCISE and 'C' not in RECORD['arms_run']:
    print('Skipped: the canonical arm C was not run (' + '; '.join(RECORD['generation_status']['shortfall']) + '), so there is no canonical C to compare with. '
          'Fitting an exploratory C here would be a redesigned study, not a completion of the omitted condition.')
elif RUN_SYNTHETIC_FRACTION_EXERCISE:
    p = float(EXERCISE_SYNTHETIC_PROBABILITY)
    if not 0.0 <= p <= 1.0:
        raise ValueError('EXERCISE_SYNTHETIC_PROBABILITY must be between 0 and 1')
    tag = f'exercise_p{p:g}'
    run_stage(tag, 'lab', 'stage_fit.py', '--synthetic-probability', p, '--tag', tag)
    EXERCISE = load(f'extensions/{tag}.json')
    table(['seed', 'C at p=0.5 (canonical, validation)', f'C at p={p:g} (validation)', 'B (validation)'],
          [[s, fmt(RECORD['validation']['C'][s]['val']['macro_f1']), fmt(r['val']['macro_f1']), fmt(RECORD['validation']['B'][s]['val']['macro_f1'])] for s, r in EXERCISE['results']['C'].items()])
else:
    print('RUN_SYNTHETIC_FRACTION_EXERCISE is False; skipped.')


### 13.2 Optional human review of generated candidates

`outputs/.../human_review_template.csv` lists every candidate. Fill `decision` with `accept`, `reject` or `uncertain` and give a short `reason`, upload the file, set `HUMAN_REVIEW_CSV` to its path and `RUN_HUMAN_REVIEW_EXTENSION = True`, then run the next cell. Only accepted candidates enter a separately identified arm C (tag `human_review`), scored on validation only. The canonical frozen results are untouched. Ask yourself whether your decisions were independent of anything you saw in Sections 9–10.


In [ ]:
if RUN_HUMAN_REVIEW_EXTENSION and 'C' not in RECORD['arms_run']:
    print('Skipped: the canonical arm C was not run (generation shortfall), so a human-reviewed C has no canonical counterpart.')
elif RUN_HUMAN_REVIEW_EXTENSION:
    if not HUMAN_REVIEW_CSV or not Path(HUMAN_REVIEW_CSV).is_file():
        print(f"Set HUMAN_REVIEW_CSV to a completed copy of {OUT / 'human_review_template.csv'}")
    else:
        run_stage('human_review', 'lab', 'stage_fit.py', '--review', HUMAN_REVIEW_CSV, '--tag', 'human_review')
        REVIEW = load('extensions/human_review.json')
        table(['seed', 'canonical C (validation)', 'human-reviewed C (validation)'], [[s, fmt(RECORD['validation']['C'][s]['val']['macro_f1']), fmt(r['val']['macro_f1'])] for s, r in REVIEW['results']['C'].items()])
else:
    print('RUN_HUMAN_REVIEW_EXTENSION is False; skipped.')


### 13.3 Bring your own data

The BYOD path runs the **whole** experiment on your images, not just inference: validation, grouping and split, the budget, Phi-4 descriptions, prompts using your defect class names, FLUX generation, the three arms, validation selection, freezing, one test evaluation, export and reload. The contract is in Section 1. To check a zip first without running any model, set `BYOD_CHECK_ZIP` to its path and run the next cell; it applies the same extraction, manifest, image, grouping, split and budget rules and reports the first broken rule. To run the experiment, set `DATA_SOURCE = 'byod'` and `BYOD_ZIP_PATH` (or leave the path empty to get an upload dialog), then Run all. Outputs go to `outputs/byod_capstone/`. If your manifest has no `group` column, grouping falls back to duplicate detection and the same limited interpretation applies as for Bosch.


In [ ]:
if BYOD_CHECK_ZIP:
    try:
        run_stage('byod_check', 'lab', 'stage_data.py', '--check-byod', BYOD_CHECK_ZIP)
    except RuntimeError as exc:
        print(f'BYOD check did not pass: {exc}')
else:
    print('BYOD_CHECK_ZIP is empty; skipped.')


### 13.4 Troubleshooting

| Symptom | Likely cause | What to do |
|---|---|---|
| Runtime check: no GPU or < 15 GB | CPU or small-GPU runtime | Runtime → Change runtime type → T4 GPU, then Run all |
| Runtime check: not enough disk | a previous run's files, or a small disk | Runtime → Disconnect and delete runtime, then Run all; keep `DELETE_MODEL_WEIGHTS_AFTER_USE = True` |
| Environment "cannot see the GPU" | driver older than the CUDA 12 wheels need | record the driver shown by the runtime check and report it; do not switch to unpinned packages |
| `uv` or package download errors | transient network or index outage | re-run the environment cell; completed environments are reused via their lock digest |
| `hash mismatch` / `uv wheel ... SHA-256` | a substituted or truncated download | re-run the environment cell; never remove `--require-hashes` or the uv pin |
| `Linux x86_64 only` | macOS, Windows or ARM runtime | use Colab, Kaggle or a Linux x86_64 Jupyter host |
| `download ... did not complete` | the dataset host timed out | re-run the data cell; completed bytes are resumed. A timeout is an access problem, not evidence of corrupt data |
| `CONTRACT FAILURE: ... digest` | a corrupted or substituted file | delete the named file under `work/sdi_capstone/` and re-run; never skip the check |
| exit code -9 or "Killed" during a model load | host memory exhausted | note `lowest host memory available` from the stage log; use a high-RAM runtime; the notebook does not silently change precision or sizes |
| CUDA out of memory | another process holds GPU memory | Runtime → Restart session, then Run all |
| `generation shortfall` | fewer than 24 usable candidates in a class | a reportable result: the synthetic comparison is not presented. Do not increase the sample count to get past it |
| `frozen input ... changed` | a file changed after freezing | re-run from Section 8; do not edit outputs by hand |
| `reload parity ... FAIL` | the artifact or backbone file differs | re-run Section 11; if it persists, the export is not trustworthy |
| BYOD `CONTRACT FAILURE` | the manifest or images break the contract | the message names the line, column, class, group or limit; fix the zip and use the BYOD check cell |
| HF warning about `HF_TOKEN` | unauthenticated downloads | harmless: every model used here is ungated, and the notebook removes tokens from child processes |

### 13.5 Further exploration

- **P1:** repeat the experiment with independently generated pools (new seed bases) and with products B and C; does C − B keep its sign?
- **P1:** run the human-reviewed arm and compare it with the canonical arm on validation.
- **P2:** train on product A and test on product B; does synthetic augmentation change the transfer gap?
- Apply the protocol to another inspection dataset with specimen identifiers, so groups mean physical parts.


## AI assistance disclosure

This notebook's code and text were developed with generative AI assistance for code development and technical writing under maintainer direction. The maintainer is responsible for reviewing the implementation, validating results and making release decisions. AI assistance is not independent verification, provider endorsement or release approval.

## References

Abouelenin, A., Ashfaq, A., Atkinson, A., Awadalla, H., Bach, N., et al. (2025). *Phi-4-Mini technical report: Compact yet powerful multimodal language models via mixture-of-LoRAs*. arXiv. https://doi.org/10.48550/arXiv.2503.01743

Black Forest Labs. (2024). *FLUX.1 [schnell]* [Model card]. Hugging Face. https://huggingface.co/black-forest-labs/FLUX.1-schnell

Deng, J., Dong, W., Socher, R., Li, L.-J., Li, K., & Fei-Fei, L. (2009). ImageNet: A large-scale hierarchical image database. In *2009 IEEE Conference on Computer Vision and Pattern Recognition* (pp. 248–255). https://doi.org/10.1109/CVPR.2009.5206848

Dettmers, T., Pagnoni, A., Holtzman, A., & Zettlemoyer, L. (2023). *QLoRA: Efficient finetuning of quantized LLMs*. arXiv. https://doi.org/10.48550/arXiv.2305.14314

Efron, B., & Tibshirani, R. J. (1993). *An introduction to the bootstrap*. Chapman & Hall/CRC. https://doi.org/10.1201/9780429246593

He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. In *2016 IEEE Conference on Computer Vision and Pattern Recognition* (pp. 770–778). https://doi.org/10.1109/CVPR.2016.90

Sokolova, M., & Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. *Information Processing & Management, 45*(4), 427–437. https://doi.org/10.1016/j.ipm.2009.03.002

Trabucco, B., Doherty, K., Gurinas, M., & Salakhutdinov, R. (2023). *Effective data augmentation with diffusion models*. arXiv. https://doi.org/10.48550/arXiv.2302.07944

Wang, R., Hoppe, S., Monari, E., & Huber, M. F. (2023). *Defect Transfer GAN: Diverse defect synthesis for data augmentation*. arXiv. https://doi.org/10.48550/arXiv.2302.08366
